## 4. Optymalizacja modelu

## 4.1 Kontrakt wejścia punktu 4 i aliasy plików

In [2]:
# 4.1.1 Definicja wejścia sekcji 4.1 i wczytanie manifestu z 3.14

import json
from pathlib import Path

import pandas as pd

step4_1_project_root = Path(
    "/root/projects/6_samodzielny_projekt_koncowy_wsl/3_Model Klasyfikacyjny"
).resolve()

step4_1_release_dir = step4_1_project_root / "artifacts" / "3_release_package"
step4_1_release_manifest_path = step4_1_release_dir / "b3_14_release_manifest.json"

assert step4_1_project_root.exists(), f"Missing project root: {step4_1_project_root}"
assert (step4_1_project_root / "3_model_klasyfikacyjny.ipynb").exists(), (
    "Missing notebook file in project root."
)
assert step4_1_release_manifest_path.exists(), (
    f"Missing release manifest from 3.14: {step4_1_release_manifest_path}"
)

with open(step4_1_release_manifest_path, "r", encoding="utf-8") as f:
    step4_1_release_manifest = json.load(f)

step4_1_required_input_artifacts = [
    "b3_14_release_manifest.json",
    "b3_13_model_ready_dataset.parquet",
    "b3_13_keep_cols.parquet",
    "b3_13_drop_cols.parquet",
    "b3_13_model_ready_columns.parquet",
    "b3_13_feature_decision_master.parquet",
    "b3_13_null_semantics.parquet",
]

step4_1_manifest_opening_check_df = pd.DataFrame(
    [
        {
            "project_root": str(step4_1_project_root),
            "release_dir": str(step4_1_release_dir),
            "release_manifest_path": str(step4_1_release_manifest_path),
            "required_input_artifact_count": int(len(step4_1_required_input_artifacts)),
            "manifest_block_code": str(step4_1_release_manifest.get("block_code", pd.NA)),
            "manifest_primary_block4_input_count": int(
                len(step4_1_release_manifest.get("primary_block4_inputs", []))
            ),
        }
    ]
)

display(step4_1_manifest_opening_check_df)

,project_root,release_dir,release_manifest_path,required_input_artifact_count,manifest_block_code,manifest_primary_block4_input_count
0,/root/projects/6_samodzielny_projekt_koncowy_w...,/root/projects/6_samodzielny_projekt_koncowy_w...,/root/projects/6_samodzielny_projekt_koncowy_w...,7,3,4


In [3]:
# 4.1.2 Weryfikacja obecności wymaganych plików wejściowych do punktu 4

from pathlib import Path
import pandas as pd

required_objects = [
    "step4_1_project_root",
    "step4_1_release_manifest_path",
    "step4_1_release_manifest",
    "step4_1_required_input_artifacts",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.1.2: {missing_objects}"

step4_1_manifest_release_artifacts = step4_1_release_manifest.get("release_artifacts", {})

step4_1_inventory_rows = []

for artifact_name in step4_1_required_input_artifacts:
    if artifact_name == "b3_14_release_manifest.json":
        matched_paths = [step4_1_release_manifest_path.resolve()]
    else:
        matched_paths = sorted(
            [path.resolve() for path in step4_1_project_root.rglob(artifact_name) if path.is_file()],
            key=lambda x: str(x),
        )

    if len(matched_paths) == 0:
        step4_1_inventory_rows.append(
            {
                "artifact_name": artifact_name,
                "artifact_path": pd.NA,
                "relative_path": pd.NA,
                "exists": 0,
                "found_count": 0,
                "path_source": pd.NA,
            }
        )
    else:
        for artifact_path in matched_paths:
            path_source = "manifest_release_artifacts"
            if artifact_name != "b3_14_release_manifest.json":
                path_source = (
                    "manifest_release_artifacts"
                    if str(artifact_path) in step4_1_manifest_release_artifacts.values()
                    else "project_tree_lookup"
                )

            step4_1_inventory_rows.append(
                {
                    "artifact_name": artifact_name,
                    "artifact_path": str(artifact_path),
                    "relative_path": str(artifact_path.relative_to(step4_1_project_root)),
                    "exists": 1,
                    "found_count": len(matched_paths),
                    "path_source": path_source,
                }
            )

step4_1_input_inventory_df = (
    pd.DataFrame(step4_1_inventory_rows)
    .sort_values(["artifact_name", "relative_path"], na_position="last")
    .reset_index(drop=True)
)

step4_1_input_inventory_summary_df = pd.DataFrame(
    [
        {
            "required_input_artifact_count": int(len(step4_1_required_input_artifacts)),
            "found_input_artifact_count": int(
                step4_1_input_inventory_df.loc[
                    step4_1_input_inventory_df["exists"] == 1, "artifact_name"
                ].nunique()
            ),
            "missing_input_artifact_count": int(
                step4_1_input_inventory_df.loc[
                    step4_1_input_inventory_df["exists"] == 0, "artifact_name"
                ].nunique()
            ),
            "duplicated_input_artifact_count": int(
                (
                    step4_1_input_inventory_df.loc[step4_1_input_inventory_df["exists"] == 1]
                    .groupby("artifact_name")["artifact_path"]
                    .count()
                    .gt(1)
                    .sum()
                )
            ),
        }
    ]
)

step4_1_missing_input_artifacts_df = (
    step4_1_input_inventory_df
    .loc[step4_1_input_inventory_df["exists"] == 0, ["artifact_name"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

display(step4_1_input_inventory_summary_df)
display(step4_1_input_inventory_df)

assert int(step4_1_input_inventory_summary_df.loc[0, "missing_input_artifact_count"]) == 0, (
    f"4.1.2 found missing required input artifacts: "
    f"{step4_1_missing_input_artifacts_df['artifact_name'].tolist()}"
)
assert int(step4_1_input_inventory_summary_df.loc[0, "duplicated_input_artifact_count"]) == 0, (
    "4.1.2 found duplicated required input artifacts."
)

,required_input_artifact_count,found_input_artifact_count,missing_input_artifact_count,duplicated_input_artifact_count
0,7,7,0,0


,artifact_name,artifact_path,relative_path,exists,found_count,path_source
0,b3_13_drop_cols.parquet,/root/projects/6_samodzielny_projekt_koncowy_w...,artifacts/3_feature_screening/b3_13_drop_cols....,1,1,project_tree_lookup
1,b3_13_feature_decision_master.parquet,/root/projects/6_samodzielny_projekt_koncowy_w...,artifacts/3_feature_screening/b3_13_feature_de...,1,1,project_tree_lookup
2,b3_13_keep_cols.parquet,/root/projects/6_samodzielny_projekt_koncowy_w...,artifacts/3_feature_screening/b3_13_keep_cols....,1,1,project_tree_lookup
3,b3_13_model_ready_columns.parquet,/root/projects/6_samodzielny_projekt_koncowy_w...,artifacts/3_feature_screening/b3_13_model_read...,1,1,project_tree_lookup
4,b3_13_model_ready_dataset.parquet,/root/projects/6_samodzielny_projekt_koncowy_w...,artifacts/3_feature_screening/b3_13_model_read...,1,1,project_tree_lookup
5,b3_13_null_semantics.parquet,/root/projects/6_samodzielny_projekt_koncowy_w...,artifacts/3_feature_screening/b3_13_null_seman...,1,1,project_tree_lookup
6,b3_14_release_manifest.json,/root/projects/6_samodzielny_projekt_koncowy_w...,artifacts/3_release_package/b3_14_release_mani...,1,1,manifest_release_artifacts


In [4]:
# 4.1.3 Role artefaktów i alias mapa wejścia punktu 4

import pandas as pd

required_objects = [
    "step4_1_input_inventory_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.1.3: {missing_objects}"

step4_1_artifact_role_map = {
    "b3_14_release_manifest.json": {
        "artifact_role": "block3_release_manifest",
        "artifact_alias": "release_manifest",
        "alias_order": 1,
    },
    "b3_13_model_ready_dataset.parquet": {
        "artifact_role": "primary_model_input_dataset",
        "artifact_alias": "model_ready_dataset",
        "alias_order": 2,
    },
    "b3_13_keep_cols.parquet": {
        "artifact_role": "feature_keep_contract",
        "artifact_alias": "keep_cols",
        "alias_order": 3,
    },
    "b3_13_drop_cols.parquet": {
        "artifact_role": "feature_drop_contract",
        "artifact_alias": "drop_cols",
        "alias_order": 4,
    },
    "b3_13_model_ready_columns.parquet": {
        "artifact_role": "model_column_contract",
        "artifact_alias": "model_ready_columns",
        "alias_order": 5,
    },
    "b3_13_feature_decision_master.parquet": {
        "artifact_role": "feature_decision_master",
        "artifact_alias": "feature_decision_master",
        "alias_order": 6,
    },
    "b3_13_null_semantics.parquet": {
        "artifact_role": "null_semantics_contract",
        "artifact_alias": "null_semantics",
        "alias_order": 7,
    },
}

step4_1_role_map_df = (
    pd.DataFrame.from_dict(step4_1_artifact_role_map, orient="index")
    .reset_index(names="artifact_name")
)

step4_1_input_inventory_with_roles_df = (
    step4_1_input_inventory_df
    .merge(step4_1_role_map_df, on="artifact_name", how="left")
    .assign(
        required_for_4_2=1,
        ready_for_4_2=lambda df: (
            df["exists"].eq(1)
            & df["artifact_role"].notna()
            & df["artifact_alias"].notna()
        ).astype(int),
    )
    .sort_values(["alias_order", "artifact_name"], ascending=[True, True])
    .reset_index(drop=True)
)

step4_1_alias_map_df = (
    step4_1_input_inventory_with_roles_df[
        [
            "artifact_alias",
            "artifact_name",
            "artifact_role",
            "relative_path",
            "artifact_path",
        ]
    ]
    .sort_values("artifact_alias")
    .reset_index(drop=True)
)

step4_1_role_alias_summary_df = pd.DataFrame(
    [
        {
            "input_inventory_row_count": int(step4_1_input_inventory_with_roles_df.shape[0]),
            "mapped_role_count": int(step4_1_input_inventory_with_roles_df["artifact_role"].notna().sum()),
            "mapped_alias_count": int(step4_1_input_inventory_with_roles_df["artifact_alias"].notna().sum()),
            "ready_for_4_2_count": int(step4_1_input_inventory_with_roles_df["ready_for_4_2"].sum()),
            "missing_role_or_alias_count": int(
                (
                    step4_1_input_inventory_with_roles_df["artifact_role"].isna()
                    | step4_1_input_inventory_with_roles_df["artifact_alias"].isna()
                ).sum()
            ),
        }
    ]
)

display(step4_1_role_alias_summary_df)
display(
    step4_1_input_inventory_with_roles_df[
        [
            "artifact_name",
            "artifact_role",
            "artifact_alias",
            "relative_path",
            "ready_for_4_2",
        ]
    ]
)

assert int(step4_1_role_alias_summary_df.loc[0, "missing_role_or_alias_count"]) == 0, (
    "4.1.3 found missing artifact role or alias."
)
assert int(step4_1_role_alias_summary_df.loc[0, "ready_for_4_2_count"]) == 7, (
    "4.1.3 expected all 7 input artifacts to be ready for 4.2."
)

,input_inventory_row_count,mapped_role_count,mapped_alias_count,ready_for_4_2_count,missing_role_or_alias_count
0,7,7,7,7,0


,artifact_name,artifact_role,artifact_alias,relative_path,ready_for_4_2
0,b3_14_release_manifest.json,block3_release_manifest,release_manifest,artifacts/3_release_package/b3_14_release_mani...,1
1,b3_13_model_ready_dataset.parquet,primary_model_input_dataset,model_ready_dataset,artifacts/3_feature_screening/b3_13_model_read...,1
2,b3_13_keep_cols.parquet,feature_keep_contract,keep_cols,artifacts/3_feature_screening/b3_13_keep_cols....,1
3,b3_13_drop_cols.parquet,feature_drop_contract,drop_cols,artifacts/3_feature_screening/b3_13_drop_cols....,1
4,b3_13_model_ready_columns.parquet,model_column_contract,model_ready_columns,artifacts/3_feature_screening/b3_13_model_read...,1
5,b3_13_feature_decision_master.parquet,feature_decision_master,feature_decision_master,artifacts/3_feature_screening/b3_13_feature_de...,1
6,b3_13_null_semantics.parquet,null_semantics_contract,null_semantics,artifacts/3_feature_screening/b3_13_null_seman...,1


In [5]:
# 4.1.4 Zapis artefaktów sekcji 4.1

import json
from pathlib import Path

import pandas as pd

required_objects = [
    "step4_1_project_root",
    "step4_1_input_inventory_with_roles_df",
    "step4_1_alias_map_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.1.4: {missing_objects}"

step4_1_output_dir = step4_1_project_root / "artifacts" / "4_model_input_contract"
step4_1_output_dir.mkdir(parents=True, exist_ok=True)

step4_1_input_inventory_path = step4_1_output_dir / "b4_01_input_inventory.parquet"
step4_1_input_contract_path = step4_1_output_dir / "b4_01_input_contract.json"

step4_1_input_inventory_save_cols = [
    "artifact_name",
    "artifact_role",
    "artifact_alias",
    "artifact_path",
    "relative_path",
    "exists",
    "found_count",
    "path_source",
    "required_for_4_2",
    "ready_for_4_2",
]

step4_1_missing_save_cols = [
    col for col in step4_1_input_inventory_save_cols
    if col not in step4_1_input_inventory_with_roles_df.columns
]
assert not step4_1_missing_save_cols, (
    f"Missing save columns in 4.1.4: {step4_1_missing_save_cols}"
)

step4_1_input_inventory_save_df = (
    step4_1_input_inventory_with_roles_df[step4_1_input_inventory_save_cols]
    .sort_values(["artifact_alias", "artifact_name"], ascending=[True, True])
    .reset_index(drop=True)
    .copy()
)

step4_1_input_inventory_save_df.to_parquet(step4_1_input_inventory_path, index=False)

step4_1_input_contract_payload = {
    "block_code": "4",
    "section_code": "4.1",
    "section_name": "input_contract_and_file_aliases",
    "project_root": str(step4_1_project_root),
    "input_artifacts": {
        "b4_01_input_inventory.parquet": str(step4_1_input_inventory_path),
        "b4_01_input_contract.json": str(step4_1_input_contract_path),
    },
    "required_input_artifact_count": int(step4_1_input_inventory_save_df.shape[0]),
    "ready_for_4_2_count": int(step4_1_input_inventory_save_df["ready_for_4_2"].sum()),
    "alias_map": step4_1_alias_map_df.to_dict(orient="records"),
    "required_inputs": step4_1_input_inventory_save_df.to_dict(orient="records"),
}

with open(step4_1_input_contract_path, "w", encoding="utf-8") as f:
    json.dump(
        step4_1_input_contract_payload,
        f,
        ensure_ascii=False,
        indent=2,
        default=str,
    )

step4_1_saved_artifacts_registry_rows = []

for artifact_path in [step4_1_input_inventory_path, step4_1_input_contract_path]:
    if artifact_path.suffix.lower() == ".parquet":
        artifact_df = pd.read_parquet(artifact_path)
        rows, cols = artifact_df.shape
    elif artifact_path.suffix.lower() == ".json":
        with open(artifact_path, "r", encoding="utf-8") as f:
            artifact_json = json.load(f)
        rows = 1
        cols = len(artifact_json) if isinstance(artifact_json, dict) else 1
    else:
        raise ValueError(f"Unsupported artifact type in 4.1.4: {artifact_path.suffix}")

    step4_1_saved_artifacts_registry_rows.append(
        {
            "artifact_name": artifact_path.name,
            "rows": int(rows),
            "cols": int(cols),
            "path": str(artifact_path),
        }
    )

step4_1_saved_artifacts_registry_df = (
    pd.DataFrame(step4_1_saved_artifacts_registry_rows)
    .sort_values("artifact_name")
    .reset_index(drop=True)
)

display(step4_1_saved_artifacts_registry_df)

assert step4_1_input_inventory_path.exists(), (
    "4.1.4 failed to save b4_01_input_inventory.parquet"
)
assert step4_1_input_contract_path.exists(), (
    "4.1.4 failed to save b4_01_input_contract.json"
)
assert step4_1_input_inventory_save_df.shape[0] == 7, (
    "4.1.4 expected 7 rows in saved input inventory."
)

,artifact_name,rows,cols,path
0,b4_01_input_contract.json,1,9,/root/projects/6_samodzielny_projekt_koncowy_w...
1,b4_01_input_inventory.parquet,7,10,/root/projects/6_samodzielny_projekt_koncowy_w...


## 4.2 Wczytanie artefaktów i kontrola schematu modelowego

In [6]:
# 4.2.1 Wczytanie kontraktu wejścia i inventory z 4.1

import json
from pathlib import Path

import pandas as pd

step4_2_project_root = Path(
    "/root/projects/6_samodzielny_projekt_koncowy_wsl/3_Model Klasyfikacyjny"
).resolve()

step4_2_input_dir = step4_2_project_root / "artifacts" / "4_model_input_contract"
step4_2_input_contract_path = step4_2_input_dir / "b4_01_input_contract.json"
step4_2_input_inventory_path = step4_2_input_dir / "b4_01_input_inventory.parquet"

assert step4_2_project_root.exists(), f"Missing project root: {step4_2_project_root}"
assert (step4_2_project_root / "3_model_klasyfikacyjny.ipynb").exists(), (
    "Missing notebook file in project root."
)
assert step4_2_input_contract_path.exists(), (
    f"Missing input contract from 4.1: {step4_2_input_contract_path}"
)
assert step4_2_input_inventory_path.exists(), (
    f"Missing input inventory from 4.1: {step4_2_input_inventory_path}"
)

with open(step4_2_input_contract_path, "r", encoding="utf-8") as f:
    step4_2_input_contract = json.load(f)

step4_2_input_inventory_df = pd.read_parquet(step4_2_input_inventory_path)

step4_2_required_aliases = [
    "model_ready_dataset",
    "keep_cols",
    "drop_cols",
    "model_ready_columns",
    "feature_decision_master",
    "null_semantics",
    "release_manifest",
]

step4_2_opening_check_df = pd.DataFrame(
    [
        {
            "input_contract_path": str(step4_2_input_contract_path),
            "input_inventory_path": str(step4_2_input_inventory_path),
            "input_inventory_row_count": int(step4_2_input_inventory_df.shape[0]),
            "ready_for_4_2_count": int(step4_2_input_inventory_df["ready_for_4_2"].sum()),
            "required_alias_count": int(len(step4_2_required_aliases)),
            "present_alias_count": int(
                step4_2_input_inventory_df["artifact_alias"]
                .isin(step4_2_required_aliases)
                .sum()
            ),
        }
    ]
)

display(step4_2_opening_check_df)

,input_contract_path,input_inventory_path,input_inventory_row_count,ready_for_4_2_count,required_alias_count,present_alias_count
0,/root/projects/6_samodzielny_projekt_koncowy_w...,/root/projects/6_samodzielny_projekt_koncowy_w...,7,7,7,7


In [7]:
# 4.2.2 Wczytanie artefaktów modelowych do kontroli schematu

from pathlib import Path
import pandas as pd

required_objects = [
    "step4_2_project_root",
    "step4_2_input_inventory_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.2.2: {missing_objects}"

step4_2_required_model_aliases = [
    "model_ready_dataset",
    "keep_cols",
    "drop_cols",
    "model_ready_columns",
    "feature_decision_master",
    "null_semantics",
]

step4_2_alias_map_df = (
    step4_2_input_inventory_df[
        ["artifact_alias", "artifact_name", "artifact_path", "ready_for_4_2"]
    ]
    .sort_values("artifact_alias")
    .reset_index(drop=True)
    .copy()
)

step4_2_missing_aliases = sorted(
    set(step4_2_required_model_aliases) - set(step4_2_alias_map_df["artifact_alias"].tolist())
)
assert not step4_2_missing_aliases, (
    f"Missing required model aliases in 4.2.2: {step4_2_missing_aliases}"
)

assert step4_2_alias_map_df.loc[
    step4_2_alias_map_df["artifact_alias"].isin(step4_2_required_model_aliases),
    "ready_for_4_2"
].eq(1).all(), "4.2.2 found aliases not ready for 4.2."

step4_2_alias_to_path = {
    row["artifact_alias"]: Path(row["artifact_path"]).resolve()
    for _, row in step4_2_alias_map_df.iterrows()
}

step4_2_model_ready_dataset_df = pd.read_parquet(
    step4_2_alias_to_path["model_ready_dataset"]
)
step4_2_keep_cols_df = pd.read_parquet(
    step4_2_alias_to_path["keep_cols"]
)
step4_2_drop_cols_df = pd.read_parquet(
    step4_2_alias_to_path["drop_cols"]
)
step4_2_model_ready_columns_df = pd.read_parquet(
    step4_2_alias_to_path["model_ready_columns"]
)
step4_2_feature_decision_master_df = pd.read_parquet(
    step4_2_alias_to_path["feature_decision_master"]
)
step4_2_null_semantics_df = pd.read_parquet(
    step4_2_alias_to_path["null_semantics"]
)

step4_2_loaded_artifacts_summary_df = pd.DataFrame(
    [
        {
            "artifact_alias": "model_ready_dataset",
            "artifact_name": "b3_13_model_ready_dataset.parquet",
            "rows": int(step4_2_model_ready_dataset_df.shape[0]),
            "cols": int(step4_2_model_ready_dataset_df.shape[1]),
            "path": str(step4_2_alias_to_path["model_ready_dataset"]),
        },
        {
            "artifact_alias": "keep_cols",
            "artifact_name": "b3_13_keep_cols.parquet",
            "rows": int(step4_2_keep_cols_df.shape[0]),
            "cols": int(step4_2_keep_cols_df.shape[1]),
            "path": str(step4_2_alias_to_path["keep_cols"]),
        },
        {
            "artifact_alias": "drop_cols",
            "artifact_name": "b3_13_drop_cols.parquet",
            "rows": int(step4_2_drop_cols_df.shape[0]),
            "cols": int(step4_2_drop_cols_df.shape[1]),
            "path": str(step4_2_alias_to_path["drop_cols"]),
        },
        {
            "artifact_alias": "model_ready_columns",
            "artifact_name": "b3_13_model_ready_columns.parquet",
            "rows": int(step4_2_model_ready_columns_df.shape[0]),
            "cols": int(step4_2_model_ready_columns_df.shape[1]),
            "path": str(step4_2_alias_to_path["model_ready_columns"]),
        },
        {
            "artifact_alias": "feature_decision_master",
            "artifact_name": "b3_13_feature_decision_master.parquet",
            "rows": int(step4_2_feature_decision_master_df.shape[0]),
            "cols": int(step4_2_feature_decision_master_df.shape[1]),
            "path": str(step4_2_alias_to_path["feature_decision_master"]),
        },
        {
            "artifact_alias": "null_semantics",
            "artifact_name": "b3_13_null_semantics.parquet",
            "rows": int(step4_2_null_semantics_df.shape[0]),
            "cols": int(step4_2_null_semantics_df.shape[1]),
            "path": str(step4_2_alias_to_path["null_semantics"]),
        },
    ]
)

display(step4_2_loaded_artifacts_summary_df)

assert step4_2_model_ready_dataset_df.shape[0] > 0, "4.2.2 loaded empty model_ready_dataset."
assert step4_2_keep_cols_df.shape[0] > 0, "4.2.2 loaded empty keep_cols."
assert step4_2_model_ready_columns_df.shape[0] > 0, "4.2.2 loaded empty model_ready_columns."

,artifact_alias,artifact_name,rows,cols,path
0,model_ready_dataset,b3_13_model_ready_dataset.parquet,218222,38,/root/projects/6_samodzielny_projekt_koncowy_w...
1,keep_cols,b3_13_keep_cols.parquet,34,23,/root/projects/6_samodzielny_projekt_koncowy_w...
2,drop_cols,b3_13_drop_cols.parquet,93,1,/root/projects/6_samodzielny_projekt_koncowy_w...
3,model_ready_columns,b3_13_model_ready_columns.parquet,38,3,/root/projects/6_samodzielny_projekt_koncowy_w...
4,feature_decision_master,b3_13_feature_decision_master.parquet,127,151,/root/projects/6_samodzielny_projekt_koncowy_w...
5,null_semantics,b3_13_null_semantics.parquet,34,17,/root/projects/6_samodzielny_projekt_koncowy_w...


In [8]:
# 4.2.3 Kontrola kontraktu kolumn i schematu modelowego

import pandas as pd

required_objects = [
    "step4_2_model_ready_dataset_df",
    "step4_2_keep_cols_df",
    "step4_2_drop_cols_df",
    "step4_2_model_ready_columns_df",
    "step4_2_feature_decision_master_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.2.3: {missing_objects}"

required_column_name_sources = {
    "keep_cols": step4_2_keep_cols_df,
    "drop_cols": step4_2_drop_cols_df,
    "model_ready_columns": step4_2_model_ready_columns_df,
    "feature_decision_master": step4_2_feature_decision_master_df,
}

step4_2_missing_column_name_sources = [
    source_name
    for source_name, df in required_column_name_sources.items()
    if "column_name" not in df.columns
]
assert not step4_2_missing_column_name_sources, (
    f"Missing 'column_name' in 4.2.3 sources: {step4_2_missing_column_name_sources}"
)

step4_2_control_columns = [
    "activity_date",
    "station_id",
    "target_alert_day",
    "split_name",
]

step4_2_dataset_columns = step4_2_model_ready_dataset_df.columns.tolist()
step4_2_dataset_column_set = set(step4_2_dataset_columns)
step4_2_feature_column_set = set(
    col for col in step4_2_dataset_columns if col not in step4_2_control_columns
)

step4_2_keep_col_set = set(
    step4_2_keep_cols_df["column_name"].astype("string").str.strip().tolist()
)
step4_2_drop_col_set = set(
    step4_2_drop_cols_df["column_name"].astype("string").str.strip().tolist()
)
step4_2_model_ready_column_set = set(
    step4_2_model_ready_columns_df["column_name"].astype("string").str.strip().tolist()
)
step4_2_feature_decision_master_set = set(
    step4_2_feature_decision_master_df["column_name"].astype("string").str.strip().tolist()
)

step4_2_missing_control_columns = [
    col for col in step4_2_control_columns if col not in step4_2_dataset_column_set
]
step4_2_keep_not_in_dataset = sorted(step4_2_keep_col_set - step4_2_feature_column_set)
step4_2_drop_not_in_feature_decision_master = sorted(
    step4_2_drop_col_set - step4_2_feature_decision_master_set
)
step4_2_keep_not_in_feature_decision_master = sorted(
    step4_2_keep_col_set - step4_2_feature_decision_master_set
)
step4_2_dataset_feature_not_in_keep = sorted(
    step4_2_feature_column_set - step4_2_keep_col_set
)
step4_2_keep_not_in_model_ready_columns = sorted(
    step4_2_keep_col_set - step4_2_model_ready_column_set
)
step4_2_model_ready_columns_not_in_dataset = sorted(
    step4_2_model_ready_column_set - step4_2_dataset_column_set
)

step4_2_schema_check_overview_df = pd.DataFrame(
    [
        {
            "dataset_row_count": int(step4_2_model_ready_dataset_df.shape[0]),
            "dataset_column_count": int(len(step4_2_dataset_columns)),
            "control_column_count": int(len(step4_2_control_columns)),
            "dataset_feature_column_count": int(len(step4_2_feature_column_set)),
            "keep_cols_count": int(len(step4_2_keep_col_set)),
            "drop_cols_count": int(len(step4_2_drop_col_set)),
            "model_ready_columns_count": int(len(step4_2_model_ready_column_set)),
            "feature_decision_master_count": int(len(step4_2_feature_decision_master_set)),
            "missing_control_column_count": int(len(step4_2_missing_control_columns)),
            "split_name_present_in_dataset": int("split_name" in step4_2_dataset_column_set),
            "split_name_present_in_keep_cols": int("split_name" in step4_2_keep_col_set),
            "keep_not_in_dataset_count": int(len(step4_2_keep_not_in_dataset)),
            "dataset_feature_not_in_keep_count": int(len(step4_2_dataset_feature_not_in_keep)),
            "keep_not_in_model_ready_columns_count": int(len(step4_2_keep_not_in_model_ready_columns)),
            "model_ready_columns_not_in_dataset_count": int(len(step4_2_model_ready_columns_not_in_dataset)),
            "keep_not_in_feature_decision_master_count": int(len(step4_2_keep_not_in_feature_decision_master)),
            "drop_not_in_feature_decision_master_count": int(len(step4_2_drop_not_in_feature_decision_master)),
        }
    ]
)

step4_2_schema_mismatch_rows = []

for column_name in step4_2_missing_control_columns:
    step4_2_schema_mismatch_rows.append(
        {"issue_type": "missing_control_column", "column_name": column_name}
    )

for column_name in step4_2_keep_not_in_dataset:
    step4_2_schema_mismatch_rows.append(
        {"issue_type": "keep_not_in_dataset_feature_columns", "column_name": column_name}
    )

for column_name in step4_2_dataset_feature_not_in_keep:
    step4_2_schema_mismatch_rows.append(
        {"issue_type": "dataset_feature_not_in_keep_cols", "column_name": column_name}
    )

for column_name in step4_2_keep_not_in_model_ready_columns:
    step4_2_schema_mismatch_rows.append(
        {"issue_type": "keep_not_in_model_ready_columns", "column_name": column_name}
    )

for column_name in step4_2_model_ready_columns_not_in_dataset:
    step4_2_schema_mismatch_rows.append(
        {"issue_type": "model_ready_columns_not_in_dataset", "column_name": column_name}
    )

for column_name in step4_2_keep_not_in_feature_decision_master:
    step4_2_schema_mismatch_rows.append(
        {"issue_type": "keep_not_in_feature_decision_master", "column_name": column_name}
    )

for column_name in step4_2_drop_not_in_feature_decision_master:
    step4_2_schema_mismatch_rows.append(
        {"issue_type": "drop_not_in_feature_decision_master", "column_name": column_name}
    )

step4_2_schema_mismatch_df = (
    pd.DataFrame(step4_2_schema_mismatch_rows)
    .sort_values(["issue_type", "column_name"], ascending=[True, True])
    .reset_index(drop=True)
    if step4_2_schema_mismatch_rows
    else pd.DataFrame(columns=["issue_type", "column_name"])
)

display(step4_2_schema_check_overview_df)
display(step4_2_schema_mismatch_df)

assert len(step4_2_missing_control_columns) == 0, (
    f"4.2.3 missing control columns in dataset: {step4_2_missing_control_columns}"
)
assert "split_name" not in step4_2_keep_col_set, (
    "4.2.3 found split_name inside keep_cols."
)
assert len(step4_2_keep_not_in_dataset) == 0, (
    f"4.2.3 found keep_cols not present in dataset feature columns: {step4_2_keep_not_in_dataset}"
)
assert len(step4_2_dataset_feature_not_in_keep) == 0, (
    f"4.2.3 found dataset feature columns outside keep_cols: {step4_2_dataset_feature_not_in_keep}"
)
assert len(step4_2_keep_not_in_model_ready_columns) == 0, (
    f"4.2.3 found keep_cols not present in model_ready_columns: {step4_2_keep_not_in_model_ready_columns}"
)
assert len(step4_2_model_ready_columns_not_in_dataset) == 0, (
    f"4.2.3 found model_ready_columns not present in dataset: {step4_2_model_ready_columns_not_in_dataset}"
)
assert len(step4_2_keep_not_in_feature_decision_master) == 0, (
    f"4.2.3 found keep_cols not present in feature_decision_master: {step4_2_keep_not_in_feature_decision_master}"
)
assert len(step4_2_drop_not_in_feature_decision_master) == 0, (
    f"4.2.3 found drop_cols not present in feature_decision_master: {step4_2_drop_not_in_feature_decision_master}"
)

,dataset_row_count,dataset_column_count,control_column_count,dataset_feature_column_count,keep_cols_count,drop_cols_count,model_ready_columns_count,feature_decision_master_count,missing_control_column_count,split_name_present_in_dataset,split_name_present_in_keep_cols,keep_not_in_dataset_count,dataset_feature_not_in_keep_count,keep_not_in_model_ready_columns_count,model_ready_columns_not_in_dataset_count,keep_not_in_feature_decision_master_count,drop_not_in_feature_decision_master_count
0,218222,38,4,34,34,93,38,127,0,1,0,0,0,0,0,0,0


,issue_type,column_name


In [9]:
# 4.2.4 Kontrola null semantics dla finalnych cech modelowych

import pandas as pd

required_objects = [
    "step4_2_model_ready_dataset_df",
    "step4_2_keep_cols_df",
    "step4_2_null_semantics_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.2.4: {missing_objects}"

required_column_name_sources = {
    "keep_cols": step4_2_keep_cols_df,
    "null_semantics": step4_2_null_semantics_df,
}

step4_2_missing_column_name_sources = [
    source_name
    for source_name, df in required_column_name_sources.items()
    if "column_name" not in df.columns
]
assert not step4_2_missing_column_name_sources, (
    f"Missing 'column_name' in 4.2.4 sources: {step4_2_missing_column_name_sources}"
)

step4_2_required_null_semantics_cols = [
    "column_name",
    "null_semantics_label",
    "null_semantics_review_flag",
]
step4_2_missing_null_semantics_cols = [
    col
    for col in step4_2_required_null_semantics_cols
    if col not in step4_2_null_semantics_df.columns
]
assert not step4_2_missing_null_semantics_cols, (
    f"Missing null semantics columns in 4.2.4: {step4_2_missing_null_semantics_cols}"
)

step4_2_keep_cols_list = (
    step4_2_keep_cols_df["column_name"]
    .astype("string")
    .str.strip()
    .tolist()
)

step4_2_null_semantics_clean_df = (
    step4_2_null_semantics_df.copy()
)
step4_2_null_semantics_clean_df["column_name"] = (
    step4_2_null_semantics_clean_df["column_name"]
    .astype("string")
    .str.strip()
)

step4_2_null_semantics_duplicate_count = int(
    step4_2_null_semantics_clean_df["column_name"].duplicated().sum()
)

step4_2_keep_col_set = set(step4_2_keep_cols_list)
step4_2_null_semantics_set = set(step4_2_null_semantics_clean_df["column_name"].tolist())

step4_2_keep_missing_in_null_semantics = sorted(
    step4_2_keep_col_set - step4_2_null_semantics_set
)
step4_2_null_semantics_not_in_keep = sorted(
    step4_2_null_semantics_set - step4_2_keep_col_set
)

step4_2_keep_missing_profile_df = (
    step4_2_model_ready_dataset_df[step4_2_keep_cols_list]
    .isna()
    .sum()
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "column_name"})
)

step4_2_keep_missing_profile_df["missing_rate"] = (
    step4_2_keep_missing_profile_df["missing_count"]
    / len(step4_2_model_ready_dataset_df)
)

step4_2_null_semantics_check_df = (
    step4_2_keep_missing_profile_df
    .merge(
        step4_2_null_semantics_clean_df[
            ["column_name", "null_semantics_label", "null_semantics_review_flag"]
        ],
        on="column_name",
        how="left",
    )
    .sort_values(["missing_count", "column_name"], ascending=[False, True])
    .reset_index(drop=True)
)

step4_2_null_semantics_overview_df = pd.DataFrame(
    [
        {
            "keep_cols_count": int(len(step4_2_keep_col_set)),
            "null_semantics_count": int(len(step4_2_null_semantics_set)),
            "null_semantics_duplicate_count": int(step4_2_null_semantics_duplicate_count),
            "keep_missing_in_null_semantics_count": int(len(step4_2_keep_missing_in_null_semantics)),
            "null_semantics_not_in_keep_count": int(len(step4_2_null_semantics_not_in_keep)),
            "keep_cols_with_missing_count": int(
                (step4_2_null_semantics_check_df["missing_count"] > 0).sum()
            ),
            "null_semantics_review_flag_count": int(
                step4_2_null_semantics_check_df["null_semantics_review_flag"].fillna(0).astype(int).sum()
            ),
        }
    ]
)

step4_2_null_semantics_mismatch_rows = []

for column_name in step4_2_keep_missing_in_null_semantics:
    step4_2_null_semantics_mismatch_rows.append(
        {"issue_type": "keep_missing_in_null_semantics", "column_name": column_name}
    )

for column_name in step4_2_null_semantics_not_in_keep:
    step4_2_null_semantics_mismatch_rows.append(
        {"issue_type": "null_semantics_not_in_keep", "column_name": column_name}
    )

step4_2_null_semantics_mismatch_df = (
    pd.DataFrame(step4_2_null_semantics_mismatch_rows)
    .sort_values(["issue_type", "column_name"], ascending=[True, True])
    .reset_index(drop=True)
    if step4_2_null_semantics_mismatch_rows
    else pd.DataFrame(columns=["issue_type", "column_name"])
)

display(step4_2_null_semantics_overview_df)
display(step4_2_null_semantics_check_df)
display(step4_2_null_semantics_mismatch_df)

assert step4_2_null_semantics_duplicate_count == 0, (
    "4.2.4 found duplicated column_name in null_semantics."
)
assert len(step4_2_keep_missing_in_null_semantics) == 0, (
    f"4.2.4 found keep_cols missing in null_semantics: {step4_2_keep_missing_in_null_semantics}"
)
assert len(step4_2_null_semantics_not_in_keep) == 0, (
    f"4.2.4 found null_semantics rows outside keep_cols: {step4_2_null_semantics_not_in_keep}"
)

,keep_cols_count,null_semantics_count,null_semantics_duplicate_count,keep_missing_in_null_semantics_count,null_semantics_not_in_keep_count,keep_cols_with_missing_count,null_semantics_review_flag_count
0,34,34,0,0,0,11,1


,column_name,missing_count,missing_rate,null_semantics_label,null_semantics_review_flag
0,alert_lag_7,8235,0.037737,expected_history_window_or_cold_start_missing,0
1,alert_lag_3,3939,0.018050,expected_history_window_or_cold_start_missing,0
2,alert_lag_2,2865,0.013129,expected_history_window_or_cold_start_missing,0
3,alert_hours_lag_1,1792,0.008212,expected_history_window_or_cold_start_missing,0
4,alert_lag_1,1792,0.008212,expected_history_window_or_cold_start_missing,0
5,deficit_alert_lag_1,1792,0.008212,expected_history_window_or_cold_start_missing,0
6,alert_hours_roll_sum_14,1075,0.004926,expected_history_window_or_cold_start_missing,0
7,alert_severity_roll_max_7,1075,0.004926,expected_history_window_or_cold_start_missing,0
8,high_severity_alert_last_7d,1075,0.004926,expected_history_window_or_cold_start_missing,0
9,consecutive_alert_days_before_t,344,0.001576,expected_history_window_or_cold_start_missing,0


,issue_type,column_name


In [10]:
# 4.2.5 Kontrola duplikatów klucza i gotowości do 4.3

import pandas as pd

required_objects = [
    "step4_2_model_ready_dataset_df",
    "step4_2_schema_check_overview_df",
    "step4_2_null_semantics_overview_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.2.5: {missing_objects}"

step4_2_key_columns = ["activity_date", "station_id"]

step4_2_duplicate_key_df = (
    step4_2_model_ready_dataset_df
    .groupby(step4_2_key_columns, dropna=False)
    .size()
    .reset_index(name="row_count")
    .sort_values(["row_count", "activity_date", "station_id"], ascending=[False, True, True])
    .reset_index(drop=True)
)

step4_2_duplicate_key_df = (
    step4_2_duplicate_key_df
    .loc[step4_2_duplicate_key_df["row_count"] > 1]
    .reset_index(drop=True)
)

step4_2_split_name_unique_count = int(
    step4_2_model_ready_dataset_df["split_name"].nunique(dropna=False)
)

step4_2_split_profile_df = (
    step4_2_model_ready_dataset_df
    .groupby("split_name", dropna=False)
    .agg(
        row_count=("target_alert_day", "size"),
        positive_count=("target_alert_day", "sum"),
        unique_station_count=("station_id", "nunique"),
        min_date=("activity_date", "min"),
        max_date=("activity_date", "max"),
    )
    .reset_index()
    .sort_values("split_name")
    .reset_index(drop=True)
)

step4_2_split_profile_df["positive_rate"] = (
    step4_2_split_profile_df["positive_count"] / step4_2_split_profile_df["row_count"]
)

step4_2_readiness_check_df = pd.DataFrame(
    [
        {
            "duplicate_day_station_key_count": int(step4_2_duplicate_key_df.shape[0]),
            "split_name_unique_count": int(step4_2_split_name_unique_count),
            "split_name_missing_count": int(step4_2_model_ready_dataset_df["split_name"].isna().sum()),
            "schema_issue_count": int(step4_2_schema_mismatch_df.shape[0]) if "step4_2_schema_mismatch_df" in globals() else pd.NA,
            "null_semantics_issue_count": int(step4_2_null_semantics_mismatch_df.shape[0]) if "step4_2_null_semantics_mismatch_df" in globals() else pd.NA,
            "null_semantics_review_flag_count": int(step4_2_null_semantics_overview_df.loc[0, "null_semantics_review_flag_count"]),
            "ready_for_4_3": int(
                (step4_2_duplicate_key_df.shape[0] == 0)
                and (step4_2_split_name_unique_count == 3)
                and (step4_2_model_ready_dataset_df["split_name"].isna().sum() == 0)
                and (int(step4_2_schema_check_overview_df.loc[0, "missing_control_column_count"]) == 0)
                and (int(step4_2_schema_check_overview_df.loc[0, "split_name_present_in_dataset"]) == 1)
                and (int(step4_2_schema_check_overview_df.loc[0, "split_name_present_in_keep_cols"]) == 0)
            ),
        }
    ]
)

display(step4_2_readiness_check_df)
display(step4_2_split_profile_df)
display(step4_2_duplicate_key_df)

assert step4_2_duplicate_key_df.shape[0] == 0, (
    "4.2.5 found duplicated activity_date + station_id keys in model_ready_dataset."
)
assert step4_2_split_name_unique_count == 3, (
    f"4.2.5 expected exactly 3 split_name values, got {step4_2_split_name_unique_count}."
)
assert int(step4_2_model_ready_dataset_df["split_name"].isna().sum()) == 0, (
    "4.2.5 found missing split_name values."
)
assert int(step4_2_readiness_check_df.loc[0, "ready_for_4_3"]) == 1, (
    "4.2.5 did not confirm readiness for 4.3."
)

,duplicate_day_station_key_count,split_name_unique_count,split_name_missing_count,schema_issue_count,null_semantics_issue_count,null_semantics_review_flag_count,ready_for_4_3
0,0,3,0,0,0,1,1


,split_name,row_count,positive_count,unique_station_count,min_date,max_date,positive_rate
0,test,75980,61198,344,2020-03-23,2020-10-31,0.805449
1,train,71363,59071,247,2017-05-02,2018-10-31,0.827754
2,validation,70879,58268,341,2019-04-01,2019-10-31,0.822077


,activity_date,station_id,row_count


In [11]:
# 4.2.6 Zapis artefaktów sekcji 4.2

import json
from pathlib import Path

import pandas as pd

required_objects = [
    "step4_2_project_root",
    "step4_2_schema_check_overview_df",
    "step4_2_null_semantics_overview_df",
    "step4_2_readiness_check_df",
    "step4_2_split_profile_df",
    "step4_2_null_semantics_check_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.2.6: {missing_objects}"

step4_2_output_dir = step4_2_project_root / "artifacts" / "4_schema_check"
step4_2_output_dir.mkdir(parents=True, exist_ok=True)

step4_2_schema_check_path = step4_2_output_dir / "b4_02_schema_check.parquet"
step4_2_schema_contract_path = step4_2_output_dir / "b4_02_schema_contract.json"

step4_2_schema_check_save_df = pd.DataFrame(
    [
        {
            "schema_dataset_row_count": int(step4_2_schema_check_overview_df.loc[0, "dataset_row_count"]),
            "schema_dataset_column_count": int(step4_2_schema_check_overview_df.loc[0, "dataset_column_count"]),
            "schema_control_column_count": int(step4_2_schema_check_overview_df.loc[0, "control_column_count"]),
            "schema_dataset_feature_column_count": int(step4_2_schema_check_overview_df.loc[0, "dataset_feature_column_count"]),
            "schema_keep_cols_count": int(step4_2_schema_check_overview_df.loc[0, "keep_cols_count"]),
            "schema_drop_cols_count": int(step4_2_schema_check_overview_df.loc[0, "drop_cols_count"]),
            "schema_model_ready_columns_count": int(step4_2_schema_check_overview_df.loc[0, "model_ready_columns_count"]),
            "schema_feature_decision_master_count": int(step4_2_schema_check_overview_df.loc[0, "feature_decision_master_count"]),
            "schema_missing_control_column_count": int(step4_2_schema_check_overview_df.loc[0, "missing_control_column_count"]),
            "schema_split_name_present_in_dataset": int(step4_2_schema_check_overview_df.loc[0, "split_name_present_in_dataset"]),
            "schema_split_name_present_in_keep_cols": int(step4_2_schema_check_overview_df.loc[0, "split_name_present_in_keep_cols"]),
            "schema_keep_not_in_dataset_count": int(step4_2_schema_check_overview_df.loc[0, "keep_not_in_dataset_count"]),
            "schema_dataset_feature_not_in_keep_count": int(step4_2_schema_check_overview_df.loc[0, "dataset_feature_not_in_keep_count"]),
            "schema_keep_not_in_model_ready_columns_count": int(step4_2_schema_check_overview_df.loc[0, "keep_not_in_model_ready_columns_count"]),
            "schema_model_ready_columns_not_in_dataset_count": int(step4_2_schema_check_overview_df.loc[0, "model_ready_columns_not_in_dataset_count"]),
            "schema_keep_not_in_feature_decision_master_count": int(step4_2_schema_check_overview_df.loc[0, "keep_not_in_feature_decision_master_count"]),
            "schema_drop_not_in_feature_decision_master_count": int(step4_2_schema_check_overview_df.loc[0, "drop_not_in_feature_decision_master_count"]),
            "nulls_keep_cols_count": int(step4_2_null_semantics_overview_df.loc[0, "keep_cols_count"]),
            "nulls_null_semantics_count": int(step4_2_null_semantics_overview_df.loc[0, "null_semantics_count"]),
            "nulls_null_semantics_duplicate_count": int(step4_2_null_semantics_overview_df.loc[0, "null_semantics_duplicate_count"]),
            "nulls_keep_missing_in_null_semantics_count": int(step4_2_null_semantics_overview_df.loc[0, "keep_missing_in_null_semantics_count"]),
            "nulls_null_semantics_not_in_keep_count": int(step4_2_null_semantics_overview_df.loc[0, "null_semantics_not_in_keep_count"]),
            "nulls_keep_cols_with_missing_count": int(step4_2_null_semantics_overview_df.loc[0, "keep_cols_with_missing_count"]),
            "nulls_review_flag_count": int(step4_2_null_semantics_overview_df.loc[0, "null_semantics_review_flag_count"]),
            "readiness_duplicate_day_station_key_count": int(step4_2_readiness_check_df.loc[0, "duplicate_day_station_key_count"]),
            "readiness_split_name_unique_count": int(step4_2_readiness_check_df.loc[0, "split_name_unique_count"]),
            "readiness_split_name_missing_count": int(step4_2_readiness_check_df.loc[0, "split_name_missing_count"]),
            "readiness_schema_issue_count": int(step4_2_readiness_check_df.loc[0, "schema_issue_count"]),
            "readiness_null_semantics_issue_count": int(step4_2_readiness_check_df.loc[0, "null_semantics_issue_count"]),
            "readiness_null_semantics_review_flag_count": int(step4_2_readiness_check_df.loc[0, "null_semantics_review_flag_count"]),
            "ready_for_4_3": int(step4_2_readiness_check_df.loc[0, "ready_for_4_3"]),
        }
    ]
)

assert step4_2_schema_check_save_df.columns.is_unique, (
    f"Duplicate columns in 4.2.6 schema check save df: {step4_2_schema_check_save_df.columns.tolist()}"
)

step4_2_schema_check_save_df.to_parquet(step4_2_schema_check_path, index=False)

step4_2_control_columns = [
    "activity_date",
    "station_id",
    "target_alert_day",
    "split_name",
]

step4_2_schema_contract_payload = {
    "block_code": "4",
    "section_code": "4.2",
    "section_name": "load_artifacts_and_model_schema_check",
    "project_root": str(step4_2_project_root),
    "output_artifacts": {
        "b4_02_schema_check.parquet": str(step4_2_schema_check_path),
        "b4_02_schema_contract.json": str(step4_2_schema_contract_path),
    },
    "control_columns": step4_2_control_columns,
    "hard_rules": {
        "split_name_present_as_control_column": True,
        "split_name_not_allowed_in_X": True,
    },
    "schema_overview": step4_2_schema_check_overview_df.iloc[0].to_dict(),
    "null_semantics_overview": step4_2_null_semantics_overview_df.iloc[0].to_dict(),
    "readiness_overview": step4_2_readiness_check_df.iloc[0].to_dict(),
    "split_profile": step4_2_split_profile_df.to_dict(orient="records"),
    "null_semantics_review_features": (
        step4_2_null_semantics_check_df
        .loc[
            step4_2_null_semantics_check_df["null_semantics_review_flag"].fillna(0).astype(int) == 1,
            ["column_name", "missing_count", "missing_rate", "null_semantics_label"],
        ]
        .to_dict(orient="records")
    ),
}

with open(step4_2_schema_contract_path, "w", encoding="utf-8") as f:
    json.dump(
        step4_2_schema_contract_payload,
        f,
        ensure_ascii=False,
        indent=2,
        default=str,
    )

step4_2_saved_artifacts_registry_rows = []

for artifact_path in [step4_2_schema_check_path, step4_2_schema_contract_path]:
    assert artifact_path.exists(), f"Missing saved artifact in 4.2.6: {artifact_path}"

    if artifact_path.suffix.lower() == ".parquet":
        artifact_df = pd.read_parquet(artifact_path)
        rows, cols = artifact_df.shape
    elif artifact_path.suffix.lower() == ".json":
        with open(artifact_path, "r", encoding="utf-8") as f:
            artifact_json = json.load(f)
        rows = 1
        cols = len(artifact_json) if isinstance(artifact_json, dict) else 1
    else:
        raise ValueError(f"Unsupported artifact type in 4.2.6: {artifact_path.suffix}")

    step4_2_saved_artifacts_registry_rows.append(
        {
            "artifact_name": artifact_path.name,
            "rows": int(rows),
            "cols": int(cols),
            "path": str(artifact_path),
        }
    )

step4_2_saved_artifacts_registry_df = (
    pd.DataFrame(step4_2_saved_artifacts_registry_rows)
    .sort_values("artifact_name")
    .reset_index(drop=True)
)

display(step4_2_saved_artifacts_registry_df)

assert step4_2_schema_check_path.exists(), (
    "4.2.6 failed to save b4_02_schema_check.parquet"
)
assert step4_2_schema_contract_path.exists(), (
    "4.2.6 failed to save b4_02_schema_contract.json"
)
assert step4_2_schema_check_save_df.shape[0] == 1, (
    "4.2.6 expected 1 row in saved schema check."
)

,artifact_name,rows,cols,path
0,b4_02_schema_check.parquet,1,31,/root/projects/6_samodzielny_projekt_koncowy_w...
1,b4_02_schema_contract.json,1,12,/root/projects/6_samodzielny_projekt_koncowy_w...


## Podsumowanie działu 4.2 Wczytanie artefaktów i kontrola schematu modelowego

**Kontekst inżynieryjny:** W dziale wczytano komplet artefaktów wejściowych do punktu 4: `b3_13_model_ready_dataset.parquet`, `b3_13_keep_cols.parquet`, `b3_13_drop_cols.parquet`, `b3_13_model_ready_columns.parquet`, `b3_13_feature_decision_master.parquet` oraz `b3_13_null_semantics.parquet`. Następnie zweryfikowano zgodność schematu modelowego, kontrolę kolumn, spójność kontraktów cech, semantykę braków danych, unikalność klucza `activity_date + station_id` oraz gotowość wejścia do sekcji `4.3`. Zapisano artefakty: `b4_02_schema_check.parquet` oraz `b4_02_schema_contract.json`.

**Interpretacja wyniku:** Dataset modelowy zawiera `218222` wiersze oraz `38` kolumn, w tym `34` finalne cechy modelowe i `4` kolumny kontrolne: `activity_date`, `station_id`, `target_alert_day` oraz `split_name`. Potwierdzono pełną zgodność z `keep_cols`, `drop_cols`, `feature_decision_master` oraz `model_ready_columns`. Kolumna `split_name` jest obecna w datasetcie jako kolumna kontrolna i nie występuje w `keep_cols`. Nie stwierdzono duplikatów klucza `activity_date + station_id`. Kontrakt `null semantics` jest spójny dla wszystkich `34` cech końcowych; `11` cech zawiera braki danych opisane zgodnie z kontraktem, a pojedyncza cecha `is_cold_start` pozostaje oznaczona flagą przeglądu semantyki braków. Kontrola końcowa potwierdziła gotowość do `4.3`.

**Znaczenie biznesowe:** Dział potwierdza, że blok 4 startuje z uporządkowanego i spójnego wejścia modelowego. Oznacza to, że dalsze etapy optymalizacji modelu będą prowadzone na stabilnym zbiorze danych, z zachowaniem jawnego kontraktu cech, kontroli braków oraz rozdziału między kolumnami modelowymi i kontrolnymi.

**Wniosek:** Wejście modelowe do punktu 4 jest kompletne, spójne i gotowe do dalszej pracy. Dział `4.2` został wykonany zgodnie z założeniami i przygotowuje projekt do przejścia do `4.3`.

## 4.3 Podział czasowy train / validation / test i tuning holdout

In [12]:
# 4.3.1 Wczytanie wejścia sekcji 4.3 i kontrola odziedziczonego splitu

import json
from pathlib import Path

import pandas as pd

step4_3_project_root = Path(
    "/root/projects/6_samodzielny_projekt_koncowy_wsl/3_Model Klasyfikacyjny"
).resolve()

step4_3_model_ready_dataset_path = (
    step4_3_project_root
    / "artifacts"
    / "3_feature_screening"
    / "b3_13_model_ready_dataset.parquet"
)
step4_3_model_ready_columns_path = (
    step4_3_project_root
    / "artifacts"
    / "3_feature_screening"
    / "b3_13_model_ready_columns.parquet"
)
step4_3_release_manifest_path = (
    step4_3_project_root
    / "artifacts"
    / "3_release_package"
    / "b3_14_release_manifest.json"
)

assert step4_3_project_root.exists(), f"Missing project root: {step4_3_project_root}"
assert step4_3_model_ready_dataset_path.exists(), (
    f"Missing model_ready_dataset: {step4_3_model_ready_dataset_path}"
)
assert step4_3_model_ready_columns_path.exists(), (
    f"Missing model_ready_columns: {step4_3_model_ready_columns_path}"
)
assert step4_3_release_manifest_path.exists(), (
    f"Missing release manifest: {step4_3_release_manifest_path}"
)

step4_3_model_ready_dataset_df = pd.read_parquet(step4_3_model_ready_dataset_path)
step4_3_model_ready_columns_df = pd.read_parquet(step4_3_model_ready_columns_path)

with open(step4_3_release_manifest_path, "r", encoding="utf-8") as f:
    step4_3_release_manifest = json.load(f)

step4_3_required_dataset_cols = [
    "activity_date",
    "station_id",
    "target_alert_day",
    "split_name",
]
step4_3_missing_dataset_cols = [
    col for col in step4_3_required_dataset_cols
    if col not in step4_3_model_ready_dataset_df.columns
]
assert not step4_3_missing_dataset_cols, (
    f"Missing required dataset columns in 4.3.1: {step4_3_missing_dataset_cols}"
)

step4_3_opening_check_df = pd.DataFrame(
    [
        {
            "model_ready_row_count": int(step4_3_model_ready_dataset_df.shape[0]),
            "model_ready_column_count": int(step4_3_model_ready_dataset_df.shape[1]),
            "model_ready_columns_row_count": int(step4_3_model_ready_columns_df.shape[0]),
            "split_name_unique_count": int(
                step4_3_model_ready_dataset_df["split_name"].nunique(dropna=False)
            ),
            "split_name_missing_count": int(
                step4_3_model_ready_dataset_df["split_name"].isna().sum()
            ),
            "min_activity_date": str(step4_3_model_ready_dataset_df["activity_date"].min()),
            "max_activity_date": str(step4_3_model_ready_dataset_df["activity_date"].max()),
            "manifest_block_code": str(step4_3_release_manifest.get("block_code", "")),
        }
    ]
)

display(step4_3_opening_check_df)

,model_ready_row_count,model_ready_column_count,model_ready_columns_row_count,split_name_unique_count,split_name_missing_count,min_activity_date,max_activity_date,manifest_block_code
0,218222,38,38,3,0,2017-05-02 00:00:00,2020-10-31 00:00:00,3


In [13]:
# 4.3.2 Profil odziedziczonego splitu po datach

import pandas as pd

required_objects = [
    "step4_3_model_ready_dataset_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.3.2: {missing_objects}"

step4_3_split_date_profile_df = (
    step4_3_model_ready_dataset_df
    .groupby("split_name", dropna=False)
    .agg(
        row_count=("target_alert_day", "size"),
        positive_count=("target_alert_day", "sum"),
        unique_station_count=("station_id", "nunique"),
        min_activity_date=("activity_date", "min"),
        max_activity_date=("activity_date", "max"),
    )
    .reset_index()
)

step4_3_split_date_profile_df["positive_rate"] = (
    step4_3_split_date_profile_df["positive_count"]
    / step4_3_split_date_profile_df["row_count"]
)

step4_3_split_date_profile_df["date_span_days"] = (
    pd.to_datetime(step4_3_split_date_profile_df["max_activity_date"])
    - pd.to_datetime(step4_3_split_date_profile_df["min_activity_date"])
).dt.days + 1

step4_3_split_date_profile_df = (
    step4_3_split_date_profile_df[
        [
            "split_name",
            "row_count",
            "positive_count",
            "positive_rate",
            "unique_station_count",
            "min_activity_date",
            "max_activity_date",
            "date_span_days",
        ]
    ]
    .sort_values("split_name")
    .reset_index(drop=True)
)

display(step4_3_split_date_profile_df)

assert step4_3_split_date_profile_df.shape[0] == 3, (
    "4.3.2 expected exactly 3 inherited split groups."
)

,split_name,row_count,positive_count,positive_rate,unique_station_count,min_activity_date,max_activity_date,date_span_days
0,test,75980,61198,0.805449,344,2020-03-23,2020-10-31,223
1,train,71363,59071,0.827754,247,2017-05-02,2018-10-31,548
2,validation,70879,58268,0.822077,341,2019-04-01,2019-10-31,214


In [14]:
# 4.3.3 Profil odziedziczonego splitu po miesiącach

import pandas as pd

required_objects = [
    "step4_3_model_ready_dataset_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.3.3: {missing_objects}"

step4_3_monthly_profile_input_df = step4_3_model_ready_dataset_df.copy()

step4_3_monthly_profile_input_df["activity_date"] = pd.to_datetime(
    step4_3_monthly_profile_input_df["activity_date"]
)
step4_3_monthly_profile_input_df["year_month"] = (
    step4_3_monthly_profile_input_df["activity_date"]
    .dt.to_period("M")
    .astype(str)
)

step4_3_split_month_profile_df = (
    step4_3_monthly_profile_input_df
    .groupby(["split_name", "year_month"], dropna=False)
    .agg(
        row_count=("target_alert_day", "size"),
        positive_count=("target_alert_day", "sum"),
        unique_station_count=("station_id", "nunique"),
        min_activity_date=("activity_date", "min"),
        max_activity_date=("activity_date", "max"),
    )
    .reset_index()
    .sort_values(["split_name", "year_month"], ascending=[True, True])
    .reset_index(drop=True)
)

step4_3_split_month_profile_df["positive_rate"] = (
    step4_3_split_month_profile_df["positive_count"]
    / step4_3_split_month_profile_df["row_count"]
)

step4_3_split_month_summary_df = (
    step4_3_split_month_profile_df
    .groupby("split_name", dropna=False)
    .agg(
        month_count=("year_month", "nunique"),
        min_year_month=("year_month", "min"),
        max_year_month=("year_month", "max"),
    )
    .reset_index()
    .sort_values("split_name")
    .reset_index(drop=True)
)

display(step4_3_split_month_summary_df)
display(step4_3_split_month_profile_df)

assert step4_3_split_month_summary_df.shape[0] == 3, (
    "4.3.3 expected exactly 3 split groups in monthly profile."
)

,split_name,month_count,min_year_month,max_year_month
0,test,8,2020-03,2020-10
1,train,13,2017-05,2018-10
2,validation,7,2019-04,2019-10


,split_name,year_month,row_count,positive_count,unique_station_count,min_activity_date,max_activity_date,positive_rate
0,test,2020-03,2828,2089,331,2020-03-23,2020-03-31,0.738685
1,test,2020-04,10042,8198,336,2020-04-01,2020-04-30,0.816371
2,test,2020-05,10564,8801,343,2020-05-01,2020-05-31,0.833112
3,test,2020-06,10290,8465,343,2020-06-01,2020-06-30,0.822643
4,test,2020-07,10656,8608,344,2020-07-01,2020-07-31,0.807808
5,test,2020-08,10664,8782,344,2020-08-01,2020-08-31,0.823518
6,test,2020-09,10320,8298,344,2020-09-01,2020-09-30,0.804070
7,test,2020-10,10616,7957,344,2020-10-01,2020-10-31,0.749529
8,train,2017-05,4135,3469,144,2017-05-02,2017-05-30,0.838936
9,train,2017-06,4176,3480,144,2017-06-01,2017-06-29,0.833333


In [15]:
# 4.3.4 Profil miesięcy train do wydzielenia tuning_holdout

import pandas as pd

required_objects = [
    "step4_3_split_month_profile_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.3.4: {missing_objects}"

step4_3_train_month_profile_df = (
    step4_3_split_month_profile_df
    .loc[step4_3_split_month_profile_df["split_name"] == "train"]
    .copy()
    .reset_index(drop=True)
)

assert step4_3_train_month_profile_df.shape[0] > 0, (
    "4.3.4 found empty monthly train profile."
)

step4_3_train_month_profile_df["train_month_order"] = range(
    1, len(step4_3_train_month_profile_df) + 1
)
step4_3_train_month_profile_df["is_first_train_month"] = (
    step4_3_train_month_profile_df["train_month_order"] == 1
).astype(int)
step4_3_train_month_profile_df["is_last_train_month"] = (
    step4_3_train_month_profile_df["train_month_order"]
    == step4_3_train_month_profile_df["train_month_order"].max()
).astype(int)

step4_3_train_month_summary_df = pd.DataFrame(
    [
        {
            "train_month_count": int(step4_3_train_month_profile_df.shape[0]),
            "first_train_month": str(step4_3_train_month_profile_df["year_month"].min()),
            "last_train_month": str(step4_3_train_month_profile_df["year_month"].max()),
            "min_train_station_count": int(step4_3_train_month_profile_df["unique_station_count"].min()),
            "max_train_station_count": int(step4_3_train_month_profile_df["unique_station_count"].max()),
            "min_train_positive_rate": float(step4_3_train_month_profile_df["positive_rate"].min()),
            "max_train_positive_rate": float(step4_3_train_month_profile_df["positive_rate"].max()),
        }
    ]
)

display(step4_3_train_month_summary_df)
display(step4_3_train_month_profile_df)

,train_month_count,first_train_month,last_train_month,min_train_station_count,max_train_station_count,min_train_positive_rate,max_train_positive_rate
0,13,2017-05,2018-10,143,245,0.789335,0.849452


,split_name,year_month,row_count,positive_count,unique_station_count,min_activity_date,max_activity_date,positive_rate,train_month_order,is_first_train_month,is_last_train_month
0,train,2017-05,4135,3469,144,2017-05-02,2017-05-30,0.838936,1,1,0
1,train,2017-06,4176,3480,144,2017-06-01,2017-06-29,0.833333,2,0,0
2,train,2017-07,4301,3652,144,2017-07-01,2017-07-30,0.849105,3,0,0
3,train,2017-08,4290,3590,143,2017-08-01,2017-08-30,0.836830,4,0,0
4,train,2017-09,4147,3447,143,2017-09-01,2017-09-29,0.831203,5,0,0
5,train,2017-10,4265,3385,143,2017-10-01,2017-10-30,0.793669,6,0,0
6,train,2018-04,3788,2990,155,2018-04-03,2018-04-30,0.789335,7,0,0
7,train,2018-05,6521,5457,213,2018-05-01,2018-05-31,0.836835,8,0,0
8,train,2018-06,6390,5428,213,2018-06-01,2018-06-30,0.849452,9,0,0
9,train,2018-07,6907,5691,245,2018-07-01,2018-07-31,0.823947,10,0,0


In [16]:
# 4.3.5 Wydzielenie statycznego tuning_holdout wewnątrz train

import pandas as pd

required_objects = [
    "step4_3_model_ready_dataset_df",
    "step4_3_train_month_profile_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.3.5: {missing_objects}"

step4_3_tuning_holdout_month = str(
    step4_3_train_month_profile_df["year_month"].max()
)

step4_3_time_split_work_df = step4_3_model_ready_dataset_df.copy()
step4_3_time_split_work_df["activity_date"] = pd.to_datetime(
    step4_3_time_split_work_df["activity_date"]
)
step4_3_time_split_work_df["year_month"] = (
    step4_3_time_split_work_df["activity_date"]
    .dt.to_period("M")
    .astype(str)
)

step4_3_time_split_work_df["inherited_split_name"] = (
    step4_3_time_split_work_df["split_name"].astype("string")
)
step4_3_time_split_work_df["is_tuning_holdout"] = (
    (step4_3_time_split_work_df["inherited_split_name"] == "train")
    & (step4_3_time_split_work_df["year_month"] == step4_3_tuning_holdout_month)
).astype(int)

step4_3_time_split_work_df["train_subsplit_name"] = "not_train"
step4_3_time_split_work_df.loc[
    step4_3_time_split_work_df["inherited_split_name"] == "train",
    "train_subsplit_name",
] = "primary_train"
step4_3_time_split_work_df.loc[
    step4_3_time_split_work_df["is_tuning_holdout"] == 1,
    "train_subsplit_name",
] = "tuning_holdout"

step4_3_tuning_holdout_summary_df = pd.DataFrame(
    [
        {
            "tuning_holdout_month": step4_3_tuning_holdout_month,
            "tuning_holdout_row_count": int(
                step4_3_time_split_work_df["is_tuning_holdout"].sum()
            ),
            "tuning_holdout_positive_count": int(
                step4_3_time_split_work_df.loc[
                    step4_3_time_split_work_df["is_tuning_holdout"] == 1,
                    "target_alert_day",
                ].sum()
            ),
            "tuning_holdout_unique_station_count": int(
                step4_3_time_split_work_df.loc[
                    step4_3_time_split_work_df["is_tuning_holdout"] == 1,
                    "station_id",
                ].nunique()
            ),
            "primary_train_row_count_after_holdout": int(
                step4_3_time_split_work_df.loc[
                    step4_3_time_split_work_df["train_subsplit_name"] == "primary_train"
                ].shape[0]
            ),
            "primary_train_unique_station_count_after_holdout": int(
                step4_3_time_split_work_df.loc[
                    step4_3_time_split_work_df["train_subsplit_name"] == "primary_train",
                    "station_id",
                ].nunique()
            ),
        }
    ]
)

step4_3_train_subsplit_profile_df = (
    step4_3_time_split_work_df
    .loc[step4_3_time_split_work_df["inherited_split_name"] == "train"]
    .groupby("train_subsplit_name", dropna=False)
    .agg(
        row_count=("target_alert_day", "size"),
        positive_count=("target_alert_day", "sum"),
        unique_station_count=("station_id", "nunique"),
        min_activity_date=("activity_date", "min"),
        max_activity_date=("activity_date", "max"),
    )
    .reset_index()
    .sort_values("train_subsplit_name")
    .reset_index(drop=True)
)

step4_3_train_subsplit_profile_df["positive_rate"] = (
    step4_3_train_subsplit_profile_df["positive_count"]
    / step4_3_train_subsplit_profile_df["row_count"]
)

display(step4_3_tuning_holdout_summary_df)
display(step4_3_train_subsplit_profile_df)

assert int(step4_3_tuning_holdout_summary_df.loc[0, "tuning_holdout_row_count"]) > 0, (
    "4.3.5 produced empty tuning_holdout."
)
assert int(step4_3_tuning_holdout_summary_df.loc[0, "primary_train_row_count_after_holdout"]) > 0, (
    "4.3.5 produced empty primary_train after holdout split."
)

,tuning_holdout_month,tuning_holdout_row_count,tuning_holdout_positive_count,tuning_holdout_unique_station_count,primary_train_row_count_after_holdout,primary_train_unique_station_count_after_holdout
0,2018-10,7530,6093,243,63833,247


,train_subsplit_name,row_count,positive_count,unique_station_count,min_activity_date,max_activity_date,positive_rate
0,primary_train,63833,52978,247,2017-05-02,2018-09-30,0.829947
1,tuning_holdout,7530,6093,243,2018-10-01,2018-10-31,0.809163


In [17]:
# 4.3.6 Budowa oficjalnego splitu punktu 4 i końcowa kontrola

import pandas as pd

required_objects = [
    "step4_3_time_split_work_df",
    "step4_3_train_subsplit_profile_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.3.6: {missing_objects}"

step4_3_time_split_df = step4_3_time_split_work_df.copy()

step4_3_time_split_df["point4_split_name"] = step4_3_time_split_df["inherited_split_name"].astype("string")
step4_3_time_split_df.loc[
    step4_3_time_split_df["train_subsplit_name"] == "primary_train",
    "point4_split_name",
] = "train"
step4_3_time_split_df.loc[
    step4_3_time_split_df["train_subsplit_name"] == "tuning_holdout",
    "point4_split_name",
] = "tuning_holdout"

step4_3_point4_split_profile_df = (
    step4_3_time_split_df
    .groupby("point4_split_name", dropna=False)
    .agg(
        row_count=("target_alert_day", "size"),
        positive_count=("target_alert_day", "sum"),
        unique_station_count=("station_id", "nunique"),
        min_activity_date=("activity_date", "min"),
        max_activity_date=("activity_date", "max"),
    )
    .reset_index()
    .sort_values("point4_split_name")
    .reset_index(drop=True)
)

step4_3_point4_split_profile_df["positive_rate"] = (
    step4_3_point4_split_profile_df["positive_count"]
    / step4_3_point4_split_profile_df["row_count"]
)

step4_3_final_split_check_df = pd.DataFrame(
    [
        {
            "point4_split_unique_count": int(step4_3_time_split_df["point4_split_name"].nunique(dropna=False)),
            "tuning_holdout_row_count": int((step4_3_time_split_df["point4_split_name"] == "tuning_holdout").sum()),
            "validation_row_count": int((step4_3_time_split_df["point4_split_name"] == "validation").sum()),
            "test_row_count": int((step4_3_time_split_df["point4_split_name"] == "test").sum()),
            "train_row_count_after_holdout": int((step4_3_time_split_df["point4_split_name"] == "train").sum()),
            "tuning_holdout_only_inside_inherited_train": int(
                step4_3_time_split_df.loc[
                    step4_3_time_split_df["point4_split_name"] == "tuning_holdout",
                    "inherited_split_name",
                ].eq("train").all()
            ),
            "validation_preserved": int(
                step4_3_time_split_df.loc[
                    step4_3_time_split_df["inherited_split_name"] == "validation",
                    "point4_split_name",
                ].eq("validation").all()
            ),
            "test_preserved": int(
                step4_3_time_split_df.loc[
                    step4_3_time_split_df["inherited_split_name"] == "test",
                    "point4_split_name",
                ].eq("test").all()
            ),
            "ready_to_save_4_3": int(
                step4_3_time_split_df["point4_split_name"].isin(
                    ["train", "tuning_holdout", "validation", "test"]
                ).all()
                and (step4_3_time_split_df["point4_split_name"] == "tuning_holdout").sum() > 0
            ),
        }
    ]
)

display(step4_3_final_split_check_df)
display(step4_3_point4_split_profile_df)

assert int(step4_3_final_split_check_df.loc[0, "point4_split_unique_count"]) == 4, (
    "4.3.6 expected exactly 4 point4 split groups."
)
assert int(step4_3_final_split_check_df.loc[0, "tuning_holdout_only_inside_inherited_train"]) == 1, (
    "4.3.6 found tuning_holdout rows outside inherited train."
)
assert int(step4_3_final_split_check_df.loc[0, "validation_preserved"]) == 1, (
    "4.3.6 validation split was not preserved."
)
assert int(step4_3_final_split_check_df.loc[0, "test_preserved"]) == 1, (
    "4.3.6 test split was not preserved."
)
assert int(step4_3_final_split_check_df.loc[0, "ready_to_save_4_3"]) == 1, (
    "4.3.6 did not confirm readiness to save section 4.3."
)

,point4_split_unique_count,tuning_holdout_row_count,validation_row_count,test_row_count,train_row_count_after_holdout,tuning_holdout_only_inside_inherited_train,validation_preserved,test_preserved,ready_to_save_4_3
0,4,7530,70879,75980,63833,1,1,1,1


,point4_split_name,row_count,positive_count,unique_station_count,min_activity_date,max_activity_date,positive_rate
0,test,75980,61198,344,2020-03-23,2020-10-31,0.805449
1,train,63833,52978,247,2017-05-02,2018-09-30,0.829947
2,tuning_holdout,7530,6093,243,2018-10-01,2018-10-31,0.809163
3,validation,70879,58268,341,2019-04-01,2019-10-31,0.822077


In [18]:
# 4.3.7 Zapis artefaktów sekcji 4.3

import json
from pathlib import Path

import pandas as pd

required_objects = [
    "step4_3_project_root",
    "step4_3_time_split_df",
    "step4_3_split_date_profile_df",
    "step4_3_split_month_summary_df",
    "step4_3_split_month_profile_df",
    "step4_3_tuning_holdout_summary_df",
    "step4_3_train_subsplit_profile_df",
    "step4_3_final_split_check_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.3.7: {missing_objects}"

step4_3_output_dir = step4_3_project_root / "artifacts" / "4_time_split"
step4_3_output_dir.mkdir(parents=True, exist_ok=True)

step4_3_time_split_path = step4_3_output_dir / "b4_03_time_split.parquet"
step4_3_split_contract_path = step4_3_output_dir / "b4_03_split_contract.json"

step4_3_time_split_save_df = step4_3_time_split_df.copy()

step4_3_required_save_cols = [
    "activity_date",
    "station_id",
    "target_alert_day",
    "split_name",
    "inherited_split_name",
    "year_month",
    "is_tuning_holdout",
    "train_subsplit_name",
    "point4_split_name",
]
step4_3_missing_save_cols = [
    col for col in step4_3_required_save_cols
    if col not in step4_3_time_split_save_df.columns
]
assert not step4_3_missing_save_cols, (
    f"Missing save columns in 4.3.7: {step4_3_missing_save_cols}"
)

step4_3_time_split_save_df.to_parquet(step4_3_time_split_path, index=False)

step4_3_split_contract_payload = {
    "block_code": "4",
    "section_code": "4.3",
    "section_name": "time_split_and_tuning_holdout",
    "project_root": str(step4_3_project_root),
    "output_artifacts": {
        "b4_03_time_split.parquet": str(step4_3_time_split_path),
        "b4_03_split_contract.json": str(step4_3_split_contract_path),
    },
    "hard_rules": {
        "inherits_split_from_block3": True,
        "does_not_build_new_main_split_from_scratch": True,
        "tuning_holdout_only_inside_train": True,
    },
    "split_date_profile": step4_3_split_date_profile_df.to_dict(orient="records"),
    "split_month_summary": step4_3_split_month_summary_df.to_dict(orient="records"),
    "tuning_holdout_summary": step4_3_tuning_holdout_summary_df.to_dict(orient="records"),
    "train_subsplit_profile": step4_3_train_subsplit_profile_df.to_dict(orient="records"),
    "point4_split_profile": step4_3_point4_split_profile_df.to_dict(orient="records"),
    "final_split_check": step4_3_final_split_check_df.iloc[0].to_dict(),
}

with open(step4_3_split_contract_path, "w", encoding="utf-8") as f:
    json.dump(
        step4_3_split_contract_payload,
        f,
        ensure_ascii=False,
        indent=2,
        default=str,
    )

step4_3_saved_artifacts_registry_rows = []

for artifact_path in [step4_3_time_split_path, step4_3_split_contract_path]:
    assert artifact_path.exists(), f"Missing saved artifact in 4.3.7: {artifact_path}"

    if artifact_path.suffix.lower() == ".parquet":
        artifact_df = pd.read_parquet(artifact_path)
        rows, cols = artifact_df.shape
    elif artifact_path.suffix.lower() == ".json":
        with open(artifact_path, "r", encoding="utf-8") as f:
            artifact_json = json.load(f)
        rows = 1
        cols = len(artifact_json) if isinstance(artifact_json, dict) else 1
    else:
        raise ValueError(f"Unsupported artifact type in 4.3.7: {artifact_path.suffix}")

    step4_3_saved_artifacts_registry_rows.append(
        {
            "artifact_name": artifact_path.name,
            "rows": int(rows),
            "cols": int(cols),
            "path": str(artifact_path),
        }
    )

step4_3_saved_artifacts_registry_df = (
    pd.DataFrame(step4_3_saved_artifacts_registry_rows)
    .sort_values("artifact_name")
    .reset_index(drop=True)
)

display(step4_3_saved_artifacts_registry_df)

assert step4_3_time_split_path.exists(), (
    "4.3.7 failed to save b4_03_time_split.parquet"
)
assert step4_3_split_contract_path.exists(), (
    "4.3.7 failed to save b4_03_split_contract.json"
)
assert step4_3_time_split_save_df.shape[0] == step4_3_time_split_df.shape[0], (
    "4.3.7 saved time split row count mismatch."
)

,artifact_name,rows,cols,path
0,b4_03_split_contract.json,1,12,/root/projects/6_samodzielny_projekt_koncowy_w...
1,b4_03_time_split.parquet,218222,43,/root/projects/6_samodzielny_projekt_koncowy_w...


## Podsumowanie działu 4.3 Podział czasowy train / validation / test i tuning holdout

**Kontekst inżynieryjny:** W dziale zmaterializowano odziedziczony podział `train / validation / test` z bloku 3 oraz wydzielono dodatkowy `tuning_holdout` wyłącznie wewnątrz części `train`. Zweryfikowano profil splitu po datach i po miesiącach, potwierdzono chronologiczny porządek podziału oraz przygotowano oficjalny split do dalszych etapów modelowania. Zapisano artefakty `b4_03_time_split.parquet` oraz `b4_03_split_contract.json`.

**Interpretacja wyniku:** Dataset wejściowy do punktu 4.3 zawiera `218222` rekordy i `38` kolumn, a kolumna `split_name` jest kompletna i przyjmuje dokładnie trzy wartości: `train`, `validation` oraz `test`. Profil po datach potwierdził rozdzielność czasową splitów. Część `train` obejmuje okres od `2017-05-02` do `2018-10-31`, `validation` od `2019-04-01` do `2019-10-31`, a `test` od `2020-03-23` do `2020-10-31`. Profil miesięczny dla `train` obejmuje `13` miesięcy obserwacji. Statyczny `tuning_holdout` został wydzielony jako miesiąc `2018-10`, co dało `7530` rekordów w holdoucie oraz `63833` rekordy w `primary_train`. Końcowa kontrola potwierdziła obecność czterech końcowych grup splitu punktu 4: `train`, `tuning_holdout`, `validation` i `test`, przy zachowaniu pełnej zgodności z odziedziczonym podziałem z bloku 3.

**Znaczenie biznesowe:** Sekcja porządkuje oficjalny schemat uczenia i walidacji modelu w punkcie 4. Dzięki wydzieleniu `tuning_holdout` możliwe jest strojenie modelu wewnątrz części treningowej bez naruszania niezależnych zbiorów `validation` i `test`. Zapewnia to spójny i kontrolowany proces dalszego benchmarkowania, tuningu i oceny jakości modelu.

**Wniosek:** Podział czasowy punktu 4 został przygotowany poprawnie, z zachowaniem dziedziczenia splitu z bloku 3 oraz z wydzieleniem statycznego `tuning_holdout` wewnątrz części treningowej. Dział `4.3` został wykonany zgodnie z założeniami i przygotowuje projekt do przejścia do kolejnych etapów modelowania.

## 4.4 Dumb Baseline operacyjny

In [19]:
# 4.4.1 Wybór realnej cechy baseline'owej z finalnego keep_cols

import pandas as pd

required_objects = [
    "step4_2_keep_cols_df",
    "step4_2_model_ready_dataset_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.4.1: {missing_objects}"

step4_4_keep_cols_work_df = step4_2_keep_cols_df.copy()
step4_4_keep_cols_work_df["column_name"] = (
    step4_4_keep_cols_work_df["column_name"]
    .astype("string")
    .str.strip()
)

step4_4_final_feature_list_df = pd.DataFrame(
    {
        "column_name": step4_4_keep_cols_work_df["column_name"].tolist()
    }
)

step4_4_final_feature_list_df["in_model_ready_dataset"] = (
    step4_4_final_feature_list_df["column_name"]
    .isin(step4_2_model_ready_dataset_df.columns)
    .astype(int)
)

step4_4_baseline_candidate_patterns = [
    r"^alert_lag_1$",
    r"^alert_lag_2$",
    r"^alert_lag_3$",
    r"^alert_lag_7$",
    r"^deficit_alert_lag_1$",
    r"^high_severity_alert_last_7d$",
    r"^is_missing_alert_history$",
]

step4_4_candidate_baseline_df = (
    step4_4_final_feature_list_df
    .loc[
        step4_4_final_feature_list_df["column_name"].str.contains(
            "|".join(step4_4_baseline_candidate_patterns),
            regex=True,
            na=False,
        )
    ]
    .reset_index(drop=True)
)

step4_4_candidate_baseline_summary_df = pd.DataFrame(
    [
        {
            "keep_cols_count": int(step4_4_final_feature_list_df.shape[0]),
            "candidate_baseline_count": int(step4_4_candidate_baseline_df.shape[0]),
            "candidate_present_in_model_ready_count": int(
                step4_4_candidate_baseline_df["in_model_ready_dataset"].sum()
            ),
        }
    ]
)

display(step4_4_candidate_baseline_summary_df)
display(step4_4_candidate_baseline_df)

,keep_cols_count,candidate_baseline_count,candidate_present_in_model_ready_count
0,34,7,7


,column_name,in_model_ready_dataset
0,alert_lag_1,1
1,alert_lag_2,1
2,alert_lag_3,1
3,alert_lag_7,1
4,deficit_alert_lag_1,1
5,high_severity_alert_last_7d,1
6,is_missing_alert_history,1


In [20]:
# 4.4.2 Wybór finalnej cechy baseline'owej i kontrola kompletności per split

import pandas as pd

required_objects = [
    "step4_2_model_ready_dataset_df",
    "step4_3_time_split_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.4.2: {missing_objects}"

step4_4_baseline_feature = "alert_lag_1"

assert step4_4_baseline_feature in step4_2_model_ready_dataset_df.columns, (
    f"Missing baseline feature in model_ready_dataset: {step4_4_baseline_feature}"
)
assert step4_4_baseline_feature in step4_3_time_split_df.columns, (
    f"Missing baseline feature in time_split dataset: {step4_4_baseline_feature}"
)
assert "point4_split_name" in step4_3_time_split_df.columns, (
    "Missing point4_split_name in time_split dataset."
)

step4_4_baseline_profile_df = (
    step4_3_time_split_df
    .groupby("point4_split_name", dropna=False)
    .agg(
        row_count=("target_alert_day", "size"),
        non_null_count=(step4_4_baseline_feature, lambda s: int(s.notna().sum())),
        null_count=(step4_4_baseline_feature, lambda s: int(s.isna().sum())),
        value_1_count=(step4_4_baseline_feature, lambda s: int(s.eq(1).sum())),
        value_0_count=(step4_4_baseline_feature, lambda s: int(s.eq(0).sum())),
        invalid_value_count=(
            step4_4_baseline_feature,
            lambda s: int((~s.isin([0, 1]) & s.notna()).sum()),
        ),
    )
    .reset_index()
    .sort_values("point4_split_name")
    .reset_index(drop=True)
)

step4_4_baseline_profile_df["non_null_rate"] = (
    step4_4_baseline_profile_df["non_null_count"]
    / step4_4_baseline_profile_df["row_count"]
)
step4_4_baseline_profile_df["null_rate"] = (
    step4_4_baseline_profile_df["null_count"]
    / step4_4_baseline_profile_df["row_count"]
)
step4_4_baseline_profile_df["value_1_rate"] = (
    step4_4_baseline_profile_df["value_1_count"]
    / step4_4_baseline_profile_df["row_count"]
)
step4_4_baseline_profile_df["value_0_rate"] = (
    step4_4_baseline_profile_df["value_0_count"]
    / step4_4_baseline_profile_df["row_count"]
)

step4_4_baseline_summary_df = pd.DataFrame(
    [
        {
            "baseline_feature": step4_4_baseline_feature,
            "split_count": int(step4_4_baseline_profile_df.shape[0]),
            "total_row_count": int(step4_4_baseline_profile_df["row_count"].sum()),
            "total_non_null_count": int(step4_4_baseline_profile_df["non_null_count"].sum()),
            "total_null_count": int(step4_4_baseline_profile_df["null_count"].sum()),
            "total_invalid_value_count": int(step4_4_baseline_profile_df["invalid_value_count"].sum()),
        }
    ]
)

display(step4_4_baseline_summary_df)
display(step4_4_baseline_profile_df)

assert int(step4_4_baseline_summary_df.loc[0, "total_invalid_value_count"]) == 0, (
    f"4.4.2 found invalid values in baseline feature: {step4_4_baseline_feature}"
)

,baseline_feature,split_count,total_row_count,total_non_null_count,total_null_count,total_invalid_value_count
0,alert_lag_1,4,218222,216430,1792,0


,point4_split_name,row_count,non_null_count,null_count,value_1_count,value_0_count,invalid_value_count,non_null_rate,null_rate,value_1_rate,value_0_rate
0,test,75980,75636,344,60966,14670,0,0.995472,0.004528,0.802395,0.193077
1,train,63833,62726,1107,52062,10664,0,0.982658,0.017342,0.815597,0.167061
2,tuning_holdout,7530,7530,0,6091,1439,0,1.0,0.0,0.808898,0.191102
3,validation,70879,70538,341,58024,12514,0,0.995189,0.004811,0.818635,0.176554


In [21]:
# 4.4.3 Reguła baseline i predykcje per split

import pandas as pd

required_objects = [
    "step4_3_time_split_df",
    "step4_4_baseline_feature",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.4.3: {missing_objects}"

assert step4_4_baseline_feature in step4_3_time_split_df.columns, (
    f"Missing baseline feature in time split dataset: {step4_4_baseline_feature}"
)
assert "point4_split_name" in step4_3_time_split_df.columns, (
    "Missing point4_split_name in time split dataset."
)
assert "target_alert_day" in step4_3_time_split_df.columns, (
    "Missing target_alert_day in time split dataset."
)

step4_4_baseline_prediction_df = step4_3_time_split_df[
    [
        "activity_date",
        "station_id",
        "target_alert_day",
        "point4_split_name",
        step4_4_baseline_feature,
    ]
].copy()

step4_4_baseline_prediction_df["baseline_feature_is_null"] = (
    step4_4_baseline_prediction_df[step4_4_baseline_feature].isna()
).astype(int)

step4_4_baseline_prediction_df["dumb_baseline_pred"] = (
    step4_4_baseline_prediction_df[step4_4_baseline_feature]
    .fillna(0)
    .astype("int8")
)

step4_4_baseline_prediction_df["dumb_baseline_score"] = (
    step4_4_baseline_prediction_df["dumb_baseline_pred"].astype("float32")
)

step4_4_baseline_rule_summary_df = pd.DataFrame(
    [
        {
            "baseline_feature": step4_4_baseline_feature,
            "baseline_rule": "predict_1_if_alert_lag_1_equals_1_else_0",
            "null_handling_rule": "fill_null_with_0",
            "prediction_row_count": int(step4_4_baseline_prediction_df.shape[0]),
            "prediction_positive_count": int(step4_4_baseline_prediction_df["dumb_baseline_pred"].sum()),
            "prediction_positive_rate": float(step4_4_baseline_prediction_df["dumb_baseline_pred"].mean()),
            "null_input_count": int(step4_4_baseline_prediction_df["baseline_feature_is_null"].sum()),
        }
    ]
)

step4_4_baseline_prediction_profile_df = (
    step4_4_baseline_prediction_df
    .groupby("point4_split_name", dropna=False)
    .agg(
        row_count=("target_alert_day", "size"),
        target_positive_count=("target_alert_day", "sum"),
        prediction_positive_count=("dumb_baseline_pred", "sum"),
        input_null_count=("baseline_feature_is_null", "sum"),
    )
    .reset_index()
    .sort_values("point4_split_name")
    .reset_index(drop=True)
)

step4_4_baseline_prediction_profile_df["target_positive_rate"] = (
    step4_4_baseline_prediction_profile_df["target_positive_count"]
    / step4_4_baseline_prediction_profile_df["row_count"]
)
step4_4_baseline_prediction_profile_df["prediction_positive_rate"] = (
    step4_4_baseline_prediction_profile_df["prediction_positive_count"]
    / step4_4_baseline_prediction_profile_df["row_count"]
)
step4_4_baseline_prediction_profile_df["input_null_rate"] = (
    step4_4_baseline_prediction_profile_df["input_null_count"]
    / step4_4_baseline_prediction_profile_df["row_count"]
)

display(step4_4_baseline_rule_summary_df)
display(step4_4_baseline_prediction_profile_df)

assert step4_4_baseline_prediction_df["dumb_baseline_pred"].isin([0, 1]).all(), (
    "4.4.3 found invalid dumb baseline predictions outside 0/1."
)

,baseline_feature,baseline_rule,null_handling_rule,prediction_row_count,prediction_positive_count,prediction_positive_rate,null_input_count
0,alert_lag_1,predict_1_if_alert_lag_1_equals_1_else_0,fill_null_with_0,218222,177143,0.811756,1792


,point4_split_name,row_count,target_positive_count,prediction_positive_count,input_null_count,target_positive_rate,prediction_positive_rate,input_null_rate
0,test,75980,61198,60966,344,0.805449,0.802395,0.004528
1,train,63833,52978,52062,1107,0.829947,0.815597,0.017342
2,tuning_holdout,7530,6093,6091,0,0.809163,0.808898,0.000000
3,validation,70879,58268,58024,341,0.822077,0.818635,0.004811


In [22]:
# 4.4.4 Metryki dumb baseline per split

import pandas as pd

required_objects = [
    "step4_4_baseline_prediction_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.4.4: {missing_objects}"

step4_4_metrics_rows = []

for split_name, split_df in (
    step4_4_baseline_prediction_df
    .groupby("point4_split_name", dropna=False)
):
    y_true = split_df["target_alert_day"].astype(int)
    y_pred = split_df["dumb_baseline_pred"].astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    row_count = int(len(split_df))
    positive_count = int((y_true == 1).sum())
    negative_count = int((y_true == 0).sum())

    accuracy = (tp + tn) / row_count if row_count > 0 else pd.NA
    precision = tp / (tp + fp) if (tp + fp) > 0 else pd.NA
    recall = tp / (tp + fn) if (tp + fn) > 0 else pd.NA
    specificity = tn / (tn + fp) if (tn + fp) > 0 else pd.NA
    f1 = (2 * precision * recall / (precision + recall)) if (precision is not pd.NA and recall is not pd.NA and (precision + recall) > 0) else pd.NA
    balanced_accuracy = ((recall + specificity) / 2) if (recall is not pd.NA and specificity is not pd.NA) else pd.NA
    pred_positive_rate = y_pred.mean() if row_count > 0 else pd.NA

    step4_4_metrics_rows.append(
        {
            "point4_split_name": str(split_name),
            "row_count": row_count,
            "positive_count": positive_count,
            "negative_count": negative_count,
            "tp": tp,
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "accuracy": float(accuracy),
            "precision": float(precision),
            "recall": float(recall),
            "specificity": float(specificity),
            "f1": float(f1),
            "balanced_accuracy": float(balanced_accuracy),
            "pred_positive_rate": float(pred_positive_rate),
        }
    )

step4_4_dumb_baseline_metrics_df = (
    pd.DataFrame(step4_4_metrics_rows)
    .sort_values("point4_split_name")
    .reset_index(drop=True)
)

display(step4_4_dumb_baseline_metrics_df)

assert step4_4_dumb_baseline_metrics_df.shape[0] == 4, (
    "4.4.4 expected exactly 4 split rows in dumb baseline metrics."
)

,point4_split_name,row_count,positive_count,negative_count,tp,tn,fp,fn,accuracy,precision,recall,specificity,f1,balanced_accuracy,pred_positive_rate
0,test,75980,61198,14782,51039,4855,9927,10159,0.735641,0.837172,0.833998,0.328440,0.835582,0.581219,0.802395
1,train,63833,52978,10855,45140,3933,6922,7838,0.768772,0.867043,0.852052,0.362322,0.859482,0.607187,0.815597
2,tuning_holdout,7530,6093,1437,5119,465,972,974,0.741567,0.840420,0.840144,0.323591,0.840282,0.581868,0.808898
3,validation,70879,58268,12611,49910,4497,8114,8358,0.767604,0.860161,0.856559,0.356593,0.858357,0.606576,0.818635


In [23]:
# 4.4.5 Zapis artefaktu sekcji 4.4

from pathlib import Path

import pandas as pd

required_objects = [
    "step4_3_project_root",
    "step4_4_dumb_baseline_metrics_df",
    "step4_4_baseline_rule_summary_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.4.5: {missing_objects}"

step4_4_output_dir = step4_3_project_root / "artifacts" / "4_dumb_baseline"
step4_4_output_dir.mkdir(parents=True, exist_ok=True)

step4_4_dumb_baseline_metrics_path = (
    step4_4_output_dir / "b4_04_dumb_baseline_metrics.parquet"
)

step4_4_dumb_baseline_metrics_save_df = step4_4_dumb_baseline_metrics_df.copy()

step4_4_dumb_baseline_metrics_save_df.insert(
    0,
    "baseline_feature",
    step4_4_baseline_rule_summary_df.loc[0, "baseline_feature"],
)
step4_4_dumb_baseline_metrics_save_df.insert(
    1,
    "baseline_rule",
    step4_4_baseline_rule_summary_df.loc[0, "baseline_rule"],
)
step4_4_dumb_baseline_metrics_save_df.insert(
    2,
    "null_handling_rule",
    step4_4_baseline_rule_summary_df.loc[0, "null_handling_rule"],
)

step4_4_dumb_baseline_metrics_save_df.to_parquet(
    step4_4_dumb_baseline_metrics_path,
    index=False,
)

step4_4_saved_artifacts_registry_df = pd.DataFrame(
    [
        {
            "artifact_name": step4_4_dumb_baseline_metrics_path.name,
            "rows": int(step4_4_dumb_baseline_metrics_save_df.shape[0]),
            "cols": int(step4_4_dumb_baseline_metrics_save_df.shape[1]),
            "path": str(step4_4_dumb_baseline_metrics_path),
        }
    ]
)

display(step4_4_saved_artifacts_registry_df)

assert step4_4_dumb_baseline_metrics_path.exists(), (
    "4.4.5 failed to save b4_04_dumb_baseline_metrics.parquet"
)
assert step4_4_dumb_baseline_metrics_save_df.shape[0] == 4, (
    "4.4.5 expected 4 rows in saved dumb baseline metrics."
)

,artifact_name,rows,cols,path
0,b4_04_dumb_baseline_metrics.parquet,4,18,/root/projects/6_samodzielny_projekt_koncowy_w...


## Podsumowanie dziłu 4.4 Dumb Baseline operacyjny

**Kontekst inżynieryjny:** W dziale zbudowano obowiązkowy baseline operacyjny oparty na finalnej cesze `alert_lag_1`, dostępnej w końcowym zbiorze modelowym. Najpierw zweryfikowano dostępność realnych kandydatów baseline’owych w `keep_cols`, a następnie wybrano cechę obecną w finalnym `model_ready_dataset`. Sprawdzono kompletność tej cechy w podziale `train`, `tuning_holdout`, `validation` i `test`, po czym zdefiniowano prostą regułę baseline: predykcja `1`, gdy `alert_lag_1 = 1`, w przeciwnym razie `0`, przy obsłudze braków przez imputację `0`. Na końcu obliczono metryki jakości per split i zapisano artefakt `b4_04_dumb_baseline_metrics.parquet`.

**Interpretacja wyniku:** Cecha `alert_lag_1` jest obecna w finalnym zbiorze wejściowym i stanowi spójny punkt startowy do budowy baseline’u. Łącznie wygenerowano predykcje dla `218222` rekordów, z czego `177143` otrzymało predykcję pozytywną. Wystąpiło `1792` braków wejściowych, które zostały obsłużone zgodnie z regułą baseline. Wyniki per split są stabilne. Accuracy mieści się w zakresie od `0.735641` do `0.768772`, precision od `0.837172` do `0.867043`, recall od `0.833998` do `0.856559`, a F1 od `0.835582` do `0.859482`. Balanced accuracy mieści się w przedziale od `0.581219` do `0.607187`. Jednocześnie specificity pozostaje niższa niż recall, co oznacza przewagę predykcji pozytywnych nad ostrożnym rozpoznawaniem klasy negatywnej.

**Znaczenie biznesowe:** Dział dostarcza prosty i jawny benchmark operacyjny dla dalszych modeli. Dzięki temu kolejne baseline’y ML i modele docelowe można porównywać nie z pustym punktem odniesienia, lecz z regułą opartą na realnym sygnale historycznym. Taki baseline pełni funkcję minimalnego standardu jakości, który powinien zostać przekroczony przez dalsze etapy modelowania.

**Wniosek:** Dumb baseline został zbudowany poprawnie, a jego wyniki są stabilne między splitami i użyteczne jako punkt odniesienia do dalszego benchmarkowania. Dział `4.4` został wykonany zgodnie z założeniami i przygotowuje projekt do przejścia do lekkich baseline’ów ML.

## 4.5 Lekkie baseline’y ML

In [24]:
# 4.5.1 Wejście modelowe do lekkich baseline'ów ML

import pandas as pd

required_objects = [
    "step4_3_time_split_df",
    "step4_2_keep_cols_df",
    "step4_2_model_ready_columns_df",
    "step4_4_dumb_baseline_metrics_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.5.1: {missing_objects}"

step4_5_control_columns = [
    "activity_date",
    "station_id",
    "target_alert_day",
    "split_name",
]

step4_5_forbidden_x_columns = [
    "activity_date",
    "station_id",
    "target_alert_day",
    "split_name",
    "inherited_split_name",
    "year_month",
    "is_tuning_holdout",
    "train_subsplit_name",
    "point4_split_name",
]

assert "column_name" in step4_2_keep_cols_df.columns, (
    "Missing column_name in step4_2_keep_cols_df."
)

step4_5_feature_columns = (
    step4_2_keep_cols_df["column_name"]
    .astype("string")
    .str.strip()
    .tolist()
)

step4_5_missing_feature_columns = [
    col for col in step4_5_feature_columns
    if col not in step4_3_time_split_df.columns
]
assert not step4_5_missing_feature_columns, (
    f"Missing keep_cols in time split dataset: {step4_5_missing_feature_columns}"
)

step4_5_forbidden_present_in_feature_columns = [
    col for col in step4_5_feature_columns
    if col in step4_5_forbidden_x_columns
]
assert not step4_5_forbidden_present_in_feature_columns, (
    f"Forbidden columns present in feature_columns: {step4_5_forbidden_present_in_feature_columns}"
)

step4_5_allowed_split_names = {
    "train",
    "tuning_holdout",
    "validation",
    "test",
}
step4_5_present_split_names = set(
    step4_3_time_split_df["point4_split_name"]
    .astype("string")
    .dropna()
    .unique()
    .tolist()
)
assert step4_5_present_split_names == step4_5_allowed_split_names, (
    f"Unexpected point4 split names: {sorted(step4_5_present_split_names)}"
)

step4_5_ml_input_df = step4_3_time_split_df[
    step4_5_control_columns + ["point4_split_name"] + step4_5_feature_columns
].copy()

step4_5_input_summary_df = pd.DataFrame(
    [
        {
            "ml_input_row_count": int(step4_5_ml_input_df.shape[0]),
            "ml_input_column_count": int(step4_5_ml_input_df.shape[1]),
            "feature_column_count": int(len(step4_5_feature_columns)),
            "control_column_count": int(len(step4_5_control_columns)),
            "split_name_in_feature_columns": int("split_name" in step4_5_feature_columns),
            "train_row_count": int((step4_5_ml_input_df["point4_split_name"] == "train").sum()),
            "tuning_holdout_row_count": int((step4_5_ml_input_df["point4_split_name"] == "tuning_holdout").sum()),
            "validation_row_count": int((step4_5_ml_input_df["point4_split_name"] == "validation").sum()),
            "test_row_count": int((step4_5_ml_input_df["point4_split_name"] == "test").sum()),
            "dumb_baseline_split_count": int(step4_4_dumb_baseline_metrics_df.shape[0]),
        }
    ]
)

display(step4_5_input_summary_df)

,ml_input_row_count,ml_input_column_count,feature_column_count,control_column_count,split_name_in_feature_columns,train_row_count,tuning_holdout_row_count,validation_row_count,test_row_count,dumb_baseline_split_count
0,218222,39,34,4,0,63833,7530,70879,75980,4


In [25]:
# 4.5.2 Surowe macierze X / y dla lekkich baseline'ów ML

import pandas as pd

required_objects = [
    "step4_5_ml_input_df",
    "step4_5_feature_columns",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.5.2: {missing_objects}"

step4_5_train_df = (
    step4_5_ml_input_df
    .loc[step4_5_ml_input_df["point4_split_name"] == "train"]
    .copy()
    .reset_index(drop=True)
)

step4_5_tuning_holdout_df = (
    step4_5_ml_input_df
    .loc[step4_5_ml_input_df["point4_split_name"] == "tuning_holdout"]
    .copy()
    .reset_index(drop=True)
)

step4_5_validation_df = (
    step4_5_ml_input_df
    .loc[step4_5_ml_input_df["point4_split_name"] == "validation"]
    .copy()
    .reset_index(drop=True)
)

step4_5_test_df = (
    step4_5_ml_input_df
    .loc[step4_5_ml_input_df["point4_split_name"] == "test"]
    .copy()
    .reset_index(drop=True)
)

step4_5_X_train_raw_df = step4_5_train_df[step4_5_feature_columns].copy()
step4_5_y_train = step4_5_train_df["target_alert_day"].astype("int8").copy()

step4_5_X_tuning_holdout_raw_df = step4_5_tuning_holdout_df[step4_5_feature_columns].copy()
step4_5_y_tuning_holdout = step4_5_tuning_holdout_df["target_alert_day"].astype("int8").copy()

step4_5_X_validation_raw_df = step4_5_validation_df[step4_5_feature_columns].copy()
step4_5_y_validation = step4_5_validation_df["target_alert_day"].astype("int8").copy()

step4_5_X_test_raw_df = step4_5_test_df[step4_5_feature_columns].copy()
step4_5_y_test = step4_5_test_df["target_alert_day"].astype("int8").copy()

step4_5_raw_matrix_summary_df = pd.DataFrame(
    [
        {
            "split_name": "train",
            "X_row_count": int(step4_5_X_train_raw_df.shape[0]),
            "X_col_count": int(step4_5_X_train_raw_df.shape[1]),
            "y_row_count": int(step4_5_y_train.shape[0]),
            "positive_count": int(step4_5_y_train.sum()),
            "feature_missing_cell_count": int(step4_5_X_train_raw_df.isna().sum().sum()),
        },
        {
            "split_name": "tuning_holdout",
            "X_row_count": int(step4_5_X_tuning_holdout_raw_df.shape[0]),
            "X_col_count": int(step4_5_X_tuning_holdout_raw_df.shape[1]),
            "y_row_count": int(step4_5_y_tuning_holdout.shape[0]),
            "positive_count": int(step4_5_y_tuning_holdout.sum()),
            "feature_missing_cell_count": int(step4_5_X_tuning_holdout_raw_df.isna().sum().sum()),
        },
        {
            "split_name": "validation",
            "X_row_count": int(step4_5_X_validation_raw_df.shape[0]),
            "X_col_count": int(step4_5_X_validation_raw_df.shape[1]),
            "y_row_count": int(step4_5_y_validation.shape[0]),
            "positive_count": int(step4_5_y_validation.sum()),
            "feature_missing_cell_count": int(step4_5_X_validation_raw_df.isna().sum().sum()),
        },
        {
            "split_name": "test",
            "X_row_count": int(step4_5_X_test_raw_df.shape[0]),
            "X_col_count": int(step4_5_X_test_raw_df.shape[1]),
            "y_row_count": int(step4_5_y_test.shape[0]),
            "positive_count": int(step4_5_y_test.sum()),
            "feature_missing_cell_count": int(step4_5_X_test_raw_df.isna().sum().sum()),
        },
    ]
)

display(step4_5_raw_matrix_summary_df)

assert step4_5_X_train_raw_df.shape[1] == len(step4_5_feature_columns), (
    "4.5.2 train feature matrix column count mismatch."
)
assert step4_5_X_train_raw_df.shape[0] == step4_5_y_train.shape[0], (
    "4.5.2 train X/y row count mismatch."
)
assert step4_5_X_tuning_holdout_raw_df.shape[0] == step4_5_y_tuning_holdout.shape[0], (
    "4.5.2 tuning_holdout X/y row count mismatch."
)
assert step4_5_X_validation_raw_df.shape[0] == step4_5_y_validation.shape[0], (
    "4.5.2 validation X/y row count mismatch."
)
assert step4_5_X_test_raw_df.shape[0] == step4_5_y_test.shape[0], (
    "4.5.2 test X/y row count mismatch."
)

,split_name,X_row_count,X_col_count,y_row_count,positive_count,feature_missing_cell_count
0,train,63833,34,63833,52978,11797
1,tuning_holdout,7530,34,7530,6093,0
2,validation,70879,34,70879,58268,6243
3,test,75980,34,75980,61198,6249


In [26]:
# 4.5.3 Train-only plan imputacji dla lekkich baseline'ów ML

import pandas as pd
from pandas.api.types import (
    is_bool_dtype,
    is_numeric_dtype,
)

required_objects = [
    "step4_5_X_train_raw_df",
    "step4_5_X_tuning_holdout_raw_df",
    "step4_5_X_validation_raw_df",
    "step4_5_X_test_raw_df",
    "step4_5_feature_columns",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.5.3: {missing_objects}"

step4_5_imputation_rows = []

for column_name in step4_5_feature_columns:
    train_series = step4_5_X_train_raw_df[column_name]

    if is_bool_dtype(train_series):
        feature_type = "binary_like"
        imputation_strategy = "mode_train_only"
        non_null_series = train_series.dropna()
        if len(non_null_series) == 0:
            imputation_value = 0
        else:
            imputation_value = int(non_null_series.mode(dropna=True).iloc[0])

    elif is_numeric_dtype(train_series):
        unique_non_null_values = set(train_series.dropna().astype("float64").unique().tolist())

        if unique_non_null_values.issubset({0.0, 1.0}):
            feature_type = "binary_like"
            imputation_strategy = "mode_train_only"
            non_null_series = train_series.dropna()
            if len(non_null_series) == 0:
                imputation_value = 0
            else:
                imputation_value = float(non_null_series.mode(dropna=True).iloc[0])
        else:
            feature_type = "numeric"
            imputation_strategy = "median_train_only"
            non_null_series = train_series.dropna()
            if len(non_null_series) == 0:
                imputation_value = 0.0
            else:
                imputation_value = float(non_null_series.median())

    else:
        feature_type = "categorical_or_other"
        imputation_strategy = "mode_train_only"
        non_null_series = train_series.dropna()
        if len(non_null_series) == 0:
            imputation_value = "__missing__"
        else:
            imputation_value = str(non_null_series.mode(dropna=True).iloc[0])

    step4_5_imputation_rows.append(
        {
            "column_name": column_name,
            "raw_dtype": str(train_series.dtype),
            "feature_type": feature_type,
            "train_missing_count": int(train_series.isna().sum()),
            "train_missing_rate": float(train_series.isna().mean()),
            "imputation_strategy": imputation_strategy,
            "imputation_value": imputation_value,
        }
    )

step4_5_imputation_plan_df = (
    pd.DataFrame(step4_5_imputation_rows)
    .sort_values(["train_missing_count", "column_name"], ascending=[False, True])
    .reset_index(drop=True)
)

step4_5_imputation_summary_df = pd.DataFrame(
    [
        {
            "feature_count": int(step4_5_imputation_plan_df.shape[0]),
            "numeric_feature_count": int((step4_5_imputation_plan_df["feature_type"] == "numeric").sum()),
            "binary_like_feature_count": int((step4_5_imputation_plan_df["feature_type"] == "binary_like").sum()),
            "categorical_or_other_feature_count": int((step4_5_imputation_plan_df["feature_type"] == "categorical_or_other").sum()),
            "features_with_missing_count": int((step4_5_imputation_plan_df["train_missing_count"] > 0).sum()),
            "total_train_missing_cell_count": int(step4_5_imputation_plan_df["train_missing_count"].sum()),
        }
    ]
)

display(step4_5_imputation_summary_df)
display(step4_5_imputation_plan_df)

assert int(step4_5_imputation_summary_df.loc[0, "feature_count"]) == len(step4_5_feature_columns), (
    "4.5.3 feature count mismatch in imputation plan."
)

,feature_count,numeric_feature_count,binary_like_feature_count,categorical_or_other_feature_count,features_with_missing_count,total_train_missing_cell_count
0,34,4,30,0,11,11797


,column_name,raw_dtype,feature_type,train_missing_count,train_missing_rate,imputation_strategy,imputation_value
0,alert_lag_7,Float32,binary_like,3440,0.053891,mode_train_only,1.0
1,alert_lag_3,Float32,binary_like,1884,0.029515,mode_train_only,1.0
2,alert_lag_2,Float32,binary_like,1495,0.023420,mode_train_only,1.0
3,alert_hours_lag_1,Float32,numeric,1107,0.017342,median_train_only,17.0
4,alert_lag_1,Float32,binary_like,1107,0.017342,mode_train_only,1.0
5,deficit_alert_lag_1,Float32,binary_like,1107,0.017342,mode_train_only,1.0
6,alert_hours_roll_sum_14,Float32,numeric,390,0.006110,median_train_only,211.0
7,alert_severity_roll_max_7,Float32,numeric,390,0.006110,median_train_only,12.0
8,high_severity_alert_last_7d,Float32,binary_like,390,0.006110,mode_train_only,0.0
9,consecutive_alert_days_before_t,Float32,numeric,247,0.003869,median_train_only,4.0


In [27]:
# 4.5.4 Zastosowanie imputacji train-only do wszystkich splitów

import pandas as pd

required_objects = [
    "step4_5_imputation_plan_df",
    "step4_5_X_train_raw_df",
    "step4_5_X_tuning_holdout_raw_df",
    "step4_5_X_validation_raw_df",
    "step4_5_X_test_raw_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.5.4: {missing_objects}"

step4_5_imputation_value_map = dict(
    zip(
        step4_5_imputation_plan_df["column_name"].astype("string"),
        step4_5_imputation_plan_df["imputation_value"],
    )
)

step4_5_X_train_imputed_df = step4_5_X_train_raw_df.copy()
step4_5_X_tuning_holdout_imputed_df = step4_5_X_tuning_holdout_raw_df.copy()
step4_5_X_validation_imputed_df = step4_5_X_validation_raw_df.copy()
step4_5_X_test_imputed_df = step4_5_X_test_raw_df.copy()

for column_name, imputation_value in step4_5_imputation_value_map.items():
    step4_5_X_train_imputed_df[column_name] = (
        step4_5_X_train_imputed_df[column_name].fillna(imputation_value)
    )
    step4_5_X_tuning_holdout_imputed_df[column_name] = (
        step4_5_X_tuning_holdout_imputed_df[column_name].fillna(imputation_value)
    )
    step4_5_X_validation_imputed_df[column_name] = (
        step4_5_X_validation_imputed_df[column_name].fillna(imputation_value)
    )
    step4_5_X_test_imputed_df[column_name] = (
        step4_5_X_test_imputed_df[column_name].fillna(imputation_value)
    )

step4_5_imputed_matrix_summary_df = pd.DataFrame(
    [
        {
            "split_name": "train",
            "row_count": int(step4_5_X_train_imputed_df.shape[0]),
            "col_count": int(step4_5_X_train_imputed_df.shape[1]),
            "missing_cell_count_after_imputation": int(step4_5_X_train_imputed_df.isna().sum().sum()),
        },
        {
            "split_name": "tuning_holdout",
            "row_count": int(step4_5_X_tuning_holdout_imputed_df.shape[0]),
            "col_count": int(step4_5_X_tuning_holdout_imputed_df.shape[1]),
            "missing_cell_count_after_imputation": int(step4_5_X_tuning_holdout_imputed_df.isna().sum().sum()),
        },
        {
            "split_name": "validation",
            "row_count": int(step4_5_X_validation_imputed_df.shape[0]),
            "col_count": int(step4_5_X_validation_imputed_df.shape[1]),
            "missing_cell_count_after_imputation": int(step4_5_X_validation_imputed_df.isna().sum().sum()),
        },
        {
            "split_name": "test",
            "row_count": int(step4_5_X_test_imputed_df.shape[0]),
            "col_count": int(step4_5_X_test_imputed_df.shape[1]),
            "missing_cell_count_after_imputation": int(step4_5_X_test_imputed_df.isna().sum().sum()),
        },
    ]
)

display(step4_5_imputed_matrix_summary_df)

assert int(step4_5_imputed_matrix_summary_df["missing_cell_count_after_imputation"].sum()) == 0, (
    "4.5.4 found remaining missing cells after train-only imputation."
)

,split_name,row_count,col_count,missing_cell_count_after_imputation
0,train,63833,34,0
1,tuning_holdout,7530,34,0
2,validation,70879,34,0
3,test,75980,34,0


In [28]:
# 4.5.5 Finalne macierze modelowe po imputacji i kontroli kodowania train-only

import pandas as pd
from pandas.api.types import is_numeric_dtype

required_objects = [
    "step4_5_X_train_imputed_df",
    "step4_5_X_tuning_holdout_imputed_df",
    "step4_5_X_validation_imputed_df",
    "step4_5_X_test_imputed_df",
    "step4_5_y_train",
    "step4_5_y_tuning_holdout",
    "step4_5_y_validation",
    "step4_5_y_test",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.5.5: {missing_objects}"

step4_5_X_train_model_df = step4_5_X_train_imputed_df.copy()
step4_5_X_tuning_holdout_model_df = step4_5_X_tuning_holdout_imputed_df.copy()
step4_5_X_validation_model_df = step4_5_X_validation_imputed_df.copy()
step4_5_X_test_model_df = step4_5_X_test_imputed_df.copy()

step4_5_non_numeric_feature_columns = [
    col for col in step4_5_X_train_model_df.columns
    if not is_numeric_dtype(step4_5_X_train_model_df[col])
]

assert not step4_5_non_numeric_feature_columns, (
    f"4.5.5 found non-numeric feature columns before model baselines: "
    f"{step4_5_non_numeric_feature_columns}"
)

for col in step4_5_X_train_model_df.columns:
    step4_5_X_train_model_df[col] = pd.to_numeric(step4_5_X_train_model_df[col], errors="raise").astype("float32")
    step4_5_X_tuning_holdout_model_df[col] = pd.to_numeric(step4_5_X_tuning_holdout_model_df[col], errors="raise").astype("float32")
    step4_5_X_validation_model_df[col] = pd.to_numeric(step4_5_X_validation_model_df[col], errors="raise").astype("float32")
    step4_5_X_test_model_df[col] = pd.to_numeric(step4_5_X_test_model_df[col], errors="raise").astype("float32")

step4_5_model_matrix_summary_df = pd.DataFrame(
    [
        {
            "split_name": "train",
            "X_row_count": int(step4_5_X_train_model_df.shape[0]),
            "X_col_count": int(step4_5_X_train_model_df.shape[1]),
            "y_row_count": int(step4_5_y_train.shape[0]),
            "missing_cell_count": int(step4_5_X_train_model_df.isna().sum().sum()),
            "dtype_example": str(step4_5_X_train_model_df.dtypes.iloc[0]),
        },
        {
            "split_name": "tuning_holdout",
            "X_row_count": int(step4_5_X_tuning_holdout_model_df.shape[0]),
            "X_col_count": int(step4_5_X_tuning_holdout_model_df.shape[1]),
            "y_row_count": int(step4_5_y_tuning_holdout.shape[0]),
            "missing_cell_count": int(step4_5_X_tuning_holdout_model_df.isna().sum().sum()),
            "dtype_example": str(step4_5_X_tuning_holdout_model_df.dtypes.iloc[0]),
        },
        {
            "split_name": "validation",
            "X_row_count": int(step4_5_X_validation_model_df.shape[0]),
            "X_col_count": int(step4_5_X_validation_model_df.shape[1]),
            "y_row_count": int(step4_5_y_validation.shape[0]),
            "missing_cell_count": int(step4_5_X_validation_model_df.isna().sum().sum()),
            "dtype_example": str(step4_5_X_validation_model_df.dtypes.iloc[0]),
        },
        {
            "split_name": "test",
            "X_row_count": int(step4_5_X_test_model_df.shape[0]),
            "X_col_count": int(step4_5_X_test_model_df.shape[1]),
            "y_row_count": int(step4_5_y_test.shape[0]),
            "missing_cell_count": int(step4_5_X_test_model_df.isna().sum().sum()),
            "dtype_example": str(step4_5_X_test_model_df.dtypes.iloc[0]),
        },
    ]
)

display(step4_5_model_matrix_summary_df)

assert int(step4_5_model_matrix_summary_df["missing_cell_count"].sum()) == 0, (
    "4.5.5 found missing values in final model matrices."
)
assert (step4_5_model_matrix_summary_df["X_row_count"] == step4_5_model_matrix_summary_df["y_row_count"]).all(), (
    "4.5.5 found X/y row count mismatch in final model matrices."
)

,split_name,X_row_count,X_col_count,y_row_count,missing_cell_count,dtype_example
0,train,63833,34,63833,0,float32
1,tuning_holdout,7530,34,7530,0,float32
2,validation,70879,34,70879,0,float32
3,test,75980,34,75980,0,float32


In [29]:
# 4.5.6 Logistic Regression baseline

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

required_objects = [
    "step4_5_X_train_model_df",
    "step4_5_X_tuning_holdout_model_df",
    "step4_5_X_validation_model_df",
    "step4_5_y_train",
    "step4_5_y_tuning_holdout",
    "step4_5_y_validation",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.5.6: {missing_objects}"

step4_5_logreg_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                solver="liblinear",
                max_iter=1000,
                random_state=42,
            ),
        ),
    ]
)

step4_5_logreg_pipeline.fit(step4_5_X_train_model_df, step4_5_y_train)

def compute_binary_metrics(y_true, y_pred, y_score):
    y_true = pd.Series(y_true).astype(int)
    y_pred = pd.Series(y_pred).astype(int)
    y_score = pd.Series(y_score).astype(float)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    row_count = int(len(y_true))
    positive_count = int((y_true == 1).sum())
    negative_count = int((y_true == 0).sum())

    accuracy = (tp + tn) / row_count if row_count > 0 else np.nan
    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    f1 = (
        2 * precision * recall / (precision + recall)
        if pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0
        else np.nan
    )
    balanced_accuracy = (
        (recall + specificity) / 2
        if pd.notna(recall) and pd.notna(specificity)
        else np.nan
    )
    pred_positive_rate = float(y_pred.mean()) if row_count > 0 else np.nan
    roc_auc = float(roc_auc_score(y_true, y_score)) if positive_count > 0 and negative_count > 0 else np.nan
    pr_auc = float(average_precision_score(y_true, y_score)) if positive_count > 0 else np.nan

    return {
        "row_count": row_count,
        "positive_count": positive_count,
        "negative_count": negative_count,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "specificity": float(specificity),
        "f1": float(f1),
        "balanced_accuracy": float(balanced_accuracy),
        "pred_positive_rate": float(pred_positive_rate),
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
    }

step4_5_logreg_eval_plan = [
    ("train", step4_5_X_train_model_df, step4_5_y_train),
    ("tuning_holdout", step4_5_X_tuning_holdout_model_df, step4_5_y_tuning_holdout),
    ("validation", step4_5_X_validation_model_df, step4_5_y_validation),
]

step4_5_logreg_metric_rows = []

for split_name, X_split, y_split in step4_5_logreg_eval_plan:
    y_score = step4_5_logreg_pipeline.predict_proba(X_split)[:, 1]
    y_pred = (y_score >= 0.5).astype("int8")

    metric_row = compute_binary_metrics(y_split, y_pred, y_score)
    metric_row["model_name"] = "logistic_regression"
    metric_row["split_name"] = split_name
    step4_5_logreg_metric_rows.append(metric_row)

step4_5_logreg_metrics_df = (
    pd.DataFrame(step4_5_logreg_metric_rows)
    .sort_values("split_name")
    .reset_index(drop=True)
)

display(step4_5_logreg_metrics_df)

assert step4_5_logreg_metrics_df.shape[0] == 3, (
    "4.5.6 expected exactly 3 split rows for logistic regression."
)

,row_count,positive_count,negative_count,tp,tn,fp,fn,accuracy,precision,recall,specificity,f1,balanced_accuracy,pred_positive_rate,roc_auc,pr_auc,model_name,split_name
0,63833,52978,10855,52136,1136,9719,842,0.834553,0.842874,0.984107,0.104652,0.908032,0.544379,0.969013,0.744695,0.926484,logistic_regression,train
1,7530,6093,1437,5960,125,1312,133,0.808101,0.819582,0.978172,0.086987,0.891882,0.532579,0.965737,0.725664,0.909122,logistic_regression,tuning_holdout
2,70879,58268,12611,57248,1378,11233,1020,0.827128,0.835969,0.982495,0.109270,0.903329,0.545882,0.966168,0.743043,0.922323,logistic_regression,validation


In [30]:
# 4.5.7 RandomForest baseline

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

required_objects = [
    "step4_5_X_train_model_df",
    "step4_5_X_tuning_holdout_model_df",
    "step4_5_X_validation_model_df",
    "step4_5_y_train",
    "step4_5_y_tuning_holdout",
    "step4_5_y_validation",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.5.7: {missing_objects}"

if "compute_binary_metrics" not in globals():
    def compute_binary_metrics(y_true, y_pred, y_score):
        y_true = pd.Series(y_true).astype(int)
        y_pred = pd.Series(y_pred).astype(int)
        y_score = pd.Series(y_score).astype(float)

        tp = int(((y_true == 1) & (y_pred == 1)).sum())
        tn = int(((y_true == 0) & (y_pred == 0)).sum())
        fp = int(((y_true == 0) & (y_pred == 1)).sum())
        fn = int(((y_true == 1) & (y_pred == 0)).sum())

        row_count = int(len(y_true))
        positive_count = int((y_true == 1).sum())
        negative_count = int((y_true == 0).sum())

        accuracy = (tp + tn) / row_count if row_count > 0 else np.nan
        precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
        recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = (
            2 * precision * recall / (precision + recall)
            if pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0
            else np.nan
        )
        balanced_accuracy = (
            (recall + specificity) / 2
            if pd.notna(recall) and pd.notna(specificity)
            else np.nan
        )
        pred_positive_rate = float(y_pred.mean()) if row_count > 0 else np.nan
        roc_auc = float(roc_auc_score(y_true, y_score)) if positive_count > 0 and negative_count > 0 else np.nan
        pr_auc = float(average_precision_score(y_true, y_score)) if positive_count > 0 else np.nan

        return {
            "row_count": row_count,
            "positive_count": positive_count,
            "negative_count": negative_count,
            "tp": tp,
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "accuracy": float(accuracy),
            "precision": float(precision),
            "recall": float(recall),
            "specificity": float(specificity),
            "f1": float(f1),
            "balanced_accuracy": float(balanced_accuracy),
            "pred_positive_rate": float(pred_positive_rate),
            "roc_auc": float(roc_auc),
            "pr_auc": float(pr_auc),
        }

step4_5_random_forest_model = RandomForestClassifier(
    n_estimators=200,
    criterion="gini",
    max_features="sqrt",
    random_state=42,
    n_jobs=-1,
)

step4_5_random_forest_model.fit(step4_5_X_train_model_df, step4_5_y_train)

step4_5_random_forest_eval_plan = [
    ("train", step4_5_X_train_model_df, step4_5_y_train),
    ("tuning_holdout", step4_5_X_tuning_holdout_model_df, step4_5_y_tuning_holdout),
    ("validation", step4_5_X_validation_model_df, step4_5_y_validation),
]

step4_5_random_forest_metric_rows = []

for split_name, X_split, y_split in step4_5_random_forest_eval_plan:
    y_score = step4_5_random_forest_model.predict_proba(X_split)[:, 1]
    y_pred = (y_score >= 0.5).astype("int8")

    metric_row = compute_binary_metrics(y_split, y_pred, y_score)
    metric_row["model_name"] = "random_forest"
    metric_row["split_name"] = split_name
    step4_5_random_forest_metric_rows.append(metric_row)

step4_5_random_forest_metrics_df = (
    pd.DataFrame(step4_5_random_forest_metric_rows)
    .sort_values("split_name")
    .reset_index(drop=True)
)

display(step4_5_random_forest_metrics_df)

assert step4_5_random_forest_metrics_df.shape[0] == 3, (
    "4.5.7 expected exactly 3 split rows for random forest."
)

,row_count,positive_count,negative_count,tp,tn,fp,fn,accuracy,precision,recall,specificity,f1,balanced_accuracy,pred_positive_rate,roc_auc,pr_auc,model_name,split_name
0,63833,52978,10855,52795,10517,338,183,0.991838,0.993639,0.996546,0.968862,0.995090,0.982704,0.832375,0.998958,0.999796,random_forest,train
1,7530,6093,1437,5632,288,1149,461,0.786189,0.830556,0.924339,0.200418,0.874942,0.562378,0.900531,0.694877,0.900336,random_forest,tuning_holdout
2,70879,58268,12611,54783,2594,10017,3485,0.809506,0.845417,0.940190,0.205693,0.890288,0.572942,0.914234,0.714382,0.913683,random_forest,validation


In [31]:
# 4.5.8 ExtraTrees baseline

import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

required_objects = [
    "step4_5_X_train_model_df",
    "step4_5_X_tuning_holdout_model_df",
    "step4_5_X_validation_model_df",
    "step4_5_y_train",
    "step4_5_y_tuning_holdout",
    "step4_5_y_validation",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.5.8: {missing_objects}"

if "compute_binary_metrics" not in globals():
    def compute_binary_metrics(y_true, y_pred, y_score):
        y_true = pd.Series(y_true).astype(int)
        y_pred = pd.Series(y_pred).astype(int)
        y_score = pd.Series(y_score).astype(float)

        tp = int(((y_true == 1) & (y_pred == 1)).sum())
        tn = int(((y_true == 0) & (y_pred == 0)).sum())
        fp = int(((y_true == 0) & (y_pred == 1)).sum())
        fn = int(((y_true == 1) & (y_pred == 0)).sum())

        row_count = int(len(y_true))
        positive_count = int((y_true == 1).sum())
        negative_count = int((y_true == 0).sum())

        accuracy = (tp + tn) / row_count if row_count > 0 else np.nan
        precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
        recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = (
            2 * precision * recall / (precision + recall)
            if pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0
            else np.nan
        )
        balanced_accuracy = (
            (recall + specificity) / 2
            if pd.notna(recall) and pd.notna(specificity)
            else np.nan
        )
        pred_positive_rate = float(y_pred.mean()) if row_count > 0 else np.nan
        roc_auc = float(roc_auc_score(y_true, y_score)) if positive_count > 0 and negative_count > 0 else np.nan
        pr_auc = float(average_precision_score(y_true, y_score)) if positive_count > 0 else np.nan

        return {
            "row_count": row_count,
            "positive_count": positive_count,
            "negative_count": negative_count,
            "tp": tp,
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "accuracy": float(accuracy),
            "precision": float(precision),
            "recall": float(recall),
            "specificity": float(specificity),
            "f1": float(f1),
            "balanced_accuracy": float(balanced_accuracy),
            "pred_positive_rate": float(pred_positive_rate),
            "roc_auc": float(roc_auc),
            "pr_auc": float(pr_auc),
        }

step4_5_extra_trees_model = ExtraTreesClassifier(
    n_estimators=300,
    criterion="gini",
    max_features="sqrt",
    random_state=42,
    n_jobs=-1,
)

step4_5_extra_trees_model.fit(step4_5_X_train_model_df, step4_5_y_train)

step4_5_extra_trees_eval_plan = [
    ("train", step4_5_X_train_model_df, step4_5_y_train),
    ("tuning_holdout", step4_5_X_tuning_holdout_model_df, step4_5_y_tuning_holdout),
    ("validation", step4_5_X_validation_model_df, step4_5_y_validation),
]

step4_5_extra_trees_metric_rows = []

for split_name, X_split, y_split in step4_5_extra_trees_eval_plan:
    y_score = step4_5_extra_trees_model.predict_proba(X_split)[:, 1]
    y_pred = (y_score >= 0.5).astype("int8")

    metric_row = compute_binary_metrics(y_split, y_pred, y_score)
    metric_row["model_name"] = "extra_trees"
    metric_row["split_name"] = split_name
    step4_5_extra_trees_metric_rows.append(metric_row)

step4_5_extra_trees_metrics_df = (
    pd.DataFrame(step4_5_extra_trees_metric_rows)
    .sort_values("split_name")
    .reset_index(drop=True)
)

display(step4_5_extra_trees_metrics_df)

assert step4_5_extra_trees_metrics_df.shape[0] == 3, (
    "4.5.8 expected exactly 3 split rows for extra trees."
)

,row_count,positive_count,negative_count,tp,tn,fp,fn,accuracy,precision,recall,specificity,f1,balanced_accuracy,pred_positive_rate,roc_auc,pr_auc,model_name,split_name
0,63833,52978,10855,52876,10436,419,102,0.991838,0.992138,0.998075,0.961400,0.995098,0.979737,0.834913,0.999739,0.999928,extra_trees,train
1,7530,6093,1437,5504,316,1121,589,0.772908,0.830792,0.903332,0.219903,0.865545,0.561617,0.879814,0.666739,0.882139,extra_trees,tuning_holdout
2,70879,58268,12611,53775,2874,9737,4493,0.799235,0.846690,0.922891,0.227896,0.883150,0.575394,0.896062,0.683346,0.894511,extra_trees,validation


In [32]:
# 4.5.9 HistGradientBoosting baseline

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

required_objects = [
    "step4_5_X_train_model_df",
    "step4_5_X_tuning_holdout_model_df",
    "step4_5_X_validation_model_df",
    "step4_5_y_train",
    "step4_5_y_tuning_holdout",
    "step4_5_y_validation",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.5.9: {missing_objects}"

if "compute_binary_metrics" not in globals():
    def compute_binary_metrics(y_true, y_pred, y_score):
        y_true = pd.Series(y_true).astype(int)
        y_pred = pd.Series(y_pred).astype(int)
        y_score = pd.Series(y_score).astype(float)

        tp = int(((y_true == 1) & (y_pred == 1)).sum())
        tn = int(((y_true == 0) & (y_pred == 0)).sum())
        fp = int(((y_true == 0) & (y_pred == 1)).sum())
        fn = int(((y_true == 1) & (y_pred == 0)).sum())

        row_count = int(len(y_true))
        positive_count = int((y_true == 1).sum())
        negative_count = int((y_true == 0).sum())

        accuracy = (tp + tn) / row_count if row_count > 0 else np.nan
        precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
        recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = (
            2 * precision * recall / (precision + recall)
            if pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0
            else np.nan
        )
        balanced_accuracy = (
            (recall + specificity) / 2
            if pd.notna(recall) and pd.notna(specificity)
            else np.nan
        )
        pred_positive_rate = float(y_pred.mean()) if row_count > 0 else np.nan
        roc_auc = float(roc_auc_score(y_true, y_score)) if positive_count > 0 and negative_count > 0 else np.nan
        pr_auc = float(average_precision_score(y_true, y_score)) if positive_count > 0 else np.nan

        return {
            "row_count": row_count,
            "positive_count": positive_count,
            "negative_count": negative_count,
            "tp": tp,
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "accuracy": float(accuracy),
            "precision": float(precision),
            "recall": float(recall),
            "specificity": float(specificity),
            "f1": float(f1),
            "balanced_accuracy": float(balanced_accuracy),
            "pred_positive_rate": float(pred_positive_rate),
            "roc_auc": float(roc_auc),
            "pr_auc": float(pr_auc),
        }

step4_5_hist_gb_model = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=200,
    max_depth=6,
    min_samples_leaf=20,
    random_state=42,
)

step4_5_hist_gb_model.fit(step4_5_X_train_model_df, step4_5_y_train)

step4_5_hist_gb_eval_plan = [
    ("train", step4_5_X_train_model_df, step4_5_y_train),
    ("tuning_holdout", step4_5_X_tuning_holdout_model_df, step4_5_y_tuning_holdout),
    ("validation", step4_5_X_validation_model_df, step4_5_y_validation),
]

step4_5_hist_gb_metric_rows = []

for split_name, X_split, y_split in step4_5_hist_gb_eval_plan:
    y_score = step4_5_hist_gb_model.predict_proba(X_split)[:, 1]
    y_pred = (y_score >= 0.5).astype("int8")

    metric_row = compute_binary_metrics(y_split, y_pred, y_score)
    metric_row["model_name"] = "hist_gradient_boosting"
    metric_row["split_name"] = split_name
    step4_5_hist_gb_metric_rows.append(metric_row)

step4_5_hist_gb_metrics_df = (
    pd.DataFrame(step4_5_hist_gb_metric_rows)
    .sort_values("split_name")
    .reset_index(drop=True)
)

display(step4_5_hist_gb_metrics_df)

assert step4_5_hist_gb_metrics_df.shape[0] == 3, (
    "4.5.9 expected exactly 3 split rows for hist gradient boosting."
)

,row_count,positive_count,negative_count,tp,tn,fp,fn,accuracy,precision,recall,specificity,f1,balanced_accuracy,pred_positive_rate,roc_auc,pr_auc,model_name,split_name
0,63833,52978,10855,52409,1328,9527,569,0.841837,0.846180,0.989260,0.122340,0.912143,0.555800,0.970282,0.796646,0.947935,hist_gradient_boosting,train
1,7530,6093,1437,5975,134,1303,118,0.811288,0.820967,0.980634,0.093250,0.893725,0.536942,0.966534,0.742099,0.920459,hist_gradient_boosting,tuning_holdout
2,70879,58268,12611,57450,1352,11259,818,0.829611,0.836135,0.985961,0.107208,0.904888,0.546585,0.969384,0.765273,0.934171,hist_gradient_boosting,validation


In [33]:
# 4.5.10 Porównanie lekkich baseline'ów ML z dumb baseline

import pandas as pd

expected_metric_objects = [
    "step4_5_logreg_metrics_df",
    "step4_5_random_forest_metrics_df",
    "step4_5_extra_trees_metrics_df",
    "step4_5_hist_gb_metrics_df",
    "step4_4_dumb_baseline_metrics_df",
]

missing_metric_objects = [name for name in expected_metric_objects if name not in globals()]
assert not missing_metric_objects, f"Missing metric objects in 4.5.10: {missing_metric_objects}"

step4_5_dumb_baseline_metrics_for_compare_df = (
    step4_4_dumb_baseline_metrics_df.copy()
    .rename(columns={"point4_split_name": "split_name"})
)

step4_5_dumb_baseline_metrics_for_compare_df["model_name"] = "dumb_baseline"

step4_5_dumb_baseline_metrics_for_compare_df = step4_5_dumb_baseline_metrics_for_compare_df[
    step4_5_dumb_baseline_metrics_for_compare_df["split_name"].isin(
        ["train", "tuning_holdout", "validation"]
    )
].reset_index(drop=True)

step4_5_logreg_metrics_for_compare_df = step4_5_logreg_metrics_df.copy()
step4_5_random_forest_metrics_for_compare_df = step4_5_random_forest_metrics_df.copy()
step4_5_extra_trees_metrics_for_compare_df = step4_5_extra_trees_metrics_df.copy()
step4_5_hist_gb_metrics_for_compare_df = step4_5_hist_gb_metrics_df.copy()

step4_5_light_ml_baselines_df = pd.concat(
    [
        step4_5_dumb_baseline_metrics_for_compare_df,
        step4_5_logreg_metrics_for_compare_df,
        step4_5_random_forest_metrics_for_compare_df,
        step4_5_extra_trees_metrics_for_compare_df,
        step4_5_hist_gb_metrics_for_compare_df,
    ],
    axis=0,
    ignore_index=True,
)

step4_5_light_ml_baselines_df["model_name"] = step4_5_light_ml_baselines_df["model_name"].astype(str)
step4_5_light_ml_baselines_df["split_name"] = step4_5_light_ml_baselines_df["split_name"].astype(str)

step4_5_light_ml_baselines_df = (
    step4_5_light_ml_baselines_df[
        [
            "model_name",
            "split_name",
            "row_count",
            "positive_count",
            "negative_count",
            "accuracy",
            "precision",
            "recall",
            "specificity",
            "f1",
            "balanced_accuracy",
            "pred_positive_rate",
            "roc_auc",
            "pr_auc",
            "tp",
            "tn",
            "fp",
            "fn",
        ]
    ]
    .sort_values(["split_name", "model_name"])
    .reset_index(drop=True)
)

step4_5_validation_comparison_df = (
    step4_5_light_ml_baselines_df.loc[
        step4_5_light_ml_baselines_df["split_name"].eq("validation")
    ]
    .sort_values(
        ["pr_auc", "roc_auc", "balanced_accuracy", "specificity", "recall"],
        ascending=[False, False, False, False, False],
    )
    .reset_index(drop=True)
)

display(step4_5_validation_comparison_df)
display(step4_5_light_ml_baselines_df)

assert step4_5_light_ml_baselines_df.shape[0] == 15, (
    "4.5.10 expected exactly 15 rows: 5 models x 3 splits."
)

,model_name,split_name,row_count,positive_count,negative_count,accuracy,precision,recall,specificity,f1,balanced_accuracy,pred_positive_rate,roc_auc,pr_auc,tp,tn,fp,fn
0,hist_gradient_boosting,validation,70879,58268,12611,0.829611,0.836135,0.985961,0.107208,0.904888,0.546585,0.969384,0.765273,0.934171,57450,1352,11259,818
1,logistic_regression,validation,70879,58268,12611,0.827128,0.835969,0.982495,0.109270,0.903329,0.545882,0.966168,0.743043,0.922323,57248,1378,11233,1020
2,random_forest,validation,70879,58268,12611,0.809506,0.845417,0.940190,0.205693,0.890288,0.572942,0.914234,0.714382,0.913683,54783,2594,10017,3485
3,extra_trees,validation,70879,58268,12611,0.799235,0.846690,0.922891,0.227896,0.883150,0.575394,0.896062,0.683346,0.894511,53775,2874,9737,4493
4,dumb_baseline,validation,70879,58268,12611,0.767604,0.860161,0.856559,0.356593,0.858357,0.606576,0.818635,NaN,NaN,49910,4497,8114,8358


,model_name,split_name,row_count,positive_count,negative_count,accuracy,precision,recall,specificity,f1,balanced_accuracy,pred_positive_rate,roc_auc,pr_auc,tp,tn,fp,fn
0,dumb_baseline,train,63833,52978,10855,0.768772,0.867043,0.852052,0.362322,0.859482,0.607187,0.815597,NaN,NaN,45140,3933,6922,7838
1,extra_trees,train,63833,52978,10855,0.991838,0.992138,0.998075,0.961400,0.995098,0.979737,0.834913,0.999739,0.999928,52876,10436,419,102
2,hist_gradient_boosting,train,63833,52978,10855,0.841837,0.846180,0.989260,0.122340,0.912143,0.555800,0.970282,0.796646,0.947935,52409,1328,9527,569
3,logistic_regression,train,63833,52978,10855,0.834553,0.842874,0.984107,0.104652,0.908032,0.544379,0.969013,0.744695,0.926484,52136,1136,9719,842
4,random_forest,train,63833,52978,10855,0.991838,0.993639,0.996546,0.968862,0.995090,0.982704,0.832375,0.998958,0.999796,52795,10517,338,183
5,dumb_baseline,tuning_holdout,7530,6093,1437,0.741567,0.840420,0.840144,0.323591,0.840282,0.581868,0.808898,NaN,NaN,5119,465,972,974
6,extra_trees,tuning_holdout,7530,6093,1437,0.772908,0.830792,0.903332,0.219903,0.865545,0.561617,0.879814,0.666739,0.882139,5504,316,1121,589
7,hist_gradient_boosting,tuning_holdout,7530,6093,1437,0.811288,0.820967,0.980634,0.093250,0.893725,0.536942,0.966534,0.742099,0.920459,5975,134,1303,118
8,logistic_regression,tuning_holdout,7530,6093,1437,0.808101,0.819582,0.978172,0.086987,0.891882,0.532579,0.965737,0.725664,0.909122,5960,125,1312,133
9,random_forest,tuning_holdout,7530,6093,1437,0.786189,0.830556,0.924339,0.200418,0.874942,0.562378,0.900531,0.694877,0.900336,5632,288,1149,461


In [34]:
# 4.5.11 Zapis artefaktu sekcji 4.5

from pathlib import Path

import pandas as pd

required_objects = [
    "step4_3_project_root",
    "step4_5_light_ml_baselines_df",
    "step4_5_validation_comparison_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.5.11: {missing_objects}"

step4_5_output_dir = step4_3_project_root / "artifacts" / "4_light_ml_baselines"
step4_5_output_dir.mkdir(parents=True, exist_ok=True)

step4_5_light_ml_baselines_path = (
    step4_5_output_dir / "b4_05_light_ml_baselines.parquet"
)

step4_5_light_ml_baselines_save_df = (
    step4_5_light_ml_baselines_df.copy()
    .sort_values(["split_name", "model_name"])
    .reset_index(drop=True)
)

step4_5_required_save_cols = [
    "model_name",
    "split_name",
    "row_count",
    "positive_count",
    "negative_count",
    "accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "balanced_accuracy",
    "pred_positive_rate",
    "roc_auc",
    "pr_auc",
    "tp",
    "tn",
    "fp",
    "fn",
]
step4_5_missing_save_cols = [
    col for col in step4_5_required_save_cols
    if col not in step4_5_light_ml_baselines_save_df.columns
]
assert not step4_5_missing_save_cols, (
    f"Missing save columns in 4.5.11: {step4_5_missing_save_cols}"
)

step4_5_light_ml_baselines_save_df.to_parquet(
    step4_5_light_ml_baselines_path,
    index=False,
)

step4_5_saved_artifacts_registry_df = pd.DataFrame(
    [
        {
            "artifact_name": step4_5_light_ml_baselines_path.name,
            "rows": int(step4_5_light_ml_baselines_save_df.shape[0]),
            "cols": int(step4_5_light_ml_baselines_save_df.shape[1]),
            "path": str(step4_5_light_ml_baselines_path),
            "validation_model_count": int(
                step4_5_validation_comparison_df["model_name"].nunique()
            ),
        }
    ]
)

display(step4_5_saved_artifacts_registry_df)

assert step4_5_light_ml_baselines_path.exists(), (
    "4.5.11 failed to save b4_05_light_ml_baselines.parquet"
)
assert step4_5_light_ml_baselines_save_df.shape[0] == 15, (
    "4.5.11 expected 15 rows in saved light ML baselines artifact."
)
assert step4_5_validation_comparison_df.shape[0] == 5, (
    "4.5.11 expected 5 validation rows in baseline comparison."
)

,artifact_name,rows,cols,path,validation_model_count
0,b4_05_light_ml_baselines.parquet,15,18,/root/projects/6_samodzielny_projekt_koncowy_w...,5


## Podsumowanie działu 4.5 Lekkie baseline’y ML

**Kontekst inżynieryjny:** W dziale przygotowano lekkie baseline’y ML na odziedziczonym wejściu modelowym z punktu `4.3`. Najpierw zweryfikowano kompletność wejścia, rozdział cech modelowych i kolumn kontrolnych oraz zbudowano surowe macierze `X / y` dla splitów `train`, `tuning_holdout`, `validation` i `test`. Następnie przygotowano plan imputacji `train-only`, zastosowano imputację do wszystkich splitów oraz potwierdzono gotowość końcowych macierzy modelowych do benchmarkowania. Na tej bazie uruchomiono cztery lekkie baseline’y ML: `LogisticRegression`, `RandomForest`, `ExtraTrees` oraz `HistGradientBoosting`. Na końcu zestawiono ich wyniki z `dumb baseline` z punktu `4.4` i zapisano artefakt `b4_05_light_ml_baselines.parquet`.

**Interpretacja wyniku:** Wejście do benchmarków ML objęło `218222` rekordy, `39` kolumn całkowitych, `34` finalne cechy modelowe i `4` kolumny kontrolne. Split wejściowy do lekkich baseline’ów obejmuje `63833` rekordy w `train`, `7530` w `tuning_holdout`, `70879` w `validation` i `75980` w `test`. W surowych macierzach występowały braki danych w `train` (`11797` komórek), `validation` (`6243`) i `test` (`6249`), podczas gdy `tuning_holdout` był kompletny; po zastosowaniu imputacji `train-only` wszystkie końcowe macierze modelowe miały `0` braków i wspólny format `34` cech typu wejściowego gotowego do modelowania. W benchmarku jakościowym najlepszy profil rankingowy na `validation` uzyskał `HistGradientBoosting` z `ROC AUC = 0.765273`, `PR AUC = 0.934171` i `recall = 0.985961`. `LogisticRegression` również osiągnął bardzo wysoki `recall = 0.982495` oraz `PR AUC = 0.922323`. `RandomForest` i `ExtraTrees` dały bardziej umiarkowany profil decyzyjny, z wyższą `specificity` niż modele liniowo-boostingowe, ale ze słabszym wynikiem rankingowym. Jednocześnie `dumb baseline` zachował najbardziej konserwatywny profil operacyjny przy progu `0.5`, z najwyższą `specificity = 0.356593` oraz `balanced_accuracy = 0.606576` na `validation`, podczas gdy lekkie baseline’y ML poprawiły przede wszystkim metryki rankingowe i `recall`.

**Znaczenie biznesowe:** Dział pokazuje, że lekkie modele ML wnoszą wartość już na etapie prostego benchmarku: lepiej porządkują ryzyko alertu i mocniej wychwytują klasę pozytywną niż prosty baseline regułowy. Jednocześnie wyniki potwierdzają, że domyślny próg `0.5` nie jest jeszcze punktem docelowym dla decyzji operacyjnej, ponieważ modele o najwyższym `recall` pozostają bardzo agresywne i generują szeroką predykcję klasy pozytywnej. Oznacza to, że blok `4.5` spełnił swoją rolę benchmarkingową: zidentyfikował najmocniejszego kandydata rankingowego w lekkiej klasie modeli oraz uzasadnił przejście do mocniejszego modelu głównego i dalszej pracy nad kalibracją oraz progiem decyzyjnym.

**Wniosek:** Dział `4.5` został wykonany poprawnie i zgodnie z założeniami. Przygotowano komplet lekkich baseline’ów ML, uporządkowano wejście modelowe w logice `train-only`, porównano modele z baseline’em operacyjnym oraz zapisano artefakt wynikowy `b4_05_light_ml_baselines.parquet`. Najmocniejszym kandydatem rankingowym w tej grupie jest `HistGradientBoosting`, natomiast pełna przewaga operacyjna nad prostym baseline’em wymaga dalszego etapu modelowania, kalibracji i strojenia progu. Sekcja `4.5` przygotowuje projekt do przejścia do `4.6`, czyli budowy głównego baseline’u `LightGBM`.

## 4.6 LightGBM baseline jako główny kandydat

In [35]:
# 4.6.0 Weryfikacja dostępności LightGBM

import lightgbm as lgb

print("LIGHTGBM_VERSION:", lgb.__version__)

LIGHTGBM_VERSION: 4.6.0


In [36]:
# 4.6.1 LightGBM baseline

import numpy as np
import pandas as pd
import lightgbm as lgb

from lightgbm import LGBMClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

required_objects = [
    "step4_5_X_train_model_df",
    "step4_5_X_tuning_holdout_model_df",
    "step4_5_X_validation_model_df",
    "step4_5_y_train",
    "step4_5_y_tuning_holdout",
    "step4_5_y_validation",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.6.1: {missing_objects}"

if "compute_binary_metrics" not in globals():
    def compute_binary_metrics(y_true, y_pred, y_score):
        y_true = pd.Series(y_true).astype(int)
        y_pred = pd.Series(y_pred).astype(int)
        y_score = pd.Series(y_score).astype(float)

        tp = int(((y_true == 1) & (y_pred == 1)).sum())
        tn = int(((y_true == 0) & (y_pred == 0)).sum())
        fp = int(((y_true == 0) & (y_pred == 1)).sum())
        fn = int(((y_true == 1) & (y_pred == 0)).sum())

        row_count = int(len(y_true))
        positive_count = int((y_true == 1).sum())
        negative_count = int((y_true == 0).sum())

        accuracy = (tp + tn) / row_count if row_count > 0 else np.nan
        precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
        recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = (
            2 * precision * recall / (precision + recall)
            if pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0
            else np.nan
        )
        balanced_accuracy = (
            (recall + specificity) / 2
            if pd.notna(recall) and pd.notna(specificity)
            else np.nan
        )
        pred_positive_rate = float(y_pred.mean()) if row_count > 0 else np.nan
        roc_auc = float(roc_auc_score(y_true, y_score)) if positive_count > 0 and negative_count > 0 else np.nan
        pr_auc = float(average_precision_score(y_true, y_score)) if positive_count > 0 else np.nan

        return {
            "row_count": row_count,
            "positive_count": positive_count,
            "negative_count": negative_count,
            "tp": tp,
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "accuracy": float(accuracy),
            "precision": float(precision),
            "recall": float(recall),
            "specificity": float(specificity),
            "f1": float(f1),
            "balanced_accuracy": float(balanced_accuracy),
            "pred_positive_rate": float(pred_positive_rate),
            "roc_auc": float(roc_auc),
            "pr_auc": float(pr_auc),
        }

step4_6_lgbm_baseline_model = LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)

step4_6_lgbm_baseline_model.fit(
    step4_5_X_train_model_df,
    step4_5_y_train,
)

step4_6_lgbm_eval_plan = [
    ("train", step4_5_X_train_model_df, step4_5_y_train),
    ("tuning_holdout", step4_5_X_tuning_holdout_model_df, step4_5_y_tuning_holdout),
    ("validation", step4_5_X_validation_model_df, step4_5_y_validation),
]

step4_6_lgbm_metric_rows = []

for split_name, X_split, y_split in step4_6_lgbm_eval_plan:
    y_score = step4_6_lgbm_baseline_model.predict_proba(X_split)[:, 1]
    y_pred = (y_score >= 0.5).astype("int8")

    metric_row = compute_binary_metrics(y_split, y_pred, y_score)
    metric_row["model_name"] = "lightgbm_baseline"
    metric_row["split_name"] = split_name
    step4_6_lgbm_metric_rows.append(metric_row)

step4_6_lgbm_baseline_metrics_df = (
    pd.DataFrame(step4_6_lgbm_metric_rows)
    .sort_values("split_name")
    .reset_index(drop=True)
)

display(step4_6_lgbm_baseline_metrics_df)

assert step4_6_lgbm_baseline_metrics_df.shape[0] == 3, (
    "4.6.1 expected exactly 3 split rows for LightGBM baseline."
)

,row_count,positive_count,negative_count,tp,tn,fp,fn,accuracy,precision,recall,specificity,f1,balanced_accuracy,pred_positive_rate,roc_auc,pr_auc,model_name,split_name
0,63833,52978,10855,52433,1520,9335,545,0.845221,0.848870,0.989713,0.140028,0.913897,0.564870,0.967650,0.815672,0.954060,lightgbm_baseline,train
1,7530,6093,1437,5963,130,1307,130,0.809163,0.820220,0.978664,0.090466,0.892464,0.534565,0.965471,0.743023,0.919105,lightgbm_baseline,tuning_holdout
2,70879,58268,12611,57414,1365,11246,854,0.829287,0.836207,0.985344,0.108239,0.904670,0.546791,0.968693,0.766115,0.934430,lightgbm_baseline,validation


In [37]:
# 4.6.2 Porównanie LightGBM baseline z dumb baseline oraz najlepszym lekkim ML

import pandas as pd

required_objects = [
    "step4_6_lgbm_baseline_metrics_df",
    "step4_5_validation_comparison_df",
    "step4_5_light_ml_baselines_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.6.2: {missing_objects}"

step4_6_best_light_ml_validation_df = (
    step4_5_validation_comparison_df.loc[
        step4_5_validation_comparison_df["model_name"] != "dumb_baseline"
    ]
    .sort_values(
        ["pr_auc", "roc_auc", "balanced_accuracy", "specificity", "recall"],
        ascending=[False, False, False, False, False],
    )
    .head(1)
    .reset_index(drop=True)
)

assert step4_6_best_light_ml_validation_df.shape[0] == 1, (
    "4.6.2 expected exactly one best light ML model on validation."
)

step4_6_best_light_ml_name = step4_6_best_light_ml_validation_df.loc[0, "model_name"]

step4_6_dumb_baseline_compare_df = (
    step4_5_light_ml_baselines_df.loc[
        step4_5_light_ml_baselines_df["model_name"] == "dumb_baseline"
    ]
    .copy()
)

step4_6_best_light_ml_compare_df = (
    step4_5_light_ml_baselines_df.loc[
        step4_5_light_ml_baselines_df["model_name"] == step4_6_best_light_ml_name
    ]
    .copy()
)

step4_6_lgbm_compare_df = step4_6_lgbm_baseline_metrics_df.copy()

step4_6_baseline_comparison_df = pd.concat(
    [
        step4_6_dumb_baseline_compare_df,
        step4_6_best_light_ml_compare_df,
        step4_6_lgbm_compare_df,
    ],
    axis=0,
    ignore_index=True,
)

step4_6_baseline_comparison_df = (
    step4_6_baseline_comparison_df[
        step4_6_baseline_comparison_df["split_name"].isin(["train", "validation"])
    ]
    .sort_values(["split_name", "model_name"])
    .reset_index(drop=True)
)

step4_6_validation_comparison_df = (
    step4_6_baseline_comparison_df.loc[
        step4_6_baseline_comparison_df["split_name"] == "validation"
    ]
    .sort_values(
        ["pr_auc", "roc_auc", "balanced_accuracy", "specificity", "recall"],
        ascending=[False, False, False, False, False],
    )
    .reset_index(drop=True)
)

display(step4_6_validation_comparison_df)
display(step4_6_baseline_comparison_df)

assert set(step4_6_baseline_comparison_df["model_name"].unique()) == {
    "dumb_baseline",
    step4_6_best_light_ml_name,
    "lightgbm_baseline",
}, "4.6.2 comparison should contain exactly dumb baseline, best light ML and LightGBM baseline."

assert set(step4_6_baseline_comparison_df["split_name"].unique()) == {
    "train",
    "validation",
}, "4.6.2 comparison should contain only train and validation splits."

,model_name,split_name,row_count,positive_count,negative_count,accuracy,precision,recall,specificity,f1,balanced_accuracy,pred_positive_rate,roc_auc,pr_auc,tp,tn,fp,fn
0,lightgbm_baseline,validation,70879,58268,12611,0.829287,0.836207,0.985344,0.108239,0.904670,0.546791,0.968693,0.766115,0.934430,57414,1365,11246,854
1,hist_gradient_boosting,validation,70879,58268,12611,0.829611,0.836135,0.985961,0.107208,0.904888,0.546585,0.969384,0.765273,0.934171,57450,1352,11259,818
2,dumb_baseline,validation,70879,58268,12611,0.767604,0.860161,0.856559,0.356593,0.858357,0.606576,0.818635,NaN,NaN,49910,4497,8114,8358


,model_name,split_name,row_count,positive_count,negative_count,accuracy,precision,recall,specificity,f1,balanced_accuracy,pred_positive_rate,roc_auc,pr_auc,tp,tn,fp,fn
0,dumb_baseline,train,63833,52978,10855,0.768772,0.867043,0.852052,0.362322,0.859482,0.607187,0.815597,NaN,NaN,45140,3933,6922,7838
1,hist_gradient_boosting,train,63833,52978,10855,0.841837,0.846180,0.989260,0.122340,0.912143,0.555800,0.970282,0.796646,0.947935,52409,1328,9527,569
2,lightgbm_baseline,train,63833,52978,10855,0.845221,0.848870,0.989713,0.140028,0.913897,0.564870,0.967650,0.815672,0.954060,52433,1520,9335,545
3,dumb_baseline,validation,70879,58268,12611,0.767604,0.860161,0.856559,0.356593,0.858357,0.606576,0.818635,NaN,NaN,49910,4497,8114,8358
4,hist_gradient_boosting,validation,70879,58268,12611,0.829611,0.836135,0.985961,0.107208,0.904888,0.546585,0.969384,0.765273,0.934171,57450,1352,11259,818
5,lightgbm_baseline,validation,70879,58268,12611,0.829287,0.836207,0.985344,0.108239,0.904670,0.546791,0.968693,0.766115,0.934430,57414,1365,11246,854


In [38]:
# 4.6.3 Zapis artefaktu sekcji 4.6

from pathlib import Path

import pandas as pd

required_objects = [
    "step4_3_project_root",
    "step4_6_lgbm_baseline_metrics_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.6.3: {missing_objects}"

step4_6_output_dir = step4_3_project_root / "artifacts" / "4_lgbm_baseline"
step4_6_output_dir.mkdir(parents=True, exist_ok=True)

step4_6_lgbm_baseline_path = step4_6_output_dir / "b4_06_lgbm_baseline.parquet"

step4_6_lgbm_baseline_save_df = (
    step4_6_lgbm_baseline_metrics_df.copy()
    .sort_values("split_name")
    .reset_index(drop=True)
)

step4_6_required_save_cols = [
    "model_name",
    "split_name",
    "row_count",
    "positive_count",
    "negative_count",
    "accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "balanced_accuracy",
    "pred_positive_rate",
    "roc_auc",
    "pr_auc",
    "tp",
    "tn",
    "fp",
    "fn",
]
step4_6_missing_save_cols = [
    col for col in step4_6_required_save_cols
    if col not in step4_6_lgbm_baseline_save_df.columns
]
assert not step4_6_missing_save_cols, (
    f"Missing save columns in 4.6.3: {step4_6_missing_save_cols}"
)

step4_6_lgbm_baseline_save_df.to_parquet(
    step4_6_lgbm_baseline_path,
    index=False,
)

step4_6_saved_artifacts_registry_df = pd.DataFrame(
    [
        {
            "artifact_name": step4_6_lgbm_baseline_path.name,
            "rows": int(step4_6_lgbm_baseline_save_df.shape[0]),
            "cols": int(step4_6_lgbm_baseline_save_df.shape[1]),
            "path": str(step4_6_lgbm_baseline_path),
            "model_name_count": int(step4_6_lgbm_baseline_save_df["model_name"].nunique()),
        }
    ]
)

display(step4_6_saved_artifacts_registry_df)

assert step4_6_lgbm_baseline_path.exists(), (
    "4.6.3 failed to save b4_06_lgbm_baseline.parquet"
)
assert step4_6_lgbm_baseline_save_df.shape[0] == 3, (
    "4.6.3 expected 3 rows in saved LightGBM baseline artifact."
)

,artifact_name,rows,cols,path,model_name_count
0,b4_06_lgbm_baseline.parquet,3,18,/root/projects/6_samodzielny_projekt_koncowy_w...,1


## Podsumowanie działu 4.6 LightGBM baseline jako główny kandydat

**Kontekst inżynieryjny:** W dziale `4.6` zweryfikowano dostępność biblioteki `LightGBM`, zbudowano bazowy model `LGBMClassifier` na tych samych macierzach wejściowych, które zostały przygotowane w `4.5`, a następnie oceniono jego jakość na splitach `train`, `tuning_holdout` oraz `validation`. Kolejnym krokiem było porównanie `LightGBM baseline` z `dumb baseline` oraz najlepszym lekkim baseline’em ML z działu `4.5`, czyli `HistGradientBoosting`. Na końcu zapisano artefakt wynikowy `b4_06_lgbm_baseline.parquet`.

**Interpretacja wyniku:** Model `LightGBM baseline` osiągnął na `validation` bardzo mocne metryki rankingowe: `ROC AUC = 0.766115`, `PR AUC = 0.934430` oraz `recall = 0.985344`. Oznacza to, że model bardzo dobrze porządkuje rekordy pod względem ryzyka klasy pozytywnej i skutecznie wychwytuje dni alertowe. Jednocześnie przy domyślnym progu `0.5` model pozostaje bardzo agresywny decyzyjnie: `specificity = 0.108239`, `balanced_accuracy = 0.546791`, a `pred_positive_rate = 0.968693`. W bezpośrednim porównaniu z najlepszym lekkim baseline’em ML `LightGBM` uzyskał minimalnie lepsze metryki rankingowe od `HistGradientBoosting`, ale różnice są niewielkie. Z kolei `dumb baseline` nadal zachowuje bardziej konserwatywny profil operacyjny przy progu `0.5`, z wyraźnie wyższą `specificity = 0.356593` oraz `balanced_accuracy = 0.606576` na `validation`.

**Znaczenie biznesowe:** Wynik działu `4.6` potwierdza, że `LightGBM` jest obecnie najmocniejszym kandydatem modelowym w projekcie pod względem jakości rankingu i wykrywania przypadków pozytywnych. Model daje bardzo dobry sygnał do dalszej pracy, ale jeszcze nie stanowi finalnej reguły decyzyjnej dla operacyjnego `0/1`, ponieważ przy progu `0.5` generuje bardzo szeroką predykcję klasy pozytywnej. Oznacza to, że kolejny etap projektu powinien koncentrować się nie na szukaniu nowego modelu od zera, lecz na dopracowaniu obecnego kandydata poprzez tuning hiperparametrów, kalibrację i dobór progu biznesowego.

**Wniosek:** Dział `4.6` został wykonany poprawnie i zgodnie z założeniami. `LightGBM baseline` został zbudowany, oceniony i zapisany jako artefakt `b4_06_lgbm_baseline.parquet`. Model minimalnie wyprzedza najlepszy lekki baseline ML i staje się oficjalnym głównym kandydatem do dalszej optymalizacji. Jednocześnie wynik uzasadnia przejście do `4.7`, czyli lekkiego tuningu hiperparametrów, bez potrzeby zmiany logiki wejścia modelowego ani kontraktu danych.

## 4.7 Lekki tuning hiperparametrów

In [39]:
# 4.7.1 Weryfikacja wejścia tuningowego

import pandas as pd

required_objects = [
    "step4_5_X_train_model_df",
    "step4_5_X_tuning_holdout_model_df",
    "step4_5_X_validation_model_df",
    "step4_5_y_train",
    "step4_5_y_tuning_holdout",
    "step4_5_y_validation",
    "step4_6_lgbm_baseline_metrics_df",
    "step4_6_baseline_comparison_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.7.1: {missing_objects}"

step4_7_input_summary_df = pd.DataFrame(
    [
        {
            "dataset_role": "train",
            "rows": int(step4_5_X_train_model_df.shape[0]),
            "cols": int(step4_5_X_train_model_df.shape[1]),
            "positive_count": int(pd.Series(step4_5_y_train).sum()),
            "negative_count": int(len(step4_5_y_train) - pd.Series(step4_5_y_train).sum()),
        },
        {
            "dataset_role": "tuning_holdout",
            "rows": int(step4_5_X_tuning_holdout_model_df.shape[0]),
            "cols": int(step4_5_X_tuning_holdout_model_df.shape[1]),
            "positive_count": int(pd.Series(step4_5_y_tuning_holdout).sum()),
            "negative_count": int(len(step4_5_y_tuning_holdout) - pd.Series(step4_5_y_tuning_holdout).sum()),
        },
        {
            "dataset_role": "validation",
            "rows": int(step4_5_X_validation_model_df.shape[0]),
            "cols": int(step4_5_X_validation_model_df.shape[1]),
            "positive_count": int(pd.Series(step4_5_y_validation).sum()),
            "negative_count": int(len(step4_5_y_validation) - pd.Series(step4_5_y_validation).sum()),
        },
    ]
)

step4_7_input_summary_df["positive_share"] = (
    step4_7_input_summary_df["positive_count"] / step4_7_input_summary_df["rows"]
)

step4_7_baseline_anchor_df = (
    step4_6_baseline_comparison_df.loc[
        step4_6_baseline_comparison_df["split_name"].isin(["train", "validation"])
    ]
    .sort_values(["split_name", "model_name"])
    .reset_index(drop=True)
)

display(step4_7_input_summary_df)
display(step4_7_baseline_anchor_df)

assert step4_7_input_summary_df["cols"].nunique() == 1, (
    "4.7.1 expected the same feature count across train, tuning_holdout and validation."
)
assert step4_7_input_summary_df.shape[0] == 3, (
    "4.7.1 expected exactly 3 input rows: train, tuning_holdout, validation."
)

,dataset_role,rows,cols,positive_count,negative_count,positive_share
0,train,63833,34,52978,10855,0.829947
1,tuning_holdout,7530,34,6093,1437,0.809163
2,validation,70879,34,58268,12611,0.822077


,model_name,split_name,row_count,positive_count,negative_count,accuracy,precision,recall,specificity,f1,balanced_accuracy,pred_positive_rate,roc_auc,pr_auc,tp,tn,fp,fn
0,dumb_baseline,train,63833,52978,10855,0.768772,0.867043,0.852052,0.362322,0.859482,0.607187,0.815597,NaN,NaN,45140,3933,6922,7838
1,hist_gradient_boosting,train,63833,52978,10855,0.841837,0.846180,0.989260,0.122340,0.912143,0.555800,0.970282,0.796646,0.947935,52409,1328,9527,569
2,lightgbm_baseline,train,63833,52978,10855,0.845221,0.848870,0.989713,0.140028,0.913897,0.564870,0.967650,0.815672,0.954060,52433,1520,9335,545
3,dumb_baseline,validation,70879,58268,12611,0.767604,0.860161,0.856559,0.356593,0.858357,0.606576,0.818635,NaN,NaN,49910,4497,8114,8358
4,hist_gradient_boosting,validation,70879,58268,12611,0.829611,0.836135,0.985961,0.107208,0.904888,0.546585,0.969384,0.765273,0.934171,57450,1352,11259,818
5,lightgbm_baseline,validation,70879,58268,12611,0.829287,0.836207,0.985344,0.108239,0.904670,0.546791,0.968693,0.766115,0.934430,57414,1365,11246,854


In [40]:
# 4.7.2 Mała siatka hiperparametrów LightGBM

import pandas as pd

required_objects = [
    "step4_7_input_summary_df",
    "step4_6_lgbm_baseline_metrics_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.7.2: {missing_objects}"

step4_7_lgbm_param_grid = [
    {
        "candidate_id": "lgbm_c01",
        "learning_rate": 0.05,
        "n_estimators": 200,
        "num_leaves": 15,
        "max_depth": -1,
        "min_child_samples": 20,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.0,
        "reg_lambda": 0.0,
    },
    {
        "candidate_id": "lgbm_c02",
        "learning_rate": 0.05,
        "n_estimators": 300,
        "num_leaves": 15,
        "max_depth": -1,
        "min_child_samples": 20,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.0,
        "reg_lambda": 0.0,
    },
    {
        "candidate_id": "lgbm_c03",
        "learning_rate": 0.05,
        "n_estimators": 200,
        "num_leaves": 31,
        "max_depth": -1,
        "min_child_samples": 20,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.0,
        "reg_lambda": 0.0,
    },
    {
        "candidate_id": "lgbm_c04",
        "learning_rate": 0.05,
        "n_estimators": 300,
        "num_leaves": 31,
        "max_depth": -1,
        "min_child_samples": 20,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.0,
        "reg_lambda": 0.0,
    },
    {
        "candidate_id": "lgbm_c05",
        "learning_rate": 0.03,
        "n_estimators": 300,
        "num_leaves": 15,
        "max_depth": -1,
        "min_child_samples": 60,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.0,
        "reg_lambda": 0.0,
    },
    {
        "candidate_id": "lgbm_c06",
        "learning_rate": 0.03,
        "n_estimators": 400,
        "num_leaves": 15,
        "max_depth": -1,
        "min_child_samples": 60,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.0,
        "reg_lambda": 0.0,
    },
    {
        "candidate_id": "lgbm_c07",
        "learning_rate": 0.03,
        "n_estimators": 300,
        "num_leaves": 31,
        "max_depth": -1,
        "min_child_samples": 60,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.0,
        "reg_lambda": 0.0,
    },
    {
        "candidate_id": "lgbm_c08",
        "learning_rate": 0.03,
        "n_estimators": 400,
        "num_leaves": 31,
        "max_depth": -1,
        "min_child_samples": 60,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.0,
        "reg_lambda": 0.0,
    },
]

step4_7_lgbm_param_grid_df = (
    pd.DataFrame(step4_7_lgbm_param_grid)
    .sort_values("candidate_id")
    .reset_index(drop=True)
)

display(step4_7_lgbm_param_grid_df)

assert step4_7_lgbm_param_grid_df.shape[0] == 8, (
    "4.7.2 expected exactly 8 LightGBM candidates in the small tuning grid."
)
assert step4_7_lgbm_param_grid_df["candidate_id"].nunique() == 8, (
    "4.7.2 expected unique candidate_id values in the tuning grid."
)

,candidate_id,learning_rate,n_estimators,num_leaves,max_depth,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda
0,lgbm_c01,0.05,200,15,-1,20,0.8,0.8,0.0,0.0
1,lgbm_c02,0.05,300,15,-1,20,0.8,0.8,0.0,0.0
2,lgbm_c03,0.05,200,31,-1,20,0.8,0.8,0.0,0.0
3,lgbm_c04,0.05,300,31,-1,20,0.8,0.8,0.0,0.0
4,lgbm_c05,0.03,300,15,-1,60,0.8,0.8,0.0,0.0
5,lgbm_c06,0.03,400,15,-1,60,0.8,0.8,0.0,0.0
6,lgbm_c07,0.03,300,31,-1,60,0.8,0.8,0.0,0.0
7,lgbm_c08,0.03,400,31,-1,60,0.8,0.8,0.0,0.0


In [41]:
# 4.7.3 Tuning kandydatów LightGBM na tuning_holdout

import numpy as np
import pandas as pd

from lightgbm import LGBMClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

required_objects = [
    "step4_7_lgbm_param_grid_df",
    "step4_5_X_train_model_df",
    "step4_5_X_tuning_holdout_model_df",
    "step4_5_y_train",
    "step4_5_y_tuning_holdout",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.7.3: {missing_objects}"

if "compute_binary_metrics" not in globals():
    def compute_binary_metrics(y_true, y_pred, y_score):
        y_true = pd.Series(y_true).astype(int)
        y_pred = pd.Series(y_pred).astype(int)
        y_score = pd.Series(y_score).astype(float)

        tp = int(((y_true == 1) & (y_pred == 1)).sum())
        tn = int(((y_true == 0) & (y_pred == 0)).sum())
        fp = int(((y_true == 0) & (y_pred == 1)).sum())
        fn = int(((y_true == 1) & (y_pred == 0)).sum())

        row_count = int(len(y_true))
        positive_count = int((y_true == 1).sum())
        negative_count = int((y_true == 0).sum())

        accuracy = (tp + tn) / row_count if row_count > 0 else np.nan
        precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
        recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = (
            2 * precision * recall / (precision + recall)
            if pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0
            else np.nan
        )
        balanced_accuracy = (
            (recall + specificity) / 2
            if pd.notna(recall) and pd.notna(specificity)
            else np.nan
        )
        pred_positive_rate = float(y_pred.mean()) if row_count > 0 else np.nan
        roc_auc = float(roc_auc_score(y_true, y_score)) if positive_count > 0 and negative_count > 0 else np.nan
        pr_auc = float(average_precision_score(y_true, y_score)) if positive_count > 0 else np.nan

        return {
            "row_count": row_count,
            "positive_count": positive_count,
            "negative_count": negative_count,
            "tp": tp,
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "accuracy": float(accuracy),
            "precision": float(precision),
            "recall": float(recall),
            "specificity": float(specificity),
            "f1": float(f1),
            "balanced_accuracy": float(balanced_accuracy),
            "pred_positive_rate": float(pred_positive_rate),
            "roc_auc": float(roc_auc),
            "pr_auc": float(pr_auc),
        }

step4_7_tuning_metric_rows = []

for _, candidate_row in step4_7_lgbm_param_grid_df.iterrows():
    step4_7_candidate_id = str(candidate_row["candidate_id"])

    step4_7_candidate_params = {
        "objective": "binary",
        "boosting_type": "gbdt",
        "learning_rate": float(candidate_row["learning_rate"]),
        "n_estimators": int(candidate_row["n_estimators"]),
        "num_leaves": int(candidate_row["num_leaves"]),
        "max_depth": int(candidate_row["max_depth"]),
        "min_child_samples": int(candidate_row["min_child_samples"]),
        "subsample": float(candidate_row["subsample"]),
        "colsample_bytree": float(candidate_row["colsample_bytree"]),
        "reg_alpha": float(candidate_row["reg_alpha"]),
        "reg_lambda": float(candidate_row["reg_lambda"]),
        "random_state": 42,
        "n_jobs": -1,
        "verbosity": -1,
    }

    step4_7_candidate_model = LGBMClassifier(**step4_7_candidate_params)
    step4_7_candidate_model.fit(
        step4_5_X_train_model_df,
        step4_5_y_train,
    )

    step4_7_y_score = step4_7_candidate_model.predict_proba(
        step4_5_X_tuning_holdout_model_df
    )[:, 1]
    step4_7_y_pred = (step4_7_y_score >= 0.5).astype("int8")

    step4_7_metric_row = compute_binary_metrics(
        step4_5_y_tuning_holdout,
        step4_7_y_pred,
        step4_7_y_score,
    )
    step4_7_metric_row["candidate_id"] = step4_7_candidate_id
    step4_7_metric_row["model_name"] = "lightgbm_tuned_candidate"
    step4_7_metric_row["split_name"] = "tuning_holdout"

    for col in [
        "learning_rate",
        "n_estimators",
        "num_leaves",
        "max_depth",
        "min_child_samples",
        "subsample",
        "colsample_bytree",
        "reg_alpha",
        "reg_lambda",
    ]:
        step4_7_metric_row[col] = candidate_row[col]

    step4_7_tuning_metric_rows.append(step4_7_metric_row)

step4_7_tuning_results_df = (
    pd.DataFrame(step4_7_tuning_metric_rows)
    .sort_values(
        ["pr_auc", "roc_auc", "balanced_accuracy", "specificity", "recall"],
        ascending=[False, False, False, False, False],
    )
    .reset_index(drop=True)
)

step4_7_best_candidate_id = str(step4_7_tuning_results_df.loc[0, "candidate_id"])

step4_7_best_candidate_params_dict = (
    step4_7_tuning_results_df.loc[
        step4_7_tuning_results_df["candidate_id"].eq(step4_7_best_candidate_id),
        [
            "candidate_id",
            "learning_rate",
            "n_estimators",
            "num_leaves",
            "max_depth",
            "min_child_samples",
            "subsample",
            "colsample_bytree",
            "reg_alpha",
            "reg_lambda",
        ],
    ]
    .iloc[0]
    .to_dict()
)

display(step4_7_tuning_results_df)

assert step4_7_tuning_results_df.shape[0] == step4_7_lgbm_param_grid_df.shape[0], (
    "4.7.3 expected one tuning result row per candidate."
)

,row_count,positive_count,negative_count,tp,tn,fp,fn,accuracy,precision,recall,...,split_name,learning_rate,n_estimators,num_leaves,max_depth,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda
0,7530,6093,1437,5985,127,1310,108,0.811687,0.820425,0.982275,...,tuning_holdout,0.03,300,15,-1,60,0.8,0.8,0.0,0.0
1,7530,6093,1437,5978,135,1302,115,0.811819,0.821154,0.981126,...,tuning_holdout,0.03,400,15,-1,60,0.8,0.8,0.0,0.0
2,7530,6093,1437,5990,129,1308,103,0.812616,0.820773,0.983095,...,tuning_holdout,0.03,300,31,-1,60,0.8,0.8,0.0,0.0
3,7530,6093,1437,5986,133,1304,107,0.812616,0.821125,0.982439,...,tuning_holdout,0.05,200,15,-1,20,0.8,0.8,0.0,0.0
4,7530,6093,1437,5983,136,1301,110,0.812616,0.821389,0.981946,...,tuning_holdout,0.03,400,31,-1,60,0.8,0.8,0.0,0.0
5,7530,6093,1437,5977,136,1301,116,0.811819,0.821242,0.980962,...,tuning_holdout,0.05,300,15,-1,20,0.8,0.8,0.0,0.0
6,7530,6093,1437,5986,125,1312,107,0.811554,0.820225,0.982439,...,tuning_holdout,0.05,200,31,-1,20,0.8,0.8,0.0,0.0
7,7530,6093,1437,5963,130,1307,130,0.809163,0.820220,0.978664,...,tuning_holdout,0.05,300,31,-1,20,0.8,0.8,0.0,0.0


In [42]:
# 4.7.4 Refit najlepszego kandydata LightGBM i ocena na train / tuning_holdout / validation

import numpy as np
import pandas as pd

from lightgbm import LGBMClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

required_objects = [
    "step4_7_tuning_results_df",
    "step4_7_best_candidate_id",
    "step4_5_X_train_model_df",
    "step4_5_X_tuning_holdout_model_df",
    "step4_5_X_validation_model_df",
    "step4_5_y_train",
    "step4_5_y_tuning_holdout",
    "step4_5_y_validation",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.7.4: {missing_objects}"

if "compute_binary_metrics" not in globals():
    def compute_binary_metrics(y_true, y_pred, y_score):
        y_true = pd.Series(y_true).astype(int)
        y_pred = pd.Series(y_pred).astype(int)
        y_score = pd.Series(y_score).astype(float)

        tp = int(((y_true == 1) & (y_pred == 1)).sum())
        tn = int(((y_true == 0) & (y_pred == 0)).sum())
        fp = int(((y_true == 0) & (y_pred == 1)).sum())
        fn = int(((y_true == 1) & (y_pred == 0)).sum())

        row_count = int(len(y_true))
        positive_count = int((y_true == 1).sum())
        negative_count = int((y_true == 0).sum())

        accuracy = (tp + tn) / row_count if row_count > 0 else np.nan
        precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
        recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = (
            2 * precision * recall / (precision + recall)
            if pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0
            else np.nan
        )
        balanced_accuracy = (
            (recall + specificity) / 2
            if pd.notna(recall) and pd.notna(specificity)
            else np.nan
        )
        pred_positive_rate = float(y_pred.mean()) if row_count > 0 else np.nan
        roc_auc = float(roc_auc_score(y_true, y_score)) if positive_count > 0 and negative_count > 0 else np.nan
        pr_auc = float(average_precision_score(y_true, y_score)) if positive_count > 0 else np.nan

        return {
            "row_count": row_count,
            "positive_count": positive_count,
            "negative_count": negative_count,
            "tp": tp,
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "accuracy": float(accuracy),
            "precision": float(precision),
            "recall": float(recall),
            "specificity": float(specificity),
            "f1": float(f1),
            "balanced_accuracy": float(balanced_accuracy),
            "pred_positive_rate": float(pred_positive_rate),
            "roc_auc": float(roc_auc),
            "pr_auc": float(pr_auc),
        }

step4_7_best_candidate_row = (
    step4_7_tuning_results_df.loc[
        step4_7_tuning_results_df["candidate_id"].eq(step4_7_best_candidate_id)
    ]
    .iloc[0]
)

step4_7_best_candidate_params = {
    "objective": "binary",
    "boosting_type": "gbdt",
    "learning_rate": float(step4_7_best_candidate_row["learning_rate"]),
    "n_estimators": int(step4_7_best_candidate_row["n_estimators"]),
    "num_leaves": int(step4_7_best_candidate_row["num_leaves"]),
    "max_depth": int(step4_7_best_candidate_row["max_depth"]),
    "min_child_samples": int(step4_7_best_candidate_row["min_child_samples"]),
    "subsample": float(step4_7_best_candidate_row["subsample"]),
    "colsample_bytree": float(step4_7_best_candidate_row["colsample_bytree"]),
    "reg_alpha": float(step4_7_best_candidate_row["reg_alpha"]),
    "reg_lambda": float(step4_7_best_candidate_row["reg_lambda"]),
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1,
}

step4_7_best_lgbm_model = LGBMClassifier(**step4_7_best_candidate_params)
step4_7_best_lgbm_model.fit(
    step4_5_X_train_model_df,
    step4_5_y_train,
)

step4_7_best_eval_plan = [
    ("train", step4_5_X_train_model_df, step4_5_y_train),
    ("tuning_holdout", step4_5_X_tuning_holdout_model_df, step4_5_y_tuning_holdout),
    ("validation", step4_5_X_validation_model_df, step4_5_y_validation),
]

step4_7_best_metric_rows = []

for split_name, X_split, y_split in step4_7_best_eval_plan:
    y_score = step4_7_best_lgbm_model.predict_proba(X_split)[:, 1]
    y_pred = (y_score >= 0.5).astype("int8")

    metric_row = compute_binary_metrics(y_split, y_pred, y_score)
    metric_row["candidate_id"] = step4_7_best_candidate_id
    metric_row["model_name"] = "lightgbm_tuned"
    metric_row["split_name"] = split_name

    for col in [
        "learning_rate",
        "n_estimators",
        "num_leaves",
        "max_depth",
        "min_child_samples",
        "subsample",
        "colsample_bytree",
        "reg_alpha",
        "reg_lambda",
    ]:
        metric_row[col] = step4_7_best_candidate_row[col]

    step4_7_best_metric_rows.append(metric_row)

step4_7_tuned_lgbm_metrics_df = (
    pd.DataFrame(step4_7_best_metric_rows)
    .sort_values("split_name")
    .reset_index(drop=True)
)

display(step4_7_tuned_lgbm_metrics_df)

assert step4_7_tuned_lgbm_metrics_df.shape[0] == 3, (
    "4.7.4 expected exactly 3 split rows for the tuned LightGBM model."
)

,row_count,positive_count,negative_count,tp,tn,fp,fn,accuracy,precision,recall,...,split_name,learning_rate,n_estimators,num_leaves,max_depth,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda
0,63833,52978,10855,52427,1207,9648,551,0.840224,0.844575,0.989599,...,train,0.03,300,15,-1,60,0.8,0.8,0.0,0.0
1,7530,6093,1437,5985,127,1310,108,0.811687,0.820425,0.982275,...,tuning_holdout,0.03,300,15,-1,60,0.8,0.8,0.0,0.0
2,70879,58268,12611,57494,1328,11283,774,0.829893,0.835948,0.986717,...,validation,0.03,300,15,-1,60,0.8,0.8,0.0,0.0


In [43]:
# 4.7.5 Porównanie tuned LightGBM z dumb baseline, LightGBM baseline i najlepszym lekkim ML

import pandas as pd

required_objects = [
    "step4_6_baseline_comparison_df",
    "step4_7_tuned_lgbm_metrics_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.7.5: {missing_objects}"

step4_7_tuned_compare_df = step4_7_tuned_lgbm_metrics_df.copy()

step4_7_validation_comparison_df = pd.concat(
    [
        step4_6_baseline_comparison_df.copy(),
        step4_7_tuned_compare_df.copy(),
    ],
    axis=0,
    ignore_index=True,
)

step4_7_validation_comparison_df = (
    step4_7_validation_comparison_df.loc[
        step4_7_validation_comparison_df["split_name"].isin(["train", "validation"])
    ]
    .sort_values(["split_name", "model_name"])
    .reset_index(drop=True)
)

step4_7_validation_ranking_df = (
    step4_7_validation_comparison_df.loc[
        step4_7_validation_comparison_df["split_name"].eq("validation")
    ]
    .sort_values(
        ["pr_auc", "roc_auc", "balanced_accuracy", "specificity", "recall"],
        ascending=[False, False, False, False, False],
    )
    .reset_index(drop=True)
)

display(step4_7_validation_ranking_df)
display(step4_7_validation_comparison_df)

assert set(step4_7_validation_comparison_df["model_name"].unique()) == {
    "dumb_baseline",
    "hist_gradient_boosting",
    "lightgbm_baseline",
    "lightgbm_tuned",
}, "4.7.5 comparison should contain exactly 4 models."

assert set(step4_7_validation_comparison_df["split_name"].unique()) == {
    "train",
    "validation",
}, "4.7.5 comparison should contain only train and validation splits."

,model_name,split_name,row_count,positive_count,negative_count,accuracy,precision,recall,specificity,f1,...,candidate_id,learning_rate,n_estimators,num_leaves,max_depth,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda
0,lightgbm_tuned,validation,70879,58268,12611,0.829893,0.835948,0.986717,0.105305,0.905097,...,lgbm_c05,0.03,300.0,15.0,-1.0,60.0,0.8,0.8,0.0,0.0
1,lightgbm_baseline,validation,70879,58268,12611,0.829287,0.836207,0.985344,0.108239,0.904670,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,hist_gradient_boosting,validation,70879,58268,12611,0.829611,0.836135,0.985961,0.107208,0.904888,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,dumb_baseline,validation,70879,58268,12611,0.767604,0.860161,0.856559,0.356593,0.858357,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,model_name,split_name,row_count,positive_count,negative_count,accuracy,precision,recall,specificity,f1,...,candidate_id,learning_rate,n_estimators,num_leaves,max_depth,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda
0,dumb_baseline,train,63833,52978,10855,0.768772,0.867043,0.852052,0.362322,0.859482,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,hist_gradient_boosting,train,63833,52978,10855,0.841837,0.846180,0.989260,0.122340,0.912143,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,lightgbm_baseline,train,63833,52978,10855,0.845221,0.848870,0.989713,0.140028,0.913897,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,lightgbm_tuned,train,63833,52978,10855,0.840224,0.844575,0.989599,0.111193,0.911354,...,lgbm_c05,0.03,300.0,15.0,-1.0,60.0,0.8,0.8,0.0,0.0
4,dumb_baseline,validation,70879,58268,12611,0.767604,0.860161,0.856559,0.356593,0.858357,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,hist_gradient_boosting,validation,70879,58268,12611,0.829611,0.836135,0.985961,0.107208,0.904888,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,lightgbm_baseline,validation,70879,58268,12611,0.829287,0.836207,0.985344,0.108239,0.904670,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,lightgbm_tuned,validation,70879,58268,12611,0.829893,0.835948,0.986717,0.105305,0.905097,...,lgbm_c05,0.03,300.0,15.0,-1.0,60.0,0.8,0.8,0.0,0.0


In [44]:
# 4.7.6 Zapis artefaktów sekcji 4.7

from pathlib import Path
import json

import pandas as pd

required_objects = [
    "step4_3_project_root",
    "step4_7_tuning_results_df",
    "step4_7_validation_comparison_df",
    "step4_7_best_candidate_id",
    "step4_7_best_candidate_params",
    "step4_7_tuned_lgbm_metrics_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.7.6: {missing_objects}"

step4_7_output_dir = step4_3_project_root / "artifacts" / "4_lgbm_tuning"
step4_7_output_dir.mkdir(parents=True, exist_ok=True)

step4_7_tuning_results_path = step4_7_output_dir / "b4_07_tuning_results.parquet"
step4_7_validation_comparison_path = step4_7_output_dir / "b4_07_validation_comparison.parquet"
step4_7_model_checkpoint_path = step4_7_output_dir / "b4_07_model_checkpoint.json"

step4_7_tuning_results_save_df = (
    step4_7_tuning_results_df.copy()
    .sort_values(
        ["pr_auc", "roc_auc", "balanced_accuracy", "specificity", "recall"],
        ascending=[False, False, False, False, False],
    )
    .reset_index(drop=True)
)

step4_7_validation_comparison_save_df = (
    step4_7_validation_comparison_df.copy()
    .sort_values(["split_name", "model_name"])
    .reset_index(drop=True)
)

step4_7_tuning_results_save_df.to_parquet(
    step4_7_tuning_results_path,
    index=False,
)

step4_7_validation_comparison_save_df.to_parquet(
    step4_7_validation_comparison_path,
    index=False,
)

step4_7_best_validation_row = (
    step4_7_tuned_lgbm_metrics_df.loc[
        step4_7_tuned_lgbm_metrics_df["split_name"].eq("validation")
    ]
    .iloc[0]
)

step4_7_model_checkpoint_dict = {
    "selected_model_name": "lightgbm_tuned",
    "selected_candidate_id": str(step4_7_best_candidate_id),
    "selected_params": {
        key: (
            int(value) if key in {"n_estimators", "num_leaves", "max_depth", "min_child_samples"}
            else float(value) if key not in {"objective", "boosting_type"} else value
        )
        for key, value in step4_7_best_candidate_params.items()
    },
    "validation_metrics": {
        "accuracy": float(step4_7_best_validation_row["accuracy"]),
        "precision": float(step4_7_best_validation_row["precision"]),
        "recall": float(step4_7_best_validation_row["recall"]),
        "specificity": float(step4_7_best_validation_row["specificity"]),
        "f1": float(step4_7_best_validation_row["f1"]),
        "balanced_accuracy": float(step4_7_best_validation_row["balanced_accuracy"]),
        "pred_positive_rate": float(step4_7_best_validation_row["pred_positive_rate"]),
        "roc_auc": float(step4_7_best_validation_row["roc_auc"]),
        "pr_auc": float(step4_7_best_validation_row["pr_auc"]),
    },
    "artifacts": {
        "tuning_results": str(step4_7_tuning_results_path),
        "validation_comparison": str(step4_7_validation_comparison_path),
    },
}

with open(step4_7_model_checkpoint_path, "w", encoding="utf-8") as f:
    json.dump(step4_7_model_checkpoint_dict, f, ensure_ascii=False, indent=2)

step4_7_saved_artifacts_registry_df = pd.DataFrame(
    [
        {
            "artifact_name": step4_7_tuning_results_path.name,
            "rows": int(step4_7_tuning_results_save_df.shape[0]),
            "cols": int(step4_7_tuning_results_save_df.shape[1]),
            "path": str(step4_7_tuning_results_path),
        },
        {
            "artifact_name": step4_7_validation_comparison_path.name,
            "rows": int(step4_7_validation_comparison_save_df.shape[0]),
            "cols": int(step4_7_validation_comparison_save_df.shape[1]),
            "path": str(step4_7_validation_comparison_path),
        },
        {
            "artifact_name": step4_7_model_checkpoint_path.name,
            "rows": 1,
            "cols": len(step4_7_model_checkpoint_dict),
            "path": str(step4_7_model_checkpoint_path),
        },
    ]
)

display(step4_7_saved_artifacts_registry_df)

assert step4_7_tuning_results_path.exists(), (
    "4.7.6 failed to save b4_07_tuning_results.parquet"
)
assert step4_7_validation_comparison_path.exists(), (
    "4.7.6 failed to save b4_07_validation_comparison.parquet"
)
assert step4_7_model_checkpoint_path.exists(), (
    "4.7.6 failed to save b4_07_model_checkpoint.json"
)
assert step4_7_tuning_results_save_df.shape[0] == 8, (
    "4.7.6 expected 8 rows in saved tuning results artifact."
)
assert step4_7_validation_comparison_save_df.shape[0] == 8, (
    "4.7.6 expected 8 rows in saved validation comparison artifact."
)

,artifact_name,rows,cols,path
0,b4_07_tuning_results.parquet,8,28,/root/projects/6_samodzielny_projekt_koncowy_w...
1,b4_07_validation_comparison.parquet,8,28,/root/projects/6_samodzielny_projekt_koncowy_w...
2,b4_07_model_checkpoint.json,1,5,/root/projects/6_samodzielny_projekt_koncowy_w...


## Podsumowanie działu 4.7 Lekki tuning hiperparametrów

**Kontekst inżynieryjny:** W dziale `4.7` przygotowano wejście tuningowe dla modelu `LightGBM`, potwierdzono spójność macierzy `train`, `tuning_holdout` i `validation` oraz zbudowano małą, kontrolowaną siatkę `8` kandydatów hiperparametrów. Następnie przeprowadzono tuning na `tuning_holdout`, wybrano najlepszego kandydata, wykonano refit modelu na `train`, oceniono go na `train`, `tuning_holdout` i `validation`, a na końcu zestawiono `lightgbm_tuned` z `dumb baseline`, `lightgbm_baseline` i najlepszym lekkim baseline’em ML. Dział został domknięty zapisem artefaktów: `b4_07_tuning_results.parquet`, `b4_07_validation_comparison.parquet` oraz `b4_07_model_checkpoint.json`.

**Interpretacja wyniku:** Wejście tuningowe było spójne i gotowe do dalszej optymalizacji: wszystkie trzy zbiory miały po `34` cechy modelowe, a udział klasy pozytywnej pozostał stabilny między splitami (`train = 0.829947`, `tuning_holdout = 0.809163`, `validation = 0.822077`). Mała siatka tuningowa potwierdziła, że najbardziej obiecujące są warianty bardziej konserwatywne, z niższym `learning_rate` oraz wyższym `min_child_samples`. Najlepszym kandydatem został `lgbm_c05` z parametrami: `learning_rate = 0.03`, `n_estimators = 300`, `num_leaves = 15`, `min_child_samples = 60`, `subsample = 0.8`, `colsample_bytree = 0.8`. Po reficie model `lightgbm_tuned` na `validation` osiągnął `accuracy = 0.829893`, `precision = 0.835948`, `recall = 0.986717`, `specificity = 0.105305` oraz `f1 = 0.905097`. W porównaniu z `lightgbm_baseline` tuning dał niewielką poprawę profilu jakościowego, głównie po stronie `accuracy`, `recall` i końcowego `f1`, przy zachowaniu bardzo mocnego wychwytywania klasy pozytywnej.

**Znaczenie biznesowe:** Wynik działu `4.7` potwierdza, że `LightGBM` pozostaje głównym kandydatem modelowym projektu i że lekki tuning hiperparametrów potrafi uporządkować wybór bardziej stabilnej konfiguracji bez zmiany kontraktu danych i bez przebudowy pipeline’u. Jednocześnie porównanie z wcześniejszymi benchmarkami pokazuje, że największą siłą tego modelu pozostaje bardzo dobry profil rankingowy i wysoki `recall`, natomiast profil decyzyjny przy progu `0.5` nadal pozostaje szeroki. Oznacza to, że projekt jest gotowy do kolejnego etapu optymalizacji, w którym kluczowe znaczenie będą miały kalibracja prawdopodobieństw oraz dobór progu biznesowego, a nie dalsze przypadkowe rozszerzanie siatki tuningowej.

**Wniosek:** Dział `4.7` został wykonany poprawnie i zgodnie z założeniami. Przeprowadzono mały, kontrolowany tuning `LightGBM`, wybrano najlepszego kandydata `lgbm_c05`, wykonano refit, porównano model z głównymi benchmarkami i zapisano komplet artefaktów wyjściowych sekcji. Tuning przyniósł uporządkowany, inkrementalny zysk jakościowy i potwierdził, że właściwy dalszy kierunek projektu to `4.8`, czyli kalibracja prawdopodobieństw, a następnie `4.9`, czyli threshold tuning oparty o logikę biznesową.

## 4.8 Kalibracja prawdopodobieństw

In [45]:
# 4.8.1 Weryfikacja finalistów do kalibracji

import pandas as pd

required_objects = [
    "step4_7_validation_comparison_df",
    "step4_5_hist_gb_model",
    "step4_6_lgbm_baseline_model",
    "step4_7_best_lgbm_model",
    "step4_5_X_tuning_holdout_model_df",
    "step4_5_X_validation_model_df",
    "step4_5_y_tuning_holdout",
    "step4_5_y_validation",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.8.1: {missing_objects}"

step4_8_candidate_models_dict = {
    "hist_gradient_boosting": step4_5_hist_gb_model,
    "lightgbm_baseline": step4_6_lgbm_baseline_model,
    "lightgbm_tuned": step4_7_best_lgbm_model,
}

step4_8_finalists_input_df = (
    step4_7_validation_comparison_df.loc[
        step4_7_validation_comparison_df["split_name"].eq("validation")
    ]
    .copy()
    .reset_index(drop=True)
)

step4_8_finalists_input_df["is_probability_model"] = (
    step4_8_finalists_input_df["model_name"].isin(step4_8_candidate_models_dict.keys())
).astype(int)

step4_8_finalists_for_calibration_df = (
    step4_8_finalists_input_df.loc[
        step4_8_finalists_input_df["is_probability_model"].eq(1)
    ]
    .sort_values(
        ["pr_auc", "roc_auc", "balanced_accuracy", "specificity", "recall"],
        ascending=[False, False, False, False, False],
    )
    .reset_index(drop=True)
)

step4_8_calibration_input_summary_df = pd.DataFrame(
    [
        {
            "dataset_role": "tuning_holdout",
            "rows": int(step4_5_X_tuning_holdout_model_df.shape[0]),
            "cols": int(step4_5_X_tuning_holdout_model_df.shape[1]),
            "positive_count": int(pd.Series(step4_5_y_tuning_holdout).sum()),
            "negative_count": int(len(step4_5_y_tuning_holdout) - pd.Series(step4_5_y_tuning_holdout).sum()),
        },
        {
            "dataset_role": "validation",
            "rows": int(step4_5_X_validation_model_df.shape[0]),
            "cols": int(step4_5_X_validation_model_df.shape[1]),
            "positive_count": int(pd.Series(step4_5_y_validation).sum()),
            "negative_count": int(len(step4_5_y_validation) - pd.Series(step4_5_y_validation).sum()),
        },
    ]
)

step4_8_calibration_input_summary_df["positive_share"] = (
    step4_8_calibration_input_summary_df["positive_count"]
    / step4_8_calibration_input_summary_df["rows"]
)

display(step4_8_finalists_for_calibration_df)
display(step4_8_calibration_input_summary_df)

assert step4_8_finalists_for_calibration_df.shape[0] == 3, (
    "4.8.1 expected exactly 3 probability-based finalists for calibration."
)
assert step4_8_calibration_input_summary_df["cols"].nunique() == 1, (
    "4.8.1 expected the same feature count in tuning_holdout and validation."
)

,model_name,split_name,row_count,positive_count,negative_count,accuracy,precision,recall,specificity,f1,...,learning_rate,n_estimators,num_leaves,max_depth,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda,is_probability_model
0,lightgbm_tuned,validation,70879,58268,12611,0.829893,0.835948,0.986717,0.105305,0.905097,...,0.03,300.0,15.0,-1.0,60.0,0.8,0.8,0.0,0.0,1
1,lightgbm_baseline,validation,70879,58268,12611,0.829287,0.836207,0.985344,0.108239,0.904670,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,hist_gradient_boosting,validation,70879,58268,12611,0.829611,0.836135,0.985961,0.107208,0.904888,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1


,dataset_role,rows,cols,positive_count,negative_count,positive_share
0,tuning_holdout,7530,34,6093,1437,0.809163
1,validation,70879,34,58268,12611,0.822077


In [46]:
# 4.8.2 Kalibracja raw / sigmoid / isotonic i ranking finalistów na validation

import numpy as np
import pandas as pd

from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

required_objects = [
    "step4_8_candidate_models_dict",
    "step4_5_X_tuning_holdout_model_df",
    "step4_5_X_validation_model_df",
    "step4_5_y_tuning_holdout",
    "step4_5_y_validation",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.8.2: {missing_objects}"

step4_8_y_tuning_series = pd.Series(step4_5_y_tuning_holdout).astype(int).reset_index(drop=True)
step4_8_y_validation_series = pd.Series(step4_5_y_validation).astype(int).reset_index(drop=True)

step4_8_validation_positive_share = float(step4_8_y_validation_series.mean())

step4_8_calibration_report_rows = []
step4_8_calibration_profile_rows = []

step4_8_bin_edges = np.linspace(0.0, 1.0, 11)

for step4_8_model_name, step4_8_model in step4_8_candidate_models_dict.items():
    step4_8_raw_tuning_proba = pd.Series(
        step4_8_model.predict_proba(step4_5_X_tuning_holdout_model_df)[:, 1],
        name="raw_tuning_proba",
    ).clip(0.0, 1.0)

    step4_8_raw_validation_proba = pd.Series(
        step4_8_model.predict_proba(step4_5_X_validation_model_df)[:, 1],
        name="raw_validation_proba",
    ).clip(0.0, 1.0)

    step4_8_sigmoid_calibrator = LogisticRegression(
        solver="lbfgs",
        max_iter=1000,
        random_state=42,
    )
    step4_8_sigmoid_calibrator.fit(
        step4_8_raw_tuning_proba.to_numpy().reshape(-1, 1),
        step4_8_y_tuning_series,
    )

    step4_8_isotonic_calibrator = IsotonicRegression(
        out_of_bounds="clip",
        y_min=0.0,
        y_max=1.0,
    )
    step4_8_isotonic_calibrator.fit(
        step4_8_raw_tuning_proba.to_numpy(),
        step4_8_y_tuning_series.to_numpy(),
    )

    step4_8_probability_variants = {
        "raw": step4_8_raw_validation_proba.to_numpy(),
        "sigmoid": step4_8_sigmoid_calibrator.predict_proba(
            step4_8_raw_validation_proba.to_numpy().reshape(-1, 1)
        )[:, 1],
        "isotonic": step4_8_isotonic_calibrator.predict(
            step4_8_raw_validation_proba.to_numpy()
        ),
    }

    for step4_8_calibration_method, step4_8_validation_proba in step4_8_probability_variants.items():
        step4_8_validation_proba = np.clip(step4_8_validation_proba, 0.0, 1.0)

        step4_8_mean_pred_proba = float(np.mean(step4_8_validation_proba))
        step4_8_brier = float(
            brier_score_loss(step4_8_y_validation_series, step4_8_validation_proba)
        )
        step4_8_roc_auc = float(
            roc_auc_score(step4_8_y_validation_series, step4_8_validation_proba)
        )
        step4_8_pr_auc = float(
            average_precision_score(step4_8_y_validation_series, step4_8_validation_proba)
        )
        step4_8_absolute_mean_gap = float(
            abs(step4_8_mean_pred_proba - step4_8_validation_positive_share)
        )

        step4_8_calibration_report_rows.append(
            {
                "model_name": step4_8_model_name,
                "calibration_method": step4_8_calibration_method,
                "split_name": "validation",
                "row_count": int(len(step4_8_validation_proba)),
                "positive_share": step4_8_validation_positive_share,
                "mean_pred_proba": step4_8_mean_pred_proba,
                "absolute_mean_gap": step4_8_absolute_mean_gap,
                "brier_score": step4_8_brier,
                "roc_auc": step4_8_roc_auc,
                "pr_auc": step4_8_pr_auc,
            }
        )

        step4_8_profile_work_df = pd.DataFrame(
            {
                "y_true": step4_8_y_validation_series.to_numpy(),
                "y_proba": step4_8_validation_proba,
            }
        )

        step4_8_profile_work_df["probability_bin"] = pd.cut(
            step4_8_profile_work_df["y_proba"],
            bins=step4_8_bin_edges,
            include_lowest=True,
            right=True,
            duplicates="drop",
        )

        step4_8_profile_grouped_df = (
            step4_8_profile_work_df.groupby("probability_bin", observed=False)
            .agg(
                row_count=("y_true", "size"),
                mean_pred_proba=("y_proba", "mean"),
                observed_positive_rate=("y_true", "mean"),
            )
            .reset_index()
        )

        step4_8_profile_grouped_df = step4_8_profile_grouped_df.loc[
            step4_8_profile_grouped_df["row_count"] > 0
        ].reset_index(drop=True)

        step4_8_profile_grouped_df["absolute_bin_gap"] = (
            step4_8_profile_grouped_df["mean_pred_proba"]
            - step4_8_profile_grouped_df["observed_positive_rate"]
        ).abs()

        for _, step4_8_profile_row in step4_8_profile_grouped_df.iterrows():
            step4_8_calibration_profile_rows.append(
                {
                    "model_name": step4_8_model_name,
                    "calibration_method": step4_8_calibration_method,
                    "split_name": "validation",
                    "probability_bin": str(step4_8_profile_row["probability_bin"]),
                    "row_count": int(step4_8_profile_row["row_count"]),
                    "mean_pred_proba": float(step4_8_profile_row["mean_pred_proba"]),
                    "observed_positive_rate": float(step4_8_profile_row["observed_positive_rate"]),
                    "absolute_bin_gap": float(step4_8_profile_row["absolute_bin_gap"]),
                }
            )

step4_8_calibration_report_df = (
    pd.DataFrame(step4_8_calibration_report_rows)
    .sort_values(
        ["brier_score", "absolute_mean_gap", "pr_auc", "roc_auc"],
        ascending=[True, True, False, False],
    )
    .reset_index(drop=True)
)

step4_8_calibration_profile_df = (
    pd.DataFrame(step4_8_calibration_profile_rows)
    .sort_values(["model_name", "calibration_method", "probability_bin"])
    .reset_index(drop=True)
)

display(step4_8_calibration_report_df)
display(step4_8_calibration_profile_df)

assert step4_8_calibration_report_df.shape[0] == 9, (
    "4.8.2 expected exactly 9 calibration rows: 3 models x 3 calibration methods."
)
assert step4_8_calibration_profile_df.shape[0] > 0, (
    "4.8.2 expected a non-empty calibration profile dataframe."
)

,model_name,calibration_method,split_name,row_count,positive_share,mean_pred_proba,absolute_mean_gap,brier_score,roc_auc,pr_auc
0,lightgbm_tuned,raw,validation,70879,0.822077,0.826014,0.003937,0.124602,0.767928,0.935374
1,lightgbm_tuned,isotonic,validation,70879,0.822077,0.817163,0.004914,0.124893,0.766521,0.930218
2,lightgbm_baseline,raw,validation,70879,0.822077,0.826260,0.004183,0.124899,0.766115,0.934430
3,hist_gradient_boosting,raw,validation,70879,0.822077,0.825680,0.003603,0.124962,0.765273,0.934171
4,lightgbm_baseline,isotonic,validation,70879,0.822077,0.817949,0.004128,0.125145,0.765076,0.929542
5,hist_gradient_boosting,isotonic,validation,70879,0.822077,0.817657,0.004420,0.125328,0.762741,0.927718
6,lightgbm_tuned,sigmoid,validation,70879,0.822077,0.817029,0.005049,0.125661,0.767928,0.935374
7,lightgbm_baseline,sigmoid,validation,70879,0.822077,0.817491,0.004586,0.125936,0.766115,0.934430
8,hist_gradient_boosting,sigmoid,validation,70879,0.822077,0.817539,0.004538,0.126014,0.765273,0.934171


,model_name,calibration_method,split_name,probability_bin,row_count,mean_pred_proba,observed_positive_rate,absolute_bin_gap
0,hist_gradient_boosting,isotonic,validation,"(-0.001, 0.1]",276,0.000412,0.021739,0.021327
1,hist_gradient_boosting,isotonic,validation,"(0.1, 0.2]",68,0.178214,0.235294,0.057080
2,hist_gradient_boosting,isotonic,validation,"(0.2, 0.3]",190,0.300000,0.263158,0.036842
3,hist_gradient_boosting,isotonic,validation,"(0.3, 0.4]",332,0.317647,0.331325,0.013678
4,hist_gradient_boosting,isotonic,validation,"(0.4, 0.5]",208,0.499094,0.442308,0.056787
...,...,...,...,...,...,...,...,...
85,lightgbm_tuned,sigmoid,validation,"(0.5, 0.6]",2599,0.555080,0.555598,0.000518
86,lightgbm_tuned,sigmoid,validation,"(0.6, 0.7]",5069,0.655060,0.619254,0.035805
87,lightgbm_tuned,sigmoid,validation,"(0.7, 0.8]",11045,0.755708,0.726573,0.029135
88,lightgbm_tuned,sigmoid,validation,"(0.8, 0.9]",27180,0.859790,0.851141,0.008649


In [47]:
# 4.8.3 Wybór zwycięzcy kalibracji

import pandas as pd

required_objects = [
    "step4_8_calibration_report_df",
    "step4_8_calibration_profile_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.8.3: {missing_objects}"

step4_8_best_calibration_row = step4_8_calibration_report_df.iloc[0].copy()

step4_8_calibration_choice_dict = {
    "selected_model_name": str(step4_8_best_calibration_row["model_name"]),
    "selected_calibration_method": str(step4_8_best_calibration_row["calibration_method"]),
    "selection_split": str(step4_8_best_calibration_row["split_name"]),
    "selection_rule": [
        "lowest_brier_score",
        "lowest_absolute_mean_gap",
        "highest_pr_auc",
        "highest_roc_auc",
    ],
    "validation_metrics": {
        "row_count": int(step4_8_best_calibration_row["row_count"]),
        "positive_share": float(step4_8_best_calibration_row["positive_share"]),
        "mean_pred_proba": float(step4_8_best_calibration_row["mean_pred_proba"]),
        "absolute_mean_gap": float(step4_8_best_calibration_row["absolute_mean_gap"]),
        "brier_score": float(step4_8_best_calibration_row["brier_score"]),
        "roc_auc": float(step4_8_best_calibration_row["roc_auc"]),
        "pr_auc": float(step4_8_best_calibration_row["pr_auc"]),
    },
}

step4_8_calibration_choice_df = pd.DataFrame(
    [
        {
            "selected_model_name": step4_8_calibration_choice_dict["selected_model_name"],
            "selected_calibration_method": step4_8_calibration_choice_dict["selected_calibration_method"],
            "selection_split": step4_8_calibration_choice_dict["selection_split"],
            "brier_score": step4_8_calibration_choice_dict["validation_metrics"]["brier_score"],
            "absolute_mean_gap": step4_8_calibration_choice_dict["validation_metrics"]["absolute_mean_gap"],
            "roc_auc": step4_8_calibration_choice_dict["validation_metrics"]["roc_auc"],
            "pr_auc": step4_8_calibration_choice_dict["validation_metrics"]["pr_auc"],
        }
    ]
)

display(step4_8_calibration_choice_df)

assert step4_8_calibration_choice_df.shape[0] == 1, (
    "4.8.3 expected exactly one selected calibration winner."
)
assert step4_8_calibration_choice_dict["selected_model_name"] in {
    "hist_gradient_boosting",
    "lightgbm_baseline",
    "lightgbm_tuned",
}, "4.8.3 selected model must be one of the calibration finalists."

assert step4_8_calibration_choice_dict["selected_calibration_method"] in {
    "raw",
    "sigmoid",
    "isotonic",
}, "4.8.3 selected calibration method must be raw, sigmoid or isotonic."

,selected_model_name,selected_calibration_method,selection_split,brier_score,absolute_mean_gap,roc_auc,pr_auc
0,lightgbm_tuned,raw,validation,0.124602,0.003937,0.767928,0.935374


In [48]:
# 4.8.4 Zapis artefaktów sekcji 4.8

from pathlib import Path
import json

import pandas as pd

required_objects = [
    "step4_3_project_root",
    "step4_8_calibration_report_df",
    "step4_8_calibration_choice_dict",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.8.4: {missing_objects}"

step4_8_output_dir = step4_3_project_root / "artifacts" / "4_calibration"
step4_8_output_dir.mkdir(parents=True, exist_ok=True)

step4_8_calibration_report_path = step4_8_output_dir / "b4_08_calibration_report.parquet"
step4_8_calibration_choice_path = step4_8_output_dir / "b4_08_calibration_choice.json"

step4_8_calibration_report_save_df = (
    step4_8_calibration_report_df.copy()
    .sort_values(
        ["brier_score", "absolute_mean_gap", "pr_auc", "roc_auc"],
        ascending=[True, True, False, False],
    )
    .reset_index(drop=True)
)

step4_8_calibration_report_save_df.to_parquet(
    step4_8_calibration_report_path,
    index=False,
)

with open(step4_8_calibration_choice_path, "w", encoding="utf-8") as f:
    json.dump(step4_8_calibration_choice_dict, f, ensure_ascii=False, indent=2)

step4_8_saved_artifacts_registry_df = pd.DataFrame(
    [
        {
            "artifact_name": step4_8_calibration_report_path.name,
            "rows": int(step4_8_calibration_report_save_df.shape[0]),
            "cols": int(step4_8_calibration_report_save_df.shape[1]),
            "path": str(step4_8_calibration_report_path),
        },
        {
            "artifact_name": step4_8_calibration_choice_path.name,
            "rows": 1,
            "cols": len(step4_8_calibration_choice_dict),
            "path": str(step4_8_calibration_choice_path),
        },
    ]
)

display(step4_8_saved_artifacts_registry_df)

assert step4_8_calibration_report_path.exists(), (
    "4.8.4 failed to save b4_08_calibration_report.parquet"
)
assert step4_8_calibration_choice_path.exists(), (
    "4.8.4 failed to save b4_08_calibration_choice.json"
)
assert step4_8_calibration_report_save_df.shape[0] == 9, (
    "4.8.4 expected 9 rows in saved calibration report."
)

,artifact_name,rows,cols,path
0,b4_08_calibration_report.parquet,9,10,/root/projects/6_samodzielny_projekt_koncowy_w...
1,b4_08_calibration_choice.json,1,5,/root/projects/6_samodzielny_projekt_koncowy_w...


## Podsumowanie działu 4.8 Kalibracja prawdopodobieństw

**Kontekst inżynieryjny:** W dziale `4.8` wybrano finalistów probabilistycznych do kalibracji: `hist_gradient_boosting`, `lightgbm_baseline` oraz `lightgbm_tuned`. Następnie przygotowano wejście kalibracyjne na `tuning_holdout` i ocenę na `validation`, przy zachowaniu tej samej liczby cech modelowych (`34`) w obu zbiorach. Dla każdego finalisty porównano trzy warianty probabilistyczne: `raw`, `sigmoid` oraz `isotonic`. Ocenę przeprowadzono przez raport kalibracyjny oraz profil binowy prawdopodobieństw, a na końcu zapisano artefakty `b4_08_calibration_report.parquet` oraz `b4_08_calibration_choice.json`.

**Interpretacja wyniku:** Najlepszy wynik kalibracyjny uzyskał wariant `lightgbm_tuned + raw`. Na `validation` osiągnął `Brier Score = 0.124602`, `absolute_mean_gap = 0.003937`, `ROC AUC = 0.767928` oraz `PR AUC = 0.935374`. Oznacza to, że spośród wszystkich sprawdzonych konfiguracji właśnie surowe prawdopodobieństwa modelu `lightgbm_tuned` dawały najlepszy kompromis między jakością rankingu a jakością probabilistyczną. Zarówno `sigmoid`, jak i `isotonic` nie poprawiły wyniku końcowego — w badanym układzie prowadziły do słabszego `Brier Score` albo pogorszenia jakości rankingowej. Profil binowy potwierdził, że zwycięski wariant zachowuje sensowną monotonię i nie wymaga wymuszonej korekty probabilistycznej.

**Znaczenie biznesowe:** Wynik działu `4.8` upraszcza dalszy pipeline. Projekt nie musi dokładać dodatkowej warstwy kalibracyjnej tylko dlatego, że jest dostępna technicznie. Zostało sprawdzone uczciwie, że `raw` działa najlepiej, więc dalsze decyzje biznesowe mogą opierać się bezpośrednio na surowych prawdopodobieństwach najlepszego modelu. To jest korzystne zarówno operacyjnie, jak i projektowo: mniej złożoności, mniej ryzyka nadmiernego dopasowania i jaśniejsza logika przed przejściem do strojenia progu decyzyjnego.

**Wniosek:** Dział `4.8` został wykonany poprawnie i dał jednoznaczny rezultat. Zwycięzcą kalibracji został `lightgbm_tuned` w wariancie `raw`, czyli bez dodatkowej kalibracji `sigmoid` ani `isotonic`. Zapisano komplet artefaktów sekcji, a projekt jest gotowy do przejścia do `4.9`, czyli `Business Cost Matrix` i doboru finalnego progu decyzyjnego.

## 4.9 Business Cost Matrix i threshold tuning

In [49]:
# 4.9.1 Weryfikacja finalisty do threshold tuningu i jawnej Business Cost Matrix

import pandas as pd

required_objects = [
    "step4_8_calibration_choice_dict",
    "step4_8_candidate_models_dict",
    "step4_5_X_validation_model_df",
    "step4_5_y_validation",
    "step4_7_validation_comparison_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.9.1: {missing_objects}"

step4_9_selected_model_name = str(step4_8_calibration_choice_dict["selected_model_name"])
step4_9_selected_calibration_method = str(step4_8_calibration_choice_dict["selected_calibration_method"])

assert step4_9_selected_model_name in step4_8_candidate_models_dict, (
    f"4.9.1 selected model not found in candidate models: {step4_9_selected_model_name}"
)
assert step4_9_selected_calibration_method == "raw", (
    f"4.9.1 currently expects raw probabilities after 4.8 winner selection, got: {step4_9_selected_calibration_method}"
)

step4_9_selected_model = step4_8_candidate_models_dict[step4_9_selected_model_name]

step4_9_validation_y_true = pd.Series(step4_5_y_validation).astype(int).reset_index(drop=True)
step4_9_validation_y_score = pd.Series(
    step4_9_selected_model.predict_proba(step4_5_X_validation_model_df)[:, 1],
    name="y_score",
).clip(0.0, 1.0)

step4_9_business_cost_matrix_dict = {
    "true_positive_cost": 0.0,
    "true_negative_cost": 0.0,
    "false_positive_cost": 1.0,
    "false_negative_cost": 6.0,
}

step4_9_threshold_input_summary_df = pd.DataFrame(
    [
        {
            "selected_model_name": step4_9_selected_model_name,
            "selected_calibration_method": step4_9_selected_calibration_method,
            "selection_split": str(step4_8_calibration_choice_dict["selection_split"]),
            "validation_rows": int(len(step4_9_validation_y_true)),
            "validation_positive_count": int(step4_9_validation_y_true.sum()),
            "validation_negative_count": int(len(step4_9_validation_y_true) - step4_9_validation_y_true.sum()),
            "validation_positive_share": float(step4_9_validation_y_true.mean()),
            "score_min": float(step4_9_validation_y_score.min()),
            "score_mean": float(step4_9_validation_y_score.mean()),
            "score_max": float(step4_9_validation_y_score.max()),
            "false_positive_cost": float(step4_9_business_cost_matrix_dict["false_positive_cost"]),
            "false_negative_cost": float(step4_9_business_cost_matrix_dict["false_negative_cost"]),
            "fn_to_fp_cost_ratio": float(
                step4_9_business_cost_matrix_dict["false_negative_cost"]
                / step4_9_business_cost_matrix_dict["false_positive_cost"]
            ),
        }
    ]
)

step4_9_threshold_anchor_df = (
    step4_7_validation_comparison_df.loc[
        step4_7_validation_comparison_df["split_name"].eq("validation")
    ]
    .copy()
    .sort_values(
        ["pr_auc", "roc_auc", "balanced_accuracy", "specificity", "recall"],
        ascending=[False, False, False, False, False],
    )
    .reset_index(drop=True)
)

display(step4_9_threshold_input_summary_df)
display(step4_9_threshold_anchor_df)

assert step4_9_threshold_input_summary_df.shape[0] == 1, (
    "4.9.1 expected exactly one threshold tuning winner summary row."
)
assert float(step4_9_validation_y_score.min()) >= 0.0 and float(step4_9_validation_y_score.max()) <= 1.0, (
    "4.9.1 expected validation scores to be in [0, 1]."
)

,selected_model_name,selected_calibration_method,selection_split,validation_rows,validation_positive_count,validation_negative_count,validation_positive_share,score_min,score_mean,score_max,false_positive_cost,false_negative_cost,fn_to_fp_cost_ratio
0,lightgbm_tuned,raw,validation,70879,58268,12611,0.822077,0.04825,0.826014,0.994142,1.0,6.0,6.0


,model_name,split_name,row_count,positive_count,negative_count,accuracy,precision,recall,specificity,f1,...,candidate_id,learning_rate,n_estimators,num_leaves,max_depth,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda
0,lightgbm_tuned,validation,70879,58268,12611,0.829893,0.835948,0.986717,0.105305,0.905097,...,lgbm_c05,0.03,300.0,15.0,-1.0,60.0,0.8,0.8,0.0,0.0
1,lightgbm_baseline,validation,70879,58268,12611,0.829287,0.836207,0.985344,0.108239,0.904670,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,hist_gradient_boosting,validation,70879,58268,12611,0.829611,0.836135,0.985961,0.107208,0.904888,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,dumb_baseline,validation,70879,58268,12611,0.767604,0.860161,0.856559,0.356593,0.858357,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [50]:
# 4.9.2 Siatka progów na validation i Business Cost Matrix

import numpy as np
import pandas as pd

required_objects = [
    "step4_9_selected_model_name",
    "step4_9_selected_calibration_method",
    "step4_9_validation_y_true",
    "step4_9_validation_y_score",
    "step4_9_business_cost_matrix_dict",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.9.2: {missing_objects}"

if "compute_binary_metrics" not in globals():
    from sklearn.metrics import average_precision_score, roc_auc_score

    def compute_binary_metrics(y_true, y_pred, y_score):
        y_true = pd.Series(y_true).astype(int)
        y_pred = pd.Series(y_pred).astype(int)
        y_score = pd.Series(y_score).astype(float)

        tp = int(((y_true == 1) & (y_pred == 1)).sum())
        tn = int(((y_true == 0) & (y_pred == 0)).sum())
        fp = int(((y_true == 0) & (y_pred == 1)).sum())
        fn = int(((y_true == 1) & (y_pred == 0)).sum())

        row_count = int(len(y_true))
        positive_count = int((y_true == 1).sum())
        negative_count = int((y_true == 0).sum())

        accuracy = (tp + tn) / row_count if row_count > 0 else np.nan
        precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
        recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = (
            2 * precision * recall / (precision + recall)
            if pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0
            else np.nan
        )
        balanced_accuracy = (
            (recall + specificity) / 2
            if pd.notna(recall) and pd.notna(specificity)
            else np.nan
        )
        pred_positive_rate = float(y_pred.mean()) if row_count > 0 else np.nan
        roc_auc = float(roc_auc_score(y_true, y_score)) if positive_count > 0 and negative_count > 0 else np.nan
        pr_auc = float(average_precision_score(y_true, y_score)) if positive_count > 0 else np.nan

        return {
            "row_count": row_count,
            "positive_count": positive_count,
            "negative_count": negative_count,
            "tp": tp,
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "accuracy": float(accuracy),
            "precision": float(precision),
            "recall": float(recall),
            "specificity": float(specificity),
            "f1": float(f1),
            "balanced_accuracy": float(balanced_accuracy),
            "pred_positive_rate": float(pred_positive_rate),
            "roc_auc": float(roc_auc),
            "pr_auc": float(pr_auc),
        }

step4_9_threshold_grid = [round(x, 2) for x in np.arange(0.05, 1.00, 0.05)]

step4_9_threshold_grid_rows = []

for step4_9_threshold in step4_9_threshold_grid:
    step4_9_y_pred = (step4_9_validation_y_score >= step4_9_threshold).astype("int8")

    step4_9_metric_row = compute_binary_metrics(
        step4_9_validation_y_true,
        step4_9_y_pred,
        step4_9_validation_y_score,
    )

    step4_9_business_cost = (
        step4_9_metric_row["fp"] * step4_9_business_cost_matrix_dict["false_positive_cost"]
        + step4_9_metric_row["fn"] * step4_9_business_cost_matrix_dict["false_negative_cost"]
    )

    step4_9_metric_row["model_name"] = step4_9_selected_model_name
    step4_9_metric_row["calibration_method"] = step4_9_selected_calibration_method
    step4_9_metric_row["split_name"] = "validation"
    step4_9_metric_row["threshold"] = float(step4_9_threshold)
    step4_9_metric_row["false_positive_cost"] = float(step4_9_business_cost_matrix_dict["false_positive_cost"])
    step4_9_metric_row["false_negative_cost"] = float(step4_9_business_cost_matrix_dict["false_negative_cost"])
    step4_9_metric_row["business_cost_total"] = float(step4_9_business_cost)
    step4_9_metric_row["business_cost_per_row"] = float(
        step4_9_business_cost / step4_9_metric_row["row_count"]
    )

    step4_9_threshold_grid_rows.append(step4_9_metric_row)

step4_9_threshold_grid_validation_df = (
    pd.DataFrame(step4_9_threshold_grid_rows)
    .sort_values(
        ["business_cost_per_row", "balanced_accuracy", "specificity", "recall"],
        ascending=[True, False, False, False],
    )
    .reset_index(drop=True)
)

display(step4_9_threshold_grid_validation_df)

assert step4_9_threshold_grid_validation_df.shape[0] == len(step4_9_threshold_grid), (
    "4.9.2 expected one result row per threshold in the validation grid."
)

,row_count,positive_count,negative_count,tp,tn,fp,fn,accuracy,precision,recall,...,roc_auc,pr_auc,model_name,calibration_method,split_name,threshold,false_positive_cost,false_negative_cost,business_cost_total,business_cost_per_row
0,70879,58268,12611,58258,273,12338,10,0.825788,0.825231,0.999828,...,0.767928,0.935374,lightgbm_tuned,raw,validation,0.20,1.0,6.0,12398.0,0.174918
1,70879,58268,12611,58263,238,12373,5,0.825364,0.824834,0.999914,...,0.767928,0.935374,lightgbm_tuned,raw,validation,0.15,1.0,6.0,12403.0,0.174988
2,70879,58268,12611,58245,331,12280,23,0.826422,0.825877,0.999605,...,0.767928,0.935374,lightgbm_tuned,raw,validation,0.25,1.0,6.0,12418.0,0.175200
3,70879,58268,12611,58264,197,12414,4,0.824800,0.824358,0.999931,...,0.767928,0.935374,lightgbm_tuned,raw,validation,0.10,1.0,6.0,12438.0,0.175482
4,70879,58268,12611,58226,382,12229,42,0.826874,0.826428,0.999279,...,0.767928,0.935374,lightgbm_tuned,raw,validation,0.30,1.0,6.0,12481.0,0.176089
5,70879,58268,12611,58268,2,12609,0,0.822105,0.822100,1.000000,...,0.767928,0.935374,lightgbm_tuned,raw,validation,0.05,1.0,6.0,12609.0,0.177895
6,70879,58268,12611,58180,470,12141,88,0.827467,0.827349,0.998490,...,0.767928,0.935374,lightgbm_tuned,raw,validation,0.35,1.0,6.0,12669.0,0.178741
7,70879,58268,12611,58065,685,11926,203,0.828877,0.829607,0.996516,...,0.767928,0.935374,lightgbm_tuned,raw,validation,0.40,1.0,6.0,13144.0,0.185443
8,70879,58268,12611,57856,945,11666,412,0.829597,0.832197,0.992929,...,0.767928,0.935374,lightgbm_tuned,raw,validation,0.45,1.0,6.0,14138.0,0.199467
9,70879,58268,12611,57494,1328,11283,774,0.829893,0.835948,0.986717,...,0.767928,0.935374,lightgbm_tuned,raw,validation,0.50,1.0,6.0,15927.0,0.224707


In [51]:
# 4.9.3 Feasible thresholds i guardraile operacyjne

import pandas as pd

required_objects = [
    "step4_9_threshold_grid_validation_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.9.3: {missing_objects}"

step4_9_guardrails_dict = {
    "min_recall": 0.95,
    "min_specificity": 0.10,
    "min_precision": 0.83,
    "max_pred_positive_rate": 0.97,
}

step4_9_threshold_guardrails_df = step4_9_threshold_grid_validation_df.copy()

step4_9_threshold_guardrails_df["passes_recall_guardrail"] = (
    step4_9_threshold_guardrails_df["recall"] >= step4_9_guardrails_dict["min_recall"]
).astype(int)

step4_9_threshold_guardrails_df["passes_specificity_guardrail"] = (
    step4_9_threshold_guardrails_df["specificity"] >= step4_9_guardrails_dict["min_specificity"]
).astype(int)

step4_9_threshold_guardrails_df["passes_precision_guardrail"] = (
    step4_9_threshold_guardrails_df["precision"] >= step4_9_guardrails_dict["min_precision"]
).astype(int)

step4_9_threshold_guardrails_df["passes_pred_positive_rate_guardrail"] = (
    step4_9_threshold_guardrails_df["pred_positive_rate"] <= step4_9_guardrails_dict["max_pred_positive_rate"]
).astype(int)

step4_9_threshold_guardrails_df["is_feasible_threshold"] = (
    step4_9_threshold_guardrails_df[
        [
            "passes_recall_guardrail",
            "passes_specificity_guardrail",
            "passes_precision_guardrail",
            "passes_pred_positive_rate_guardrail",
        ]
    ].sum(axis=1)
    == 4
).astype(int)

step4_9_feasible_thresholds_df = (
    step4_9_threshold_guardrails_df.loc[
        step4_9_threshold_guardrails_df["is_feasible_threshold"].eq(1)
    ]
    .sort_values(
        ["business_cost_per_row", "balanced_accuracy", "specificity", "recall"],
        ascending=[True, False, False, False],
    )
    .reset_index(drop=True)
)

step4_9_guardrails_summary_df = pd.DataFrame(
    [
        {
            "min_recall": step4_9_guardrails_dict["min_recall"],
            "min_specificity": step4_9_guardrails_dict["min_specificity"],
            "min_precision": step4_9_guardrails_dict["min_precision"],
            "max_pred_positive_rate": step4_9_guardrails_dict["max_pred_positive_rate"],
            "n_thresholds_checked": int(step4_9_threshold_guardrails_df.shape[0]),
            "n_feasible_thresholds": int(step4_9_feasible_thresholds_df.shape[0]),
        }
    ]
)

display(step4_9_guardrails_summary_df)
display(step4_9_feasible_thresholds_df)
display(step4_9_threshold_guardrails_df)

assert step4_9_feasible_thresholds_df.shape[0] > 0, (
    "4.9.3 expected at least one feasible threshold after applying operational guardrails."
)

,min_recall,min_specificity,min_precision,max_pred_positive_rate,n_thresholds_checked,n_feasible_thresholds
0,0.95,0.1,0.83,0.97,19,2


,row_count,positive_count,negative_count,tp,tn,fp,fn,accuracy,precision,recall,...,threshold,false_positive_cost,false_negative_cost,business_cost_total,business_cost_per_row,passes_recall_guardrail,passes_specificity_guardrail,passes_precision_guardrail,passes_pred_positive_rate_guardrail,is_feasible_threshold
0,70879,58268,12611,56732,1999,10612,1536,0.828609,0.842421,0.973639,...,0.55,1.0,6.0,19828.0,0.279744,1,1,1,1,1
1,70879,58268,12611,55631,2847,9764,2637,0.825040,0.850692,0.954744,...,0.60,1.0,6.0,25586.0,0.360981,1,1,1,1,1


,row_count,positive_count,negative_count,tp,tn,fp,fn,accuracy,precision,recall,...,threshold,false_positive_cost,false_negative_cost,business_cost_total,business_cost_per_row,passes_recall_guardrail,passes_specificity_guardrail,passes_precision_guardrail,passes_pred_positive_rate_guardrail,is_feasible_threshold
0,70879,58268,12611,58258,273,12338,10,0.825788,0.825231,0.999828,...,0.20,1.0,6.0,12398.0,0.174918,1,0,0,0,0
1,70879,58268,12611,58263,238,12373,5,0.825364,0.824834,0.999914,...,0.15,1.0,6.0,12403.0,0.174988,1,0,0,0,0
2,70879,58268,12611,58245,331,12280,23,0.826422,0.825877,0.999605,...,0.25,1.0,6.0,12418.0,0.175200,1,0,0,0,0
3,70879,58268,12611,58264,197,12414,4,0.824800,0.824358,0.999931,...,0.10,1.0,6.0,12438.0,0.175482,1,0,0,0,0
4,70879,58268,12611,58226,382,12229,42,0.826874,0.826428,0.999279,...,0.30,1.0,6.0,12481.0,0.176089,1,0,0,0,0
5,70879,58268,12611,58268,2,12609,0,0.822105,0.822100,1.000000,...,0.05,1.0,6.0,12609.0,0.177895,1,0,0,0,0
6,70879,58268,12611,58180,470,12141,88,0.827467,0.827349,0.998490,...,0.35,1.0,6.0,12669.0,0.178741,1,0,0,0,0
7,70879,58268,12611,58065,685,11926,203,0.828877,0.829607,0.996516,...,0.40,1.0,6.0,13144.0,0.185443,1,0,0,0,0
8,70879,58268,12611,57856,945,11666,412,0.829597,0.832197,0.992929,...,0.45,1.0,6.0,14138.0,0.199467,1,0,1,0,0
9,70879,58268,12611,57494,1328,11283,774,0.829893,0.835948,0.986717,...,0.50,1.0,6.0,15927.0,0.224707,1,1,1,0,0


In [52]:
# 4.9.4 Wybór finalnego progu decyzyjnego

import pandas as pd

required_objects = [
    "step4_9_feasible_thresholds_df",
    "step4_9_guardrails_dict",
    "step4_9_selected_model_name",
    "step4_9_selected_calibration_method",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.9.4: {missing_objects}"

step4_9_best_threshold_row = step4_9_feasible_thresholds_df.iloc[0].copy()

step4_9_threshold_choice_dict = {
    "selected_model_name": str(step4_9_selected_model_name),
    "selected_calibration_method": str(step4_9_selected_calibration_method),
    "selected_threshold": float(step4_9_best_threshold_row["threshold"]),
    "selection_split": str(step4_9_best_threshold_row["split_name"]),
    "selection_rule": [
        "feasible_threshold_only",
        "lowest_business_cost_per_row",
        "highest_balanced_accuracy",
        "highest_specificity",
        "highest_recall",
    ],
    "guardrails": {
        "min_recall": float(step4_9_guardrails_dict["min_recall"]),
        "min_specificity": float(step4_9_guardrails_dict["min_specificity"]),
        "min_precision": float(step4_9_guardrails_dict["min_precision"]),
        "max_pred_positive_rate": float(step4_9_guardrails_dict["max_pred_positive_rate"]),
    },
    "validation_metrics": {
        "accuracy": float(step4_9_best_threshold_row["accuracy"]),
        "precision": float(step4_9_best_threshold_row["precision"]),
        "recall": float(step4_9_best_threshold_row["recall"]),
        "specificity": float(step4_9_best_threshold_row["specificity"]),
        "f1": float(step4_9_best_threshold_row["f1"]),
        "balanced_accuracy": float(step4_9_best_threshold_row["balanced_accuracy"]),
        "pred_positive_rate": float(step4_9_best_threshold_row["pred_positive_rate"]),
        "business_cost_total": float(step4_9_best_threshold_row["business_cost_total"]),
        "business_cost_per_row": float(step4_9_best_threshold_row["business_cost_per_row"]),
        "tp": int(step4_9_best_threshold_row["tp"]),
        "tn": int(step4_9_best_threshold_row["tn"]),
        "fp": int(step4_9_best_threshold_row["fp"]),
        "fn": int(step4_9_best_threshold_row["fn"]),
    },
}

step4_9_threshold_choice_df = pd.DataFrame(
    [
        {
            "selected_model_name": step4_9_threshold_choice_dict["selected_model_name"],
            "selected_calibration_method": step4_9_threshold_choice_dict["selected_calibration_method"],
            "selected_threshold": step4_9_threshold_choice_dict["selected_threshold"],
            "selection_split": step4_9_threshold_choice_dict["selection_split"],
            "business_cost_per_row": step4_9_threshold_choice_dict["validation_metrics"]["business_cost_per_row"],
            "balanced_accuracy": step4_9_threshold_choice_dict["validation_metrics"]["balanced_accuracy"],
            "recall": step4_9_threshold_choice_dict["validation_metrics"]["recall"],
            "specificity": step4_9_threshold_choice_dict["validation_metrics"]["specificity"],
            "precision": step4_9_threshold_choice_dict["validation_metrics"]["precision"],
            "pred_positive_rate": step4_9_threshold_choice_dict["validation_metrics"]["pred_positive_rate"],
            "tp": step4_9_threshold_choice_dict["validation_metrics"]["tp"],
            "tn": step4_9_threshold_choice_dict["validation_metrics"]["tn"],
            "fp": step4_9_threshold_choice_dict["validation_metrics"]["fp"],
            "fn": step4_9_threshold_choice_dict["validation_metrics"]["fn"],
        }
    ]
)

display(step4_9_threshold_choice_df)

assert step4_9_threshold_choice_df.shape[0] == 1, (
    "4.9.4 expected exactly one selected threshold row."
)
assert float(step4_9_threshold_choice_df.loc[0, "selected_threshold"]) > 0.0, (
    "4.9.4 selected threshold must be positive."
)

,selected_model_name,selected_calibration_method,selected_threshold,selection_split,business_cost_per_row,balanced_accuracy,recall,specificity,precision,pred_positive_rate,tp,tn,fp,fn
0,lightgbm_tuned,raw,0.55,validation,0.279744,0.566076,0.973639,0.158512,0.842421,0.950126,56732,1999,10612,1536


In [53]:
# 4.9.5 Zapis artefaktów sekcji 4.9

from pathlib import Path
import json

import pandas as pd

required_objects = [
    "step4_3_project_root",
    "step4_9_threshold_grid_validation_df",
    "step4_9_threshold_choice_dict",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.9.5: {missing_objects}"

step4_9_output_dir = step4_3_project_root / "artifacts" / "4_threshold_tuning"
step4_9_output_dir.mkdir(parents=True, exist_ok=True)

step4_9_threshold_grid_path = step4_9_output_dir / "b4_09_threshold_grid_validation.parquet"
step4_9_threshold_choice_path = step4_9_output_dir / "b4_09_threshold_choice.json"

step4_9_threshold_grid_save_df = (
    step4_9_threshold_grid_validation_df.copy()
    .sort_values(
        ["business_cost_per_row", "balanced_accuracy", "specificity", "recall"],
        ascending=[True, False, False, False],
    )
    .reset_index(drop=True)
)

step4_9_threshold_grid_save_df.to_parquet(
    step4_9_threshold_grid_path,
    index=False,
)

with open(step4_9_threshold_choice_path, "w", encoding="utf-8") as f:
    json.dump(step4_9_threshold_choice_dict, f, ensure_ascii=False, indent=2)

step4_9_saved_artifacts_registry_df = pd.DataFrame(
    [
        {
            "artifact_name": step4_9_threshold_grid_path.name,
            "rows": int(step4_9_threshold_grid_save_df.shape[0]),
            "cols": int(step4_9_threshold_grid_save_df.shape[1]),
            "path": str(step4_9_threshold_grid_path),
        },
        {
            "artifact_name": step4_9_threshold_choice_path.name,
            "rows": 1,
            "cols": len(step4_9_threshold_choice_dict),
            "path": str(step4_9_threshold_choice_path),
        },
    ]
)

display(step4_9_saved_artifacts_registry_df)

assert step4_9_threshold_grid_path.exists(), (
    "4.9.5 failed to save b4_09_threshold_grid_validation.parquet"
)
assert step4_9_threshold_choice_path.exists(), (
    "4.9.5 failed to save b4_09_threshold_choice.json"
)
assert step4_9_threshold_grid_save_df.shape[0] == 19, (
    "4.9.5 expected 19 rows in saved threshold grid artifact."
)

,artifact_name,rows,cols,path
0,b4_09_threshold_grid_validation.parquet,19,24,/root/projects/6_samodzielny_projekt_koncowy_w...
1,b4_09_threshold_choice.json,1,7,/root/projects/6_samodzielny_projekt_koncowy_w...


## Podsumowanie działu 4.9 Business Cost Matrix i threshold tuning

**Kontekst inżynieryjny:** W dziale `4.9` wykorzystano zwycięzcę z `4.8`, czyli `lightgbm_tuned` w wariancie `raw`, i przygotowano jawny proces doboru progu decyzyjnego na `validation`. Najpierw potwierdzono finalistę oraz zdefiniowano Business Cost Matrix z kosztami `false_positive_cost = 1.0` oraz `false_negative_cost = 6.0`. Następnie zbudowano siatkę progów na `validation`, policzono metryki klasyfikacyjne i koszt biznesowy dla każdego progu, po czym dołożono guardraile operacyjne. Na końcu wybrano finalny próg oraz zapisano artefakty `b4_09_threshold_grid_validation.parquet` i `b4_09_threshold_choice.json`.

**Interpretacja wyniku:** Surowy wynik kosztowy wskazywał najniższy koszt biznesowy dla bardzo niskich progów, co było spójne z macierzą kosztów silnie karzącą `false negatives`. Jednocześnie po nałożeniu guardraili operacyjnych liczba dopuszczalnych progów zawęziła się do `2` wariantów: `0.55` i `0.60`. Finalnie wybrano próg `0.55`, ponieważ był najlepszym progiem feasible według przyjętej logiki selekcji. Dla `validation` uzyskano: `business_cost_per_row = 0.279744`, `balanced_accuracy = 0.566076`, `recall = 0.973639`, `specificity = 0.158512`, `precision = 0.842421`, `pred_positive_rate = 0.950126`, przy macierzy pomyłek `TP = 56732`, `TN = 1999`, `FP = 10612`, `FN = 1536`.

**Znaczenie biznesowe:** Dział `4.9` pokazał, że sam najniższy koszt biznesowy nie może być jedynym kryterium wyboru progu, ponieważ prowadziłby do zbyt agresywnego profilu predykcji dodatniej. Zastosowanie guardraili operacyjnych uporządkowało decyzję i pozwoliło wybrać próg, który zachowuje bardzo wysoki `recall`, ale jednocześnie spełnia minimalne warunki jakości operacyjnej. Oznacza to, że projekt przeszedł od modelu rankingowego do jawnej reguły decyzyjnej, którą można dalej uczciwie testować i raportować.

**Wniosek:** Dział `4.9` został wykonany poprawnie i zakończył się wyborem finalnego progu `0.55` dla modelu `lightgbm_tuned` w wariancie `raw`. Próg ten jest najlepszym kompromisem spośród progów dopuszczonych przez guardraile operacyjne i Business Cost Matrix. Zapisano komplet artefaktów sekcji, a projekt jest gotowy do przejścia do `4.10`, czyli finalnego porównania modeli na `validation`.

## 4.10 Finalne porównanie modeli na validation

In [54]:
# 4.10.1 Finalne porównanie modeli na validation

import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss

required_objects = [
    "step4_5_light_ml_baselines_df",
    "step4_7_validation_comparison_df",
    "step4_8_calibration_report_df",
    "step4_8_calibration_choice_dict",
    "step4_9_threshold_choice_dict",
    "step4_5_X_validation_model_df",
    "step4_5_y_validation",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.10.1: {missing_objects}"

step4_10_y_validation_series = pd.Series(step4_5_y_validation).astype(int).reset_index(drop=True)

step4_10_model_object_candidates = {
    "logistic_regression": [
        "step4_5_logreg_model",
        "step4_5_logistic_regression_model",
    ],
    "random_forest": [
        "step4_5_random_forest_model",
    ],
    "extra_trees": [
        "step4_5_extra_trees_model",
    ],
    "hist_gradient_boosting": [
        "step4_5_hist_gb_model",
    ],
    "lightgbm_baseline": [
        "step4_6_lgbm_baseline_model",
    ],
    "lightgbm_tuned": [
        "step4_7_best_lgbm_model",
    ],
}

step4_10_resolved_model_objects = {}
for step4_10_model_name, step4_10_candidate_names in step4_10_model_object_candidates.items():
    for step4_10_candidate_name in step4_10_candidate_names:
        if step4_10_candidate_name in globals():
            step4_10_resolved_model_objects[step4_10_model_name] = globals()[step4_10_candidate_name]
            break

step4_10_raw_brier_from_models = {}
for step4_10_model_name, step4_10_model_obj in step4_10_resolved_model_objects.items():
    step4_10_y_score = pd.Series(
        step4_10_model_obj.predict_proba(step4_5_X_validation_model_df)[:, 1]
    ).clip(0.0, 1.0)
    step4_10_raw_brier_from_models[step4_10_model_name] = float(
        brier_score_loss(step4_10_y_validation_series, step4_10_y_score)
    )

step4_10_raw_calibration_report_df = (
    step4_8_calibration_report_df.loc[
        step4_8_calibration_report_df["calibration_method"].eq("raw")
    ]
    .copy()
    .reset_index(drop=True)
)

step4_10_raw_report_map = {}
for _, step4_10_row in step4_10_raw_calibration_report_df.iterrows():
    step4_10_raw_report_map[str(step4_10_row["model_name"])] = {
        "brier_score": float(step4_10_row["brier_score"]),
        "roc_auc": float(step4_10_row["roc_auc"]),
        "pr_auc": float(step4_10_row["pr_auc"]),
    }

step4_10_base_validation_df = (
    step4_5_light_ml_baselines_df.loc[
        step4_5_light_ml_baselines_df["split_name"].eq("validation")
    ]
    .copy()
    .reset_index(drop=True)
)

step4_10_base_validation_df = step4_10_base_validation_df.loc[
    step4_10_base_validation_df["model_name"].isin(
        [
            "dumb_baseline",
            "logistic_regression",
            "random_forest",
            "extra_trees",
            "hist_gradient_boosting",
        ]
    )
].reset_index(drop=True)

step4_10_rows = []

for _, step4_10_row in step4_10_base_validation_df.iterrows():
    step4_10_model_name = str(step4_10_row["model_name"])

    if step4_10_model_name == "dumb_baseline":
        step4_10_brier = float(1.0 - float(step4_10_row["accuracy"]))
        step4_10_roc_auc = np.nan
        step4_10_pr_auc = np.nan
    else:
        step4_10_brier = step4_10_raw_brier_from_models.get(step4_10_model_name, np.nan)
        step4_10_roc_auc = float(step4_10_row["roc_auc"])
        step4_10_pr_auc = float(step4_10_row["pr_auc"])

    step4_10_rows.append(
        {
            "comparison_variant": step4_10_model_name,
            "model_name": step4_10_model_name,
            "calibration_method": "raw_or_not_applicable",
            "decision_threshold": 0.50,
            "accuracy": float(step4_10_row["accuracy"]),
            "precision": float(step4_10_row["precision"]),
            "recall": float(step4_10_row["recall"]),
            "specificity": float(step4_10_row["specificity"]),
            "f1": float(step4_10_row["f1"]),
            "balanced_accuracy": float(step4_10_row["balanced_accuracy"]),
            "pred_positive_rate": float(step4_10_row["pred_positive_rate"]),
            "roc_auc": step4_10_roc_auc,
            "pr_auc": step4_10_pr_auc,
            "brier_score": step4_10_brier,
            "business_cost_per_row": np.nan,
        }
    )

step4_10_lgbm_baseline_validation_row = (
    step4_7_validation_comparison_df.loc[
        (step4_7_validation_comparison_df["split_name"].eq("validation"))
        & (step4_7_validation_comparison_df["model_name"].eq("lightgbm_baseline"))
    ]
    .iloc[0]
)

step4_10_rows.append(
    {
        "comparison_variant": "lightgbm_baseline",
        "model_name": "lightgbm_baseline",
        "calibration_method": "raw",
        "decision_threshold": 0.50,
        "accuracy": float(step4_10_lgbm_baseline_validation_row["accuracy"]),
        "precision": float(step4_10_lgbm_baseline_validation_row["precision"]),
        "recall": float(step4_10_lgbm_baseline_validation_row["recall"]),
        "specificity": float(step4_10_lgbm_baseline_validation_row["specificity"]),
        "f1": float(step4_10_lgbm_baseline_validation_row["f1"]),
        "balanced_accuracy": float(step4_10_lgbm_baseline_validation_row["balanced_accuracy"]),
        "pred_positive_rate": float(step4_10_lgbm_baseline_validation_row["pred_positive_rate"]),
        "roc_auc": step4_10_raw_report_map["lightgbm_baseline"]["roc_auc"],
        "pr_auc": step4_10_raw_report_map["lightgbm_baseline"]["pr_auc"],
        "brier_score": step4_10_raw_report_map["lightgbm_baseline"]["brier_score"],
        "business_cost_per_row": np.nan,
    }
)

step4_10_lgbm_tuned_validation_row = (
    step4_7_validation_comparison_df.loc[
        (step4_7_validation_comparison_df["split_name"].eq("validation"))
        & (step4_7_validation_comparison_df["model_name"].eq("lightgbm_tuned"))
    ]
    .iloc[0]
)

step4_10_rows.append(
    {
        "comparison_variant": "lightgbm_tuned",
        "model_name": "lightgbm_tuned",
        "calibration_method": "raw",
        "decision_threshold": 0.50,
        "accuracy": float(step4_10_lgbm_tuned_validation_row["accuracy"]),
        "precision": float(step4_10_lgbm_tuned_validation_row["precision"]),
        "recall": float(step4_10_lgbm_tuned_validation_row["recall"]),
        "specificity": float(step4_10_lgbm_tuned_validation_row["specificity"]),
        "f1": float(step4_10_lgbm_tuned_validation_row["f1"]),
        "balanced_accuracy": float(step4_10_lgbm_tuned_validation_row["balanced_accuracy"]),
        "pred_positive_rate": float(step4_10_lgbm_tuned_validation_row["pred_positive_rate"]),
        "roc_auc": step4_10_raw_report_map["lightgbm_tuned"]["roc_auc"],
        "pr_auc": step4_10_raw_report_map["lightgbm_tuned"]["pr_auc"],
        "brier_score": step4_10_raw_report_map["lightgbm_tuned"]["brier_score"],
        "business_cost_per_row": np.nan,
    }
)

step4_10_rows.append(
    {
        "comparison_variant": "lightgbm_tuned_calibration_winner",
        "model_name": str(step4_8_calibration_choice_dict["selected_model_name"]),
        "calibration_method": str(step4_8_calibration_choice_dict["selected_calibration_method"]),
        "decision_threshold": 0.50,
        "accuracy": float(step4_10_lgbm_tuned_validation_row["accuracy"]),
        "precision": float(step4_10_lgbm_tuned_validation_row["precision"]),
        "recall": float(step4_10_lgbm_tuned_validation_row["recall"]),
        "specificity": float(step4_10_lgbm_tuned_validation_row["specificity"]),
        "f1": float(step4_10_lgbm_tuned_validation_row["f1"]),
        "balanced_accuracy": float(step4_10_lgbm_tuned_validation_row["balanced_accuracy"]),
        "pred_positive_rate": float(step4_10_lgbm_tuned_validation_row["pred_positive_rate"]),
        "roc_auc": float(step4_8_calibration_choice_dict["validation_metrics"]["roc_auc"]),
        "pr_auc": float(step4_8_calibration_choice_dict["validation_metrics"]["pr_auc"]),
        "brier_score": float(step4_8_calibration_choice_dict["validation_metrics"]["brier_score"]),
        "business_cost_per_row": np.nan,
    }
)

step4_10_rows.append(
    {
        "comparison_variant": "lightgbm_tuned_selected_threshold",
        "model_name": str(step4_9_threshold_choice_dict["selected_model_name"]),
        "calibration_method": str(step4_9_threshold_choice_dict["selected_calibration_method"]),
        "decision_threshold": float(step4_9_threshold_choice_dict["selected_threshold"]),
        "accuracy": float(step4_9_threshold_choice_dict["validation_metrics"]["accuracy"]),
        "precision": float(step4_9_threshold_choice_dict["validation_metrics"]["precision"]),
        "recall": float(step4_9_threshold_choice_dict["validation_metrics"]["recall"]),
        "specificity": float(step4_9_threshold_choice_dict["validation_metrics"]["specificity"]),
        "f1": float(step4_9_threshold_choice_dict["validation_metrics"]["f1"]),
        "balanced_accuracy": float(step4_9_threshold_choice_dict["validation_metrics"]["balanced_accuracy"]),
        "pred_positive_rate": float(step4_9_threshold_choice_dict["validation_metrics"]["pred_positive_rate"]),
        "roc_auc": float(step4_8_calibration_choice_dict["validation_metrics"]["roc_auc"]),
        "pr_auc": float(step4_8_calibration_choice_dict["validation_metrics"]["pr_auc"]),
        "brier_score": float(step4_8_calibration_choice_dict["validation_metrics"]["brier_score"]),
        "business_cost_per_row": float(step4_9_threshold_choice_dict["validation_metrics"]["business_cost_per_row"]),
    }
)

step4_10_model_comparison_validation_df = pd.DataFrame(step4_10_rows)

step4_10_order_map = {
    "dumb_baseline": 1,
    "logistic_regression": 2,
    "random_forest": 3,
    "extra_trees": 4,
    "hist_gradient_boosting": 5,
    "lightgbm_baseline": 6,
    "lightgbm_tuned": 7,
    "lightgbm_tuned_calibration_winner": 8,
    "lightgbm_tuned_selected_threshold": 9,
}

step4_10_model_comparison_validation_df["comparison_order"] = (
    step4_10_model_comparison_validation_df["comparison_variant"]
    .map(step4_10_order_map)
    .astype(int)
)

step4_10_model_comparison_validation_df = (
    step4_10_model_comparison_validation_df
    .sort_values("comparison_order")
    .drop(columns=["comparison_order"])
    .reset_index(drop=True)
)

display(step4_10_model_comparison_validation_df)

assert step4_10_model_comparison_validation_df.shape[0] == 9, (
    "4.10.1 expected exactly 9 comparison rows on validation."
)

,comparison_variant,model_name,calibration_method,decision_threshold,accuracy,precision,recall,specificity,f1,balanced_accuracy,pred_positive_rate,roc_auc,pr_auc,brier_score,business_cost_per_row
0,dumb_baseline,dumb_baseline,raw_or_not_applicable,0.50,0.767604,0.860161,0.856559,0.356593,0.858357,0.606576,0.818635,NaN,NaN,0.232396,NaN
1,logistic_regression,logistic_regression,raw_or_not_applicable,0.50,0.827128,0.835969,0.982495,0.109270,0.903329,0.545882,0.966168,0.743043,0.922323,NaN,NaN
2,random_forest,random_forest,raw_or_not_applicable,0.50,0.809506,0.845417,0.940190,0.205693,0.890288,0.572942,0.914234,0.714382,0.913683,0.140825,NaN
3,extra_trees,extra_trees,raw_or_not_applicable,0.50,0.799235,0.846690,0.922891,0.227896,0.883150,0.575394,0.896062,0.683346,0.894511,0.154182,NaN
4,hist_gradient_boosting,hist_gradient_boosting,raw_or_not_applicable,0.50,0.829611,0.836135,0.985961,0.107208,0.904888,0.546585,0.969384,0.765273,0.934171,0.124962,NaN
5,lightgbm_baseline,lightgbm_baseline,raw,0.50,0.829287,0.836207,0.985344,0.108239,0.904670,0.546791,0.968693,0.766115,0.934430,0.124899,NaN
6,lightgbm_tuned,lightgbm_tuned,raw,0.50,0.829893,0.835948,0.986717,0.105305,0.905097,0.546011,0.970344,0.767928,0.935374,0.124602,NaN
7,lightgbm_tuned_calibration_winner,lightgbm_tuned,raw,0.50,0.829893,0.835948,0.986717,0.105305,0.905097,0.546011,0.970344,0.767928,0.935374,0.124602,NaN
8,lightgbm_tuned_selected_threshold,lightgbm_tuned,raw,0.55,0.828609,0.842421,0.973639,0.158512,0.903289,0.566076,0.950126,0.767928,0.935374,0.124602,0.279744


In [55]:
# 4.10.2 Oczyszczenie finalnej tabeli porównawczej na validation

import pandas as pd

required_objects = [
    "step4_10_model_comparison_validation_df",
    "step4_8_calibration_choice_dict",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.10.2: {missing_objects}"

step4_10_model_comparison_validation_clean_df = (
    step4_10_model_comparison_validation_df.copy()
)

step4_10_selected_model_name = str(step4_8_calibration_choice_dict["selected_model_name"])
step4_10_selected_calibration_method = str(step4_8_calibration_choice_dict["selected_calibration_method"])

if (
    step4_10_selected_model_name == "lightgbm_tuned"
    and step4_10_selected_calibration_method == "raw"
):
    step4_10_model_comparison_validation_clean_df = (
        step4_10_model_comparison_validation_clean_df.loc[
            step4_10_model_comparison_validation_clean_df["comparison_variant"]
            != "lightgbm_tuned_calibration_winner"
        ]
        .reset_index(drop=True)
    )

display(step4_10_model_comparison_validation_clean_df)

assert "lightgbm_tuned_selected_threshold" in set(
    step4_10_model_comparison_validation_clean_df["comparison_variant"]
), "4.10.2 expected selected-threshold variant in final comparison."

assert step4_10_model_comparison_validation_clean_df.shape[0] == 8, (
    "4.10.2 expected 8 rows after removing the duplicate calibration-winner row."
)

,comparison_variant,model_name,calibration_method,decision_threshold,accuracy,precision,recall,specificity,f1,balanced_accuracy,pred_positive_rate,roc_auc,pr_auc,brier_score,business_cost_per_row
0,dumb_baseline,dumb_baseline,raw_or_not_applicable,0.50,0.767604,0.860161,0.856559,0.356593,0.858357,0.606576,0.818635,NaN,NaN,0.232396,NaN
1,logistic_regression,logistic_regression,raw_or_not_applicable,0.50,0.827128,0.835969,0.982495,0.109270,0.903329,0.545882,0.966168,0.743043,0.922323,NaN,NaN
2,random_forest,random_forest,raw_or_not_applicable,0.50,0.809506,0.845417,0.940190,0.205693,0.890288,0.572942,0.914234,0.714382,0.913683,0.140825,NaN
3,extra_trees,extra_trees,raw_or_not_applicable,0.50,0.799235,0.846690,0.922891,0.227896,0.883150,0.575394,0.896062,0.683346,0.894511,0.154182,NaN
4,hist_gradient_boosting,hist_gradient_boosting,raw_or_not_applicable,0.50,0.829611,0.836135,0.985961,0.107208,0.904888,0.546585,0.969384,0.765273,0.934171,0.124962,NaN
5,lightgbm_baseline,lightgbm_baseline,raw,0.50,0.829287,0.836207,0.985344,0.108239,0.904670,0.546791,0.968693,0.766115,0.934430,0.124899,NaN
6,lightgbm_tuned,lightgbm_tuned,raw,0.50,0.829893,0.835948,0.986717,0.105305,0.905097,0.546011,0.970344,0.767928,0.935374,0.124602,NaN
7,lightgbm_tuned_selected_threshold,lightgbm_tuned,raw,0.55,0.828609,0.842421,0.973639,0.158512,0.903289,0.566076,0.950126,0.767928,0.935374,0.124602,0.279744


In [56]:
# 4.10.3 Zapis artefaktu sekcji 4.10

from pathlib import Path

import pandas as pd

required_objects = [
    "step4_3_project_root",
    "step4_10_model_comparison_validation_clean_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.10.3: {missing_objects}"

step4_10_output_dir = step4_3_project_root / "artifacts" / "4_model_comparison_validation"
step4_10_output_dir.mkdir(parents=True, exist_ok=True)

step4_10_model_comparison_validation_path = (
    step4_10_output_dir / "b4_10_model_comparison_validation.parquet"
)

step4_10_model_comparison_validation_save_df = (
    step4_10_model_comparison_validation_clean_df.copy()
    .reset_index(drop=True)
)

step4_10_model_comparison_validation_save_df.to_parquet(
    step4_10_model_comparison_validation_path,
    index=False,
)

step4_10_saved_artifacts_registry_df = pd.DataFrame(
    [
        {
            "artifact_name": step4_10_model_comparison_validation_path.name,
            "rows": int(step4_10_model_comparison_validation_save_df.shape[0]),
            "cols": int(step4_10_model_comparison_validation_save_df.shape[1]),
            "path": str(step4_10_model_comparison_validation_path),
        }
    ]
)

display(step4_10_saved_artifacts_registry_df)

assert step4_10_model_comparison_validation_path.exists(), (
    "4.10.3 failed to save b4_10_model_comparison_validation.parquet"
)
assert step4_10_model_comparison_validation_save_df.shape[0] == 8, (
    "4.10.3 expected 8 rows in saved validation comparison artifact."
)

,artifact_name,rows,cols,path
0,b4_10_model_comparison_validation.parquet,8,15,/root/projects/6_samodzielny_projekt_koncowy_w...


## Podsumowanie działu 4.10 Finalne porównanie modeli na validation

**Kontekst inżynieryjny:** W dziale `4.10` zbudowano końcową tabelę porównawczą jakości modeli na zbiorze `validation`. Do wspólnego porównania włączono: `dumb_baseline`, lekkie baseline’y ML (`logistic_regression`, `random_forest`, `extra_trees`, `hist_gradient_boosting`), `lightgbm_baseline`, `lightgbm_tuned` oraz finalną wersję decyzyjną `lightgbm_tuned_selected_threshold`. Tabela została oczyszczona z duplikatu wariantu kalibracyjnego, ponieważ zwycięzca kalibracji pokrywał się z `lightgbm_tuned` w wersji `raw`. Na końcu zapisano artefakt `b4_10_model_comparison_validation.parquet`.

**Interpretacja wyniku:** Końcowe porównanie pokazało trzy różne profile jakości. `lightgbm_tuned` przy progu `0.50` osiągnął najlepsze metryki rankingowe i probabilistyczne: `ROC AUC = 0.767928`, `PR AUC = 0.935374` oraz `Brier Score = 0.124602`. `dumb_baseline` pozostał najmocniejszy pod względem czystej równowagi operacyjnej przy domyślnym progu, z `balanced_accuracy = 0.606576` i `specificity = 0.356593`. Z kolei finalna wersja projektowa `lightgbm_tuned_selected_threshold` z progiem `0.55` poprawiła profil decyzyjny względem `lightgbm_tuned` przy `0.50`: `precision = 0.842421`, `specificity = 0.158512`, `balanced_accuracy = 0.566076`, `pred_positive_rate = 0.950126`, przy zachowaniu wysokiego `recall = 0.973639`. Oznacza to, że końcowy próg biznesowy uporządkował zachowanie modelu, choć nie zmienił go w model konserwatywny.

**Znaczenie biznesowe:** Dział `4.10` rozdziela trzy role modeli w projekcie. `LightGBM tuned` jest najlepszym kandydatem do rankingu i pracy na prawdopodobieństwach. `Dumb baseline` pozostaje ważnym punktem odniesienia, ponieważ pokazuje najmocniejszy prosty balans operacyjny. Finalna wersja `lightgbm_tuned_selected_threshold` stanowi natomiast najlepszy kompromis projektowy po uwzględnieniu kalibracji, Business Cost Matrix i guardraili operacyjnych. Dzięki temu projekt nie kończy się wyborem modelu „najlepszego na jednej metryce”, tylko wyborem wariantu najlepiej uzasadnionego biznesowo.

**Wniosek:** Dział `4.10` został wykonany poprawnie i doprowadził do czytelnego, końcowego porównania modeli na `validation`. Najmocniejszym modelem rankingowym jest `lightgbm_tuned`, natomiast finalną wersją decyzyjną projektu jest `lightgbm_tuned` w wariancie `raw` z progiem `0.55`. Zapisano artefakt `b4_10_model_comparison_validation.parquet`, a projekt jest gotowy do przejścia do `4.11`, czyli refitu zwycięzcy na `train + validation` oraz jednej uczciwej oceny na `test`.

## 4.11 Refit zwycięzcy i uczciwa ocena na test

In [57]:
# 4.11.1 Weryfikacja wejścia do refitu zwycięzcy i oceny na test

import pandas as pd

required_objects = [
    "step4_5_X_train_model_df",
    "step4_5_X_validation_model_df",
    "step4_5_X_test_model_df",
    "step4_5_y_train",
    "step4_5_y_validation",
    "step4_5_y_test",
    "step4_7_best_candidate_row",
    "step4_9_threshold_choice_dict",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.11.1: {missing_objects}"

step4_11_selected_model_name = str(step4_9_threshold_choice_dict["selected_model_name"])
step4_11_selected_calibration_method = str(step4_9_threshold_choice_dict["selected_calibration_method"])
step4_11_selected_threshold = float(step4_9_threshold_choice_dict["selected_threshold"])

assert step4_11_selected_model_name == "lightgbm_tuned", (
    f"4.11.1 expected selected model lightgbm_tuned, got: {step4_11_selected_model_name}"
)
assert step4_11_selected_calibration_method == "raw", (
    f"4.11.1 expected raw calibration method, got: {step4_11_selected_calibration_method}"
)

step4_11_refit_input_summary_df = pd.DataFrame(
    [
        {
            "dataset_role": "train",
            "rows": int(step4_5_X_train_model_df.shape[0]),
            "cols": int(step4_5_X_train_model_df.shape[1]),
            "positive_count": int(pd.Series(step4_5_y_train).sum()),
            "negative_count": int(len(step4_5_y_train) - pd.Series(step4_5_y_train).sum()),
        },
        {
            "dataset_role": "validation",
            "rows": int(step4_5_X_validation_model_df.shape[0]),
            "cols": int(step4_5_X_validation_model_df.shape[1]),
            "positive_count": int(pd.Series(step4_5_y_validation).sum()),
            "negative_count": int(len(step4_5_y_validation) - pd.Series(step4_5_y_validation).sum()),
        },
        {
            "dataset_role": "test",
            "rows": int(step4_5_X_test_model_df.shape[0]),
            "cols": int(step4_5_X_test_model_df.shape[1]),
            "positive_count": int(pd.Series(step4_5_y_test).sum()),
            "negative_count": int(len(step4_5_y_test) - pd.Series(step4_5_y_test).sum()),
        },
    ]
)

step4_11_refit_input_summary_df["positive_share"] = (
    step4_11_refit_input_summary_df["positive_count"]
    / step4_11_refit_input_summary_df["rows"]
)

step4_11_selected_model_summary_df = pd.DataFrame(
    [
        {
            "selected_model_name": step4_11_selected_model_name,
            "selected_calibration_method": step4_11_selected_calibration_method,
            "selected_threshold": step4_11_selected_threshold,
            "learning_rate": float(step4_7_best_candidate_row["learning_rate"]),
            "n_estimators": int(step4_7_best_candidate_row["n_estimators"]),
            "num_leaves": int(step4_7_best_candidate_row["num_leaves"]),
            "max_depth": int(step4_7_best_candidate_row["max_depth"]),
            "min_child_samples": int(step4_7_best_candidate_row["min_child_samples"]),
            "subsample": float(step4_7_best_candidate_row["subsample"]),
            "colsample_bytree": float(step4_7_best_candidate_row["colsample_bytree"]),
            "reg_alpha": float(step4_7_best_candidate_row["reg_alpha"]),
            "reg_lambda": float(step4_7_best_candidate_row["reg_lambda"]),
        }
    ]
)

display(step4_11_refit_input_summary_df)
display(step4_11_selected_model_summary_df)

assert step4_11_refit_input_summary_df["cols"].nunique() == 1, (
    "4.11.1 expected the same feature count across train, validation and test."
)
assert step4_11_refit_input_summary_df.shape[0] == 3, (
    "4.11.1 expected exactly 3 input rows: train, validation, test."
)

,dataset_role,rows,cols,positive_count,negative_count,positive_share
0,train,63833,34,52978,10855,0.829947
1,validation,70879,34,58268,12611,0.822077
2,test,75980,34,61198,14782,0.805449


,selected_model_name,selected_calibration_method,selected_threshold,learning_rate,n_estimators,num_leaves,max_depth,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda
0,lightgbm_tuned,raw,0.55,0.03,300,15,-1,60,0.8,0.8,0.0,0.0


In [58]:
# 4.11.2 Refit zwycięzcy na train + validation i uczciwa ocena na test

import numpy as np
import pandas as pd

from lightgbm import LGBMClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

required_objects = [
    "step4_5_X_train_model_df",
    "step4_5_X_validation_model_df",
    "step4_5_X_test_model_df",
    "step4_5_y_train",
    "step4_5_y_validation",
    "step4_5_y_test",
    "step4_11_selected_model_summary_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.11.2: {missing_objects}"

if "compute_binary_metrics" not in globals():
    def compute_binary_metrics(y_true, y_pred, y_score):
        y_true = pd.Series(y_true).astype(int)
        y_pred = pd.Series(y_pred).astype(int)
        y_score = pd.Series(y_score).astype(float)

        tp = int(((y_true == 1) & (y_pred == 1)).sum())
        tn = int(((y_true == 0) & (y_pred == 0)).sum())
        fp = int(((y_true == 0) & (y_pred == 1)).sum())
        fn = int(((y_true == 1) & (y_pred == 0)).sum())

        row_count = int(len(y_true))
        positive_count = int((y_true == 1).sum())
        negative_count = int((y_true == 0).sum())

        accuracy = (tp + tn) / row_count if row_count > 0 else np.nan
        precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
        recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = (
            2 * precision * recall / (precision + recall)
            if pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0
            else np.nan
        )
        balanced_accuracy = (
            (recall + specificity) / 2
            if pd.notna(recall) and pd.notna(specificity)
            else np.nan
        )
        pred_positive_rate = float(y_pred.mean()) if row_count > 0 else np.nan
        roc_auc = float(roc_auc_score(y_true, y_score)) if positive_count > 0 and negative_count > 0 else np.nan
        pr_auc = float(average_precision_score(y_true, y_score)) if positive_count > 0 else np.nan

        return {
            "row_count": row_count,
            "positive_count": positive_count,
            "negative_count": negative_count,
            "tp": tp,
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "accuracy": float(accuracy),
            "precision": float(precision),
            "recall": float(recall),
            "specificity": float(specificity),
            "f1": float(f1),
            "balanced_accuracy": float(balanced_accuracy),
            "pred_positive_rate": float(pred_positive_rate),
            "roc_auc": float(roc_auc),
            "pr_auc": float(pr_auc),
        }

step4_11_selected_model_row = step4_11_selected_model_summary_df.iloc[0]

step4_11_refit_params = {
    "objective": "binary",
    "boosting_type": "gbdt",
    "learning_rate": float(step4_11_selected_model_row["learning_rate"]),
    "n_estimators": int(step4_11_selected_model_row["n_estimators"]),
    "num_leaves": int(step4_11_selected_model_row["num_leaves"]),
    "max_depth": int(step4_11_selected_model_row["max_depth"]),
    "min_child_samples": int(step4_11_selected_model_row["min_child_samples"]),
    "subsample": float(step4_11_selected_model_row["subsample"]),
    "colsample_bytree": float(step4_11_selected_model_row["colsample_bytree"]),
    "reg_alpha": float(step4_11_selected_model_row["reg_alpha"]),
    "reg_lambda": float(step4_11_selected_model_row["reg_lambda"]),
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1,
}

step4_11_selected_threshold = float(step4_11_selected_model_row["selected_threshold"])

step4_11_X_refit_model_df = pd.concat(
    [
        step4_5_X_train_model_df,
        step4_5_X_validation_model_df,
    ],
    axis=0,
).reset_index(drop=True)

step4_11_y_refit = pd.concat(
    [
        pd.Series(step4_5_y_train),
        pd.Series(step4_5_y_validation),
    ],
    axis=0,
).reset_index(drop=True).astype(int)

step4_11_refit_model = LGBMClassifier(**step4_11_refit_params)
step4_11_refit_model.fit(
    step4_11_X_refit_model_df,
    step4_11_y_refit,
)

step4_11_test_y_score = pd.Series(
    step4_11_refit_model.predict_proba(step4_5_X_test_model_df)[:, 1],
    name="predicted_probability",
).clip(0.0, 1.0)

step4_11_test_y_pred = pd.Series(
    (step4_11_test_y_score >= step4_11_selected_threshold).astype("int8"),
    name="predicted_label",
)

step4_11_test_metrics_dict = compute_binary_metrics(
    step4_5_y_test,
    step4_11_test_y_pred,
    step4_11_test_y_score,
)

step4_11_test_metrics_dict["selected_model_name"] = str(step4_11_selected_model_row["selected_model_name"])
step4_11_test_metrics_dict["selected_calibration_method"] = str(step4_11_selected_model_row["selected_calibration_method"])
step4_11_test_metrics_dict["selected_threshold"] = step4_11_selected_threshold
step4_11_test_metrics_dict["refit_rows"] = int(step4_11_X_refit_model_df.shape[0])
step4_11_test_metrics_dict["refit_positive_count"] = int(step4_11_y_refit.sum())
step4_11_test_metrics_dict["refit_negative_count"] = int(len(step4_11_y_refit) - step4_11_y_refit.sum())

step4_11_test_metrics_df = pd.DataFrame([step4_11_test_metrics_dict])

step4_11_predictions_test_df = pd.DataFrame(
    {
        "y_true": pd.Series(step4_5_y_test).astype(int).reset_index(drop=True),
        "predicted_probability": step4_11_test_y_score.reset_index(drop=True),
        "predicted_label": step4_11_test_y_pred.reset_index(drop=True),
        "selected_threshold": step4_11_selected_threshold,
        "selected_model_name": str(step4_11_selected_model_row["selected_model_name"]),
        "selected_calibration_method": str(step4_11_selected_model_row["selected_calibration_method"]),
    }
)

display(step4_11_test_metrics_df)
display(step4_11_predictions_test_df.head(10))

assert step4_11_test_metrics_df.shape[0] == 1, (
    "4.11.2 expected exactly one test metrics row."
)
assert step4_11_predictions_test_df.shape[0] == step4_5_X_test_model_df.shape[0], (
    "4.11.2 expected one prediction row per test record."
)

,row_count,positive_count,negative_count,tp,tn,fp,fn,accuracy,precision,recall,...,balanced_accuracy,pred_positive_rate,roc_auc,pr_auc,selected_model_name,selected_calibration_method,selected_threshold,refit_rows,refit_positive_count,refit_negative_count
0,75980,61198,14782,59178,2110,12672,2020,0.806633,0.823633,0.966992,...,0.554867,0.945644,0.732529,0.915951,lightgbm_tuned,raw,0.55,134712,111246,23466


,y_true,predicted_probability,predicted_label,selected_threshold,selected_model_name,selected_calibration_method
0,0,0.290928,0,0.55,lightgbm_tuned,raw
1,1,0.290928,0,0.55,lightgbm_tuned,raw
2,1,0.290928,0,0.55,lightgbm_tuned,raw
3,1,0.271971,0,0.55,lightgbm_tuned,raw
4,0,0.290928,0,0.55,lightgbm_tuned,raw
5,1,0.290928,0,0.55,lightgbm_tuned,raw
6,1,0.290928,0,0.55,lightgbm_tuned,raw
7,1,0.290928,0,0.55,lightgbm_tuned,raw
8,0,0.290928,0,0.55,lightgbm_tuned,raw
9,1,0.290928,0,0.55,lightgbm_tuned,raw


In [59]:
# 4.11.3 Zapis artefaktów sekcji 4.11

from pathlib import Path
import json

import pandas as pd

required_objects = [
    "step4_3_project_root",
    "step4_11_test_metrics_df",
    "step4_11_predictions_test_df",
    "step4_11_selected_model_summary_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.11.3: {missing_objects}"

step4_11_output_dir = step4_3_project_root / "artifacts" / "4_test_evaluation"
step4_11_output_dir.mkdir(parents=True, exist_ok=True)

step4_11_test_metrics_path = step4_11_output_dir / "b4_11_test_metrics.parquet"
step4_11_predictions_test_path = step4_11_output_dir / "b4_11_predictions_test.parquet"
step4_11_selected_model_path = step4_11_output_dir / "b4_11_selected_model.json"

step4_11_test_metrics_save_df = (
    step4_11_test_metrics_df.copy()
    .reset_index(drop=True)
)

step4_11_predictions_test_save_df = (
    step4_11_predictions_test_df.copy()
    .reset_index(drop=True)
)

step4_11_test_metrics_save_df.to_parquet(
    step4_11_test_metrics_path,
    index=False,
)

step4_11_predictions_test_save_df.to_parquet(
    step4_11_predictions_test_path,
    index=False,
)

step4_11_selected_model_row = step4_11_selected_model_summary_df.iloc[0]

step4_11_selected_model_dict = {
    "selected_model_name": str(step4_11_selected_model_row["selected_model_name"]),
    "selected_calibration_method": str(step4_11_selected_model_row["selected_calibration_method"]),
    "selected_threshold": float(step4_11_selected_model_row["selected_threshold"]),
    "model_params": {
        "learning_rate": float(step4_11_selected_model_row["learning_rate"]),
        "n_estimators": int(step4_11_selected_model_row["n_estimators"]),
        "num_leaves": int(step4_11_selected_model_row["num_leaves"]),
        "max_depth": int(step4_11_selected_model_row["max_depth"]),
        "min_child_samples": int(step4_11_selected_model_row["min_child_samples"]),
        "subsample": float(step4_11_selected_model_row["subsample"]),
        "colsample_bytree": float(step4_11_selected_model_row["colsample_bytree"]),
        "reg_alpha": float(step4_11_selected_model_row["reg_alpha"]),
        "reg_lambda": float(step4_11_selected_model_row["reg_lambda"]),
    },
    "artifacts": {
        "test_metrics": str(step4_11_test_metrics_path),
        "predictions_test": str(step4_11_predictions_test_path),
    },
}

with open(step4_11_selected_model_path, "w", encoding="utf-8") as f:
    json.dump(step4_11_selected_model_dict, f, ensure_ascii=False, indent=2)

step4_11_saved_artifacts_registry_df = pd.DataFrame(
    [
        {
            "artifact_name": step4_11_test_metrics_path.name,
            "rows": int(step4_11_test_metrics_save_df.shape[0]),
            "cols": int(step4_11_test_metrics_save_df.shape[1]),
            "path": str(step4_11_test_metrics_path),
        },
        {
            "artifact_name": step4_11_predictions_test_path.name,
            "rows": int(step4_11_predictions_test_save_df.shape[0]),
            "cols": int(step4_11_predictions_test_save_df.shape[1]),
            "path": str(step4_11_predictions_test_path),
        },
        {
            "artifact_name": step4_11_selected_model_path.name,
            "rows": 1,
            "cols": len(step4_11_selected_model_dict),
            "path": str(step4_11_selected_model_path),
        },
    ]
)

display(step4_11_saved_artifacts_registry_df)

assert step4_11_test_metrics_path.exists(), (
    "4.11.3 failed to save b4_11_test_metrics.parquet"
)
assert step4_11_predictions_test_path.exists(), (
    "4.11.3 failed to save b4_11_predictions_test.parquet"
)
assert step4_11_selected_model_path.exists(), (
    "4.11.3 failed to save b4_11_selected_model.json"
)
assert step4_11_test_metrics_save_df.shape[0] == 1, (
    "4.11.3 expected 1 row in saved test metrics artifact."
)
assert step4_11_predictions_test_save_df.shape[0] == 75980, (
    "4.11.3 expected 75980 rows in saved test predictions artifact."
)

,artifact_name,rows,cols,path
0,b4_11_test_metrics.parquet,1,22,/root/projects/6_samodzielny_projekt_koncowy_w...
1,b4_11_predictions_test.parquet,75980,6,/root/projects/6_samodzielny_projekt_koncowy_w...
2,b4_11_selected_model.json,1,5,/root/projects/6_samodzielny_projekt_koncowy_w...


## Podsumowanie działu 4.11 Refit zwycięzcy i uczciwa ocena na test

**Kontekst inżynieryjny:** W dziale `4.11` przygotowano finalny etap uczciwej oceny zwycięskiego modelu. Najpierw zweryfikowano spójność wejścia dla `train`, `validation` i `test`, potwierdzono ostateczny wybór modelu `lightgbm_tuned` w wariancie `raw` z progiem `0.55` oraz zamrożono zestaw hiperparametrów zwycięzcy. Następnie wykonano refit modelu na połączonym zbiorze `train + validation`, po czym przeprowadzono jedną, końcową ocenę na `test`. Na końcu zapisano artefakty: `b4_11_test_metrics.parquet`, `b4_11_predictions_test.parquet` oraz `b4_11_selected_model.json`.

**Interpretacja wyniku:** Wejście do refitu było spójne: wszystkie trzy zbiory miały po `34` cechy modelowe, a finalny model został oceniony na `75980` rekordach testowych. Po reficie na `train + validation` model `lightgbm_tuned` z progiem `0.55` osiągnął na `test`: `accuracy = 0.806633`, `precision = 0.823633`, `recall = 0.966992`, `balanced_accuracy = 0.554867`, `ROC AUC = 0.732529` oraz `PR AUC = 0.915951`. Macierz pomyłek na `test` wyniosła: `TP = 59178`, `TN = 2110`, `FP = 12672`, `FN = 2020`. Wynik potwierdza, że model zachował główną własność projektu — bardzo wysokie wychwytywanie klasy pozytywnej — przy jednoczesnym szerokim profilu predykcji dodatniej.

**Znaczenie biznesowe:** Test potwierdził, że wybrany model nie rozjechał się po reficie i zachowuje logicznie spójny profil względem wcześniejszych etapów `validation`. Ostateczny model nadal działa w reżimie wysokiego `recall`, co jest zgodne z wcześniej przyjętą macierzą kosztów biznesowych silnie karzącą `false negatives`. Jednocześnie wynik testowy pokazuje, że model pozostaje agresywny operacyjnie i generuje szeroki strumień predykcji dodatnich. Oznacza to, że projekt dostarczył świadomie wybrany kompromis: wysoka skuteczność wykrywania alertów kosztem ograniczonej selektywności.

**Wniosek:** Dział `4.11` został wykonany poprawnie i zakończył proces uczciwej oceny finalnego modelu na `test`. Zwycięzcą projektu pozostaje `lightgbm_tuned` w wariancie `raw` z progiem `0.55`, po reficie na `train + validation`. Zapisano komplet artefaktów sekcji, a projekt jest gotowy do przejścia do `4.12`, czyli analizy stabilności modelu w czasie oraz dalszych raportów przekrojowych.

## 4.12 Stabilność modelu w czasie

In [60]:
# 4.12.1 Wejście do analizy stabilności modelu w czasie

import pandas as pd

required_objects = [
    "step4_5_test_df",
    "step4_11_predictions_test_df",
    "step4_11_test_metrics_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.12.1: {missing_objects}"

step4_12_required_test_cols = [
    "activity_date",
    "station_id",
    "target_alert_day",
    "point4_split_name",
]
step4_12_missing_test_cols = [
    col for col in step4_12_required_test_cols
    if col not in step4_5_test_df.columns
]
assert not step4_12_missing_test_cols, (
    f"Missing required test columns in 4.12.1: {step4_12_missing_test_cols}"
)

step4_12_temporal_input_df = pd.concat(
    [
        step4_5_test_df[
            [
                "activity_date",
                "station_id",
                "target_alert_day",
                "point4_split_name",
            ]
        ].reset_index(drop=True),
        step4_11_predictions_test_df[
            [
                "predicted_probability",
                "predicted_label",
                "selected_threshold",
                "selected_model_name",
                "selected_calibration_method",
            ]
        ].reset_index(drop=True),
    ],
    axis=1,
)

step4_12_temporal_input_df["activity_date"] = pd.to_datetime(
    step4_12_temporal_input_df["activity_date"]
)
step4_12_temporal_input_df["year_month"] = (
    step4_12_temporal_input_df["activity_date"].dt.to_period("M").astype(str)
)
step4_12_temporal_input_df["year"] = (
    step4_12_temporal_input_df["activity_date"].dt.year.astype("int16")
)
step4_12_temporal_input_df["month"] = (
    step4_12_temporal_input_df["activity_date"].dt.month.astype("int8")
)
step4_12_temporal_input_df["quarter"] = (
    step4_12_temporal_input_df["activity_date"].dt.to_period("Q").astype(str)
)

step4_12_temporal_input_summary_df = pd.DataFrame(
    [
        {
            "row_count": int(step4_12_temporal_input_df.shape[0]),
            "col_count": int(step4_12_temporal_input_df.shape[1]),
            "unique_station_count": int(step4_12_temporal_input_df["station_id"].nunique()),
            "unique_date_count": int(step4_12_temporal_input_df["activity_date"].nunique()),
            "min_activity_date": step4_12_temporal_input_df["activity_date"].min(),
            "max_activity_date": step4_12_temporal_input_df["activity_date"].max(),
            "unique_year_month_count": int(step4_12_temporal_input_df["year_month"].nunique()),
            "unique_quarter_count": int(step4_12_temporal_input_df["quarter"].nunique()),
            "test_split_only": int(step4_12_temporal_input_df["point4_split_name"].eq("test").all()),
        }
    ]
)

step4_12_month_profile_df = (
    step4_12_temporal_input_df
    .groupby("year_month", dropna=False)
    .agg(
        row_count=("target_alert_day", "size"),
        positive_count=("target_alert_day", "sum"),
        mean_predicted_probability=("predicted_probability", "mean"),
        pred_positive_rate=("predicted_label", "mean"),
        unique_station_count=("station_id", "nunique"),
        min_activity_date=("activity_date", "min"),
        max_activity_date=("activity_date", "max"),
    )
    .reset_index()
    .sort_values("year_month")
    .reset_index(drop=True)
)

step4_12_month_profile_df["positive_share"] = (
    step4_12_month_profile_df["positive_count"]
    / step4_12_month_profile_df["row_count"]
)

display(step4_12_temporal_input_summary_df)
display(step4_12_month_profile_df)

assert int(step4_12_temporal_input_summary_df.loc[0, "test_split_only"]) == 1, (
    "4.12.1 expected only test rows in temporal stability input."
)
assert step4_12_temporal_input_df.shape[0] == step4_11_predictions_test_df.shape[0], (
    "4.12.1 temporal input row count mismatch vs saved test predictions."
)

,row_count,col_count,unique_station_count,unique_date_count,min_activity_date,max_activity_date,unique_year_month_count,unique_quarter_count,test_split_only
0,75980,13,344,223,2020-03-23,2020-10-31,8,4,1


,year_month,row_count,positive_count,mean_predicted_probability,pred_positive_rate,unique_station_count,min_activity_date,max_activity_date,positive_share
0,2020-03,2828,2089,0.667911,0.829208,331,2020-03-23,2020-03-31,0.738685
1,2020-04,10042,8198,0.774513,0.944832,336,2020-04-01,2020-04-30,0.816371
2,2020-05,10564,8801,0.814543,0.966869,343,2020-05-01,2020-05-31,0.833112
3,2020-06,10290,8465,0.823725,0.963460,343,2020-06-01,2020-06-30,0.822643
4,2020-07,10656,8608,0.812306,0.952703,344,2020-07-01,2020-07-31,0.807808
5,2020-08,10664,8782,0.821265,0.960615,344,2020-08-01,2020-08-31,0.823518
6,2020-09,10320,8298,0.806374,0.947190,344,2020-09-01,2020-09-30,0.804070
7,2020-10,10616,7957,0.771251,0.915411,344,2020-10-01,2020-10-31,0.749529


In [61]:
# 4.12.2 Metryki modelu per miesiąc na test

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

required_objects = [
    "step4_12_temporal_input_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.12.2: {missing_objects}"

def compute_binary_metrics(y_true, y_pred, y_score):
    y_true = pd.Series(y_true).astype(int)
    y_pred = pd.Series(y_pred).astype(int)
    y_score = pd.Series(y_score).astype(float)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    row_count = int(len(y_true))
    positive_count = int((y_true == 1).sum())
    negative_count = int((y_true == 0).sum())

    accuracy = (tp + tn) / row_count if row_count > 0 else np.nan
    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    f1 = (
        2 * precision * recall / (precision + recall)
        if pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0
        else np.nan
    )
    balanced_accuracy = (
        (recall + specificity) / 2
        if pd.notna(recall) and pd.notna(specificity)
        else np.nan
    )
    pred_positive_rate = float(y_pred.mean()) if row_count > 0 else np.nan
    roc_auc = float(roc_auc_score(y_true, y_score)) if positive_count > 0 and negative_count > 0 else np.nan
    pr_auc = float(average_precision_score(y_true, y_score)) if positive_count > 0 else np.nan
    brier_score = float(brier_score_loss(y_true, y_score)) if row_count > 0 else np.nan

    return {
        "row_count": row_count,
        "positive_count": positive_count,
        "negative_count": negative_count,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "specificity": float(specificity),
        "f1": float(f1),
        "balanced_accuracy": float(balanced_accuracy),
        "pred_positive_rate": float(pred_positive_rate),
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
        "brier_score": float(brier_score),
    }

step4_12_month_metric_rows = []

for step4_12_year_month, step4_12_group_df in (
    step4_12_temporal_input_df
    .sort_values(["activity_date", "station_id"])
    .groupby("year_month", dropna=False)
):
    step4_12_metric_row = compute_binary_metrics(
        step4_12_group_df["target_alert_day"],
        step4_12_group_df["predicted_label"],
        step4_12_group_df["predicted_probability"],
    )
    step4_12_metric_row["year_month"] = str(step4_12_year_month)
    step4_12_metric_row["min_activity_date"] = step4_12_group_df["activity_date"].min()
    step4_12_metric_row["max_activity_date"] = step4_12_group_df["activity_date"].max()
    step4_12_metric_row["positive_share"] = float(step4_12_group_df["target_alert_day"].mean())
    step4_12_metric_row["mean_predicted_probability"] = float(step4_12_group_df["predicted_probability"].mean())
    step4_12_metric_row["unique_station_count"] = int(step4_12_group_df["station_id"].nunique())
    step4_12_month_metric_rows.append(step4_12_metric_row)

step4_12_month_metrics_df = (
    pd.DataFrame(step4_12_month_metric_rows)
    .sort_values("year_month")
    .reset_index(drop=True)
)

display(step4_12_month_metrics_df)

assert step4_12_month_metrics_df.shape[0] == step4_12_temporal_input_df["year_month"].nunique(), (
    "4.12.2 expected one metrics row per year_month in test temporal input."
)

,row_count,positive_count,negative_count,tp,tn,fp,fn,accuracy,precision,recall,...,pred_positive_rate,roc_auc,pr_auc,brier_score,year_month,min_activity_date,max_activity_date,positive_share,mean_predicted_probability,unique_station_count
0,2828,2089,739,1847,241,498,242,0.738331,0.787633,0.884155,...,0.829208,0.683166,0.839818,0.183098,2020-03,2020-03-23,2020-03-31,0.738685,0.667911,331
1,10042,8198,1844,7908,264,1580,290,0.813782,0.833474,0.964626,...,0.944832,0.721728,0.919052,0.137546,2020-04,2020-04-01,2020-04-30,0.816371,0.774513,336
2,10564,8801,1763,8608,157,1606,193,0.829705,0.842765,0.978071,...,0.966869,0.733572,0.930002,0.126065,2020-05,2020-05-01,2020-05-31,0.833112,0.814543,343
3,10290,8465,1825,8281,192,1633,184,0.823421,0.835283,0.978263,...,0.963460,0.738450,0.925339,0.129993,2020-06,2020-06-01,2020-06-30,0.822643,0.823725,343
4,10656,8608,2048,8370,266,1782,238,0.810435,0.824468,0.972351,...,0.952703,0.728887,0.914100,0.138806,2020-07,2020-07-01,2020-07-31,0.807808,0.812306,344
5,10664,8782,1882,8570,208,1674,212,0.823143,0.836587,0.975860,...,0.960615,0.737012,0.925696,0.129845,2020-08,2020-08-01,2020-08-31,0.823518,0.821265,344
6,10320,8298,2022,8034,281,1741,264,0.805717,0.821893,0.968185,...,0.947190,0.740768,0.920390,0.139217,2020-09,2020-09-01,2020-09-30,0.804070,0.806374,344
7,10616,7957,2659,7560,501,2158,397,0.759326,0.777938,0.950107,...,0.915411,0.729671,0.886730,0.164813,2020-10,2020-10-01,2020-10-31,0.749529,0.771251,344


In [62]:
# 4.12.3 Metryki modelu po reżimach sezonowych na test

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

required_objects = [
    "step4_12_temporal_input_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.12.3: {missing_objects}"

def compute_binary_metrics(y_true, y_pred, y_score):
    y_true = pd.Series(y_true).astype(int)
    y_pred = pd.Series(y_pred).astype(int)
    y_score = pd.Series(y_score).astype(float)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    row_count = int(len(y_true))
    positive_count = int((y_true == 1).sum())
    negative_count = int((y_true == 0).sum())

    accuracy = (tp + tn) / row_count if row_count > 0 else np.nan
    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    f1 = (
        2 * precision * recall / (precision + recall)
        if pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0
        else np.nan
    )
    balanced_accuracy = (
        (recall + specificity) / 2
        if pd.notna(recall) and pd.notna(specificity)
        else np.nan
    )
    pred_positive_rate = float(y_pred.mean()) if row_count > 0 else np.nan
    roc_auc = float(roc_auc_score(y_true, y_score)) if positive_count > 0 and negative_count > 0 else np.nan
    pr_auc = float(average_precision_score(y_true, y_score)) if positive_count > 0 else np.nan
    brier_score = float(brier_score_loss(y_true, y_score)) if row_count > 0 else np.nan

    return {
        "row_count": row_count,
        "positive_count": positive_count,
        "negative_count": negative_count,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "specificity": float(specificity),
        "f1": float(f1),
        "balanced_accuracy": float(balanced_accuracy),
        "pred_positive_rate": float(pred_positive_rate),
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
        "brier_score": float(brier_score),
    }

step4_12_regime_input_df = step4_12_temporal_input_df.copy()

step4_12_regime_input_df["season_regime"] = np.select(
    [
        step4_12_regime_input_df["month"].eq(3),
        step4_12_regime_input_df["month"].isin([4, 5, 6, 7, 8, 9]),
        step4_12_regime_input_df["month"].eq(10),
    ],
    [
        "season_start_edge",
        "season_core",
        "season_end_edge",
    ],
    default="outside_defined_regime",
)

step4_12_regime_metric_rows = []

for step4_12_regime_name, step4_12_group_df in (
    step4_12_regime_input_df
    .sort_values(["activity_date", "station_id"])
    .groupby("season_regime", dropna=False)
):
    step4_12_metric_row = compute_binary_metrics(
        step4_12_group_df["target_alert_day"],
        step4_12_group_df["predicted_label"],
        step4_12_group_df["predicted_probability"],
    )
    step4_12_metric_row["season_regime"] = str(step4_12_regime_name)
    step4_12_metric_row["min_activity_date"] = step4_12_group_df["activity_date"].min()
    step4_12_metric_row["max_activity_date"] = step4_12_group_df["activity_date"].max()
    step4_12_metric_row["positive_share"] = float(step4_12_group_df["target_alert_day"].mean())
    step4_12_metric_row["mean_predicted_probability"] = float(step4_12_group_df["predicted_probability"].mean())
    step4_12_metric_row["unique_station_count"] = int(step4_12_group_df["station_id"].nunique())
    step4_12_regime_metric_rows.append(step4_12_metric_row)

step4_12_regime_metrics_df = (
    pd.DataFrame(step4_12_regime_metric_rows)
    .sort_values("season_regime")
    .reset_index(drop=True)
)

display(step4_12_regime_metrics_df)

assert set(step4_12_regime_metrics_df["season_regime"].unique()) == {
    "season_core",
    "season_end_edge",
    "season_start_edge",
}, "4.12.3 expected exactly 3 seasonal regimes in the test window."

,row_count,positive_count,negative_count,tp,tn,fp,fn,accuracy,precision,recall,...,pred_positive_rate,roc_auc,pr_auc,brier_score,season_regime,min_activity_date,max_activity_date,positive_share,mean_predicted_probability,unique_station_count
0,62536,51152,11384,49771,1368,10016,1381,0.817753,0.832472,0.973002,...,0.956041,0.732547,0.922050,0.133541,season_core,2020-04-01,2020-09-30,0.817961,0.809043,344
1,10616,7957,2659,7560,501,2158,397,0.759326,0.777938,0.950107,...,0.915411,0.729671,0.886730,0.164813,season_end_edge,2020-10-01,2020-10-31,0.749529,0.771251,344
2,2828,2089,739,1847,241,498,242,0.738331,0.787633,0.884155,...,0.829208,0.683166,0.839818,0.183098,season_start_edge,2020-03-23,2020-03-31,0.738685,0.667911,331


In [64]:
# 4.12.4 Scalenie stabilności czasowej i driftu jakości modelu

import pandas as pd
from sklearn.metrics import brier_score_loss

required_objects = [
    "step4_12_month_metrics_df",
    "step4_12_regime_metrics_df",
    "step4_11_test_metrics_df",
    "step4_11_predictions_test_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.12.4: {missing_objects}"

step4_12_test_anchor_row = step4_11_test_metrics_df.iloc[0]

step4_12_test_brier_score = float(
    brier_score_loss(
        step4_11_predictions_test_df["y_true"].astype(int),
        step4_11_predictions_test_df["predicted_probability"].astype(float),
    )
)

step4_12_test_positive_share = float(
    step4_11_predictions_test_df["y_true"].astype(int).mean()
)

step4_12_month_stability_df = (
    step4_12_month_metrics_df.copy()
    .rename(
        columns={
            "year_month": "bucket_name",
            "min_activity_date": "bucket_start_date",
            "max_activity_date": "bucket_end_date",
        }
    )
)
step4_12_month_stability_df["analysis_level"] = "month"

step4_12_regime_stability_df = (
    step4_12_regime_metrics_df.copy()
    .rename(
        columns={
            "season_regime": "bucket_name",
            "min_activity_date": "bucket_start_date",
            "max_activity_date": "bucket_end_date",
        }
    )
)
step4_12_regime_stability_df["analysis_level"] = "season_regime"

step4_12_temporal_stability_df = pd.concat(
    [
        step4_12_month_stability_df,
        step4_12_regime_stability_df,
    ],
    axis=0,
    ignore_index=True,
)

step4_12_temporal_stability_df["recall_vs_test_delta"] = (
    step4_12_temporal_stability_df["recall"] - float(step4_12_test_anchor_row["recall"])
)
step4_12_temporal_stability_df["specificity_vs_test_delta"] = (
    step4_12_temporal_stability_df["specificity"] - float(step4_12_test_anchor_row["specificity"])
)
step4_12_temporal_stability_df["balanced_accuracy_vs_test_delta"] = (
    step4_12_temporal_stability_df["balanced_accuracy"] - float(step4_12_test_anchor_row["balanced_accuracy"])
)
step4_12_temporal_stability_df["pr_auc_vs_test_delta"] = (
    step4_12_temporal_stability_df["pr_auc"] - float(step4_12_test_anchor_row["pr_auc"])
)
step4_12_temporal_stability_df["brier_vs_test_delta"] = (
    step4_12_temporal_stability_df["brier_score"] - step4_12_test_brier_score
)
step4_12_temporal_stability_df["pred_positive_rate_vs_test_delta"] = (
    step4_12_temporal_stability_df["pred_positive_rate"] - float(step4_12_test_anchor_row["pred_positive_rate"])
)
step4_12_temporal_stability_df["positive_share_vs_test_delta"] = (
    step4_12_temporal_stability_df["positive_share"] - step4_12_test_positive_share
)

step4_12_temporal_stability_df = (
    step4_12_temporal_stability_df[
        [
            "analysis_level",
            "bucket_name",
            "bucket_start_date",
            "bucket_end_date",
            "row_count",
            "positive_count",
            "negative_count",
            "positive_share",
            "mean_predicted_probability",
            "pred_positive_rate",
            "accuracy",
            "precision",
            "recall",
            "specificity",
            "f1",
            "balanced_accuracy",
            "roc_auc",
            "pr_auc",
            "brier_score",
            "recall_vs_test_delta",
            "specificity_vs_test_delta",
            "balanced_accuracy_vs_test_delta",
            "pr_auc_vs_test_delta",
            "brier_vs_test_delta",
            "pred_positive_rate_vs_test_delta",
            "positive_share_vs_test_delta",
            "unique_station_count",
        ]
    ]
    .sort_values(["analysis_level", "bucket_start_date", "bucket_name"])
    .reset_index(drop=True)
)

display(step4_12_temporal_stability_df)

assert step4_12_temporal_stability_df.shape[0] == (
    step4_12_month_metrics_df.shape[0] + step4_12_regime_metrics_df.shape[0]
), "4.12.4 expected month + regime rows in temporal stability table."

,analysis_level,bucket_name,bucket_start_date,bucket_end_date,row_count,positive_count,negative_count,positive_share,mean_predicted_probability,pred_positive_rate,...,pr_auc,brier_score,recall_vs_test_delta,specificity_vs_test_delta,balanced_accuracy_vs_test_delta,pr_auc_vs_test_delta,brier_vs_test_delta,pred_positive_rate_vs_test_delta,positive_share_vs_test_delta,unique_station_count
0,month,2020-03,2020-03-23,2020-03-31,2828,2089,739,0.738685,0.667911,0.829208,...,0.839818,0.183098,-0.082837,0.183375,0.050269,-0.076133,0.043343,-0.116436,-0.066764,331
1,month,2020-04,2020-04-01,2020-04-30,10042,8198,1844,0.816371,0.774513,0.944832,...,0.919052,0.137546,-0.002367,0.000426,-0.000971,0.003101,-0.002209,-0.000812,0.010922,336
2,month,2020-05,2020-05-01,2020-05-31,10564,8801,1763,0.833112,0.814543,0.966869,...,0.930002,0.126065,0.011078,-0.053688,-0.021305,0.014051,-0.013690,0.021225,0.027664,343
3,month,2020-06,2020-06-01,2020-06-30,10290,8465,1825,0.822643,0.823725,0.963460,...,0.925339,0.129993,0.011271,-0.037536,-0.013132,0.009388,-0.009762,0.017816,0.017195,343
4,month,2020-07,2020-07-01,2020-07-31,10656,8608,2048,0.807808,0.812306,0.952703,...,0.914100,0.138806,0.005359,-0.012858,-0.003750,-0.001851,-0.000949,0.007059,0.002359,344
5,month,2020-08,2020-08-01,2020-08-31,10664,8782,1882,0.823518,0.821265,0.960615,...,0.925696,0.129845,0.008867,-0.032220,-0.011677,0.009745,-0.009910,0.014972,0.018070,344
6,month,2020-09,2020-09-01,2020-09-30,10320,8298,2022,0.804070,0.806374,0.947190,...,0.920390,0.139217,0.001193,-0.003770,-0.001289,0.004439,-0.000538,0.001546,-0.001379,344
7,month,2020-10,2020-10-01,2020-10-31,10616,7957,2659,0.749529,0.771251,0.915411,...,0.886730,0.164813,-0.016886,0.045676,0.014395,-0.029221,0.025058,-0.030233,-0.055920,344
8,season_regime,season_start_edge,2020-03-23,2020-03-31,2828,2089,739,0.738685,0.667911,0.829208,...,0.839818,0.183098,-0.082837,0.183375,0.050269,-0.076133,0.043343,-0.116436,-0.066764,331
9,season_regime,season_core,2020-04-01,2020-09-30,62536,51152,11384,0.817961,0.809043,0.956041,...,0.922050,0.133541,0.006010,-0.022573,-0.008281,0.006099,-0.006214,0.010398,0.012512,344


In [65]:
# 4.12.5 Zapis artefaktu sekcji 4.12

from pathlib import Path

import pandas as pd

required_objects = [
    "step4_3_project_root",
    "step4_12_temporal_stability_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.12.5: {missing_objects}"

step4_12_output_dir = step4_3_project_root / "artifacts" / "4_temporal_stability"
step4_12_output_dir.mkdir(parents=True, exist_ok=True)

step4_12_temporal_stability_path = (
    step4_12_output_dir / "b4_12_temporal_stability.parquet"
)

step4_12_temporal_stability_save_df = (
    step4_12_temporal_stability_df.copy()
    .reset_index(drop=True)
)

step4_12_temporal_stability_save_df.to_parquet(
    step4_12_temporal_stability_path,
    index=False,
)

step4_12_saved_artifacts_registry_df = pd.DataFrame(
    [
        {
            "artifact_name": step4_12_temporal_stability_path.name,
            "rows": int(step4_12_temporal_stability_save_df.shape[0]),
            "cols": int(step4_12_temporal_stability_save_df.shape[1]),
            "path": str(step4_12_temporal_stability_path),
        }
    ]
)

display(step4_12_saved_artifacts_registry_df)

assert step4_12_temporal_stability_path.exists(), (
    "4.12.5 failed to save b4_12_temporal_stability.parquet"
)
assert step4_12_temporal_stability_save_df.shape[0] == 11, (
    "4.12.5 expected 11 rows in saved temporal stability artifact."
)

,artifact_name,rows,cols,path
0,b4_12_temporal_stability.parquet,11,27,/root/projects/6_samodzielny_projekt_koncowy_w...


## Podsumowanie działu 4.12 Stabilność modelu w czasie

**Kontekst inżynieryjny:** W dziale `4.12` zbudowano analizę stabilności czasowej finalnego modelu na zbiorze `test`. Najpierw przygotowano wejście temporalne przez połączenie informacji identyfikacyjnych (`activity_date`, `station_id`, `target_alert_day`, split testowy) z zapisanymi predykcjami testowymi modelu. Następnie policzono profile miesięczne oraz profile po reżimach sezonowych, a na końcu scalono wyniki do jednej tabeli stabilności zawierającej metryki jakości oraz odchylenia względem całego wyniku testowego. Zapisano artefakt `b4_12_temporal_stability.parquet`.

**Interpretacja wyniku:** Analiza objęła `75980` rekordów testowych, `344` stacje, `223` dni i okres od `2020-03-23` do `2020-10-31`. W ujęciu miesięcznym model utrzymywał bardzo wysoki `recall`, ale jakość nie była jednakowa w całym okresie. Najlepszy profil występował w miesiącach rdzenia sezonu rowerowego, natomiast słabsze wyniki pojawiły się na krawędziach sezonu, szczególnie w marcu i październiku. W ujęciu reżimowym `season_core` (`2020-04-01` do `2020-09-30`) osiągnął najlepszą jakość, z `recall = 0.973002`, `PR AUC = 0.922050` i `Brier Score = 0.133541`. `season_start_edge` był najsłabszy jakościowo (`recall = 0.884155`, `PR AUC = 0.839818`, `Brier Score = 0.183098`), a `season_end_edge` również był słabszy od rdzenia sezonu (`recall = 0.950107`, `PR AUC = 0.886730`, `Brier Score = 0.164813`). Wyniki wskazują na sezonowy drift jakości modelu, a nie na losową niestabilność.

**Znaczenie biznesowe:** Dział `4.12` pokazał, że finalny model najlepiej pracuje w warunkach typowego, pełnego sezonu rowerowego, czyli w okresie o najbardziej regularnym i przewidywalnym wzorcu operacyjnym. Pogorszenie jakości na początku i końcu sezonu oznacza, że model jest bardziej wrażliwy na zmianę reżimu niż na sam upływ czasu. Jest to ważna informacja biznesowa, ponieważ wskazuje, że końcowa jakość modelu powinna być interpretowana nie tylko globalnie, ale również sezonowo. Taki wynik nie podważa modelu, lecz porządkuje jego realny zakres działania i pokazuje, gdzie należy spodziewać się większego ryzyka błędu.

**Wniosek:** Dział `4.12` został wykonany poprawnie i potwierdził, że model jest stabilny kierunkowo w czasie, ale nie jest jednakowo mocny w każdym reżimie sezonowym. Najlepsze wyniki osiąga w rdzeniu sezonu, a słabiej działa na jego wejściu i wyjściu. Zapisano artefakt `b4_12_temporal_stability.parquet`, a projekt jest gotowy do przejścia do `4.13`, czyli raportów przekrojowych typu huby vs reszta i innych segmentów biznesowych.

## 4.13 Huby vs reszta i raporty przekrojowe

In [68]:
# 4.13.1 Weryfikacja wejścia do slicingu biznesowego i budowa ramki segmentacyjnej testu

from pathlib import Path
import pandas as pd

required_objects = [
    "step4_3_project_root",
    "step4_5_test_df",
    "step4_11_predictions_test_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.13.1: {missing_objects}"

assert step4_5_test_df.shape[0] == step4_11_predictions_test_df.shape[0], (
    "4.13.1 test feature rows must match saved test predictions rows."
)

step4_13_data_dir = step4_3_project_root / "data"
assert step4_13_data_dir.exists(), f"Missing data dir in 4.13.1: {step4_13_data_dir}"

def resolve_first_existing(columns, candidates):
    for candidate in candidates:
        if candidate in columns:
            return candidate
    return None

step4_13_base_test_df = step4_5_test_df.copy()
step4_13_base_test_df.columns = [str(col).strip() for col in step4_13_base_test_df.columns]

step4_13_required_test_cols = [
    "activity_date",
    "station_id",
    "target_alert_day",
    "point4_split_name",
]
step4_13_missing_test_cols = [
    col for col in step4_13_required_test_cols
    if col not in step4_13_base_test_df.columns
]
assert not step4_13_missing_test_cols, (
    f"Missing required test columns in 4.13.1: {step4_13_missing_test_cols}"
)

step4_13_base_test_df["activity_date"] = pd.to_datetime(
    step4_13_base_test_df["activity_date"],
    errors="coerce",
)
step4_13_base_test_df["station_id"] = (
    step4_13_base_test_df["station_id"].astype("string").str.strip()
)
step4_13_base_test_df["representative_station_id"] = (
    step4_13_base_test_df["station_id"].astype("string").str.strip()
)

step4_13_test_columns = pd.Index(step4_13_base_test_df.columns.astype("string"))

step4_13_hub_col = resolve_first_existing(
    step4_13_test_columns,
    ["hub_flag", "is_hub"],
)
step4_13_cold_start_col = resolve_first_existing(
    step4_13_test_columns,
    ["is_cold_start"],
)

assert step4_13_hub_col is not None, "4.13.1 could not resolve hub flag in test dataset."
assert step4_13_cold_start_col is not None, "4.13.1 could not resolve cold start flag in test dataset."

step4_13_slice_input_df = pd.concat(
    [
        step4_13_base_test_df[
            [
                "activity_date",
                "station_id",
                "target_alert_day",
                "point4_split_name",
                "representative_station_id",
                step4_13_hub_col,
                step4_13_cold_start_col,
            ]
        ].reset_index(drop=True),
        step4_11_predictions_test_df[
            [
                "predicted_probability",
                "predicted_label",
                "selected_threshold",
                "selected_model_name",
                "selected_calibration_method",
            ]
        ].reset_index(drop=True),
    ],
    axis=1,
)

step4_13_slice_input_df = step4_13_slice_input_df.rename(
    columns={
        step4_13_hub_col: "hub_flag_slice",
        step4_13_cold_start_col: "is_cold_start_slice",
    }
)

# łącznik stacja -> physical_location_id
step4_13_station_location_master_df = pd.read_parquet(
    step4_13_data_dir / "station_location_master.parquet",
    columns=["representative_station_id", "physical_location_id"],
).copy()

step4_13_station_location_master_df["representative_station_id"] = (
    step4_13_station_location_master_df["representative_station_id"]
    .astype("string")
    .str.strip()
)
step4_13_station_location_master_df["physical_location_id"] = (
    step4_13_station_location_master_df["physical_location_id"]
    .astype("string")
    .str.strip()
)

step4_13_station_location_master_df = (
    step4_13_station_location_master_df
    .drop_duplicates(subset=["representative_station_id"])
    .reset_index(drop=True)
)

step4_13_slice_input_df = step4_13_slice_input_df.merge(
    step4_13_station_location_master_df,
    on="representative_station_id",
    how="left",
    validate="many_to_one",
)

# persona po physical_location_id
step4_13_station_persona_profile_df = pd.read_parquet(
    step4_13_data_dir / "station_persona_profile.parquet"
).copy()
step4_13_station_persona_profile_df.columns = [
    str(col).strip() for col in step4_13_station_persona_profile_df.columns
]

step4_13_persona_col = resolve_first_existing(
    step4_13_station_persona_profile_df.columns,
    ["dominant_persona", "station_persona", "persona", "persona_group"],
)
assert step4_13_persona_col is not None, (
    "4.13.1 could not resolve persona column in station_persona_profile.parquet."
)

step4_13_station_persona_profile_df["physical_location_id"] = (
    step4_13_station_persona_profile_df["physical_location_id"]
    .astype("string")
    .str.strip()
)

step4_13_station_persona_profile_df = (
    step4_13_station_persona_profile_df[
        ["physical_location_id", step4_13_persona_col]
    ]
    .drop_duplicates(subset=["physical_location_id"])
    .rename(columns={step4_13_persona_col: "persona_slice"})
    .reset_index(drop=True)
)

step4_13_slice_input_df = step4_13_slice_input_df.merge(
    step4_13_station_persona_profile_df,
    on="physical_location_id",
    how="left",
    validate="many_to_one",
)

# mikrostrefa po physical_location_id
step4_13_microzone_reference_df = pd.read_parquet(
    step4_13_data_dir / "microzone_reference.parquet",
    columns=["physical_location_id", "microzone_id"],
).copy()

step4_13_microzone_reference_df["physical_location_id"] = (
    step4_13_microzone_reference_df["physical_location_id"]
    .astype("string")
    .str.strip()
)

step4_13_microzone_reference_df = (
    step4_13_microzone_reference_df
    .drop_duplicates(subset=["physical_location_id"])
    .rename(columns={"microzone_id": "microzone_slice"})
    .reset_index(drop=True)
)

step4_13_slice_input_df = step4_13_slice_input_df.merge(
    step4_13_microzone_reference_df,
    on="physical_location_id",
    how="left",
    validate="many_to_one",
)

# święta i pogoda po dacie z warstwy dziennej
step4_13_daily_forecast_ready_df = pd.read_parquet(
    step4_13_data_dir / "daily_forecast_ready_data.parquet"
).copy()
step4_13_daily_forecast_ready_df.columns = [
    str(col).strip() for col in step4_13_daily_forecast_ready_df.columns
]

step4_13_daily_date_col = resolve_first_existing(
    step4_13_daily_forecast_ready_df.columns,
    ["departure_date", "activity_date", "date"],
)
assert step4_13_daily_date_col is not None, (
    "4.13.1 could not resolve date column in daily_forecast_ready_data.parquet."
)

step4_13_holiday_col = resolve_first_existing(
    step4_13_daily_forecast_ready_df.columns,
    ["is_holiday", "is_business_free_day"],
)
assert step4_13_holiday_col is not None, (
    "4.13.1 could not resolve holiday column in daily_forecast_ready_data.parquet."
)

step4_13_weather_col = resolve_first_existing(
    step4_13_daily_forecast_ready_df.columns,
    [
        "forecast_like_avg_air_temperature_degc",
        "forecast_like_sum_precipitation_amount_mm",
        "forecast_like_daily_apparent_temperature_c",
        "forecast_like_daily_wind_drag_index",
        "forecast_like_precipitation_event",
        "forecast_like_daylight_hours",
    ],
)
assert step4_13_weather_col is not None, (
    "4.13.1 could not resolve weather proxy column in daily_forecast_ready_data.parquet."
)

step4_13_daily_slice_df = (
    step4_13_daily_forecast_ready_df[
        [step4_13_daily_date_col, step4_13_holiday_col, step4_13_weather_col]
    ]
    .copy()
    .drop_duplicates(subset=[step4_13_daily_date_col])
    .rename(
        columns={
            step4_13_daily_date_col: "activity_date",
            step4_13_holiday_col: "holiday_flag_slice",
            step4_13_weather_col: "weather_proxy_slice",
        }
    )
    .reset_index(drop=True)
)

step4_13_daily_slice_df["activity_date"] = pd.to_datetime(
    step4_13_daily_slice_df["activity_date"],
    errors="coerce",
)

step4_13_slice_input_df = step4_13_slice_input_df.merge(
    step4_13_daily_slice_df,
    on="activity_date",
    how="left",
    validate="many_to_one",
)

step4_13_slice_column_map_df = pd.DataFrame(
    [
        {"slice_name": "hub_flag", "resolved_column_name": "hub_flag_slice"},
        {"slice_name": "is_cold_start", "resolved_column_name": "is_cold_start_slice"},
        {"slice_name": "microzone", "resolved_column_name": "microzone_slice"},
        {"slice_name": "holiday_flag", "resolved_column_name": "holiday_flag_slice"},
        {"slice_name": "persona", "resolved_column_name": "persona_slice"},
        {"slice_name": "weather_proxy", "resolved_column_name": "weather_proxy_slice"},
    ]
)

step4_13_slice_input_summary_df = pd.DataFrame(
    [
        {
            "row_count": int(step4_13_slice_input_df.shape[0]),
            "col_count": int(step4_13_slice_input_df.shape[1]),
            "unique_station_count": int(step4_13_slice_input_df["station_id"].nunique()),
            "unique_date_count": int(step4_13_slice_input_df["activity_date"].nunique()),
            "positive_count": int(step4_13_slice_input_df["target_alert_day"].sum()),
            "negative_count": int(
                len(step4_13_slice_input_df) - step4_13_slice_input_df["target_alert_day"].sum()
            ),
            "test_split_only": int(step4_13_slice_input_df["point4_split_name"].eq("test").all()),
            "persona_missing_count": int(step4_13_slice_input_df["persona_slice"].isna().sum()),
            "microzone_missing_count": int(step4_13_slice_input_df["microzone_slice"].isna().sum()),
            "holiday_missing_count": int(step4_13_slice_input_df["holiday_flag_slice"].isna().sum()),
            "weather_missing_count": int(step4_13_slice_input_df["weather_proxy_slice"].isna().sum()),
            "physical_location_missing_count": int(step4_13_slice_input_df["physical_location_id"].isna().sum()),
        }
    ]
)

display(step4_13_slice_column_map_df)
display(step4_13_slice_input_summary_df)

assert int(step4_13_slice_input_summary_df.loc[0, "test_split_only"]) == 1, (
    "4.13.1 expected only test rows in slice input."
)
assert step4_13_slice_input_df.shape[0] == step4_11_predictions_test_df.shape[0], (
    "4.13.1 slice input row count mismatch vs saved test predictions."
)

,slice_name,resolved_column_name
0,hub_flag,hub_flag_slice
1,is_cold_start,is_cold_start_slice
2,microzone,microzone_slice
3,holiday_flag,holiday_flag_slice
4,persona,persona_slice
5,weather_proxy,weather_proxy_slice


,row_count,col_count,unique_station_count,unique_date_count,positive_count,negative_count,test_split_only,persona_missing_count,microzone_missing_count,holiday_missing_count,weather_missing_count,physical_location_missing_count
0,75980,17,344,223,61198,14782,1,0,0,0,0,0


In [70]:
# 4.13.2 Metryki segmentowe: huby vs reszta

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

required_objects = [
    "step4_13_slice_input_df",
    "step4_11_test_metrics_df",
    "step4_11_predictions_test_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.13.2: {missing_objects}"

def compute_binary_metrics(y_true, y_pred, y_score):
    y_true = pd.Series(y_true).astype(int)
    y_pred = pd.Series(y_pred).astype(int)
    y_score = pd.Series(y_score).astype(float)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    row_count = int(len(y_true))
    positive_count = int((y_true == 1).sum())
    negative_count = int((y_true == 0).sum())

    accuracy = (tp + tn) / row_count if row_count > 0 else np.nan
    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    f1 = (
        2 * precision * recall / (precision + recall)
        if pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0
        else np.nan
    )
    balanced_accuracy = (
        (recall + specificity) / 2
        if pd.notna(recall) and pd.notna(specificity)
        else np.nan
    )
    pred_positive_rate = float(y_pred.mean()) if row_count > 0 else np.nan
    roc_auc = float(roc_auc_score(y_true, y_score)) if positive_count > 0 and negative_count > 0 else np.nan
    pr_auc = float(average_precision_score(y_true, y_score)) if positive_count > 0 else np.nan
    brier_score = float(brier_score_loss(y_true, y_score)) if row_count > 0 else np.nan

    return {
        "row_count": row_count,
        "positive_count": positive_count,
        "negative_count": negative_count,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "specificity": float(specificity),
        "f1": float(f1),
        "balanced_accuracy": float(balanced_accuracy),
        "pred_positive_rate": float(pred_positive_rate),
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
        "brier_score": float(brier_score),
    }

step4_13_hub_input_df = step4_13_slice_input_df.copy()

step4_13_hub_input_df["hub_flag_numeric"] = pd.to_numeric(
    step4_13_hub_input_df["hub_flag_slice"],
    errors="coerce",
)

step4_13_hub_input_df = step4_13_hub_input_df.loc[
    step4_13_hub_input_df["hub_flag_numeric"].isin([0, 1])
].copy()

step4_13_hub_input_df["hub_segment"] = step4_13_hub_input_df["hub_flag_numeric"].map(
    {
        1.0: "hub",
        0.0: "non_hub",
    }
)

step4_13_test_anchor_row = step4_11_test_metrics_df.iloc[0]

step4_13_test_brier_score = float(
    brier_score_loss(
        step4_11_predictions_test_df["y_true"].astype(int),
        step4_11_predictions_test_df["predicted_probability"].astype(float),
    )
)

step4_13_hub_metric_rows = []

for step4_13_hub_segment, step4_13_group_df in (
    step4_13_hub_input_df
    .sort_values(["activity_date", "station_id"])
    .groupby("hub_segment", dropna=False)
):
    step4_13_metric_row = compute_binary_metrics(
        step4_13_group_df["target_alert_day"],
        step4_13_group_df["predicted_label"],
        step4_13_group_df["predicted_probability"],
    )
    step4_13_metric_row["slice_family"] = "hub_vs_rest"
    step4_13_metric_row["slice_name"] = str(step4_13_hub_segment)
    step4_13_metric_row["positive_share"] = float(step4_13_group_df["target_alert_day"].mean())
    step4_13_metric_row["mean_predicted_probability"] = float(step4_13_group_df["predicted_probability"].mean())
    step4_13_metric_row["unique_station_count"] = int(step4_13_group_df["station_id"].nunique())
    step4_13_metric_row["recall_vs_test_delta"] = (
        step4_13_metric_row["recall"] - float(step4_13_test_anchor_row["recall"])
    )
    step4_13_metric_row["specificity_vs_test_delta"] = (
        step4_13_metric_row["specificity"] - float(step4_13_test_anchor_row["specificity"])
    )
    step4_13_metric_row["balanced_accuracy_vs_test_delta"] = (
        step4_13_metric_row["balanced_accuracy"] - float(step4_13_test_anchor_row["balanced_accuracy"])
    )
    step4_13_metric_row["pr_auc_vs_test_delta"] = (
        step4_13_metric_row["pr_auc"] - float(step4_13_test_anchor_row["pr_auc"])
    )
    step4_13_metric_row["brier_vs_test_delta"] = (
        step4_13_metric_row["brier_score"] - step4_13_test_brier_score
    )
    step4_13_hub_metric_rows.append(step4_13_metric_row)

step4_13_hub_slice_report_df = (
    pd.DataFrame(step4_13_hub_metric_rows)
    .sort_values("slice_name")
    .reset_index(drop=True)
)

display(step4_13_hub_slice_report_df)

assert set(step4_13_hub_slice_report_df["slice_name"].unique()) == {
    "hub",
    "non_hub",
}, "4.13.2 expected exactly hub and non_hub segments."

,row_count,positive_count,negative_count,tp,tn,fp,fn,accuracy,precision,recall,...,slice_family,slice_name,positive_share,mean_predicted_probability,unique_station_count,recall_vs_test_delta,specificity_vs_test_delta,balanced_accuracy_vs_test_delta,pr_auc_vs_test_delta,brier_vs_test_delta
0,11132,9365,1767,9084,272,1495,281,0.840460,0.858682,0.969995,...,hub_vs_rest,hub,0.841268,0.836061,50,0.003002,0.011192,0.007097,0.032466,-0.025380
1,64848,51833,13015,50094,1838,11177,1739,0.800827,0.817581,0.966450,...,hub_vs_rest,non_hub,0.799300,0.792063,294,-0.000542,-0.001520,-0.001031,-0.007509,0.004357


In [71]:
# 4.13.3 Metryki segmentowe: cold start vs non-cold-start

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

required_objects = [
    "step4_13_slice_input_df",
    "step4_11_test_metrics_df",
    "step4_11_predictions_test_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.13.3: {missing_objects}"

def compute_binary_metrics(y_true, y_pred, y_score):
    y_true = pd.Series(y_true).astype(int)
    y_pred = pd.Series(y_pred).astype(int)
    y_score = pd.Series(y_score).astype(float)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    row_count = int(len(y_true))
    positive_count = int((y_true == 1).sum())
    negative_count = int((y_true == 0).sum())

    accuracy = (tp + tn) / row_count if row_count > 0 else np.nan
    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    f1 = (
        2 * precision * recall / (precision + recall)
        if pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0
        else np.nan
    )
    balanced_accuracy = (
        (recall + specificity) / 2
        if pd.notna(recall) and pd.notna(specificity)
        else np.nan
    )
    pred_positive_rate = float(y_pred.mean()) if row_count > 0 else np.nan
    roc_auc = float(roc_auc_score(y_true, y_score)) if positive_count > 0 and negative_count > 0 else np.nan
    pr_auc = float(average_precision_score(y_true, y_score)) if positive_count > 0 else np.nan
    brier_score = float(brier_score_loss(y_true, y_score)) if row_count > 0 else np.nan

    return {
        "row_count": row_count,
        "positive_count": positive_count,
        "negative_count": negative_count,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "specificity": float(specificity),
        "f1": float(f1),
        "balanced_accuracy": float(balanced_accuracy),
        "pred_positive_rate": float(pred_positive_rate),
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
        "brier_score": float(brier_score),
    }

step4_13_cold_start_input_df = step4_13_slice_input_df.copy()

step4_13_cold_start_input_df["cold_start_numeric"] = pd.to_numeric(
    step4_13_cold_start_input_df["is_cold_start_slice"],
    errors="coerce",
)

step4_13_cold_start_input_df = step4_13_cold_start_input_df.loc[
    step4_13_cold_start_input_df["cold_start_numeric"].isin([0, 1])
].copy()

step4_13_cold_start_input_df["cold_start_segment"] = (
    step4_13_cold_start_input_df["cold_start_numeric"]
    .map({1.0: "cold_start", 0.0: "non_cold_start"})
)

step4_13_test_anchor_row = step4_11_test_metrics_df.iloc[0]

step4_13_test_brier_score = float(
    brier_score_loss(
        step4_11_predictions_test_df["y_true"].astype(int),
        step4_11_predictions_test_df["predicted_probability"].astype(float),
    )
)

step4_13_cold_start_metric_rows = []

for step4_13_segment_name, step4_13_group_df in (
    step4_13_cold_start_input_df
    .sort_values(["activity_date", "station_id"])
    .groupby("cold_start_segment", dropna=False)
):
    step4_13_metric_row = compute_binary_metrics(
        step4_13_group_df["target_alert_day"],
        step4_13_group_df["predicted_label"],
        step4_13_group_df["predicted_probability"],
    )
    step4_13_metric_row["slice_family"] = "cold_start_vs_rest"
    step4_13_metric_row["slice_name"] = str(step4_13_segment_name)
    step4_13_metric_row["positive_share"] = float(step4_13_group_df["target_alert_day"].mean())
    step4_13_metric_row["mean_predicted_probability"] = float(step4_13_group_df["predicted_probability"].mean())
    step4_13_metric_row["unique_station_count"] = int(step4_13_group_df["station_id"].nunique())
    step4_13_metric_row["recall_vs_test_delta"] = (
        step4_13_metric_row["recall"] - float(step4_13_test_anchor_row["recall"])
    )
    step4_13_metric_row["specificity_vs_test_delta"] = (
        step4_13_metric_row["specificity"] - float(step4_13_test_anchor_row["specificity"])
    )
    step4_13_metric_row["balanced_accuracy_vs_test_delta"] = (
        step4_13_metric_row["balanced_accuracy"] - float(step4_13_test_anchor_row["balanced_accuracy"])
    )
    step4_13_metric_row["pr_auc_vs_test_delta"] = (
        step4_13_metric_row["pr_auc"] - float(step4_13_test_anchor_row["pr_auc"])
    )
    step4_13_metric_row["brier_vs_test_delta"] = (
        step4_13_metric_row["brier_score"] - step4_13_test_brier_score
    )
    step4_13_cold_start_metric_rows.append(step4_13_metric_row)

step4_13_cold_start_slice_report_df = (
    pd.DataFrame(step4_13_cold_start_metric_rows)
    .sort_values("slice_name")
    .reset_index(drop=True)
)

display(step4_13_cold_start_slice_report_df)

assert set(step4_13_cold_start_slice_report_df["slice_name"].unique()) == {
    "cold_start",
    "non_cold_start",
}, "4.13.3 expected exactly cold_start and non_cold_start segments."

,row_count,positive_count,negative_count,tp,tn,fp,fn,accuracy,precision,recall,...,slice_family,slice_name,positive_share,mean_predicted_probability,unique_station_count,recall_vs_test_delta,specificity_vs_test_delta,balanced_accuracy_vs_test_delta,pr_auc_vs_test_delta,brier_vs_test_delta
0,14,6,8,4,0,8,2,0.285714,0.333333,0.666667,...,cold_start_vs_rest,cold_start,0.428571,0.648856,2,-0.300326,-0.142741,-0.221533,-0.562364,0.244163
1,75911,61192,14719,59174,2070,12649,2018,0.806787,0.823886,0.967022,...,cold_start_vs_rest,non_cold_start,0.806102,0.798824,344,0.000029,-0.002107,-0.001039,0.000077,-0.000092


In [72]:
# 4.13.4 Profil mikro-stref i wybór segmentów do oceny

import pandas as pd

required_objects = [
    "step4_13_slice_input_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.13.4: {missing_objects}"

step4_13_microzone_profile_df = step4_13_slice_input_df.copy()

step4_13_microzone_profile_df["microzone_slice"] = (
    step4_13_microzone_profile_df["microzone_slice"]
    .astype("string")
    .str.strip()
)

step4_13_microzone_profile_df = step4_13_microzone_profile_df.loc[
    step4_13_microzone_profile_df["microzone_slice"].notna()
].copy()

step4_13_microzone_profile_df = (
    step4_13_microzone_profile_df
    .groupby("microzone_slice", dropna=False)
    .agg(
        row_count=("target_alert_day", "size"),
        positive_count=("target_alert_day", "sum"),
        negative_count=("target_alert_day", lambda s: int((1 - s).sum())),
        unique_station_count=("station_id", "nunique"),
        min_activity_date=("activity_date", "min"),
        max_activity_date=("activity_date", "max"),
        mean_predicted_probability=("predicted_probability", "mean"),
        pred_positive_rate=("predicted_label", "mean"),
    )
    .reset_index()
    .rename(columns={"microzone_slice": "slice_name"})
)

step4_13_microzone_profile_df["positive_share"] = (
    step4_13_microzone_profile_df["positive_count"]
    / step4_13_microzone_profile_df["row_count"]
)

step4_13_microzone_profile_df["is_supported_microzone"] = (
    (step4_13_microzone_profile_df["row_count"] >= 500)
    & (step4_13_microzone_profile_df["positive_count"] >= 50)
    & (step4_13_microzone_profile_df["negative_count"] >= 50)
).astype(int)

step4_13_supported_microzones_df = (
    step4_13_microzone_profile_df.loc[
        step4_13_microzone_profile_df["is_supported_microzone"].eq(1)
    ]
    .sort_values(["row_count", "positive_count"], ascending=[False, False])
    .reset_index(drop=True)
)

step4_13_microzone_support_summary_df = pd.DataFrame(
    [
        {
            "total_microzones": int(step4_13_microzone_profile_df["slice_name"].nunique()),
            "supported_microzones": int(step4_13_supported_microzones_df["slice_name"].nunique()),
            "min_row_threshold": 500,
            "min_positive_threshold": 50,
            "min_negative_threshold": 50,
        }
    ]
)

display(step4_13_microzone_support_summary_df)
display(step4_13_supported_microzones_df.head(15))

assert step4_13_supported_microzones_df.shape[0] > 0, (
    "4.13.4 expected at least one supported microzone for segment evaluation."
)

,total_microzones,supported_microzones,min_row_threshold,min_positive_threshold,min_negative_threshold
0,304,7,500,50,50


,slice_name,row_count,positive_count,negative_count,unique_station_count,min_activity_date,max_activity_date,mean_predicted_probability,pred_positive_rate,positive_share,is_supported_microzone
0,MZ_0067,1782,1144,638,8,2020-03-23,2020-10-31,0.718207,0.900112,0.641975,1
1,MZ_0252,892,657,235,4,2020-03-23,2020-10-31,0.748911,0.923767,0.736547,1
2,MZ_0054,888,591,297,4,2020-03-23,2020-10-31,0.740965,0.941441,0.665541,1
3,MZ_0084,883,561,322,4,2020-03-23,2020-10-31,0.681782,0.849377,0.635334,1
4,MZ_0044,668,565,103,3,2020-03-23,2020-10-31,0.854417,0.994012,0.845808,1
5,MZ_0064,668,493,175,3,2020-03-23,2020-10-31,0.783566,0.947605,0.738024,1
6,MZ_0123,665,407,258,3,2020-03-23,2020-10-31,0.637261,0.769925,0.612030,1


In [73]:
# 4.13.5 Metryki segmentowe: wspierane mikro-strefy

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

required_objects = [
    "step4_13_slice_input_df",
    "step4_13_supported_microzones_df",
    "step4_11_test_metrics_df",
    "step4_11_predictions_test_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.13.5: {missing_objects}"

def compute_binary_metrics(y_true, y_pred, y_score):
    y_true = pd.Series(y_true).astype(int)
    y_pred = pd.Series(y_pred).astype(int)
    y_score = pd.Series(y_score).astype(float)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    row_count = int(len(y_true))
    positive_count = int((y_true == 1).sum())
    negative_count = int((y_true == 0).sum())

    accuracy = (tp + tn) / row_count if row_count > 0 else np.nan
    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    f1 = (
        2 * precision * recall / (precision + recall)
        if pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0
        else np.nan
    )
    balanced_accuracy = (
        (recall + specificity) / 2
        if pd.notna(recall) and pd.notna(specificity)
        else np.nan
    )
    pred_positive_rate = float(y_pred.mean()) if row_count > 0 else np.nan
    roc_auc = float(roc_auc_score(y_true, y_score)) if positive_count > 0 and negative_count > 0 else np.nan
    pr_auc = float(average_precision_score(y_true, y_score)) if positive_count > 0 else np.nan
    brier_score = float(brier_score_loss(y_true, y_score)) if row_count > 0 else np.nan

    return {
        "row_count": row_count,
        "positive_count": positive_count,
        "negative_count": negative_count,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "specificity": float(specificity),
        "f1": float(f1),
        "balanced_accuracy": float(balanced_accuracy),
        "pred_positive_rate": float(pred_positive_rate),
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
        "brier_score": float(brier_score),
    }

step4_13_microzone_input_df = step4_13_slice_input_df.copy()
step4_13_microzone_input_df["microzone_slice"] = (
    step4_13_microzone_input_df["microzone_slice"]
    .astype("string")
    .str.strip()
)

step4_13_supported_microzone_names = set(
    step4_13_supported_microzones_df["slice_name"].astype("string").str.strip().tolist()
)

step4_13_microzone_input_df = step4_13_microzone_input_df.loc[
    step4_13_microzone_input_df["microzone_slice"].isin(step4_13_supported_microzone_names)
].copy()

step4_13_test_anchor_row = step4_11_test_metrics_df.iloc[0]

step4_13_test_brier_score = float(
    brier_score_loss(
        step4_11_predictions_test_df["y_true"].astype(int),
        step4_11_predictions_test_df["predicted_probability"].astype(float),
    )
)

step4_13_microzone_metric_rows = []

for step4_13_slice_name, step4_13_group_df in (
    step4_13_microzone_input_df
    .sort_values(["activity_date", "station_id"])
    .groupby("microzone_slice", dropna=False)
):
    step4_13_metric_row = compute_binary_metrics(
        step4_13_group_df["target_alert_day"],
        step4_13_group_df["predicted_label"],
        step4_13_group_df["predicted_probability"],
    )
    step4_13_metric_row["slice_family"] = "microzone"
    step4_13_metric_row["slice_name"] = str(step4_13_slice_name)
    step4_13_metric_row["positive_share"] = float(step4_13_group_df["target_alert_day"].mean())
    step4_13_metric_row["mean_predicted_probability"] = float(step4_13_group_df["predicted_probability"].mean())
    step4_13_metric_row["unique_station_count"] = int(step4_13_group_df["station_id"].nunique())
    step4_13_metric_row["recall_vs_test_delta"] = (
        step4_13_metric_row["recall"] - float(step4_13_test_anchor_row["recall"])
    )
    step4_13_metric_row["specificity_vs_test_delta"] = (
        step4_13_metric_row["specificity"] - float(step4_13_test_anchor_row["specificity"])
    )
    step4_13_metric_row["balanced_accuracy_vs_test_delta"] = (
        step4_13_metric_row["balanced_accuracy"] - float(step4_13_test_anchor_row["balanced_accuracy"])
    )
    step4_13_metric_row["pr_auc_vs_test_delta"] = (
        step4_13_metric_row["pr_auc"] - float(step4_13_test_anchor_row["pr_auc"])
    )
    step4_13_metric_row["brier_vs_test_delta"] = (
        step4_13_metric_row["brier_score"] - step4_13_test_brier_score
    )
    step4_13_microzone_metric_rows.append(step4_13_metric_row)

step4_13_microzone_slice_report_df = (
    pd.DataFrame(step4_13_microzone_metric_rows)
    .sort_values(["pr_auc", "balanced_accuracy", "specificity", "recall"], ascending=[False, False, False, False])
    .reset_index(drop=True)
)

display(step4_13_microzone_slice_report_df)

assert step4_13_microzone_slice_report_df.shape[0] == step4_13_supported_microzones_df.shape[0], (
    "4.13.5 expected one metric row per supported microzone."
)

,row_count,positive_count,negative_count,tp,tn,fp,fn,accuracy,precision,recall,...,slice_family,slice_name,positive_share,mean_predicted_probability,unique_station_count,recall_vs_test_delta,specificity_vs_test_delta,balanced_accuracy_vs_test_delta,pr_auc_vs_test_delta,brier_vs_test_delta
0,668,565,103,562,1,102,3,0.842814,0.846386,0.994690,...,microzone,MZ_0044,0.845808,0.854417,3,0.027698,-0.133032,-0.052667,-0.021211,-0.009220
1,668,493,175,473,15,160,20,0.730539,0.747235,0.959432,...,microzone,MZ_0064,0.738024,0.783566,3,-0.007560,-0.057027,-0.032294,-0.080484,0.047226
2,892,657,235,628,39,196,29,0.747758,0.762136,0.955860,...,microzone,MZ_0252,0.736547,0.748911,4,-0.011132,0.023216,0.006042,-0.084235,0.038012
3,1782,1144,638,1052,86,552,92,0.638608,0.655860,0.919580,...,microzone,MZ_0067,0.641975,0.718207,8,-0.047412,-0.007945,-0.027678,-0.193983,0.091350
4,883,561,322,506,78,244,55,0.661382,0.674667,0.901961,...,microzone,MZ_0084,0.635334,0.681782,4,-0.065032,0.099495,0.017232,-0.196258,0.082552
5,888,591,297,569,30,267,22,0.674550,0.680622,0.962775,...,microzone,MZ_0054,0.665541,0.740965,4,-0.004217,-0.041731,-0.022974,-0.203789,0.087597
6,665,407,258,326,72,186,81,0.598496,0.636719,0.800983,...,microzone,MZ_0123,0.612030,0.637261,3,-0.166010,0.136329,-0.014840,-0.243850,0.103351


In [74]:
# 4.13.6 Metryki segmentowe: święta vs nie-święta

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

required_objects = [
    "step4_13_slice_input_df",
    "step4_11_test_metrics_df",
    "step4_11_predictions_test_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.13.6: {missing_objects}"

def compute_binary_metrics(y_true, y_pred, y_score):
    y_true = pd.Series(y_true).astype(int)
    y_pred = pd.Series(y_pred).astype(int)
    y_score = pd.Series(y_score).astype(float)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    row_count = int(len(y_true))
    positive_count = int((y_true == 1).sum())
    negative_count = int((y_true == 0).sum())

    accuracy = (tp + tn) / row_count if row_count > 0 else np.nan
    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    f1 = (
        2 * precision * recall / (precision + recall)
        if pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0
        else np.nan
    )
    balanced_accuracy = (
        (recall + specificity) / 2
        if pd.notna(recall) and pd.notna(specificity)
        else np.nan
    )
    pred_positive_rate = float(y_pred.mean()) if row_count > 0 else np.nan
    roc_auc = float(roc_auc_score(y_true, y_score)) if positive_count > 0 and negative_count > 0 else np.nan
    pr_auc = float(average_precision_score(y_true, y_score)) if positive_count > 0 else np.nan
    brier_score = float(brier_score_loss(y_true, y_score)) if row_count > 0 else np.nan

    return {
        "row_count": row_count,
        "positive_count": positive_count,
        "negative_count": negative_count,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "specificity": float(specificity),
        "f1": float(f1),
        "balanced_accuracy": float(balanced_accuracy),
        "pred_positive_rate": float(pred_positive_rate),
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
        "brier_score": float(brier_score),
    }

step4_13_holiday_input_df = step4_13_slice_input_df.copy()

step4_13_holiday_input_df["holiday_flag_numeric"] = (
    step4_13_holiday_input_df["holiday_flag_slice"]
    .replace({True: 1, False: 0, "True": 1, "False": 0, "true": 1, "false": 0})
)

step4_13_holiday_input_df["holiday_flag_numeric"] = pd.to_numeric(
    step4_13_holiday_input_df["holiday_flag_numeric"],
    errors="coerce",
)

step4_13_holiday_input_df = step4_13_holiday_input_df.loc[
    step4_13_holiday_input_df["holiday_flag_numeric"].isin([0, 1])
].copy()

step4_13_holiday_input_df["holiday_segment"] = (
    step4_13_holiday_input_df["holiday_flag_numeric"]
    .map({1.0: "holiday", 0.0: "non_holiday"})
)

step4_13_test_anchor_row = step4_11_test_metrics_df.iloc[0]

step4_13_test_brier_score = float(
    brier_score_loss(
        step4_11_predictions_test_df["y_true"].astype(int),
        step4_11_predictions_test_df["predicted_probability"].astype(float),
    )
)

step4_13_holiday_metric_rows = []

for step4_13_segment_name, step4_13_group_df in (
    step4_13_holiday_input_df
    .sort_values(["activity_date", "station_id"])
    .groupby("holiday_segment", dropna=False)
):
    step4_13_metric_row = compute_binary_metrics(
        step4_13_group_df["target_alert_day"],
        step4_13_group_df["predicted_label"],
        step4_13_group_df["predicted_probability"],
    )
    step4_13_metric_row["slice_family"] = "holiday_vs_rest"
    step4_13_metric_row["slice_name"] = str(step4_13_segment_name)
    step4_13_metric_row["positive_share"] = float(step4_13_group_df["target_alert_day"].mean())
    step4_13_metric_row["mean_predicted_probability"] = float(step4_13_group_df["predicted_probability"].mean())
    step4_13_metric_row["unique_station_count"] = int(step4_13_group_df["station_id"].nunique())
    step4_13_metric_row["recall_vs_test_delta"] = (
        step4_13_metric_row["recall"] - float(step4_13_test_anchor_row["recall"])
    )
    step4_13_metric_row["specificity_vs_test_delta"] = (
        step4_13_metric_row["specificity"] - float(step4_13_test_anchor_row["specificity"])
    )
    step4_13_metric_row["balanced_accuracy_vs_test_delta"] = (
        step4_13_metric_row["balanced_accuracy"] - float(step4_13_test_anchor_row["balanced_accuracy"])
    )
    step4_13_metric_row["pr_auc_vs_test_delta"] = (
        step4_13_metric_row["pr_auc"] - float(step4_13_test_anchor_row["pr_auc"])
    )
    step4_13_metric_row["brier_vs_test_delta"] = (
        step4_13_metric_row["brier_score"] - step4_13_test_brier_score
    )
    step4_13_holiday_metric_rows.append(step4_13_metric_row)

step4_13_holiday_slice_report_df = (
    pd.DataFrame(step4_13_holiday_metric_rows)
    .sort_values("slice_name")
    .reset_index(drop=True)
)

display(step4_13_holiday_slice_report_df)

assert set(step4_13_holiday_slice_report_df["slice_name"].unique()) == {
    "holiday",
    "non_holiday",
}, "4.13.6 expected exactly holiday and non_holiday segments."

,row_count,positive_count,negative_count,tp,tn,fp,fn,accuracy,precision,recall,...,slice_family,slice_name,positive_share,mean_predicted_probability,unique_station_count,recall_vs_test_delta,specificity_vs_test_delta,balanced_accuracy_vs_test_delta,pr_auc_vs_test_delta,brier_vs_test_delta
0,2695,2105,590,2064,39,551,41,0.780334,0.789293,0.980523,...,holiday_vs_rest,holiday,0.781076,0.775248,343,0.013530,-0.076639,-0.031555,-0.033988,0.019240
1,73285,59093,14192,57114,2071,12121,1979,0.807600,0.824930,0.966510,...,holiday_vs_rest,non_holiday,0.806345,0.799365,344,-0.000482,0.003186,0.001352,0.001081,-0.000708


In [75]:
# 4.13.7 Profil proxy pogody i budowa segmentów pogodowych

import pandas as pd

required_objects = [
    "step4_13_slice_input_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.13.7: {missing_objects}"

step4_13_weather_profile_input_df = step4_13_slice_input_df.copy()

step4_13_weather_profile_input_df["weather_proxy_numeric"] = pd.to_numeric(
    step4_13_weather_profile_input_df["weather_proxy_slice"],
    errors="coerce",
)

step4_13_weather_profile_input_df = step4_13_weather_profile_input_df.loc[
    step4_13_weather_profile_input_df["weather_proxy_numeric"].notna()
].copy()

assert step4_13_weather_profile_input_df.shape[0] > 0, (
    "4.13.7 expected non-empty weather proxy input after numeric conversion."
)

step4_13_weather_unique_values = int(
    step4_13_weather_profile_input_df["weather_proxy_numeric"].nunique()
)
assert step4_13_weather_unique_values >= 3, (
    f"4.13.7 expected at least 3 unique weather values, got: {step4_13_weather_unique_values}"
)

step4_13_weather_profile_input_df["weather_segment"] = pd.qcut(
    step4_13_weather_profile_input_df["weather_proxy_numeric"],
    q=3,
    labels=["weather_low", "weather_mid", "weather_high"],
    duplicates="drop",
)

step4_13_weather_profile_summary_df = pd.DataFrame(
    [
        {
            "row_count": int(step4_13_weather_profile_input_df.shape[0]),
            "unique_station_count": int(step4_13_weather_profile_input_df["station_id"].nunique()),
            "unique_date_count": int(step4_13_weather_profile_input_df["activity_date"].nunique()),
            "weather_proxy_min": float(step4_13_weather_profile_input_df["weather_proxy_numeric"].min()),
            "weather_proxy_median": float(step4_13_weather_profile_input_df["weather_proxy_numeric"].median()),
            "weather_proxy_max": float(step4_13_weather_profile_input_df["weather_proxy_numeric"].max()),
            "weather_proxy_unique_values": step4_13_weather_unique_values,
        }
    ]
)

step4_13_weather_segment_profile_df = (
    step4_13_weather_profile_input_df
    .groupby("weather_segment", dropna=False)
    .agg(
        row_count=("target_alert_day", "size"),
        positive_count=("target_alert_day", "sum"),
        negative_count=("target_alert_day", lambda s: int((1 - s).sum())),
        unique_station_count=("station_id", "nunique"),
        min_activity_date=("activity_date", "min"),
        max_activity_date=("activity_date", "max"),
        weather_proxy_min=("weather_proxy_numeric", "min"),
        weather_proxy_mean=("weather_proxy_numeric", "mean"),
        weather_proxy_max=("weather_proxy_numeric", "max"),
        mean_predicted_probability=("predicted_probability", "mean"),
        pred_positive_rate=("predicted_label", "mean"),
    )
    .reset_index()
)

step4_13_weather_segment_profile_df["positive_share"] = (
    step4_13_weather_segment_profile_df["positive_count"]
    / step4_13_weather_segment_profile_df["row_count"]
)

display(step4_13_weather_profile_summary_df)
display(step4_13_weather_segment_profile_df)

assert step4_13_weather_segment_profile_df.shape[0] == 3, (
    "4.13.7 expected exactly 3 weather segments: low, mid, high."
)

,row_count,unique_station_count,unique_date_count,weather_proxy_min,weather_proxy_median,weather_proxy_max,weather_proxy_unique_values
0,75980,344,223,-3.359228,13.521627,23.944096,223


,weather_segment,row_count,positive_count,negative_count,unique_station_count,min_activity_date,max_activity_date,weather_proxy_min,weather_proxy_mean,weather_proxy_max,mean_predicted_probability,pred_positive_rate,positive_share
0,weather_low,25499,20303,5196,344,2020-03-23,2020-10-31,-3.359228,5.694111,9.963870,0.771965,0.931488,0.796227
1,weather_mid,25397,20302,5095,344,2020-04-21,2020-10-29,10.087237,13.211761,15.398649,0.805059,0.945505,0.799386
2,weather_high,25084,20593,4491,344,2020-05-22,2020-09-26,15.402992,18.405939,23.944096,0.818862,0.960174,0.820962


In [76]:
# 4.13.8 Metryki segmentowe: warunki pogodowe low / mid / high

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

required_objects = [
    "step4_13_weather_profile_input_df",
    "step4_11_test_metrics_df",
    "step4_11_predictions_test_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.13.8: {missing_objects}"

def compute_binary_metrics(y_true, y_pred, y_score):
    y_true = pd.Series(y_true).astype(int)
    y_pred = pd.Series(y_pred).astype(int)
    y_score = pd.Series(y_score).astype(float)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    row_count = int(len(y_true))
    positive_count = int((y_true == 1).sum())
    negative_count = int((y_true == 0).sum())

    accuracy = (tp + tn) / row_count if row_count > 0 else np.nan
    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    f1 = (
        2 * precision * recall / (precision + recall)
        if pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0
        else np.nan
    )
    balanced_accuracy = (
        (recall + specificity) / 2
        if pd.notna(recall) and pd.notna(specificity)
        else np.nan
    )
    pred_positive_rate = float(y_pred.mean()) if row_count > 0 else np.nan
    roc_auc = float(roc_auc_score(y_true, y_score)) if positive_count > 0 and negative_count > 0 else np.nan
    pr_auc = float(average_precision_score(y_true, y_score)) if positive_count > 0 else np.nan
    brier_score = float(brier_score_loss(y_true, y_score)) if row_count > 0 else np.nan

    return {
        "row_count": row_count,
        "positive_count": positive_count,
        "negative_count": negative_count,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "specificity": float(specificity),
        "f1": float(f1),
        "balanced_accuracy": float(balanced_accuracy),
        "pred_positive_rate": float(pred_positive_rate),
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
        "brier_score": float(brier_score),
    }

step4_13_weather_metrics_input_df = step4_13_weather_profile_input_df.copy()

step4_13_test_anchor_row = step4_11_test_metrics_df.iloc[0]

step4_13_test_brier_score = float(
    brier_score_loss(
        step4_11_predictions_test_df["y_true"].astype(int),
        step4_11_predictions_test_df["predicted_probability"].astype(float),
    )
)

step4_13_weather_metric_rows = []

for step4_13_segment_name, step4_13_group_df in (
    step4_13_weather_metrics_input_df
    .sort_values(["activity_date", "station_id"])
    .groupby("weather_segment", dropna=False)
):
    step4_13_metric_row = compute_binary_metrics(
        step4_13_group_df["target_alert_day"],
        step4_13_group_df["predicted_label"],
        step4_13_group_df["predicted_probability"],
    )
    step4_13_metric_row["slice_family"] = "weather_proxy"
    step4_13_metric_row["slice_name"] = str(step4_13_segment_name)
    step4_13_metric_row["positive_share"] = float(step4_13_group_df["target_alert_day"].mean())
    step4_13_metric_row["mean_predicted_probability"] = float(step4_13_group_df["predicted_probability"].mean())
    step4_13_metric_row["unique_station_count"] = int(step4_13_group_df["station_id"].nunique())
    step4_13_metric_row["weather_proxy_min"] = float(step4_13_group_df["weather_proxy_numeric"].min())
    step4_13_metric_row["weather_proxy_mean"] = float(step4_13_group_df["weather_proxy_numeric"].mean())
    step4_13_metric_row["weather_proxy_max"] = float(step4_13_group_df["weather_proxy_numeric"].max())
    step4_13_metric_row["recall_vs_test_delta"] = (
        step4_13_metric_row["recall"] - float(step4_13_test_anchor_row["recall"])
    )
    step4_13_metric_row["specificity_vs_test_delta"] = (
        step4_13_metric_row["specificity"] - float(step4_13_test_anchor_row["specificity"])
    )
    step4_13_metric_row["balanced_accuracy_vs_test_delta"] = (
        step4_13_metric_row["balanced_accuracy"] - float(step4_13_test_anchor_row["balanced_accuracy"])
    )
    step4_13_metric_row["pr_auc_vs_test_delta"] = (
        step4_13_metric_row["pr_auc"] - float(step4_13_test_anchor_row["pr_auc"])
    )
    step4_13_metric_row["brier_vs_test_delta"] = (
        step4_13_metric_row["brier_score"] - step4_13_test_brier_score
    )
    step4_13_weather_metric_rows.append(step4_13_metric_row)

step4_13_weather_slice_report_df = (
    pd.DataFrame(step4_13_weather_metric_rows)
    .sort_values("slice_name")
    .reset_index(drop=True)
)

display(step4_13_weather_slice_report_df)

assert set(step4_13_weather_slice_report_df["slice_name"].unique()) == {
    "weather_low",
    "weather_mid",
    "weather_high",
}, "4.13.8 expected exactly weather_low, weather_mid and weather_high segments."

,row_count,positive_count,negative_count,tp,tn,fp,fn,accuracy,precision,recall,...,mean_predicted_probability,unique_station_count,weather_proxy_min,weather_proxy_mean,weather_proxy_max,recall_vs_test_delta,specificity_vs_test_delta,balanced_accuracy_vs_test_delta,pr_auc_vs_test_delta,brier_vs_test_delta
0,25084,20593,4491,20089,495,3996,504,0.820603,0.834088,0.975526,...,0.818862,344,15.402992,18.405941,23.944096,0.008533,-0.032521,-0.011994,0.006354,-0.007909
1,25499,20303,5196,19417,861,4335,886,0.795247,0.817489,0.956361,...,0.771965,344,-3.359228,5.694110,9.963870,-0.010631,0.022963,0.006166,-0.007933,0.006907
2,25397,20302,5095,19672,754,4341,630,0.804268,0.819223,0.968969,...,0.805059,344,10.087237,13.211761,15.398649,0.001976,0.005247,0.003612,0.001452,0.000876


In [77]:
# 4.13.9 Profil person i wybór segmentów do oceny

import pandas as pd

required_objects = [
    "step4_13_slice_input_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.13.9: {missing_objects}"

step4_13_persona_profile_df = step4_13_slice_input_df.copy()

step4_13_persona_profile_df["persona_slice"] = (
    step4_13_persona_profile_df["persona_slice"]
    .astype("string")
    .str.strip()
)

step4_13_persona_profile_df = step4_13_persona_profile_df.loc[
    step4_13_persona_profile_df["persona_slice"].notna()
    & step4_13_persona_profile_df["persona_slice"].ne("")
].copy()

step4_13_persona_profile_df = (
    step4_13_persona_profile_df
    .groupby("persona_slice", dropna=False)
    .agg(
        row_count=("target_alert_day", "size"),
        positive_count=("target_alert_day", "sum"),
        negative_count=("target_alert_day", lambda s: int((1 - s).sum())),
        unique_station_count=("station_id", "nunique"),
        min_activity_date=("activity_date", "min"),
        max_activity_date=("activity_date", "max"),
        mean_predicted_probability=("predicted_probability", "mean"),
        pred_positive_rate=("predicted_label", "mean"),
    )
    .reset_index()
    .rename(columns={"persona_slice": "slice_name"})
)

step4_13_persona_profile_df["positive_share"] = (
    step4_13_persona_profile_df["positive_count"]
    / step4_13_persona_profile_df["row_count"]
)

step4_13_persona_profile_df["is_supported_persona"] = (
    (step4_13_persona_profile_df["row_count"] >= 500)
    & (step4_13_persona_profile_df["positive_count"] >= 50)
    & (step4_13_persona_profile_df["negative_count"] >= 50)
).astype(int)

step4_13_supported_personas_df = (
    step4_13_persona_profile_df.loc[
        step4_13_persona_profile_df["is_supported_persona"].eq(1)
    ]
    .sort_values(["row_count", "positive_count"], ascending=[False, False])
    .reset_index(drop=True)
)

step4_13_persona_support_summary_df = pd.DataFrame(
    [
        {
            "total_personas": int(step4_13_persona_profile_df["slice_name"].nunique()),
            "supported_personas": int(step4_13_supported_personas_df["slice_name"].nunique()),
            "min_row_threshold": 500,
            "min_positive_threshold": 50,
            "min_negative_threshold": 50,
        }
    ]
)

display(step4_13_persona_support_summary_df)
display(step4_13_supported_personas_df)

assert step4_13_supported_personas_df.shape[0] > 0, (
    "4.13.9 expected at least one supported persona for segment evaluation."
)

,total_personas,supported_personas,min_row_threshold,min_positive_threshold,min_negative_threshold
0,3,3,500,50,50


,slice_name,row_count,positive_count,negative_count,unique_station_count,min_activity_date,max_activity_date,mean_predicted_probability,pred_positive_rate,positive_share,is_supported_persona
0,mixed_other,70149,57679,12470,317,2020-03-23,2020-10-31,0.812345,0.961510,0.822236,1
1,utility_short_trip,5051,3084,1967,23,2020-03-23,2020-10-31,0.635193,0.769749,0.610572,1
2,commuter,780,435,345,4,2020-03-23,2020-10-31,0.611803,0.657692,0.557692,1


In [78]:
# 4.13.10 Metryki segmentowe: persony

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

required_objects = [
    "step4_13_slice_input_df",
    "step4_13_supported_personas_df",
    "step4_11_test_metrics_df",
    "step4_11_predictions_test_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.13.10: {missing_objects}"

def compute_binary_metrics(y_true, y_pred, y_score):
    y_true = pd.Series(y_true).astype(int)
    y_pred = pd.Series(y_pred).astype(int)
    y_score = pd.Series(y_score).astype(float)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    row_count = int(len(y_true))
    positive_count = int((y_true == 1).sum())
    negative_count = int((y_true == 0).sum())

    accuracy = (tp + tn) / row_count if row_count > 0 else np.nan
    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    f1 = (
        2 * precision * recall / (precision + recall)
        if pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0
        else np.nan
    )
    balanced_accuracy = (
        (recall + specificity) / 2
        if pd.notna(recall) and pd.notna(specificity)
        else np.nan
    )
    pred_positive_rate = float(y_pred.mean()) if row_count > 0 else np.nan
    roc_auc = float(roc_auc_score(y_true, y_score)) if positive_count > 0 and negative_count > 0 else np.nan
    pr_auc = float(average_precision_score(y_true, y_score)) if positive_count > 0 else np.nan
    brier_score = float(brier_score_loss(y_true, y_score)) if row_count > 0 else np.nan

    return {
        "row_count": row_count,
        "positive_count": positive_count,
        "negative_count": negative_count,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "specificity": float(specificity),
        "f1": float(f1),
        "balanced_accuracy": float(balanced_accuracy),
        "pred_positive_rate": float(pred_positive_rate),
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
        "brier_score": float(brier_score),
    }

step4_13_persona_input_df = step4_13_slice_input_df.copy()

step4_13_persona_input_df["persona_slice"] = (
    step4_13_persona_input_df["persona_slice"]
    .astype("string")
    .str.strip()
)

step4_13_supported_persona_names = set(
    step4_13_supported_personas_df["slice_name"].astype("string").str.strip().tolist()
)

step4_13_persona_input_df = step4_13_persona_input_df.loc[
    step4_13_persona_input_df["persona_slice"].isin(step4_13_supported_persona_names)
].copy()

step4_13_test_anchor_row = step4_11_test_metrics_df.iloc[0]

step4_13_test_brier_score = float(
    brier_score_loss(
        step4_11_predictions_test_df["y_true"].astype(int),
        step4_11_predictions_test_df["predicted_probability"].astype(float),
    )
)

step4_13_persona_metric_rows = []

for step4_13_slice_name, step4_13_group_df in (
    step4_13_persona_input_df
    .sort_values(["activity_date", "station_id"])
    .groupby("persona_slice", dropna=False)
):
    step4_13_metric_row = compute_binary_metrics(
        step4_13_group_df["target_alert_day"],
        step4_13_group_df["predicted_label"],
        step4_13_group_df["predicted_probability"],
    )
    step4_13_metric_row["slice_family"] = "persona"
    step4_13_metric_row["slice_name"] = str(step4_13_slice_name)
    step4_13_metric_row["positive_share"] = float(step4_13_group_df["target_alert_day"].mean())
    step4_13_metric_row["mean_predicted_probability"] = float(step4_13_group_df["predicted_probability"].mean())
    step4_13_metric_row["unique_station_count"] = int(step4_13_group_df["station_id"].nunique())
    step4_13_metric_row["recall_vs_test_delta"] = (
        step4_13_metric_row["recall"] - float(step4_13_test_anchor_row["recall"])
    )
    step4_13_metric_row["specificity_vs_test_delta"] = (
        step4_13_metric_row["specificity"] - float(step4_13_test_anchor_row["specificity"])
    )
    step4_13_metric_row["balanced_accuracy_vs_test_delta"] = (
        step4_13_metric_row["balanced_accuracy"] - float(step4_13_test_anchor_row["balanced_accuracy"])
    )
    step4_13_metric_row["pr_auc_vs_test_delta"] = (
        step4_13_metric_row["pr_auc"] - float(step4_13_test_anchor_row["pr_auc"])
    )
    step4_13_metric_row["brier_vs_test_delta"] = (
        step4_13_metric_row["brier_score"] - step4_13_test_brier_score
    )
    step4_13_persona_metric_rows.append(step4_13_metric_row)

step4_13_persona_slice_report_df = (
    pd.DataFrame(step4_13_persona_metric_rows)
    .sort_values(["pr_auc", "balanced_accuracy", "specificity", "recall"], ascending=[False, False, False, False])
    .reset_index(drop=True)
)

display(step4_13_persona_slice_report_df)

assert step4_13_persona_slice_report_df.shape[0] == step4_13_supported_personas_df.shape[0], (
    "4.13.10 expected one metric row per supported persona."
)

,row_count,positive_count,negative_count,tp,tn,fp,fn,accuracy,precision,recall,...,slice_family,slice_name,positive_share,mean_predicted_probability,unique_station_count,recall_vs_test_delta,specificity_vs_test_delta,balanced_accuracy_vs_test_delta,pr_auc_vs_test_delta,brier_vs_test_delta
0,70149,57679,12470,56327,1348,11122,1352,0.822179,0.835105,0.976560,...,persona,mixed_other,0.822236,0.812345,317,0.009568,-0.034642,-0.012537,0.005625,-0.007649
1,5051,3084,1967,2538,617,1350,546,0.624629,0.652778,0.822957,...,persona,utility_short_trip,0.610572,0.635193,23,-0.144035,0.170934,0.013450,-0.201930,0.090543
2,780,435,345,313,145,200,122,0.587179,0.610136,0.719540,...,persona,commuter,0.557692,0.611803,4,-0.247452,0.277549,0.015048,-0.234306,0.101624


In [79]:
# 4.13.11 Scalenie raportów przekrojowych i ranking trudnych segmentów

import pandas as pd

required_objects = [
    "step4_13_hub_slice_report_df",
    "step4_13_cold_start_slice_report_df",
    "step4_13_microzone_slice_report_df",
    "step4_13_holiday_slice_report_df",
    "step4_13_weather_slice_report_df",
    "step4_13_persona_slice_report_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.13.11: {missing_objects}"

step4_13_slice_report_df = pd.concat(
    [
        step4_13_hub_slice_report_df.copy(),
        step4_13_cold_start_slice_report_df.copy(),
        step4_13_microzone_slice_report_df.copy(),
        step4_13_holiday_slice_report_df.copy(),
        step4_13_weather_slice_report_df.copy(),
        step4_13_persona_slice_report_df.copy(),
    ],
    axis=0,
    ignore_index=True,
)

step4_13_slice_report_df["is_low_support_segment"] = (
    (step4_13_slice_report_df["row_count"] < 1000)
    | (step4_13_slice_report_df["unique_station_count"] < 5)
).astype(int)

step4_13_slice_report_df["hard_segment_score"] = (
    (-1.0 * step4_13_slice_report_df["pr_auc_vs_test_delta"])
    + step4_13_slice_report_df["brier_vs_test_delta"]
    + (-1.0 * step4_13_slice_report_df["recall_vs_test_delta"]).clip(lower=0)
)

step4_13_slice_report_df = (
    step4_13_slice_report_df[
        [
            "slice_family",
            "slice_name",
            "row_count",
            "positive_count",
            "negative_count",
            "positive_share",
            "mean_predicted_probability",
            "pred_positive_rate",
            "accuracy",
            "precision",
            "recall",
            "specificity",
            "f1",
            "balanced_accuracy",
            "roc_auc",
            "pr_auc",
            "brier_score",
            "unique_station_count",
            "recall_vs_test_delta",
            "specificity_vs_test_delta",
            "balanced_accuracy_vs_test_delta",
            "pr_auc_vs_test_delta",
            "brier_vs_test_delta",
            "is_low_support_segment",
            "hard_segment_score",
        ]
    ]
    .sort_values(["slice_family", "slice_name"])
    .reset_index(drop=True)
)

step4_13_hard_segments_df = (
    step4_13_slice_report_df.loc[
        step4_13_slice_report_df["is_low_support_segment"].eq(0)
    ]
    .sort_values(
        ["hard_segment_score", "pr_auc_vs_test_delta", "brier_vs_test_delta"],
        ascending=[False, True, False],
    )
    .reset_index(drop=True)
)

display(step4_13_slice_report_df)
display(step4_13_hard_segments_df)

assert step4_13_slice_report_df.shape[0] == (
    step4_13_hub_slice_report_df.shape[0]
    + step4_13_cold_start_slice_report_df.shape[0]
    + step4_13_microzone_slice_report_df.shape[0]
    + step4_13_holiday_slice_report_df.shape[0]
    + step4_13_weather_slice_report_df.shape[0]
    + step4_13_persona_slice_report_df.shape[0]
), "4.13.11 expected all slice reports to be included in the final slice table."

,slice_family,slice_name,row_count,positive_count,negative_count,positive_share,mean_predicted_probability,pred_positive_rate,accuracy,precision,...,pr_auc,brier_score,unique_station_count,recall_vs_test_delta,specificity_vs_test_delta,balanced_accuracy_vs_test_delta,pr_auc_vs_test_delta,brier_vs_test_delta,is_low_support_segment,hard_segment_score
0,cold_start_vs_rest,cold_start,14,6,8,0.428571,0.648856,0.857143,0.285714,0.333333,...,0.353587,0.383918,2,-0.300326,-0.142741,-0.221533,-0.562364,0.244163,1,1.106853
1,cold_start_vs_rest,non_cold_start,75911,61192,14719,0.806102,0.798824,0.946147,0.806787,0.823886,...,0.916028,0.139663,344,0.000029,-0.002107,-0.001039,0.000077,-0.000092,0,-0.000169
2,holiday_vs_rest,holiday,2695,2105,590,0.781076,0.775248,0.970315,0.780334,0.789293,...,0.881963,0.158995,343,0.013530,-0.076639,-0.031555,-0.033988,0.019240,0,0.053229
3,holiday_vs_rest,non_holiday,73285,59093,14192,0.806345,0.799365,0.944736,0.807600,0.824930,...,0.917032,0.139047,344,-0.000482,0.003186,0.001352,0.001081,-0.000708,0,-0.001306
4,hub_vs_rest,hub,11132,9365,1767,0.841268,0.836061,0.950323,0.840460,0.858682,...,0.948417,0.114374,50,0.003002,0.011192,0.007097,0.032466,-0.025380,0,-0.057846
5,hub_vs_rest,non_hub,64848,51833,13015,0.799300,0.792063,0.944840,0.800827,0.817581,...,0.908442,0.144112,294,-0.000542,-0.001520,-0.001031,-0.007509,0.004357,0,0.012408
6,microzone,MZ_0044,668,565,103,0.845808,0.854417,0.994012,0.842814,0.846386,...,0.894740,0.130535,3,0.027698,-0.133032,-0.052667,-0.021211,-0.009220,1,0.011991
7,microzone,MZ_0054,888,591,297,0.665541,0.740965,0.941441,0.674550,0.680622,...,0.712162,0.227352,4,-0.004217,-0.041731,-0.022974,-0.203789,0.087597,1,0.295603
8,microzone,MZ_0064,668,493,175,0.738024,0.783566,0.947605,0.730539,0.747235,...,0.835467,0.186981,3,-0.007560,-0.057027,-0.032294,-0.080484,0.047226,1,0.135270
9,microzone,MZ_0067,1782,1144,638,0.641975,0.718207,0.900112,0.638608,0.655860,...,0.721968,0.231105,8,-0.047412,-0.007945,-0.027678,-0.193983,0.091350,0,0.332745


,slice_family,slice_name,row_count,positive_count,negative_count,positive_share,mean_predicted_probability,pred_positive_rate,accuracy,precision,...,pr_auc,brier_score,unique_station_count,recall_vs_test_delta,specificity_vs_test_delta,balanced_accuracy_vs_test_delta,pr_auc_vs_test_delta,brier_vs_test_delta,is_low_support_segment,hard_segment_score
0,persona,utility_short_trip,5051,3084,1967,0.610572,0.635193,0.769749,0.624629,0.652778,...,0.714020,0.230298,23,-0.144035,0.170934,0.013450,-0.201930,0.090543,0,0.436509
1,microzone,MZ_0067,1782,1144,638,0.641975,0.718207,0.900112,0.638608,0.655860,...,0.721968,0.231105,8,-0.047412,-0.007945,-0.027678,-0.193983,0.091350,0,0.332745
2,holiday_vs_rest,holiday,2695,2105,590,0.781076,0.775248,0.970315,0.780334,0.789293,...,0.881963,0.158995,343,0.013530,-0.076639,-0.031555,-0.033988,0.019240,0,0.053229
3,weather_proxy,weather_low,25499,20303,5196,0.796227,0.771965,0.931488,0.795247,0.817489,...,0.908017,0.146662,344,-0.010631,0.022963,0.006166,-0.007933,0.006907,0,0.025472
4,hub_vs_rest,non_hub,64848,51833,13015,0.799300,0.792063,0.944840,0.800827,0.817581,...,0.908442,0.144112,294,-0.000542,-0.001520,-0.001031,-0.007509,0.004357,0,0.012408
5,cold_start_vs_rest,non_cold_start,75911,61192,14719,0.806102,0.798824,0.946147,0.806787,0.823886,...,0.916028,0.139663,344,0.000029,-0.002107,-0.001039,0.000077,-0.000092,0,-0.000169
6,weather_proxy,weather_mid,25397,20302,5095,0.799386,0.805059,0.945505,0.804268,0.819223,...,0.917403,0.140631,344,0.001976,0.005247,0.003612,0.001452,0.000876,0,-0.000576
7,holiday_vs_rest,non_holiday,73285,59093,14192,0.806345,0.799365,0.944736,0.807600,0.824930,...,0.917032,0.139047,344,-0.000482,0.003186,0.001352,0.001081,-0.000708,0,-0.001306
8,persona,mixed_other,70149,57679,12470,0.822236,0.812345,0.961510,0.822179,0.835105,...,0.921575,0.132105,317,0.009568,-0.034642,-0.012537,0.005625,-0.007649,0,-0.013274
9,weather_proxy,weather_high,25084,20593,4491,0.820962,0.818862,0.960174,0.820603,0.834088,...,0.922304,0.131846,344,0.008533,-0.032521,-0.011994,0.006354,-0.007909,0,-0.014262


In [80]:
# 4.13.12 Zapis artefaktu sekcji 4.13

from pathlib import Path

import pandas as pd

required_objects = [
    "step4_3_project_root",
    "step4_13_slice_report_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.13.12: {missing_objects}"

step4_13_output_dir = step4_3_project_root / "artifacts" / "4_slice_report"
step4_13_output_dir.mkdir(parents=True, exist_ok=True)

step4_13_slice_report_path = step4_13_output_dir / "b4_13_slice_report.parquet"

step4_13_slice_report_save_df = (
    step4_13_slice_report_df.copy()
    .reset_index(drop=True)
)

step4_13_slice_report_save_df.to_parquet(
    step4_13_slice_report_path,
    index=False,
)

step4_13_saved_artifacts_registry_df = pd.DataFrame(
    [
        {
            "artifact_name": step4_13_slice_report_path.name,
            "rows": int(step4_13_slice_report_save_df.shape[0]),
            "cols": int(step4_13_slice_report_save_df.shape[1]),
            "path": str(step4_13_slice_report_path),
        }
    ]
)

display(step4_13_saved_artifacts_registry_df)

assert step4_13_slice_report_path.exists(), (
    "4.13.12 failed to save b4_13_slice_report.parquet"
)
assert step4_13_slice_report_save_df.shape[0] == 19, (
    "4.13.12 expected 19 rows in saved slice report artifact."
)

,artifact_name,rows,cols,path
0,b4_13_slice_report.parquet,19,25,/root/projects/6_samodzielny_projekt_koncowy_w...


## Podsumowanie działu 4.13 Huby vs reszta i raporty przekrojowe

**Kontekst inżynieryjny:** W dziale `4.13` wykonano biznesowy slicing jakości finalnego modelu na zbiorze `test`. Najpierw zbudowano kompletną ramkę segmentacyjną łączącą predykcje testowe z informacjami pomocniczymi o stacjach, mikro-strefach, personach, świętach oraz proxy pogody. Następnie policzono metryki segmentowe dla przekrojów: `hub vs non_hub`, `cold_start vs non_cold_start`, wspierane mikro-strefy, `holiday vs non_holiday`, segmenty pogodowe `weather_low / weather_mid / weather_high` oraz persony stacji. Na końcu scalono wszystkie wyniki do jednej tabeli przekrojowej, zbudowano ranking trudnych segmentów i zapisano artefakt `b4_13_slice_report.parquet`.

**Interpretacja wyniku:** Analiza pokazała, że jakość modelu nie jest jednakowa między segmentami biznesowymi. `Huby` okazały się mocniejszym segmentem niż `non_hub`, z lepszym `PR AUC` i lepszym profilem probabilistycznym. Segment `cold_start` był obecny w śladowej skali (`14` wierszy, `2` stacje), dlatego jego wynik ma wyłącznie charakter orientacyjny i nie powinien być traktowany jako twardy wniosek o jakości modelu. W mikro-strefach ujawniły się lokalne słabe punkty, szczególnie `MZ_0067`, która była największą wspieraną mikro-strefą i jednocześnie wyraźnie słabszym segmentem jakościowym. W przekroju świątecznym model zachował wysoki `recall`, ale ogólna jakość na świętach była słabsza niż poza świętami. W segmencie pogodowym najlepszy profil jakości pojawił się przy `weather_high`, słabszy przy `weather_low`, a `weather_mid` pozostał najbliżej średniego zachowania testowego. W przekroju person model najlepiej działał dla `mixed_other`, wyraźnie słabiej dla `utility_short_trip`, natomiast `commuter` wypadł najsłabiej, ale przy małej liczbie stacji.

**Znaczenie biznesowe:** Dział `4.13` pokazał, że średni wynik testowy nie opisuje w pełni zachowania modelu w realnych segmentach operacyjnych. Model jest najmocniejszy tam, gdzie wzorzec działania stacji jest bardziej przewidywalny i lepiej reprezentowany w danych, a słabnie w wybranych lokalnych oraz specyficznych segmentach biznesowych. Najważniejsze praktyczne obserwacje są trzy: `huby` nie stanowią słabego punktu modelu, święta są trudniejszym reżimem operacyjnym, a najbardziej problematyczne segmenty o sensownym wsparciu to `utility_short_trip` oraz `MZ_0067`. Dzięki temu analiza przekrojowa nie tylko ocenia model globalnie, ale wskazuje, gdzie dokładnie ryzyko błędu jest większe.

**Wniosek:** Dział `4.13` został wykonany poprawnie i zakończył się pełnym raportem przekrojowym jakości modelu. Finalna analiza potwierdziła, że model ma mocniejsze i słabsze segmenty biznesowe, a ranking trudnych segmentów pozwala te obszary jasno nazwać i uzasadnić. Zapisano artefakt `b4_13_slice_report.parquet`, a projekt jest gotowy do przejścia do `4.14`, czyli error analysis i interpretowalności modelu.

## 4.14 Error analysis i interpretowalność

In [81]:
# 4.14.1 Weryfikacja wejścia do error analysis i budowa ramki błędów na teście

import numpy as np
import pandas as pd

required_objects = [
    "step4_11_predictions_test_df",
    "step4_13_slice_input_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.14.1: {missing_objects}"

assert step4_11_predictions_test_df.shape[0] == step4_13_slice_input_df.shape[0], (
    "4.14.1 expected test predictions and slice input to have the same row count."
)

step4_14_required_slice_cols = [
    "activity_date",
    "station_id",
    "target_alert_day",
    "hub_flag_slice",
    "is_cold_start_slice",
    "microzone_slice",
    "holiday_flag_slice",
    "persona_slice",
    "weather_proxy_slice",
]
step4_14_missing_slice_cols = [
    col for col in step4_14_required_slice_cols
    if col not in step4_13_slice_input_df.columns
]
assert not step4_14_missing_slice_cols, (
    f"Missing required slice columns in 4.14.1: {step4_14_missing_slice_cols}"
)

step4_14_error_analysis_input_df = pd.concat(
    [
        step4_13_slice_input_df[step4_14_required_slice_cols].reset_index(drop=True),
        step4_11_predictions_test_df[
            [
                "y_true",
                "predicted_probability",
                "predicted_label",
                "selected_threshold",
                "selected_model_name",
                "selected_calibration_method",
            ]
        ].reset_index(drop=True),
    ],
    axis=1,
)

step4_14_error_analysis_input_df["activity_date"] = pd.to_datetime(
    step4_14_error_analysis_input_df["activity_date"],
    errors="coerce",
)

step4_14_error_analysis_input_df["hub_flag_slice"] = pd.to_numeric(
    step4_14_error_analysis_input_df["hub_flag_slice"],
    errors="coerce",
)
step4_14_error_analysis_input_df["is_cold_start_slice"] = pd.to_numeric(
    step4_14_error_analysis_input_df["is_cold_start_slice"],
    errors="coerce",
)
step4_14_error_analysis_input_df["holiday_flag_slice"] = (
    step4_14_error_analysis_input_df["holiday_flag_slice"]
    .replace({True: 1, False: 0, "True": 1, "False": 0, "true": 1, "false": 0})
)
step4_14_error_analysis_input_df["holiday_flag_slice"] = pd.to_numeric(
    step4_14_error_analysis_input_df["holiday_flag_slice"],
    errors="coerce",
)
step4_14_error_analysis_input_df["weather_proxy_slice"] = pd.to_numeric(
    step4_14_error_analysis_input_df["weather_proxy_slice"],
    errors="coerce",
)

step4_14_error_analysis_input_df["microzone_slice"] = (
    step4_14_error_analysis_input_df["microzone_slice"]
    .astype("string")
    .str.strip()
)
step4_14_error_analysis_input_df["persona_slice"] = (
    step4_14_error_analysis_input_df["persona_slice"]
    .astype("string")
    .str.strip()
)

step4_14_error_analysis_input_df["weather_segment"] = pd.qcut(
    step4_14_error_analysis_input_df["weather_proxy_slice"],
    q=3,
    labels=["weather_low", "weather_mid", "weather_high"],
    duplicates="drop",
)

step4_14_conditions = [
    (step4_14_error_analysis_input_df["y_true"] == 1) & (step4_14_error_analysis_input_df["predicted_label"] == 1),
    (step4_14_error_analysis_input_df["y_true"] == 0) & (step4_14_error_analysis_input_df["predicted_label"] == 0),
    (step4_14_error_analysis_input_df["y_true"] == 0) & (step4_14_error_analysis_input_df["predicted_label"] == 1),
    (step4_14_error_analysis_input_df["y_true"] == 1) & (step4_14_error_analysis_input_df["predicted_label"] == 0),
]
step4_14_labels = ["tp", "tn", "fp", "fn"]

step4_14_error_analysis_input_df["outcome_type"] = np.select(
    step4_14_conditions,
    step4_14_labels,
    default="unknown",
)

step4_14_error_analysis_input_df["is_error"] = (
    step4_14_error_analysis_input_df["outcome_type"].isin(["fp", "fn"])
).astype("int8")
step4_14_error_analysis_input_df["is_false_positive"] = (
    step4_14_error_analysis_input_df["outcome_type"].eq("fp")
).astype("int8")
step4_14_error_analysis_input_df["is_false_negative"] = (
    step4_14_error_analysis_input_df["outcome_type"].eq("fn")
).astype("int8")

step4_14_error_analysis_input_df["is_hub_error"] = (
    step4_14_error_analysis_input_df["is_error"].eq(1)
    & step4_14_error_analysis_input_df["hub_flag_slice"].eq(1)
).astype("int8")
step4_14_error_analysis_input_df["is_cold_start_error"] = (
    step4_14_error_analysis_input_df["is_error"].eq(1)
    & step4_14_error_analysis_input_df["is_cold_start_slice"].eq(1)
).astype("int8")
step4_14_error_analysis_input_df["is_holiday_error"] = (
    step4_14_error_analysis_input_df["is_error"].eq(1)
    & step4_14_error_analysis_input_df["holiday_flag_slice"].eq(1)
).astype("int8")
step4_14_error_analysis_input_df["is_weather_low_error"] = (
    step4_14_error_analysis_input_df["is_error"].eq(1)
    & step4_14_error_analysis_input_df["weather_segment"].astype("string").eq("weather_low")
).astype("int8")
step4_14_error_analysis_input_df["is_utility_short_trip_error"] = (
    step4_14_error_analysis_input_df["is_error"].eq(1)
    & step4_14_error_analysis_input_df["persona_slice"].eq("utility_short_trip")
).astype("int8")
step4_14_error_analysis_input_df["is_mz_0067_error"] = (
    step4_14_error_analysis_input_df["is_error"].eq(1)
    & step4_14_error_analysis_input_df["microzone_slice"].eq("MZ_0067")
).astype("int8")

step4_14_error_summary_df = pd.DataFrame(
    [
        {
            "row_count": int(step4_14_error_analysis_input_df.shape[0]),
            "error_count": int(step4_14_error_analysis_input_df["is_error"].sum()),
            "false_positive_count": int(step4_14_error_analysis_input_df["is_false_positive"].sum()),
            "false_negative_count": int(step4_14_error_analysis_input_df["is_false_negative"].sum()),
            "tp_count": int(step4_14_error_analysis_input_df["outcome_type"].eq("tp").sum()),
            "tn_count": int(step4_14_error_analysis_input_df["outcome_type"].eq("tn").sum()),
            "error_rate": float(step4_14_error_analysis_input_df["is_error"].mean()),
            "fp_share_of_errors": float(
                step4_14_error_analysis_input_df["is_false_positive"].sum()
                / step4_14_error_analysis_input_df["is_error"].sum()
            ),
            "fn_share_of_errors": float(
                step4_14_error_analysis_input_df["is_false_negative"].sum()
                / step4_14_error_analysis_input_df["is_error"].sum()
            ),
            "hub_error_count": int(step4_14_error_analysis_input_df["is_hub_error"].sum()),
            "cold_start_error_count": int(step4_14_error_analysis_input_df["is_cold_start_error"].sum()),
            "holiday_error_count": int(step4_14_error_analysis_input_df["is_holiday_error"].sum()),
            "weather_low_error_count": int(step4_14_error_analysis_input_df["is_weather_low_error"].sum()),
            "utility_short_trip_error_count": int(step4_14_error_analysis_input_df["is_utility_short_trip_error"].sum()),
            "mz_0067_error_count": int(step4_14_error_analysis_input_df["is_mz_0067_error"].sum()),
        }
    ]
)

display(step4_14_error_summary_df)
display(step4_14_error_analysis_input_df.head(10))

assert set(step4_14_error_analysis_input_df["outcome_type"].unique()) == {"tp", "tn", "fp", "fn"}, (
    "4.14.1 expected exactly tp, tn, fp and fn outcomes."
)

,row_count,error_count,false_positive_count,false_negative_count,tp_count,tn_count,error_rate,fp_share_of_errors,fn_share_of_errors,hub_error_count,cold_start_error_count,holiday_error_count,weather_low_error_count,utility_short_trip_error_count,mz_0067_error_count
0,75980,14692,12672,2020,59178,2110,0.193367,0.86251,0.13749,1776,10,592,5221,1896,644


,activity_date,station_id,target_alert_day,hub_flag_slice,is_cold_start_slice,microzone_slice,holiday_flag_slice,persona_slice,weather_proxy_slice,y_true,...,outcome_type,is_error,is_false_positive,is_false_negative,is_hub_error,is_cold_start_error,is_holiday_error,is_weather_low_error,is_utility_short_trip_error,is_mz_0067_error
0,2020-03-23,1,0,0,0.0,MZ_0012,0,mixed_other,2.699652,0,...,tn,0,0,0,0,0,0,0,0,0
1,2020-03-23,10,1,0,0.0,MZ_0049,0,utility_short_trip,2.699652,1,...,fn,1,0,1,0,0,0,1,1,0
2,2020-03-23,100,1,0,0.0,MZ_0227,0,mixed_other,2.699652,1,...,fn,1,0,1,0,0,0,1,0,0
3,2020-03-23,101,1,1,0.0,MZ_0191,0,mixed_other,2.699652,1,...,fn,1,0,1,1,0,0,1,0,0
4,2020-03-23,103,0,0,0.0,MZ_0220,0,mixed_other,2.699652,0,...,tn,0,0,0,0,0,0,0,0,0
5,2020-03-23,104,1,0,0.0,MZ_0210,0,mixed_other,2.699652,1,...,fn,1,0,1,0,0,0,1,0,0
6,2020-03-23,105,1,0,0.0,MZ_0197,0,mixed_other,2.699652,1,...,fn,1,0,1,0,0,0,1,0,0
7,2020-03-23,106,1,0,0.0,MZ_0200,0,mixed_other,2.699652,1,...,fn,1,0,1,0,0,0,1,0,0
8,2020-03-23,107,0,0,0.0,MZ_0195,0,mixed_other,2.699652,0,...,tn,0,0,0,0,0,0,0,0,0
9,2020-03-23,108,1,0,0.0,MZ_0208,0,mixed_other,2.699652,1,...,fn,1,0,1,0,0,0,1,0,0


In [82]:
# 4.14.2 Feature importance modelowe dla finalnego LightGBM

import numpy as np
import pandas as pd

required_objects = [
    "step4_11_refit_model",
    "step4_11_X_refit_model_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.14.2: {missing_objects}"

assert hasattr(step4_11_refit_model, "booster_"), (
    "4.14.2 expected fitted LightGBM model with booster_."
)

step4_14_feature_names = list(step4_11_X_refit_model_df.columns)

step4_14_gain_importance = step4_11_refit_model.booster_.feature_importance(
    importance_type="gain"
)
step4_14_split_importance = step4_11_refit_model.booster_.feature_importance(
    importance_type="split"
)

assert len(step4_14_feature_names) == len(step4_14_gain_importance) == len(step4_14_split_importance), (
    "4.14.2 feature importance length mismatch."
)

step4_14_feature_importance_df = pd.DataFrame(
    {
        "feature_name": step4_14_feature_names,
        "gain_importance": step4_14_gain_importance.astype(float),
        "split_importance": step4_14_split_importance.astype(float),
    }
)

step4_14_total_gain = float(step4_14_feature_importance_df["gain_importance"].sum())
step4_14_total_split = float(step4_14_feature_importance_df["split_importance"].sum())

step4_14_feature_importance_df["gain_importance_share"] = np.where(
    step4_14_total_gain > 0,
    step4_14_feature_importance_df["gain_importance"] / step4_14_total_gain,
    np.nan,
)
step4_14_feature_importance_df["split_importance_share"] = np.where(
    step4_14_total_split > 0,
    step4_14_feature_importance_df["split_importance"] / step4_14_total_split,
    np.nan,
)

step4_14_feature_importance_df["gain_rank"] = (
    step4_14_feature_importance_df["gain_importance"]
    .rank(method="dense", ascending=False)
    .astype(int)
)
step4_14_feature_importance_df["split_rank"] = (
    step4_14_feature_importance_df["split_importance"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

step4_14_feature_importance_df = (
    step4_14_feature_importance_df
    .sort_values(
        ["gain_importance", "split_importance", "feature_name"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

step4_14_feature_importance_summary_df = pd.DataFrame(
    [
        {
            "feature_count": int(step4_14_feature_importance_df.shape[0]),
            "nonzero_gain_features": int((step4_14_feature_importance_df["gain_importance"] > 0).sum()),
            "nonzero_split_features": int((step4_14_feature_importance_df["split_importance"] > 0).sum()),
            "top1_feature": str(step4_14_feature_importance_df.loc[0, "feature_name"]),
            "top1_gain_share": float(step4_14_feature_importance_df.loc[0, "gain_importance_share"]),
            "top5_gain_share_sum": float(step4_14_feature_importance_df["gain_importance_share"].head(5).sum()),
            "top10_gain_share_sum": float(step4_14_feature_importance_df["gain_importance_share"].head(10).sum()),
        }
    ]
)

display(step4_14_feature_importance_summary_df)
display(step4_14_feature_importance_df.head(20))

assert step4_14_feature_importance_df.shape[0] == step4_11_X_refit_model_df.shape[1], (
    "4.14.2 expected one importance row per model feature."
)

,feature_count,nonzero_gain_features,nonzero_split_features,top1_feature,top1_gain_share,top5_gain_share_sum,top10_gain_share_sum
0,34,32,32,alert_hours_roll_sum_14,0.433086,0.817816,0.937575


,feature_name,gain_importance,split_importance,gain_importance_share,split_importance_share,gain_rank,split_rank
0,alert_hours_roll_sum_14,153504.989200,734.0,0.433086,0.174762,1,1
1,alert_hours_lag_1,59710.238619,636.0,0.168461,0.151429,2,3
2,consecutive_alert_days_before_t,34693.097674,483.0,0.097880,0.115000,3,4
3,alert_severity_roll_max_7,24034.000782,683.0,0.067807,0.162619,4,2
4,alert_lag_7,17928.393329,133.0,0.050582,0.031667,5,6
5,is_business_free_day,16594.344774,296.0,0.046818,0.070476,6,5
6,is_any_missing_signal,13699.238294,118.0,0.038650,0.028095,7,7
7,is_missing_returns_history,4084.582842,108.0,0.011524,0.025714,8,8
8,is_missing_alert_history,4044.043079,44.0,0.011410,0.010476,9,19
9,alert_lag_2,4025.679412,59.0,0.011358,0.014048,10,15


In [85]:
# 4.14.3 Permutation importance na próbce testowej

import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score

required_objects = [
    "step4_11_refit_model",
    "step4_5_X_test_model_df",
    "step4_5_y_test",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.14.3: {missing_objects}"

# próbka testowa z zachowaniem proporcji klas
step4_14_perm_sample_size = min(12000, int(step4_5_X_test_model_df.shape[0]))
step4_14_perm_y_test_series = pd.Series(step4_5_y_test).reset_index(drop=True).astype(int)

step4_14_perm_pos_idx = step4_14_perm_y_test_series[step4_14_perm_y_test_series.eq(1)].index
step4_14_perm_neg_idx = step4_14_perm_y_test_series[step4_14_perm_y_test_series.eq(0)].index

step4_14_perm_pos_n = int(round(step4_14_perm_sample_size * step4_14_perm_y_test_series.mean()))
step4_14_perm_pos_n = min(step4_14_perm_pos_n, len(step4_14_perm_pos_idx))
step4_14_perm_neg_n = step4_14_perm_sample_size - step4_14_perm_pos_n
step4_14_perm_neg_n = min(step4_14_perm_neg_n, len(step4_14_perm_neg_idx))

if (step4_14_perm_pos_n + step4_14_perm_neg_n) < step4_14_perm_sample_size:
    step4_14_perm_remaining = step4_14_perm_sample_size - (step4_14_perm_pos_n + step4_14_perm_neg_n)

    step4_14_extra_pos = min(
        step4_14_perm_remaining,
        len(step4_14_perm_pos_idx) - step4_14_perm_pos_n,
    )
    step4_14_perm_pos_n += step4_14_extra_pos
    step4_14_perm_remaining -= step4_14_extra_pos

    step4_14_extra_neg = min(
        step4_14_perm_remaining,
        len(step4_14_perm_neg_idx) - step4_14_perm_neg_n,
    )
    step4_14_perm_neg_n += step4_14_extra_neg

step4_14_perm_sample_idx = pd.Index(
    np.concatenate(
        [
            step4_14_perm_pos_idx.to_series().sample(step4_14_perm_pos_n, random_state=42).to_numpy(),
            step4_14_perm_neg_idx.to_series().sample(step4_14_perm_neg_n, random_state=42).to_numpy(),
        ]
    )
).sort_values()

# pełna macierz 34 cech zgodna z treningiem modelu
step4_14_perm_X_sample_df = (
    step4_5_X_test_model_df.iloc[step4_14_perm_sample_idx]
    .reset_index(drop=True)
)
step4_14_perm_y_sample = (
    step4_14_perm_y_test_series.iloc[step4_14_perm_sample_idx]
    .reset_index(drop=True)
)

assert step4_14_perm_X_sample_df.shape[1] == step4_5_X_test_model_df.shape[1], (
    "4.14.3 expected full feature matrix for permutation importance."
)

def pr_auc_scorer(estimator, X, y):
    y_score = estimator.predict_proba(X)[:, 1]
    return average_precision_score(y, y_score)

step4_14_permutation_result = permutation_importance(
    estimator=step4_11_refit_model,
    X=step4_14_perm_X_sample_df,
    y=step4_14_perm_y_sample,
    scoring=pr_auc_scorer,
    n_repeats=5,
    random_state=42,
    n_jobs=1,
)

step4_14_permutation_importance_df = pd.DataFrame(
    {
        "feature_name": list(step4_14_perm_X_sample_df.columns),
        "permutation_importance_mean": step4_14_permutation_result.importances_mean.astype(float),
        "permutation_importance_std": step4_14_permutation_result.importances_std.astype(float),
    }
)

step4_14_permutation_importance_df["abs_permutation_importance_mean"] = (
    step4_14_permutation_importance_df["permutation_importance_mean"].abs()
)
step4_14_permutation_importance_df["permutation_rank"] = (
    step4_14_permutation_importance_df["abs_permutation_importance_mean"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

step4_14_permutation_importance_df = (
    step4_14_permutation_importance_df
    .sort_values(
        ["abs_permutation_importance_mean", "permutation_importance_std", "feature_name"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

# lekki podgląd top 12 do czytania
step4_14_permutation_importance_top12_df = (
    step4_14_permutation_importance_df
    .head(12)
    .copy()
    .reset_index(drop=True)
)

step4_14_permutation_importance_summary_df = pd.DataFrame(
    [
        {
            "sample_rows": int(step4_14_perm_X_sample_df.shape[0]),
            "feature_count_full": int(step4_14_permutation_importance_df.shape[0]),
            "feature_count_top12": int(step4_14_permutation_importance_top12_df.shape[0]),
            "positive_share_in_sample": float(step4_14_perm_y_sample.mean()),
            "top1_feature_full": str(step4_14_permutation_importance_df.loc[0, "feature_name"]),
            "top1_permutation_importance_full": float(
                step4_14_permutation_importance_df.loc[0, "permutation_importance_mean"]
            ),
            "top5_abs_permutation_sum_full": float(
                step4_14_permutation_importance_df["abs_permutation_importance_mean"].head(5).sum()
            ),
        }
    ]
)

display(step4_14_permutation_importance_summary_df)
display(step4_14_permutation_importance_df)
display(step4_14_permutation_importance_top12_df)

assert step4_14_permutation_importance_df.shape[0] == step4_5_X_test_model_df.shape[1], (
    "4.14.3 expected one permutation row per full model feature."
)
assert step4_14_permutation_importance_top12_df.shape[0] == 12, (
    "4.14.3 expected top12 preview."
)

,sample_rows,feature_count_full,feature_count_top12,positive_share_in_sample,top1_feature_full,top1_permutation_importance_full,top5_abs_permutation_sum_full
0,12000,34,12,0.805417,alert_hours_roll_sum_14,0.016926,0.045257


,feature_name,permutation_importance_mean,permutation_importance_std,abs_permutation_importance_mean,permutation_rank
0,alert_hours_roll_sum_14,0.016926,0.001485,0.016926,1
1,alert_hours_lag_1,0.010605,0.001384,0.010605,2
2,alert_severity_roll_max_7,0.006592,0.000712,0.006592,3
3,consecutive_alert_days_before_t,0.006335,0.001195,0.006335,4
4,is_business_free_day,0.004799,0.000440,0.004799,5
5,alert_lag_7,0.000924,0.000251,0.000924,6
6,hub_flag,0.000470,0.000225,0.000470,7
7,deficit_alert_lag_1,0.000331,0.000202,0.000331,8
8,is_any_missing_signal,0.000331,0.000052,0.000331,9
9,is_missing_returns_history,0.000176,0.000062,0.000176,10


,feature_name,permutation_importance_mean,permutation_importance_std,abs_permutation_importance_mean,permutation_rank
0,alert_hours_roll_sum_14,0.016926,0.001485,0.016926,1
1,alert_hours_lag_1,0.010605,0.001384,0.010605,2
2,alert_severity_roll_max_7,0.006592,0.000712,0.006592,3
3,consecutive_alert_days_before_t,0.006335,0.001195,0.006335,4
4,is_business_free_day,0.004799,0.000440,0.004799,5
5,alert_lag_7,0.000924,0.000251,0.000924,6
6,hub_flag,0.000470,0.000225,0.000470,7
7,deficit_alert_lag_1,0.000331,0.000202,0.000331,8
8,is_any_missing_signal,0.000331,0.000052,0.000331,9
9,is_missing_returns_history,0.000176,0.000062,0.000176,10


In [87]:
# 4.14.4 SHAP na próbce testowej

import numpy as np
import pandas as pd

required_objects = [
    "step4_11_refit_model",
    "step4_5_X_test_model_df",
    "step4_5_y_test",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.14.4: {missing_objects}"

assert hasattr(step4_11_refit_model, "booster_"), (
    "4.14.4 expected fitted LightGBM model with booster_."
)

# próbka testowa z zachowaniem proporcji klas
step4_14_shap_sample_size = min(3000, int(step4_5_X_test_model_df.shape[0]))
step4_14_shap_y_test_series = pd.Series(step4_5_y_test).reset_index(drop=True).astype(int)

step4_14_shap_pos_idx = step4_14_shap_y_test_series[step4_14_shap_y_test_series.eq(1)].index
step4_14_shap_neg_idx = step4_14_shap_y_test_series[step4_14_shap_y_test_series.eq(0)].index

step4_14_shap_pos_n = int(round(step4_14_shap_sample_size * step4_14_shap_y_test_series.mean()))
step4_14_shap_pos_n = min(step4_14_shap_pos_n, len(step4_14_shap_pos_idx))
step4_14_shap_neg_n = step4_14_shap_sample_size - step4_14_shap_pos_n
step4_14_shap_neg_n = min(step4_14_shap_neg_n, len(step4_14_shap_neg_idx))

if (step4_14_shap_pos_n + step4_14_shap_neg_n) < step4_14_shap_sample_size:
    step4_14_shap_remaining = step4_14_shap_sample_size - (step4_14_shap_pos_n + step4_14_shap_neg_n)

    step4_14_shap_extra_pos = min(
        step4_14_shap_remaining,
        len(step4_14_shap_pos_idx) - step4_14_shap_pos_n,
    )
    step4_14_shap_pos_n += step4_14_shap_extra_pos
    step4_14_shap_remaining -= step4_14_shap_extra_pos

    step4_14_shap_extra_neg = min(
        step4_14_shap_remaining,
        len(step4_14_shap_neg_idx) - step4_14_shap_neg_n,
    )
    step4_14_shap_neg_n += step4_14_shap_extra_neg

step4_14_shap_sample_idx = pd.Index(
    np.concatenate(
        [
            step4_14_shap_pos_idx.to_series().sample(step4_14_shap_pos_n, random_state=42).to_numpy(),
            step4_14_shap_neg_idx.to_series().sample(step4_14_shap_neg_n, random_state=42).to_numpy(),
        ]
    )
).sort_values()

step4_14_shap_X_sample_df = (
    step4_5_X_test_model_df.iloc[step4_14_shap_sample_idx]
    .reset_index(drop=True)
)
step4_14_shap_y_sample = (
    step4_14_shap_y_test_series.iloc[step4_14_shap_sample_idx]
    .reset_index(drop=True)
)

# wkłady cech typu SHAP z LightGBM bez zewnętrznej biblioteki shap
step4_14_pred_contrib = step4_11_refit_model.booster_.predict(
    step4_14_shap_X_sample_df,
    pred_contrib=True,
)

step4_14_pred_contrib = np.asarray(step4_14_pred_contrib)

assert step4_14_pred_contrib.shape[0] == step4_14_shap_X_sample_df.shape[0], (
    "4.14.4 contribution row count mismatch."
)
assert step4_14_pred_contrib.shape[1] == (step4_14_shap_X_sample_df.shape[1] + 1), (
    "4.14.4 expected feature contributions plus expected value column."
)

step4_14_shap_values = step4_14_pred_contrib[:, :-1]
step4_14_shap_expected_value = step4_14_pred_contrib[:, -1]

step4_14_shap_importance_df = pd.DataFrame(
    {
        "feature_name": list(step4_14_shap_X_sample_df.columns),
        "mean_abs_shap_value": np.abs(step4_14_shap_values).mean(axis=0).astype(float),
        "mean_shap_value": step4_14_shap_values.mean(axis=0).astype(float),
        "feature_sample_mean": step4_14_shap_X_sample_df.mean(axis=0).astype(float).to_numpy(),
    }
)

step4_14_shap_total = float(step4_14_shap_importance_df["mean_abs_shap_value"].sum())

step4_14_shap_importance_df["mean_abs_shap_share"] = np.where(
    step4_14_shap_total > 0,
    step4_14_shap_importance_df["mean_abs_shap_value"] / step4_14_shap_total,
    np.nan,
)
step4_14_shap_importance_df["shap_rank"] = (
    step4_14_shap_importance_df["mean_abs_shap_value"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

step4_14_shap_importance_df = (
    step4_14_shap_importance_df
    .sort_values(
        ["mean_abs_shap_value", "feature_name"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

step4_14_shap_summary_df = pd.DataFrame(
    [
        {
            "sample_rows": int(step4_14_shap_X_sample_df.shape[0]),
            "feature_count": int(step4_14_shap_importance_df.shape[0]),
            "positive_share_in_sample": float(step4_14_shap_y_sample.mean()),
            "expected_value_mean": float(np.mean(step4_14_shap_expected_value)),
            "top1_feature": str(step4_14_shap_importance_df.loc[0, "feature_name"]),
            "top1_mean_abs_shap_value": float(step4_14_shap_importance_df.loc[0, "mean_abs_shap_value"]),
            "top5_mean_abs_shap_share_sum": float(
                step4_14_shap_importance_df["mean_abs_shap_share"].head(5).sum()
            ),
            "top10_mean_abs_shap_share_sum": float(
                step4_14_shap_importance_df["mean_abs_shap_share"].head(10).sum()
            ),
        }
    ]
)

display(step4_14_shap_summary_df)
display(step4_14_shap_importance_df.head(20))

assert step4_14_shap_importance_df.shape[0] == step4_14_shap_X_sample_df.shape[1], (
    "4.14.4 expected one SHAP row per model feature."
)

,sample_rows,feature_count,positive_share_in_sample,expected_value_mean,top1_feature,top1_mean_abs_shap_value,top5_mean_abs_shap_share_sum,top10_mean_abs_shap_share_sum
0,3000,34,0.805333,1.91984,alert_hours_roll_sum_14,0.417202,0.779802,0.921855


,feature_name,mean_abs_shap_value,mean_shap_value,feature_sample_mean,mean_abs_shap_share,shap_rank
0,alert_hours_roll_sum_14,0.417202,-0.077871,187.511002,0.255041,1
1,alert_hours_lag_1,0.277922,-0.000148,13.793667,0.169897,2
2,consecutive_alert_days_before_t,0.219359,-0.027264,10.060000,0.134097,3
3,alert_severity_roll_max_7,0.203801,-0.119842,9.597266,0.124586,4
4,is_business_free_day,0.157333,-0.001401,0.305333,0.096180,5
5,alert_lag_7,0.115586,0.008227,0.819333,0.070660,6
6,hub_flag,0.036268,0.004999,0.155667,0.022171,7
7,alert_lag_2,0.035410,0.002985,0.819667,0.021647,8
8,is_weekend,0.025184,0.000200,0.287333,0.015395,9
9,is_weekend_x_persona__mixed_other,0.019924,0.000678,0.265667,0.012180,10


In [88]:
# 4.14.5 Scalenie interpretowalności cech do jednej tabeli

import pandas as pd

required_objects = [
    "step4_14_feature_importance_df",
    "step4_14_permutation_importance_df",
    "step4_14_shap_importance_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.14.5: {missing_objects}"

step4_14_feature_importance_report_df = (
    step4_14_feature_importance_df.copy()
    .merge(
        step4_14_permutation_importance_df[
            [
                "feature_name",
                "permutation_importance_mean",
                "permutation_importance_std",
                "permutation_rank",
            ]
        ].copy(),
        on="feature_name",
        how="left",
        validate="one_to_one",
    )
    .merge(
        step4_14_shap_importance_df[
            [
                "feature_name",
                "mean_abs_shap_value",
                "mean_shap_value",
                "mean_abs_shap_share",
                "shap_rank",
            ]
        ].copy(),
        on="feature_name",
        how="left",
        validate="one_to_one",
    )
)

step4_14_feature_importance_report_df["gain_rank"] = (
    step4_14_feature_importance_report_df["gain_rank"].astype(int)
)
step4_14_feature_importance_report_df["split_rank"] = (
    step4_14_feature_importance_report_df["split_rank"].astype(int)
)
step4_14_feature_importance_report_df["permutation_rank"] = (
    step4_14_feature_importance_report_df["permutation_rank"].astype(int)
)
step4_14_feature_importance_report_df["shap_rank"] = (
    step4_14_feature_importance_report_df["shap_rank"].astype(int)
)

step4_14_feature_importance_report_df["rank_sum"] = (
    step4_14_feature_importance_report_df["gain_rank"]
    + step4_14_feature_importance_report_df["permutation_rank"]
    + step4_14_feature_importance_report_df["shap_rank"]
)

step4_14_feature_importance_report_df["consensus_rank"] = (
    step4_14_feature_importance_report_df["rank_sum"]
    .rank(method="dense", ascending=True)
    .astype(int)
)

step4_14_feature_importance_report_df = (
    step4_14_feature_importance_report_df[
        [
            "feature_name",
            "gain_importance",
            "gain_importance_share",
            "gain_rank",
            "split_importance",
            "split_importance_share",
            "split_rank",
            "permutation_importance_mean",
            "permutation_importance_std",
            "permutation_rank",
            "mean_abs_shap_value",
            "mean_shap_value",
            "mean_abs_shap_share",
            "shap_rank",
            "rank_sum",
            "consensus_rank",
        ]
    ]
    .sort_values(
        [
            "consensus_rank",
            "gain_rank",
            "permutation_rank",
            "shap_rank",
            "feature_name",
        ],
        ascending=[True, True, True, True, True],
    )
    .reset_index(drop=True)
)

step4_14_feature_importance_consensus_summary_df = pd.DataFrame(
    [
        {
            "feature_count": int(step4_14_feature_importance_report_df.shape[0]),
            "top1_feature": str(step4_14_feature_importance_report_df.loc[0, "feature_name"]),
            "top5_consensus_features": ", ".join(
                step4_14_feature_importance_report_df["feature_name"].head(5).tolist()
            ),
            "top10_consensus_features": ", ".join(
                step4_14_feature_importance_report_df["feature_name"].head(10).tolist()
            ),
        }
    ]
)

display(step4_14_feature_importance_consensus_summary_df)
display(step4_14_feature_importance_report_df.head(20))

assert step4_14_feature_importance_report_df.shape[0] == 34, (
    "4.14.5 expected 34 rows in merged feature importance report."
)

,feature_count,top1_feature,top5_consensus_features,top10_consensus_features
0,34,alert_hours_roll_sum_14,"alert_hours_roll_sum_14, alert_hours_lag_1, co...","alert_hours_roll_sum_14, alert_hours_lag_1, co..."


,feature_name,gain_importance,gain_importance_share,gain_rank,split_importance,split_importance_share,split_rank,permutation_importance_mean,permutation_importance_std,permutation_rank,mean_abs_shap_value,mean_shap_value,mean_abs_shap_share,shap_rank,rank_sum,consensus_rank
0,alert_hours_roll_sum_14,153504.989200,0.433086,1,734.0,0.174762,1,0.016926,0.001485,1,0.417202,-0.077871,0.255041,1,3,1
1,alert_hours_lag_1,59710.238619,0.168461,2,636.0,0.151429,3,0.010605,0.001384,2,0.277922,-0.000148,0.169897,2,6,2
2,consecutive_alert_days_before_t,34693.097674,0.097880,3,483.0,0.115000,4,0.006335,0.001195,4,0.219359,-0.027264,0.134097,3,10,3
3,alert_severity_roll_max_7,24034.000782,0.067807,4,683.0,0.162619,2,0.006592,0.000712,3,0.203801,-0.119842,0.124586,4,11,4
4,is_business_free_day,16594.344774,0.046818,6,296.0,0.070476,5,0.004799,0.000440,5,0.157333,-0.001401,0.096180,5,16,5
5,alert_lag_7,17928.393329,0.050582,5,133.0,0.031667,6,0.000924,0.000251,6,0.115586,0.008227,0.070660,6,17,6
6,is_any_missing_signal,13699.238294,0.038650,7,118.0,0.028095,7,0.000331,0.000052,9,0.019486,0.005506,0.011912,11,27,7
7,alert_lag_2,4025.679412,0.011358,10,59.0,0.014048,15,0.000141,0.000060,12,0.035410,0.002985,0.021647,8,30,8
8,hub_flag,1584.825867,0.004471,16,97.0,0.023095,9,0.000470,0.000225,7,0.036268,0.004999,0.022171,7,30,8
9,is_missing_returns_history,4084.582842,0.011524,8,108.0,0.025714,8,0.000176,0.000062,10,0.014900,0.006929,0.009109,14,32,9


In [89]:
# 4.14.6 Agregacja błędów finalnego modelu po typach i trudnych segmentach

import numpy as np
import pandas as pd

required_objects = [
    "step4_14_error_analysis_input_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.14.6: {missing_objects}"

step4_14_error_base_df = step4_14_error_analysis_input_df.copy()

step4_14_total_rows = int(step4_14_error_base_df.shape[0])
step4_14_total_errors = int(step4_14_error_base_df["is_error"].sum())
step4_14_total_fp = int(step4_14_error_base_df["is_false_positive"].sum())
step4_14_total_fn = int(step4_14_error_base_df["is_false_negative"].sum())
step4_14_overall_error_rate = float(step4_14_error_base_df["is_error"].mean())

def summarize_subset(df, slice_family, slice_name):
    subset_row_count = int(df.shape[0])
    subset_error_count = int(df["is_error"].sum())
    subset_fp_count = int(df["is_false_positive"].sum())
    subset_fn_count = int(df["is_false_negative"].sum())

    return {
        "slice_family": str(slice_family),
        "slice_name": str(slice_name),
        "row_count": subset_row_count,
        "positive_count": int(df["y_true"].sum()),
        "negative_count": int((1 - df["y_true"]).sum()),
        "error_count": subset_error_count,
        "false_positive_count": subset_fp_count,
        "false_negative_count": subset_fn_count,
        "error_rate": float(subset_error_count / subset_row_count) if subset_row_count > 0 else np.nan,
        "false_positive_rate": float(subset_fp_count / subset_row_count) if subset_row_count > 0 else np.nan,
        "false_negative_rate": float(subset_fn_count / subset_row_count) if subset_row_count > 0 else np.nan,
        "fp_share_within_errors": float(subset_fp_count / subset_error_count) if subset_error_count > 0 else np.nan,
        "fn_share_within_errors": float(subset_fn_count / subset_error_count) if subset_error_count > 0 else np.nan,
        "share_of_all_rows": float(subset_row_count / step4_14_total_rows) if step4_14_total_rows > 0 else np.nan,
        "share_of_all_errors": float(subset_error_count / step4_14_total_errors) if step4_14_total_errors > 0 else np.nan,
        "error_rate_vs_test_delta": (
            float(subset_error_count / subset_row_count) - step4_14_overall_error_rate
            if subset_row_count > 0 else np.nan
        ),
        "mean_predicted_probability": float(df["predicted_probability"].mean()) if subset_row_count > 0 else np.nan,
        "pred_positive_rate": float(df["predicted_label"].mean()) if subset_row_count > 0 else np.nan,
        "unique_station_count": int(df["station_id"].nunique()) if subset_row_count > 0 else 0,
        "min_activity_date": df["activity_date"].min() if subset_row_count > 0 else pd.NaT,
        "max_activity_date": df["activity_date"].max() if subset_row_count > 0 else pd.NaT,
    }

step4_14_error_analysis_rows = []

# przegląd globalny
step4_14_error_analysis_rows.append(
    summarize_subset(
        step4_14_error_base_df,
        "global",
        "all_test_rows",
    )
)

# typy błędów jako osobne przekroje
step4_14_error_analysis_rows.append(
    summarize_subset(
        step4_14_error_base_df.loc[step4_14_error_base_df["is_false_positive"].eq(1)].copy(),
        "error_type",
        "false_positive",
    )
)
step4_14_error_analysis_rows.append(
    summarize_subset(
        step4_14_error_base_df.loc[step4_14_error_base_df["is_false_negative"].eq(1)].copy(),
        "error_type",
        "false_negative",
    )
)

# segmenty operacyjne i trudne warunki
step4_14_segment_specs = [
    ("business_segment", "hub", step4_14_error_base_df["hub_flag_slice"].eq(1)),
    ("business_segment", "cold_start", step4_14_error_base_df["is_cold_start_slice"].eq(1)),
    ("business_segment", "holiday", step4_14_error_base_df["holiday_flag_slice"].eq(1)),
    (
        "business_segment",
        "weather_low",
        step4_14_error_base_df["weather_segment"].astype("string").eq("weather_low"),
    ),
    (
        "business_segment",
        "utility_short_trip",
        step4_14_error_base_df["persona_slice"].astype("string").eq("utility_short_trip"),
    ),
    (
        "business_segment",
        "mz_0067",
        step4_14_error_base_df["microzone_slice"].astype("string").eq("MZ_0067"),
    ),
]

for step4_14_slice_family, step4_14_slice_name, step4_14_mask in step4_14_segment_specs:
    step4_14_error_analysis_rows.append(
        summarize_subset(
            step4_14_error_base_df.loc[step4_14_mask].copy(),
            step4_14_slice_family,
            step4_14_slice_name,
        )
    )

step4_14_error_analysis_df = (
    pd.DataFrame(step4_14_error_analysis_rows)
    .sort_values(
        ["slice_family", "slice_name"],
        ascending=[True, True],
    )
    .reset_index(drop=True)
)

step4_14_error_difficult_segments_df = (
    step4_14_error_analysis_df.loc[
        step4_14_error_analysis_df["slice_family"].eq("business_segment")
    ]
    .sort_values(
        ["error_rate_vs_test_delta", "share_of_all_errors", "error_rate"],
        ascending=[False, False, False],
    )
    .reset_index(drop=True)
)

display(step4_14_error_analysis_df)
display(step4_14_error_difficult_segments_df)

assert step4_14_error_analysis_df.shape[0] == 9, (
    "4.14.6 expected 9 rows in aggregated error analysis report."
)

,slice_family,slice_name,row_count,positive_count,negative_count,error_count,false_positive_count,false_negative_count,error_rate,false_positive_rate,...,fp_share_within_errors,fn_share_within_errors,share_of_all_rows,share_of_all_errors,error_rate_vs_test_delta,mean_predicted_probability,pred_positive_rate,unique_station_count,min_activity_date,max_activity_date
0,business_segment,cold_start,14,6,8,10,8,2,0.714286,0.571429,...,0.800000,0.200000,0.000184,0.000681,0.520919,0.648856,0.857143,2,2020-03-28,2020-07-15
1,business_segment,holiday,2695,2105,590,592,551,41,0.219666,0.204453,...,0.930743,0.069257,0.035470,0.040294,0.026299,0.775248,0.970315,343,2020-04-10,2020-10-31
2,business_segment,hub,11132,9365,1767,1776,1495,281,0.159540,0.134298,...,0.841779,0.158221,0.146512,0.120882,-0.033827,0.836061,0.950323,50,2020-03-23,2020-10-31
3,business_segment,mz_0067,1782,1144,638,644,552,92,0.361392,0.309764,...,0.857143,0.142857,0.023454,0.043833,0.168025,0.718207,0.900112,8,2020-03-23,2020-10-31
4,business_segment,utility_short_trip,5051,3084,1967,1896,1350,546,0.375371,0.267274,...,0.712025,0.287975,0.066478,0.129050,0.182005,0.635193,0.769749,23,2020-03-23,2020-10-31
5,business_segment,weather_low,25499,20303,5196,5221,4335,886,0.204753,0.170007,...,0.830301,0.169699,0.335601,0.355363,0.011386,0.771965,0.931488,344,2020-03-23,2020-10-31
6,error_type,false_negative,2020,2020,0,2020,0,2020,1.000000,0.000000,...,0.000000,1.000000,0.026586,0.137490,0.806633,0.465439,0.000000,281,2020-03-23,2020-10-31
7,error_type,false_positive,12672,0,12672,12672,12672,0,1.000000,1.000000,...,1.000000,0.000000,0.166781,0.862510,0.806633,0.746760,1.000000,343,2020-03-24,2020-10-31
8,global,all_test_rows,75980,61198,14782,14692,12672,2020,0.193367,0.166781,...,0.862510,0.137490,1.000000,1.000000,0.000000,0.798510,0.945644,344,2020-03-23,2020-10-31


,slice_family,slice_name,row_count,positive_count,negative_count,error_count,false_positive_count,false_negative_count,error_rate,false_positive_rate,...,fp_share_within_errors,fn_share_within_errors,share_of_all_rows,share_of_all_errors,error_rate_vs_test_delta,mean_predicted_probability,pred_positive_rate,unique_station_count,min_activity_date,max_activity_date
0,business_segment,cold_start,14,6,8,10,8,2,0.714286,0.571429,...,0.800000,0.200000,0.000184,0.000681,0.520919,0.648856,0.857143,2,2020-03-28,2020-07-15
1,business_segment,utility_short_trip,5051,3084,1967,1896,1350,546,0.375371,0.267274,...,0.712025,0.287975,0.066478,0.129050,0.182005,0.635193,0.769749,23,2020-03-23,2020-10-31
2,business_segment,mz_0067,1782,1144,638,644,552,92,0.361392,0.309764,...,0.857143,0.142857,0.023454,0.043833,0.168025,0.718207,0.900112,8,2020-03-23,2020-10-31
3,business_segment,holiday,2695,2105,590,592,551,41,0.219666,0.204453,...,0.930743,0.069257,0.035470,0.040294,0.026299,0.775248,0.970315,343,2020-04-10,2020-10-31
4,business_segment,weather_low,25499,20303,5196,5221,4335,886,0.204753,0.170007,...,0.830301,0.169699,0.335601,0.355363,0.011386,0.771965,0.931488,344,2020-03-23,2020-10-31
5,business_segment,hub,11132,9365,1767,1776,1495,281,0.159540,0.134298,...,0.841779,0.158221,0.146512,0.120882,-0.033827,0.836061,0.950323,50,2020-03-23,2020-10-31


In [90]:
# 4.14.7 Zapis artefaktów sekcji 4.14

from pathlib import Path

import pandas as pd

required_objects = [
    "step4_3_project_root",
    "step4_14_feature_importance_report_df",
    "step4_14_error_analysis_df",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.14.7: {missing_objects}"

step4_14_output_dir = step4_3_project_root / "artifacts" / "4_error_analysis"
step4_14_output_dir.mkdir(parents=True, exist_ok=True)

step4_14_feature_importance_path = step4_14_output_dir / "b4_14_feature_importance.parquet"
step4_14_error_analysis_path = step4_14_output_dir / "b4_14_error_analysis.parquet"

step4_14_feature_importance_save_df = (
    step4_14_feature_importance_report_df.copy()
    .reset_index(drop=True)
)
step4_14_error_analysis_save_df = (
    step4_14_error_analysis_df.copy()
    .reset_index(drop=True)
)

step4_14_feature_importance_save_df.to_parquet(
    step4_14_feature_importance_path,
    index=False,
)
step4_14_error_analysis_save_df.to_parquet(
    step4_14_error_analysis_path,
    index=False,
)

step4_14_saved_artifacts_registry_df = pd.DataFrame(
    [
        {
            "artifact_name": step4_14_feature_importance_path.name,
            "rows": int(step4_14_feature_importance_save_df.shape[0]),
            "cols": int(step4_14_feature_importance_save_df.shape[1]),
            "path": str(step4_14_feature_importance_path),
        },
        {
            "artifact_name": step4_14_error_analysis_path.name,
            "rows": int(step4_14_error_analysis_save_df.shape[0]),
            "cols": int(step4_14_error_analysis_save_df.shape[1]),
            "path": str(step4_14_error_analysis_path),
        },
    ]
)

display(step4_14_saved_artifacts_registry_df)

assert step4_14_feature_importance_path.exists(), (
    "4.14.7 failed to save b4_14_feature_importance.parquet"
)
assert step4_14_error_analysis_path.exists(), (
    "4.14.7 failed to save b4_14_error_analysis.parquet"
)
assert step4_14_feature_importance_save_df.shape[0] == 34, (
    "4.14.7 expected 34 rows in saved feature importance artifact."
)
assert step4_14_error_analysis_save_df.shape[0] == 9, (
    "4.14.7 expected 9 rows in saved error analysis artifact."
)

,artifact_name,rows,cols,path
0,b4_14_feature_importance.parquet,34,16,/root/projects/6_samodzielny_projekt_koncowy_w...
1,b4_14_error_analysis.parquet,9,21,/root/projects/6_samodzielny_projekt_koncowy_w...


## Podsumowanie działu 4.14 Error analysis i interpretowalność

**Kontekst inżynieryjny:** W dziale `4.14` przeprowadzono analizę błędów finalnego modelu oraz zbudowano lekką warstwę interpretowalności. Najpierw przygotowano pełną ramkę błędów na zbiorze `test`, łącząc predykcje z segmentami biznesowymi i oznaczając obserwacje jako `tp`, `tn`, `fp`, `fn`. Następnie policzono agregaty błędów globalnie oraz dla kluczowych segmentów problemowych: `hub`, `cold_start`, `holiday`, `weather_low`, `utility_short_trip` i `MZ_0067`. Równolegle zbudowano trzy warstwy interpretowalności cech: modelowe `feature importance` z LightGBM, `permutation importance` policzone na pełnych `34` cechach na próbce testowej oraz wkłady typu `SHAP` uzyskane przez `pred_contrib=True` z boostera LightGBM. Na końcu scalono wszystkie trzy perspektywy do jednego raportu konsensusowego ważności cech oraz zapisano artefakty `b4_14_feature_importance.parquet` i `b4_14_error_analysis.parquet`.

**Interpretacja wyniku:** Analiza błędów potwierdziła, że finalny model myli się głównie przez `false positives`, które stanowią około `86%` wszystkich błędów testowych. Globalny `error_rate` na teście wyniósł około `19.3%`. W przekrojach biznesowych najsilniejszymi i wiarygodnymi trudnymi segmentami okazały się `utility_short_trip` oraz `MZ_0067`, gdzie udział błędów był wyraźnie wyższy niż średnio na teście. Segment świąteczny również okazał się trudniejszy niż zwykłe dni, a `weather_low` dawał umiarkowane pogorszenie jakości. `Huby` nie były źródłem problemu — przeciwnie, miały niższy udział błędów niż średnia testowa. W warstwie interpretowalności trzy niezależne metody zgodnie wskazały ten sam rdzeń modelu: `alert_hours_roll_sum_14`, `alert_hours_lag_1`, `consecutive_alert_days_before_t`, `alert_severity_roll_max_7` oraz `is_business_free_day`. Oznacza to, że finalny model opiera decyzje przede wszystkim na świeżej historii alertów, ciągłości problemów operacyjnych i kalendarzu.

**Znaczenie biznesowe:** Dział `4.14` pokazał nie tylko średnią skuteczność modelu, ale też mechanikę jego działania i miejsca, w których faktycznie traci jakość. Z perspektywy operacyjnej najważniejszy wniosek jest taki, że model jest agresywny alarmowo: częściej generuje nadmiarowy alarm niż pomija alert. Taki profil może być akceptowalny, jeśli priorytetem biznesowym jest wysoki `recall`, ale oznacza większe obciążenie operacyjne przez nadmiar interwencji. Równocześnie analiza trudnych segmentów pozwala wskazać konkretne obszary do poprawy w kolejnych iteracjach modelu: przede wszystkim stacje/persony typu `utility_short_trip`, mikro-strefę `MZ_0067` oraz reżimy świąteczne. Z punktu widzenia interpretowalności wynik jest mocny, ponieważ trzy różne metody ważności cech zbieżnie potwierdziły ten sam, logiczny biznesowo rdzeń modelu.

**Wniosek:** Dział `4.14` został wykonany poprawnie i domknął warstwę analizy błędów oraz interpretowalności finalnego modelu. Otrzymano spójny obraz: model jest najmocniejszy tam, gdzie historia alertów dobrze opisuje przyszłe ryzyko, a najsłabszy w kilku wyraźnych segmentach operacyjnych. Jednocześnie interpretowalność potwierdziła, że predykcje nie opierają się na przypadkowych sygnałach, lecz głównie na historii alertów i kalendarzu. Zapisano dwa finalne artefakty sekcji: `b4_14_feature_importance.parquet` oraz `b4_14_error_analysis.parquet`.

## 4.15 Zapis finalnych artefaktów punktu 4

In [93]:
# 4.15.1 Zapis finalnych artefaktów punktu 4

import json
from pathlib import Path

import joblib
import pandas as pd

required_objects = [
    "step4_3_project_root",
    "step4_11_refit_model",
]
missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects in 4.15.1: {missing_objects}"

step4_15_project_root = Path(step4_3_project_root).resolve()
assert step4_15_project_root.exists(), f"Missing project root in 4.15.1: {step4_15_project_root}"


# rozwiązywanie artefaktów wejściowych po nazwie bez zgadywania katalogu
def resolve_single_artifact(project_root, artifact_name):
    matched_paths = sorted(
        [path.resolve() for path in project_root.rglob(artifact_name) if path.is_file()],
        key=lambda x: str(x),
    )
    assert len(matched_paths) > 0, f"4.15.1 missing required artifact: {artifact_name}"
    assert len(matched_paths) == 1, (
        f"4.15.1 expected exactly one path for {artifact_name}, got {len(matched_paths)}"
    )
    return matched_paths[0]


# pomocnicze wydobycie list cech/kolumn z parquetów kontraktowych
def extract_contract_feature_list(df, allowed_features=None):
    preferred_name_patterns = [
        "feature_name",
        "features",
        "feature",
        "keep_col",
        "keep_cols",
        "column_name",
        "column_names",
        "columns",
        "column",
    ]

    ordered_cols = []
    for pattern in preferred_name_patterns:
        for col in df.columns:
            col_lower = str(col).lower()
            if pattern == col_lower or pattern in col_lower:
                if col not in ordered_cols:
                    ordered_cols.append(col)

    if len(ordered_cols) == 0:
        ordered_cols = list(df.columns)

    extracted_values = []
    for col in ordered_cols:
        col_series = df[col]
        col_dtype = col_series.dtype

        is_text_like = (
            pd.api.types.is_object_dtype(col_dtype)
            or pd.api.types.is_string_dtype(col_dtype)
            or isinstance(col_dtype, pd.CategoricalDtype)
        )
        if not is_text_like:
            continue

        for value in col_series.dropna().tolist():
            value_str = str(value).strip()
            if value_str == "":
                continue
            if allowed_features is not None and value_str not in allowed_features:
                continue
            extracted_values.append(value_str)

    seen = set()
    ordered_unique = []
    for value in extracted_values:
        if value not in seen:
            seen.add(value)
            ordered_unique.append(value)

    return ordered_unique


# inwentaryzacja zapisanych plików
def build_release_inventory(release_dir, project_root):
    release_files = sorted(
        [path.resolve() for path in release_dir.iterdir() if path.is_file()],
        key=lambda x: str(x),
    )

    release_rows = []
    for path in release_files:
        suffix = path.suffix.lower()
        if suffix == ".parquet":
            df = pd.read_parquet(path)
            row_count = int(df.shape[0])
            col_count = int(df.shape[1])
        elif suffix == ".json":
            with open(path, "r", encoding="utf-8") as f:
                payload = json.load(f)
            row_count = 1
            col_count = int(len(payload)) if isinstance(payload, dict) else 1
        elif suffix == ".joblib":
            row_count = 1
            col_count = 1
        else:
            row_count = 1
            col_count = 1

        release_rows.append(
            {
                "artifact_name": path.name,
                "artifact_path": str(path),
                "relative_path": str(path.relative_to(project_root)),
                "rows": row_count,
                "cols": col_count,
                "size_mb": round(path.stat().st_size / (1024 * 1024), 6),
            }
        )

    return (
        pd.DataFrame(release_rows)
        .sort_values("artifact_name")
        .reset_index(drop=True)
    )


step4_15_input_paths = {
    "b3_13_model_ready_dataset.parquet": resolve_single_artifact(
        step4_15_project_root, "b3_13_model_ready_dataset.parquet"
    ),
    "b3_13_keep_cols.parquet": resolve_single_artifact(
        step4_15_project_root, "b3_13_keep_cols.parquet"
    ),
    "b3_13_model_ready_columns.parquet": resolve_single_artifact(
        step4_15_project_root, "b3_13_model_ready_columns.parquet"
    ),
    "b3_14_release_manifest.json": resolve_single_artifact(
        step4_15_project_root, "b3_14_release_manifest.json"
    ),
    "b4_10_model_comparison_validation.parquet": resolve_single_artifact(
        step4_15_project_root, "b4_10_model_comparison_validation.parquet"
    ),
    "b4_09_threshold_choice.json": resolve_single_artifact(
        step4_15_project_root, "b4_09_threshold_choice.json"
    ),
    "b4_11_test_metrics.parquet": resolve_single_artifact(
        step4_15_project_root, "b4_11_test_metrics.parquet"
    ),
    "b4_11_predictions_test.parquet": resolve_single_artifact(
        step4_15_project_root, "b4_11_predictions_test.parquet"
    ),
    "b4_14_feature_importance.parquet": resolve_single_artifact(
        step4_15_project_root, "b4_14_feature_importance.parquet"
    ),
    "b4_14_error_analysis.parquet": resolve_single_artifact(
        step4_15_project_root, "b4_14_error_analysis.parquet"
    ),
}

# wczytanie artefaktów wejściowych
step4_15_model_ready_dataset_df = pd.read_parquet(
    step4_15_input_paths["b3_13_model_ready_dataset.parquet"]
)
step4_15_keep_cols_df = pd.read_parquet(
    step4_15_input_paths["b3_13_keep_cols.parquet"]
)
step4_15_model_ready_columns_df = pd.read_parquet(
    step4_15_input_paths["b3_13_model_ready_columns.parquet"]
)
step4_15_model_comparison_report_df = pd.read_parquet(
    step4_15_input_paths["b4_10_model_comparison_validation.parquet"]
)
step4_15_test_metrics_df = pd.read_parquet(
    step4_15_input_paths["b4_11_test_metrics.parquet"]
)
step4_15_predictions_test_df = pd.read_parquet(
    step4_15_input_paths["b4_11_predictions_test.parquet"]
)
step4_15_feature_importance_df = pd.read_parquet(
    step4_15_input_paths["b4_14_feature_importance.parquet"]
)
step4_15_error_analysis_df = pd.read_parquet(
    step4_15_input_paths["b4_14_error_analysis.parquet"]
)

with open(step4_15_input_paths["b3_14_release_manifest.json"], "r", encoding="utf-8") as f:
    step4_15_block3_release_manifest = json.load(f)

with open(step4_15_input_paths["b4_09_threshold_choice.json"], "r", encoding="utf-8") as f:
    step4_15_threshold_choice_dict = json.load(f)

if hasattr(step4_11_refit_model, "feature_name_"):
    step4_15_model_feature_names = [str(col) for col in step4_11_refit_model.feature_name_]
elif hasattr(step4_11_refit_model, "booster_"):
    step4_15_model_feature_names = [str(col) for col in step4_11_refit_model.booster_.feature_name()]
else:
    raise AssertionError("4.15.1 could not resolve model feature names.")

step4_15_model_feature_name_set = set(step4_15_model_feature_names)

step4_15_keep_feature_list = extract_contract_feature_list(
    step4_15_keep_cols_df,
    allowed_features=step4_15_model_feature_name_set,
)
step4_15_model_ready_columns_list = extract_contract_feature_list(
    step4_15_model_ready_columns_df,
    allowed_features=None,
)
step4_15_model_ready_columns_filtered = [
    col for col in step4_15_model_ready_columns_list
    if col in step4_15_model_ready_dataset_df.columns
]
step4_15_model_ready_feature_list = [
    col for col in step4_15_model_ready_columns_filtered
    if col in step4_15_model_feature_name_set
]

assert len(step4_15_model_feature_names) > 0, "4.15.1 expected non-empty model feature list."
assert step4_15_test_metrics_df.shape[0] == 1, "4.15.1 expected exactly 1 row in test metrics."
assert step4_15_feature_importance_df.shape[0] == len(step4_15_model_feature_names), (
    "4.15.1 feature importance rows must match model feature count."
)

if len(step4_15_keep_feature_list) > 0:
    assert set(step4_15_keep_feature_list) == step4_15_model_feature_name_set, (
        "4.15.1 parsed keep_cols feature set does not match model features."
    )

if len(step4_15_model_ready_feature_list) > 0:
    assert step4_15_model_feature_name_set.issubset(set(step4_15_model_ready_feature_list)), (
        "4.15.1 model features are not a subset of parsed model_ready_columns contract."
    )

# raport progu jako parquet
step4_15_threshold_report_df = pd.json_normalize(
    step4_15_threshold_choice_dict,
    sep="__",
)

# katalog końcowego pakietu
step4_15_output_dir = step4_15_project_root / "artifacts" / "4_final_package"
step4_15_output_dir.mkdir(parents=True, exist_ok=True)

step4_15_output_paths = {
    "b4_15_model_comparison_report.parquet": step4_15_output_dir / "b4_15_model_comparison_report.parquet",
    "b4_15_threshold_report.parquet": step4_15_output_dir / "b4_15_threshold_report.parquet",
    "b4_15_test_metrics.parquet": step4_15_output_dir / "b4_15_test_metrics.parquet",
    "b4_15_predictions_test.parquet": step4_15_output_dir / "b4_15_predictions_test.parquet",
    "b4_15_feature_importance.parquet": step4_15_output_dir / "b4_15_feature_importance.parquet",
    "b4_15_error_analysis.parquet": step4_15_output_dir / "b4_15_error_analysis.parquet",
    "b4_15_model_registry.json": step4_15_output_dir / "b4_15_model_registry.json",
    "b4_15_best_model.joblib": step4_15_output_dir / "b4_15_best_model.joblib",
    "b4_15_inference_config.json": step4_15_output_dir / "b4_15_inference_config.json",
    "b4_15_handoff_manifest.json": step4_15_output_dir / "b4_15_handoff_manifest.json",
}

# zapis parquetów końcowych
step4_15_model_comparison_report_save_df = (
    step4_15_model_comparison_report_df.copy().reset_index(drop=True)
)
step4_15_threshold_report_save_df = (
    step4_15_threshold_report_df.copy().reset_index(drop=True)
)
step4_15_test_metrics_save_df = (
    step4_15_test_metrics_df.copy().reset_index(drop=True)
)
step4_15_predictions_test_save_df = (
    step4_15_predictions_test_df.copy().reset_index(drop=True)
)
step4_15_feature_importance_save_df = (
    step4_15_feature_importance_df.copy().reset_index(drop=True)
)
step4_15_error_analysis_save_df = (
    step4_15_error_analysis_df.copy().reset_index(drop=True)
)

step4_15_model_comparison_report_save_df.to_parquet(
    step4_15_output_paths["b4_15_model_comparison_report.parquet"],
    index=False,
)
step4_15_threshold_report_save_df.to_parquet(
    step4_15_output_paths["b4_15_threshold_report.parquet"],
    index=False,
)
step4_15_test_metrics_save_df.to_parquet(
    step4_15_output_paths["b4_15_test_metrics.parquet"],
    index=False,
)
step4_15_predictions_test_save_df.to_parquet(
    step4_15_output_paths["b4_15_predictions_test.parquet"],
    index=False,
)
step4_15_feature_importance_save_df.to_parquet(
    step4_15_output_paths["b4_15_feature_importance.parquet"],
    index=False,
)
step4_15_error_analysis_save_df.to_parquet(
    step4_15_output_paths["b4_15_error_analysis.parquet"],
    index=False,
)

# zapis finalnego modelu
joblib.dump(
    step4_11_refit_model,
    step4_15_output_paths["b4_15_best_model.joblib"],
)

step4_15_selected_model_name = str(
    step4_15_threshold_choice_dict.get(
        "selected_model_name",
        step4_15_test_metrics_save_df.loc[0, "selected_model_name"],
    )
)
step4_15_selected_calibration_method = str(
    step4_15_threshold_choice_dict.get(
        "selected_calibration_method",
        step4_15_test_metrics_save_df.loc[0, "selected_calibration_method"],
    )
)
step4_15_selected_threshold = float(
    step4_15_threshold_choice_dict.get(
        "selected_threshold",
        step4_15_test_metrics_save_df.loc[0, "selected_threshold"],
    )
)

# registry modelu
step4_15_model_registry_payload = {
    "block_code": "4",
    "block_name": "classification_model_final_package",
    "project_root": str(step4_15_project_root),
    "release_dir": str(step4_15_output_dir),
    "selected_model": {
        "model_name": step4_15_selected_model_name,
        "calibration_method": step4_15_selected_calibration_method,
        "decision_threshold": step4_15_selected_threshold,
        "model_class": step4_11_refit_model.__class__.__name__,
        "model_module": step4_11_refit_model.__class__.__module__,
        "model_artifact_name": "b4_15_best_model.joblib",
        "model_artifact_path": str(step4_15_output_paths["b4_15_best_model.joblib"]),
    },
    "model_hyperparameters": step4_11_refit_model.get_params(),
    "feature_contract": {
        "feature_count": int(len(step4_15_model_feature_names)),
        "feature_names": step4_15_model_feature_names,
        "keep_cols_count": int(len(step4_15_keep_feature_list)),
        "model_ready_columns_count": int(len(step4_15_model_ready_columns_filtered)),
    },
    "evaluation_summary": step4_15_test_metrics_save_df.iloc[0].to_dict(),
    "source_artifacts": {key: str(value) for key, value in step4_15_input_paths.items()},
    "release_artifacts": {key: str(value) for key, value in step4_15_output_paths.items()},
}

with open(step4_15_output_paths["b4_15_model_registry.json"], "w", encoding="utf-8") as f:
    json.dump(
        step4_15_model_registry_payload,
        f,
        ensure_ascii=False,
        indent=2,
        default=str,
    )

# config inferencyjny
step4_15_inference_config_payload = {
    "model_name": step4_15_selected_model_name,
    "calibration_method": step4_15_selected_calibration_method,
    "decision_threshold": step4_15_selected_threshold,
    "model_artifact_path": str(step4_15_output_paths["b4_15_best_model.joblib"]),
    "feature_count": int(len(step4_15_model_feature_names)),
    "input_feature_names": step4_15_model_feature_names,
    "input_contract_sources": {
        "keep_cols_path": str(step4_15_input_paths["b3_13_keep_cols.parquet"]),
        "model_ready_columns_path": str(step4_15_input_paths["b3_13_model_ready_columns.parquet"]),
    },
    "output_columns": [
        "predicted_probability",
        "predicted_label",
    ],
    "prediction_notes": {
        "probability_column": "predicted_probability",
        "label_column": "predicted_label",
        "positive_class_definition": "target_alert_day == 1",
    },
}

with open(step4_15_output_paths["b4_15_inference_config.json"], "w", encoding="utf-8") as f:
    json.dump(
        step4_15_inference_config_payload,
        f,
        ensure_ascii=False,
        indent=2,
        default=str,
    )

# pierwszy przebieg inwentaryzacji przed manifestem handoff
step4_15_release_inventory_df = build_release_inventory(
    step4_15_output_dir,
    step4_15_project_root,
)

# pierwszy zapis handoff manifest
step4_15_handoff_manifest_payload = {
    "block_code": "4",
    "block_name": "classification_model_final_handoff",
    "project_root": str(step4_15_project_root),
    "release_dir": str(step4_15_output_dir),
    "upstream_block3_manifest_path": str(step4_15_input_paths["b3_14_release_manifest.json"]),
    "upstream_block3_manifest_summary": step4_15_block3_release_manifest.get("package_summary", {}),
    "selected_model": {
        "model_name": step4_15_selected_model_name,
        "calibration_method": step4_15_selected_calibration_method,
        "decision_threshold": step4_15_selected_threshold,
    },
    "release_artifacts": {
        row["artifact_name"]: row["artifact_path"]
        for _, row in step4_15_release_inventory_df.iterrows()
    },
    "release_inventory": step4_15_release_inventory_df.to_dict(orient="records"),
    "package_summary": {
        "artifact_count": int(step4_15_release_inventory_df.shape[0]),
        "total_size_mb": round(float(step4_15_release_inventory_df["size_mb"].sum()), 6),
        "parquet_artifact_count": int(
            step4_15_release_inventory_df["artifact_name"].str.endswith(".parquet").sum()
        ),
        "json_artifact_count": int(
            step4_15_release_inventory_df["artifact_name"].str.endswith(".json").sum()
        ),
        "joblib_artifact_count": int(
            step4_15_release_inventory_df["artifact_name"].str.endswith(".joblib").sum()
        ),
    },
    "consistency_checks": {
        "all_release_paths_exist": bool(
            all(Path(path).exists() for path in step4_15_release_inventory_df["artifact_path"])
        ),
        "model_feature_count_matches_feature_importance": bool(
            len(step4_15_model_feature_names) == int(step4_15_feature_importance_save_df.shape[0])
        ),
        "model_feature_set_matches_keep_cols_when_parsed": bool(
            set(step4_15_keep_feature_list) == step4_15_model_feature_name_set
            if len(step4_15_keep_feature_list) > 0 else True
        ),
        "model_feature_set_is_subset_of_model_ready_columns_when_parsed": bool(
            step4_15_model_feature_name_set.issubset(set(step4_15_model_ready_feature_list))
            if len(step4_15_model_ready_feature_list) > 0 else True
        ),
        "prediction_row_count_matches_test_metrics_row_count": bool(
            int(step4_15_predictions_test_save_df.shape[0]) == int(step4_15_test_metrics_save_df.loc[0, "row_count"])
        ),
    },
}

with open(step4_15_output_paths["b4_15_handoff_manifest.json"], "w", encoding="utf-8") as f:
    json.dump(
        step4_15_handoff_manifest_payload,
        f,
        ensure_ascii=False,
        indent=2,
        default=str,
    )

# drugi przebieg inwentaryzacji już z handoff manifestem
step4_15_release_inventory_df = build_release_inventory(
    step4_15_output_dir,
    step4_15_project_root,
)

# finalny zapis handoff manifest z pełną listą 10 artefaktów
step4_15_handoff_manifest_payload = {
    "block_code": "4",
    "block_name": "classification_model_final_handoff",
    "project_root": str(step4_15_project_root),
    "release_dir": str(step4_15_output_dir),
    "upstream_block3_manifest_path": str(step4_15_input_paths["b3_14_release_manifest.json"]),
    "upstream_block3_manifest_summary": step4_15_block3_release_manifest.get("package_summary", {}),
    "selected_model": {
        "model_name": step4_15_selected_model_name,
        "calibration_method": step4_15_selected_calibration_method,
        "decision_threshold": step4_15_selected_threshold,
    },
    "release_artifacts": {
        row["artifact_name"]: row["artifact_path"]
        for _, row in step4_15_release_inventory_df.iterrows()
    },
    "release_inventory": step4_15_release_inventory_df.to_dict(orient="records"),
    "package_summary": {
        "artifact_count": int(step4_15_release_inventory_df.shape[0]),
        "total_size_mb": round(float(step4_15_release_inventory_df["size_mb"].sum()), 6),
        "parquet_artifact_count": int(
            step4_15_release_inventory_df["artifact_name"].str.endswith(".parquet").sum()
        ),
        "json_artifact_count": int(
            step4_15_release_inventory_df["artifact_name"].str.endswith(".json").sum()
        ),
        "joblib_artifact_count": int(
            step4_15_release_inventory_df["artifact_name"].str.endswith(".joblib").sum()
        ),
    },
    "consistency_checks": {
        "all_release_paths_exist": bool(
            all(Path(path).exists() for path in step4_15_release_inventory_df["artifact_path"])
        ),
        "model_feature_count_matches_feature_importance": bool(
            len(step4_15_model_feature_names) == int(step4_15_feature_importance_save_df.shape[0])
        ),
        "model_feature_set_matches_keep_cols_when_parsed": bool(
            set(step4_15_keep_feature_list) == step4_15_model_feature_name_set
            if len(step4_15_keep_feature_list) > 0 else True
        ),
        "model_feature_set_is_subset_of_model_ready_columns_when_parsed": bool(
            step4_15_model_feature_name_set.issubset(set(step4_15_model_ready_feature_list))
            if len(step4_15_model_ready_feature_list) > 0 else True
        ),
        "prediction_row_count_matches_test_metrics_row_count": bool(
            int(step4_15_predictions_test_save_df.shape[0]) == int(step4_15_test_metrics_save_df.loc[0, "row_count"])
        ),
    },
}

with open(step4_15_output_paths["b4_15_handoff_manifest.json"], "w", encoding="utf-8") as f:
    json.dump(
        step4_15_handoff_manifest_payload,
        f,
        ensure_ascii=False,
        indent=2,
        default=str,
    )

# finalny rejestr artefaktów
step4_15_saved_artifacts_registry_df = (
    step4_15_release_inventory_df[
        ["artifact_name", "rows", "cols", "artifact_path"]
    ]
    .rename(columns={"artifact_path": "path"})
    .sort_values("artifact_name")
    .reset_index(drop=True)
)

display(step4_15_saved_artifacts_registry_df)

assert step4_15_saved_artifacts_registry_df.shape[0] == 10, (
    "4.15.1 expected 10 final artifacts in saved registry."
)
assert step4_15_model_comparison_report_save_df.shape[0] >= 1, (
    "4.15.1 expected non-empty model comparison report."
)
assert step4_15_threshold_report_save_df.shape[0] == 1, (
    "4.15.1 expected exactly 1 row in threshold report."
)
assert step4_15_test_metrics_save_df.shape[0] == 1, (
    "4.15.1 expected exactly 1 row in test metrics report."
)
assert int(step4_15_predictions_test_save_df.shape[0]) == int(step4_15_test_metrics_save_df.loc[0, "row_count"]), (
    "4.15.1 prediction row count mismatch."
)
assert step4_15_feature_importance_save_df.shape[0] == 34, (
    "4.15.1 expected 34 rows in final feature importance report."
)
assert step4_15_error_analysis_save_df.shape[0] == 9, (
    "4.15.1 expected 9 rows in final error analysis report."
)
assert all(path.exists() for path in step4_15_output_paths.values()), (
    "4.15.1 found at least one missing final artifact after save."
)

,artifact_name,rows,cols,path
0,b4_15_best_model.joblib,1,1,/root/projects/6_samodzielny_projekt_koncowy_w...
1,b4_15_error_analysis.parquet,9,21,/root/projects/6_samodzielny_projekt_koncowy_w...
2,b4_15_feature_importance.parquet,34,16,/root/projects/6_samodzielny_projekt_koncowy_w...
3,b4_15_handoff_manifest.json,1,11,/root/projects/6_samodzielny_projekt_koncowy_w...
4,b4_15_inference_config.json,1,9,/root/projects/6_samodzielny_projekt_koncowy_w...
5,b4_15_model_comparison_report.parquet,8,15,/root/projects/6_samodzielny_projekt_koncowy_w...
6,b4_15_model_registry.json,1,10,/root/projects/6_samodzielny_projekt_koncowy_w...
7,b4_15_predictions_test.parquet,75980,6,/root/projects/6_samodzielny_projekt_koncowy_w...
8,b4_15_test_metrics.parquet,1,22,/root/projects/6_samodzielny_projekt_koncowy_w...
9,b4_15_threshold_report.parquet,1,22,/root/projects/6_samodzielny_projekt_koncowy_w...


## Podsumowanie działu 4.15 Zapis finalnych artefaktów punktu 4

**Kontekst inżynieryjny:** W dziale `4.15` złożono pełny finalny pakiet modelowy dla punktu `4`. Na wejściu wykorzystano artefakty z wcześniejszych etapów: raport porównania modeli na `validation`, wybór progu decyzyjnego, finalne metryki testowe, predykcje testowe, raport interpretowalności cech, raport error analysis oraz manifest wydania z punktu `3`. Na tej podstawie zapisano kompletny zestaw finalnych plików: raport porównania modeli, raport progu, metryki testowe, predykcje testowe, feature importance, error analysis, registry modelu, finalny model `joblib`, config inferencyjny oraz manifest przekazania `handoff`. Dodatkowo wykonano kontrole spójności kontraktu cech, zgodności liczby predykcji z liczbą rekordów testowych oraz zgodności modelu z raportem ważności cech.

**Interpretacja wyniku:** Finalna paczka została zbudowana poprawnie i zawiera dokładnie `10` artefaktów końcowych. W pakiecie znajdują się wszystkie kluczowe elementy potrzebne do odtworzenia wyniku modelowego: wytrenowany model, komplet raportów ewaluacyjnych, dane predykcyjne, metadane modelu oraz pliki konfiguracyjne do inferencji. Rozmiary zapisanych artefaktów są zgodne z wcześniejszymi etapami projektu: raport feature importance zawiera `34` cechy, raport error analysis `9` agregatów, a zbiór finalnych predykcji testowych `75 980` rekordów. Obecność `model_registry.json`, `inference_config.json` oraz `handoff_manifest.json` oznacza, że pakiet obejmuje nie tylko warstwę ewaluacyjną, ale także warstwę przekazania i dalszego użycia.

**Znaczenie biznesowe:** Dział `4.15` zamienia wynik modelowania w uporządkowany produkt końcowy. Dzięki temu końcowy rezultat nie ogranicza się do notebooka i tabel, lecz zawiera pełną paczkę artefaktów: model, raporty, konfigurację wejścia i opis wyjścia. Taki układ zwiększa użyteczność całego rozwiązania, ponieważ umożliwia jego spójne przekazanie, ponowne uruchomienie i dalszą integrację z kolejnymi etapami pracy.

**Wniosek:** Dział `4.15` został zakończony poprawnie i domknął punkt `4` na poziomie artefaktowym. Zbudowano kompletny finalny pakiet modelowy obejmujący model, raporty, metryki, predykcje, konfigurację inferencji oraz manifest przekazania. Projekt posiada spójny zestaw końcowych plików, które mogą stanowić bazę do dalszego użycia, integracji lub utrzymania.